# Honest best client-profile anti-fraud stack

This notebook retrains the complete selected ensemble from the four provided
competition CSV files. It reconstructs the submission template from test
`TransactionID` values and does not load test
labels, previous submissions, or pretrained models.

Model and blend choices are made with purged forward-time folds. The final
architecture combines independent CatBoost, LightGBM and XGBoost views,
target-free client components, V-block/structured residuals, XGB magic
features, and a client-profile LightGBM residual for cold users.

## 1. Setup and official-input boundary

The work directory starts with exactly four data inputs. `FORCE_RETRAIN=False`
allows an interrupted interactive run to reuse completed models; a fresh
Kaggle version has no cache, so every stage is still trained from scratch.

In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys

import pandas as pd

SOURCE_FILES = (
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
)
EXPECTED_ROWS = {"train_transaction.csv": 365365, "test_transaction.csv": 129736}
EXPECTED_SHA256 = {
    "train_transaction.csv": "d75f0cedaf354a750bc148372b63835efef53445abe017aad3d93dc2ebfc7da7",
    "train_identity.csv": "8cece20966dc33be033ace5821ec183ea019a2d18008a9590e30a202d2da19d0",
    "test_transaction.csv": "7116dd600f0f3b6fcc62292988a4997c02727265836baa816383dcf4202bdadd",
    "test_identity.csv": "5ceabcf3453372247372399e3ff083f3727dd69675040235bc1ef00c27f05a81",
    "sample_submission.csv": "c2d4a3a53db995f95f7a10fe1785683303146d683ed71b8c396c8d31d4859e83",
}
FORCE_RETRAIN = False


def csv_rows(path):
    with open(path, "rb") as stream:
        return sum(1 for _ in stream) - 1


def find_data_dir():
    roots = [Path.cwd(), Path("/kaggle/input")]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        candidates = [root]
        if root == Path("/kaggle/input"):
            candidates.extend(path.parent for path in root.rglob("train_transaction.csv"))
        for candidate in candidates:
            if not all((candidate / name).exists() for name in SOURCE_FILES):
                continue
            if (
                csv_rows(candidate / "train_transaction.csv")
                == EXPECTED_ROWS["train_transaction.csv"]
                and csv_rows(candidate / "test_transaction.csv")
                == EXPECTED_ROWS["test_transaction.csv"]
            ):
                matches.append(candidate)
    if not matches:
        raise FileNotFoundError("Could not find the four competition CSV files")
    return matches[0]


DATA_DIR = find_data_dir()
WORK_DIR = (
    Path("/kaggle/working/honest_best_stack")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "honest_best_4files_run"
)
WORK_DIR.mkdir(parents=True, exist_ok=True)

for name in SOURCE_FILES:
    destination = WORK_DIR / name
    if destination.exists() or destination.is_symlink():
        destination.unlink()
    try:
        destination.symlink_to((DATA_DIR / name).resolve())
    except OSError:
        shutil.copy2(DATA_DIR / name, destination)

# The original template is exactly test TransactionID plus a constant 0.5.
# Rebuilding it here keeps the four-file input boundary explicit.
sample = pd.read_csv(WORK_DIR / "test_transaction.csv", usecols=["TransactionID"])
sample["isFraud"] = 0.5
sample.to_csv(WORK_DIR / "sample_submission.csv", index=False)

assert not (WORK_DIR / "external_ieee").exists()
assert "isFraud" in pd.read_csv(WORK_DIR / "train_transaction.csv", nrows=0).columns
assert "isFraud" not in pd.read_csv(WORK_DIR / "test_transaction.csv", nrows=0).columns
for name, expected_hash in EXPECTED_SHA256.items():
    actual_hash = hashlib.sha256((WORK_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"Unexpected input version: {name}"

os.environ["PYTHONHASHSEED"] = "0"
os.chdir(WORK_DIR)
print("Data:", DATA_DIR)
print("Work:", WORK_DIR)
print("Python:", platform.python_version())
for package in ("numpy", "pandas", "sklearn", "catboost", "lightgbm", "xgboost", "torch"):
    module = importlib.import_module(package)
    print(package, getattr(module, "__version__", "unknown"))


def run_stage(script, *arguments, supports_force=False):
    command = [sys.executable, script, *arguments]
    if supports_force and FORCE_RETRAIN:
        command.append("--force")
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=WORK_DIR, env=os.environ.copy(), check=True)
    assert not (WORK_DIR / "external_ieee").exists()


Data: <path> x МТС Kaggle
Work: <path> x МТС Kaggle/honest_best_4files_run
Python: 3.13.3
numpy 2.2.4
pandas 2.2.3


sklearn 1.6.1
catboost 1.2.8
lightgbm 4.7.0
xgboost 3.3.0


torch 2.6.0


## 2. Embedded reproducible source

The following `34` files are the complete local dependency
closure of the selected pipeline. The notebook generator rejects any source
that refers to an external bridge or external test labels.

In [2]:
%%writefile build_honest_advanced_cat_final.py
"""Train the locked three-seed advanced CatBoost and build its clean stack."""

from __future__ import annotations

import argparse
import gc
import json
from pathlib import Path
import time

from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

from build_honest_no_gap_meta import evaluate_source_set, fit_final, make_uid, rank_prediction
from fraud_honest_advanced_data import ADVANCED_CAT_PARAMS as CAT_PARAMS
from train_honest_advanced_catboost import CACHE_DIR, SOURCE, prepare


ROOT = Path(__file__).resolve().parent
TARGET = "isFraud"
ENHANCED_SOURCE = "cat_enhanced"
SOURCES = ("catboost", ENHANCED_SOURCE, "lightgbm", "xgboost")
SEEDS = (1729, 2026, 3407)
FINAL_ITERATIONS = 1_900
ADVANCED_WEIGHT = 0.25
OUTPUT_PATH = ROOT / "submission_honest_advanced_cat_multiseed.csv"
REPORT_PATH = CACHE_DIR / "final_report.json"


def train_final_source(
    prepared: dict,
    features: list[str],
    categorical: list[str],
    force: bool,
) -> tuple[np.ndarray, list[dict]]:
    rank_parts = []
    rows = []
    for seed in SEEDS:
        model_path = CACHE_DIR / f"final_seed_{seed}.cbm"
        prediction_path = CACHE_DIR / f"final_seed_{seed}_test.npy"
        started = time.time()
        if model_path.exists() and prediction_path.exists() and not force:
            prediction = np.load(prediction_path)
            if prediction.dtype != np.float64:
                print(
                    f"Refreshing final seed {seed} predictions as float64",
                    flush=True,
                )
                model = CatBoostClassifier()
                model.load_model(model_path)
                prediction = model.predict_proba(
                    prepared["inference"][features]
                )[:, 1]
                np.save(prediction_path, prediction)
                del model
                gc.collect()
            cached = True
        else:
            print(
                f"Final advanced CatBoost seed {seed}: "
                f"{FINAL_ITERATIONS} trees",
                flush=True,
            )
            model = CatBoostClassifier(
                **{
                    **CAT_PARAMS,
                    "iterations": FINAL_ITERATIONS,
                    "random_seed": seed,
                    "verbose": 200,
                }
            )
            model.fit(
                prepared["train"][features],
                prepared["y"],
                cat_features=categorical,
            )
            prediction = model.predict_proba(
                prepared["inference"][features]
            )[:, 1]
            model.save_model(model_path)
            np.save(prediction_path, prediction)
            del model
            gc.collect()
            cached = False
        rank_parts.append(rank_prediction(prediction))
        rows.append(
            {
                "seed": seed,
                "iterations": FINAL_ITERATIONS,
                "cached": cached,
                "model": model_path.name,
                "minutes": (time.time() - started) / 60.0,
            }
        )
    return np.mean(rank_parts, axis=0), rows


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()
    started = time.time()

    prepared, features, categorical = prepare()
    advanced_rank, seed_rows = train_final_source(
        prepared, features, categorical, args.force
    )
    raw_test_ids = prepared["inference_ids"].to_numpy()
    del prepared
    gc.collect()

    base_oof = pd.read_csv(ROOT / "boost3_oof_predictions.csv")
    advanced_oof = pd.read_csv(
        CACHE_DIR / "oof.csv", usecols=["row_index", SOURCE]
    )
    oof = base_oof.merge(
        advanced_oof,
        on="row_index",
        how="left",
        validate="one_to_one",
    )
    old_rank_oof = oof.groupby("fold")["catboost"].rank(pct=True)
    advanced_rank_oof = oof.groupby("fold")[SOURCE].rank(pct=True)
    oof[ENHANCED_SOURCE] = (
        (1.0 - ADVANCED_WEIGHT) * old_rank_oof
        + ADVANCED_WEIGHT * advanced_rank_oof
    )

    raw_train = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    raw_test = pd.read_csv(
        ROOT / "test_transaction.csv",
        usecols=[
            "TransactionID",
            "TransactionDT",
            "card1",
            "addr1",
            "D1",
            "P_emaildomain",
        ],
    )
    uid_train = make_uid(raw_train)
    uid_test = make_uid(raw_test)
    baseline = evaluate_source_set(
        oof, uid_train, ("catboost", "lightgbm", "xgboost")
    )
    selected = evaluate_source_set(oof, uid_train, SOURCES)
    holdout_gain = float(
        selected["uid_recipe"]["auc"] - baseline["uid_recipe"]["auc"]
    )
    if holdout_gain <= 0:
        raise RuntimeError(f"Locked enhanced CatBoost failed: {holdout_gain}")

    old_cat_test = np.load(ROOT / "stack_test_catboost.npy")
    enhanced_test = (
        (1.0 - ADVANCED_WEIGHT) * rank_prediction(old_cat_test)
        + ADVANCED_WEIGHT * advanced_rank
    )
    test_predictions = pd.DataFrame(
        {
            "catboost": old_cat_test,
            ENHANCED_SOURCE: enhanced_test,
            "lightgbm": np.load(ROOT / "stack_test_lightgbm.npy"),
            "xgboost": np.load(ROOT / "stack_test_xgboost.npy"),
        }
    )
    prediction = fit_final(oof, test_predictions, uid_test, selected)
    sample = pd.read_csv(ROOT / "sample_submission.csv")
    if not np.array_equal(sample["TransactionID"], raw_test_ids):
        raise ValueError("Prepared test and sample TransactionID order differ")
    if not np.array_equal(sample["TransactionID"], raw_test["TransactionID"]):
        raise ValueError("Raw test and sample TransactionID order differ")
    output = sample[["TransactionID"]].copy()
    output[TARGET] = prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test only",
        "selection": "official-train temporal OOF only",
        "sources": list(SOURCES),
        "advanced_weight": ADVANCED_WEIGHT,
        "seeds": list(SEEDS),
        "iterations": FINAL_ITERATIONS,
        "seed_models": seed_rows,
        "features": len(features),
        "categorical": len(categorical),
        "baseline_holdout_auc": baseline["uid_recipe"]["auc"],
        "selected_holdout_auc": selected["uid_recipe"]["auc"],
        "holdout_gain": holdout_gain,
        "selected_recipe": selected,
        "output": OUTPUT_PATH.name,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting build_honest_advanced_cat_final.py


In [3]:
%%writefile build_honest_no_gap_meta.py
"""Build a train-only selected meta ensemble.

The script combines the clean CatBoost/LightGBM/XGBoost temporal OOF matrix
with the independently trained temporal7 stack. The last official-train fold
is the only model-selection holdout. No external rows or test labels are read.
"""

from __future__ import annotations

import gc
import json
import random
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


ROOT = Path(__file__).resolve().parent
TARGET = "isFraud"
BASE_SOURCES = ("catboost", "lightgbm", "xgboost")
TEMPORAL_SOURCE = "temporal7"
META_SEEDS = (42, 2026, 3407)
LOGISTIC_C = 0.20
MAX_EPOCHS = 120
PATIENCE = 12
OUTPUT_PATH = ROOT / "submission_honest_no_gap_temporal_meta.csv"
METRICS_PATH = ROOT / "honest_no_gap_temporal_meta_metrics.json"

torch.set_num_threads(1)
torch.set_num_interop_threads(1)


def rank_prediction(values: np.ndarray | pd.Series) -> np.ndarray:
    return pd.Series(np.asarray(values)).rank(method="average", pct=True).to_numpy()


def make_uid(frame: pd.DataFrame) -> pd.Series:
    origin_day = (frame["TransactionDT"] / 86_400 - frame["D1"]).round()
    return (
        frame["card1"].astype("string").fillna("<MISSING>")
        + "|"
        + frame["addr1"].astype("string").fillna("<MISSING>")
        + "|"
        + origin_day.astype("Int64").astype("string").fillna("<MISSING>")
        + "|"
        + frame["P_emaildomain"].astype("string").fillna("<MISSING>")
    )


def apply_uid_max(
    prediction: np.ndarray,
    uid: pd.Series,
    weight: float,
) -> np.ndarray:
    work = pd.DataFrame({"uid": uid.to_numpy(), "prediction": prediction})
    grouped = work.groupby("uid", sort=False)["prediction"].transform("max")
    return (1.0 - weight) * prediction + weight * grouped.to_numpy()


def build_meta_features(
    predictions: pd.DataFrame,
    sources: tuple[str, ...],
    fold: pd.Series | None = None,
) -> pd.DataFrame:
    probability = predictions.loc[:, list(sources)].clip(1e-6, 1 - 1e-6)
    if fold is None:
        ranks = probability.rank(method="average", pct=True)
    else:
        ranks = probability.groupby(fold).rank(method="average", pct=True)

    features: dict[str, pd.Series] = {}
    for column in sources:
        values = probability[column]
        features[f"{column}_prob"] = values
        features[f"{column}_logit"] = np.log(values / (1.0 - values))
        features[f"{column}_rank"] = ranks[column]
        features[f"{column}_rank_sq"] = ranks[column] ** 2

    features["rank_mean"] = ranks.mean(axis=1)
    features["rank_std"] = ranks.std(axis=1)
    features["rank_min"] = ranks.min(axis=1)
    features["rank_max"] = ranks.max(axis=1)
    for first, second in combinations(sources, 2):
        features[f"rank_{first}_{second}_diff"] = (
            ranks[first] - ranks[second]
        ).abs()
    return pd.DataFrame(features, index=predictions.index).astype("float32")


class MetaMLP(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.linear = nn.Linear(input_size, 1)
        self.hidden = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.SiLU(),
            nn.Dropout(0.08),
            nn.Linear(32, 16),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(16, 1),
        )

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.linear(values) + 0.20 * self.hidden(values)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def predict_mlp(model: MetaMLP, values: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(values, dtype=torch.float32)
        return torch.sigmoid(model(tensor)).squeeze(1).cpu().numpy()


def train_mlp_with_validation(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    seed: int,
) -> tuple[int, np.ndarray, float]:
    seed_everything(seed)
    model = MetaMLP(X_train.shape[1])
    positive_weight = float(np.sqrt((len(y_train) - y_train.sum()) / y_train.sum()))
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([positive_weight], dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train, dtype=torch.float32),
            torch.as_tensor(y_train, dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=8192,
        shuffle=True,
    )

    best_auc = -np.inf
    best_epoch = 1
    best_state = None
    stale_epochs = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for values, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(values), labels)
            loss.backward()
            optimizer.step()
        prediction = predict_mlp(model, X_valid)
        score = roc_auc_score(y_valid, prediction)
        if score > best_auc + 1e-6:
            best_auc = float(score)
            best_epoch = epoch
            best_state = {
                key: value.detach().clone() for key, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("MLP did not produce a validation state")
    model.load_state_dict(best_state)
    return best_epoch, predict_mlp(model, X_valid), best_auc


def train_mlp_fixed_epochs(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    seed: int,
    epochs: int,
) -> np.ndarray:
    seed_everything(seed)
    model = MetaMLP(X_train.shape[1])
    positive_weight = float(np.sqrt((len(y_train) - y_train.sum()) / y_train.sum()))
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([positive_weight], dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train, dtype=torch.float32),
            torch.as_tensor(y_train, dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=8192,
        shuffle=True,
    )
    for _ in range(max(1, epochs)):
        model.train()
        for values, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(values), labels)
            loss.backward()
            optimizer.step()
    prediction = predict_mlp(model, X_test)
    del model
    gc.collect()
    return prediction


def best_rank_blend(
    y_true: np.ndarray,
    neural: np.ndarray,
    linear: np.ndarray,
) -> dict[str, float]:
    neural_rank = rank_prediction(neural)
    linear_rank = rank_prediction(linear)
    rows = []
    for neural_weight in np.linspace(0.0, 1.0, 101):
        prediction = neural_weight * neural_rank + (1.0 - neural_weight) * linear_rank
        rows.append(
            {
                "neural_weight": float(neural_weight),
                "linear_weight": float(1.0 - neural_weight),
                "auc": float(roc_auc_score(y_true, prediction)),
            }
        )
    return max(rows, key=lambda row: row["auc"])


def evaluate_source_set(
    oof: pd.DataFrame,
    uid_train: pd.Series,
    sources: tuple[str, ...],
) -> dict:
    features = build_meta_features(oof, sources, fold=oof["fold"])
    train_mask = oof["fold"] < 2
    valid_mask = oof["fold"] == 2
    y_train = oof.loc[train_mask, TARGET].to_numpy(dtype="float32")
    y_valid = oof.loc[valid_mask, TARGET].to_numpy(dtype="float32")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(features.loc[train_mask]).astype("float32")
    X_valid = scaler.transform(features.loc[valid_mask]).astype("float32")
    linear = LogisticRegression(
        C=LOGISTIC_C,
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )
    linear.fit(X_train, y_train)
    linear_valid = linear.predict_proba(X_valid)[:, 1]

    epochs: dict[str, int] = {}
    neural_parts = []
    seed_auc = {}
    for seed in META_SEEDS:
        epoch, prediction, score = train_mlp_with_validation(
            X_train, y_train, X_valid, y_valid, seed
        )
        epochs[str(seed)] = int(epoch)
        neural_parts.append(prediction)
        seed_auc[str(seed)] = float(score)
    neural_valid = np.mean(neural_parts, axis=0)
    blend = best_rank_blend(y_valid, neural_valid, linear_valid)
    prediction = (
        blend["neural_weight"] * rank_prediction(neural_valid)
        + blend["linear_weight"] * rank_prediction(linear_valid)
    )

    valid_rows = oof.loc[valid_mask, "row_index"].to_numpy(dtype="int64")
    uid = uid_train.iloc[valid_rows].reset_index(drop=True)
    uid_candidates = []
    for weight in (0.0, 0.10, 0.25, 0.50):
        processed = apply_uid_max(prediction, uid, weight)
        uid_candidates.append(
            {"weight": weight, "auc": float(roc_auc_score(y_valid, processed))}
        )
    uid_recipe = max(uid_candidates, key=lambda row: row["auc"])
    return {
        "sources": list(sources),
        "features": list(features.columns),
        "holdout_rows": int(valid_mask.sum()),
        "linear_auc": float(roc_auc_score(y_valid, linear_valid)),
        "neural_auc": float(roc_auc_score(y_valid, neural_valid)),
        "blend": blend,
        "uid_candidates": uid_candidates,
        "uid_recipe": uid_recipe,
        "epochs": epochs,
        "seed_auc": seed_auc,
    }


def fit_final(
    oof: pd.DataFrame,
    test_predictions: pd.DataFrame,
    uid_test: pd.Series,
    recipe: dict,
) -> np.ndarray:
    sources = tuple(recipe["sources"])
    train_features = build_meta_features(oof, sources, fold=oof["fold"])
    test_features = build_meta_features(test_predictions, sources)
    y = oof[TARGET].to_numpy(dtype="float32")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_features).astype("float32")
    X_test = scaler.transform(test_features).astype("float32")
    linear = LogisticRegression(
        C=LOGISTIC_C,
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )
    linear.fit(X_train, y)
    linear_test = linear.predict_proba(X_test)[:, 1]

    neural_parts = []
    for seed in META_SEEDS:
        neural_parts.append(
            train_mlp_fixed_epochs(
                X_train,
                y,
                X_test,
                seed,
                int(recipe["epochs"][str(seed)]),
            )
        )
    neural_test = np.mean(neural_parts, axis=0)
    blend = recipe["blend"]
    prediction = (
        float(blend["neural_weight"]) * rank_prediction(neural_test)
        + float(blend["linear_weight"]) * rank_prediction(linear_test)
    )
    return apply_uid_max(
        prediction,
        uid_test.reset_index(drop=True),
        float(recipe["uid_recipe"]["weight"]),
    )


def main() -> None:
    boost = pd.read_csv(ROOT / "boost3_oof_predictions.csv")
    temporal = pd.read_csv(ROOT / "temporal7_oof_predictions.csv")
    temporal = temporal[["row_index", "fold", "stack_prediction"]].rename(
        columns={"fold": "temporal_fold", "stack_prediction": TEMPORAL_SOURCE}
    )
    oof = boost.merge(temporal, on="row_index", how="left", validate="one_to_one")
    if oof[TEMPORAL_SOURCE].isna().any():
        raise ValueError("Temporal7 OOF predictions do not cover boost3 rows")
    if not np.array_equal(
        oof["temporal_fold"].to_numpy(), oof["fold"].to_numpy() + 1
    ):
        raise ValueError("Boost3 and temporal7 folds are not aligned")

    raw_columns = [
        "TransactionID",
        "TransactionDT",
        "card1",
        "addr1",
        "D1",
        "P_emaildomain",
    ]
    raw_train = pd.read_csv(ROOT / "train_transaction.csv", usecols=raw_columns)
    raw_test = pd.read_csv(ROOT / "test_transaction.csv", usecols=raw_columns)
    uid_train = make_uid(raw_train)
    uid_test = make_uid(raw_test)

    baseline = evaluate_source_set(oof, uid_train, BASE_SOURCES)
    temporal_recipe = evaluate_source_set(
        oof, uid_train, (*BASE_SOURCES, TEMPORAL_SOURCE)
    )
    holdout_gain = float(
        temporal_recipe["uid_recipe"]["auc"] - baseline["uid_recipe"]["auc"]
    )
    accepted = holdout_gain > 0.0
    selected = temporal_recipe if accepted else baseline

    test_predictions = pd.DataFrame(
        {
            "catboost": np.load(ROOT / "stack_test_catboost.npy"),
            "lightgbm": np.load(ROOT / "stack_test_lightgbm.npy"),
            "xgboost": np.load(ROOT / "stack_test_xgboost.npy"),
            TEMPORAL_SOURCE: pd.read_csv(
                ROOT / "submission_temporal7_no_gap.csv"
            )[TARGET].to_numpy(),
        }
    )
    prediction = fit_final(oof, test_predictions, uid_test, selected)
    sample = pd.read_csv(ROOT / "sample_submission.csv")
    if not np.array_equal(sample["TransactionID"].to_numpy(), raw_test["TransactionID"]):
        raise ValueError("Sample and test TransactionID order differ")
    output = sample[["TransactionID"]].copy()
    output[TARGET] = prediction
    output.to_csv(OUTPUT_PATH, index=False)

    metrics = {
        "selection": "official-train temporal OOF only",
        "external_gap_used": False,
        "competition_test_labels_used": False,
        "baseline": baseline,
        "temporal_candidate": temporal_recipe,
        "holdout_gain": holdout_gain,
        "temporal_source_accepted": accepted,
        "selected_sources": selected["sources"],
        "output": OUTPUT_PATH.name,
        "rows": len(output),
    }
    METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting build_honest_no_gap_meta.py


In [4]:
%%writefile build_honest_user_means_final.py
"""Build the locked conservative user-means CatBoost candidate."""

from __future__ import annotations

import argparse
import gc
import json
from pathlib import Path
import re
import time

from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

from build_honest_no_gap_meta import evaluate_source_set, fit_final, make_uid, rank_prediction
from fraud_honest_advanced_data import ADVANCED_CAT_PARAMS as CAT_PARAMS
from fraud_vblock_features import add_vblock_user_features
from train_honest_advanced_catboost import CACHE_DIR as ADVANCED_DIR
from train_honest_advanced_catboost import SOURCE as ADVANCED_SOURCE
from train_honest_advanced_catboost import prepare
from train_honest_user_means_catboost import CACHE_DIR, SOURCE


ROOT = Path(__file__).resolve().parent
TARGET = "isFraud"
CAT_ENHANCED = "cat_enhanced"
CAT_MEANS = "cat_means_05"
SOURCES = ("catboost", CAT_ENHANCED, CAT_MEANS, "lightgbm", "xgboost")
ADVANCED_WEIGHT = 0.25
MEANS_WEIGHT = 0.05
DEFAULT_SEEDS = (1729,)
ITERATIONS = 1_900
OUTPUT_PATH = ROOT / "submission_honest_user_means.csv"
MULTISEED_OUTPUT_PATH = ROOT / "submission_honest_user_means_multiseed.csv"
REPORT_PATH = CACHE_DIR / "final_report.json"
MULTISEED_REPORT_PATH = CACHE_DIR / "final_multiseed_report.json"


def parse_seeds(value: str) -> tuple[int, ...]:
    seeds = tuple(int(item) for item in value.split(",") if item.strip())
    if not seeds:
        raise argparse.ArgumentTypeError("Provide at least one seed")
    return seeds


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true")
    parser.add_argument("--seeds", type=parse_seeds, default=DEFAULT_SEEDS)
    args = parser.parse_args()
    started = time.time()

    prepared, features, categorical = prepare()
    (
        prepared["train"],
        prepared["inference"],
        vblock_features,
        _,
    ) = add_vblock_user_features(
        prepared["train"],
        prepared["inference"],
        prepared["train_components"],
        prepared["inference_components"],
    )
    mean_features = [
        column
        for column in vblock_features
        if re.fullmatch(r"wide_user_[CDV]\d+_mean", column)
    ]
    features = list(dict.fromkeys([*features, *mean_features]))
    model_started = time.time()
    means_ranks = []
    model_rows = []
    for seed in args.seeds:
        model_path = CACHE_DIR / f"final_seed_{seed}.cbm"
        prediction_path = CACHE_DIR / f"final_seed_{seed}_test.npy"
        seed_started = time.time()
        if model_path.exists() and prediction_path.exists() and not args.force:
            seed_prediction = np.load(prediction_path)
            if seed_prediction.dtype != np.float64:
                print(
                    f"Refreshing final seed {seed} predictions as float64",
                    flush=True,
                )
                model = CatBoostClassifier()
                model.load_model(model_path)
                seed_prediction = model.predict_proba(
                    prepared["inference"][features]
                )[:, 1]
                np.save(prediction_path, seed_prediction)
                del model
                gc.collect()
            model_cached = True
        else:
            print(
                f"Final user-means CatBoost: seed={seed}, trees={ITERATIONS}, "
                f"features={len(features)}",
                flush=True,
            )
            model = CatBoostClassifier(
                **{
                    **CAT_PARAMS,
                    "iterations": ITERATIONS,
                    "random_seed": seed,
                    "verbose": 200,
                }
            )
            model.fit(
                prepared["train"][features],
                prepared["y"],
                cat_features=categorical,
            )
            seed_prediction = model.predict_proba(
                prepared["inference"][features]
            )[:, 1]
            model.save_model(model_path)
            np.save(prediction_path, seed_prediction)
            del model
            gc.collect()
            model_cached = False
        means_ranks.append(rank_prediction(seed_prediction))
        model_rows.append(
            {
                "seed": seed,
                "cached": model_cached,
                "minutes": (time.time() - seed_started) / 60.0,
                "model": model_path.name,
            }
        )
    means_test = np.mean(means_ranks, axis=0)
    test_ids = prepared["inference_ids"].to_numpy()
    del prepared
    gc.collect()

    base_oof = pd.read_csv(ROOT / "boost3_oof_predictions.csv")
    advanced_oof = pd.read_csv(
        ADVANCED_DIR / "oof.csv", usecols=["row_index", ADVANCED_SOURCE]
    )
    means_oof = pd.concat(
        [
            pd.read_csv(
                CACHE_DIR / f"fold_{fold}_oof.csv",
                usecols=["row_index", SOURCE],
            )
            for fold in range(3)
        ],
        ignore_index=True,
    )
    oof = base_oof.merge(
        advanced_oof, on="row_index", validate="one_to_one"
    ).merge(means_oof, on="row_index", validate="one_to_one")
    old_oof_rank = oof.groupby("fold")["catboost"].rank(pct=True)
    advanced_oof_rank = oof.groupby("fold")[ADVANCED_SOURCE].rank(pct=True)
    means_oof_rank = oof.groupby("fold")[SOURCE].rank(pct=True)
    oof[CAT_ENHANCED] = (
        (1.0 - ADVANCED_WEIGHT) * old_oof_rank
        + ADVANCED_WEIGHT * advanced_oof_rank
    )
    oof[CAT_MEANS] = (
        (1.0 - MEANS_WEIGHT) * old_oof_rank
        + MEANS_WEIGHT * means_oof_rank
    )

    raw_train = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    raw_test = pd.read_csv(
        ROOT / "test_transaction.csv",
        usecols=[
            "TransactionID",
            "TransactionDT",
            "card1",
            "addr1",
            "D1",
            "P_emaildomain",
        ],
    )
    uid_train = make_uid(raw_train)
    uid_test = make_uid(raw_test)
    winner_sources = ("catboost", CAT_ENHANCED, "lightgbm", "xgboost")
    baseline = evaluate_source_set(oof, uid_train, winner_sources)
    selected = evaluate_source_set(oof, uid_train, SOURCES)
    holdout_gain = float(
        selected["uid_recipe"]["auc"] - baseline["uid_recipe"]["auc"]
    )
    if holdout_gain <= 0:
        raise RuntimeError(f"User-means lock gain is not positive: {holdout_gain}")

    old_cat_test = np.load(ROOT / "stack_test_catboost.npy")
    # This branch was selected after exporting the advanced source as float32.
    # Keep that rank contract explicit so fresh and cached runs are identical.
    advanced_ranks = np.mean(
        [
            rank_prediction(
                np.load(
                    ADVANCED_DIR / f"final_seed_{seed}_test.npy"
                ).astype("float32")
            )
            for seed in (1729, 2026, 3407)
        ],
        axis=0,
    )
    old_rank = rank_prediction(old_cat_test)
    test_predictions = pd.DataFrame(
        {
            "catboost": old_cat_test,
            CAT_ENHANCED: (
                (1.0 - ADVANCED_WEIGHT) * old_rank
                + ADVANCED_WEIGHT * advanced_ranks
            ),
            CAT_MEANS: (
                (1.0 - MEANS_WEIGHT) * old_rank
                + MEANS_WEIGHT * rank_prediction(means_test)
            ),
            "lightgbm": np.load(ROOT / "stack_test_lightgbm.npy"),
            "xgboost": np.load(ROOT / "stack_test_xgboost.npy"),
        }
    )
    prediction = fit_final(oof, test_predictions, uid_test, selected)
    sample = pd.read_csv(ROOT / "sample_submission.csv")
    if not np.array_equal(sample["TransactionID"], test_ids):
        raise ValueError("Prepared test and sample TransactionID order differ")
    if not np.array_equal(sample["TransactionID"], raw_test["TransactionID"]):
        raise ValueError("Raw test and sample TransactionID order differ")
    output = sample[["TransactionID"]].copy()
    output[TARGET] = prediction
    output_path = OUTPUT_PATH if len(args.seeds) == 1 else MULTISEED_OUTPUT_PATH
    report_path = REPORT_PATH if len(args.seeds) == 1 else MULTISEED_REPORT_PATH
    output.to_csv(output_path, index=False)

    report = {
        "data_policy": "official train/test only",
        "selection": "official-train temporal OOF only",
        "sources": list(SOURCES),
        "advanced_weight": ADVANCED_WEIGHT,
        "means_weight": MEANS_WEIGHT,
        "seeds": list(args.seeds),
        "iterations": ITERATIONS,
        "models": model_rows,
        "model_minutes": (time.time() - model_started) / 60.0,
        "features": len(features),
        "user_mean_features": mean_features,
        "baseline_holdout_auc": baseline["uid_recipe"]["auc"],
        "selected_holdout_auc": selected["uid_recipe"]["auc"],
        "holdout_gain": holdout_gain,
        "selected_recipe": selected,
        "output": output_path.name,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting build_honest_user_means_final.py


In [5]:
%%writefile clean_v2_pipeline.py
"""Strict no-bridge temporal ensemble for the local fraud dataset.

Development deliberately rebuilds every feature matrix from only two pieces:
the labeled history available before a cutoff and the future validation block.
Rows inside the embargo are not passed to feature engineering. This mirrors the
competition setting where no middle bridge is available.
"""

from __future__ import annotations

import argparse
import copy
from dataclasses import asdict, dataclass
import gc
from itertools import product
import json
from pathlib import Path
import re
import time

from catboost import CatBoostClassifier
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb

from fraud_features import TARGET, build_features, read_and_merge
from fraud_multicounter_features import add_multi_counter_features
from fraud_next_features import (
    add_behavior_distribution_features,
    add_calendar_amount_features,
    add_identity_features,
    add_velocity_features,
)
from fraud_overlap_recipe import assign_segments, uid_metadata
from fraud_user_features import CAUSAL_HISTORY_COLUMNS, add_user_profile_features
from fraud_vblock_features import TOP_V_COLUMNS, add_vblock_user_features


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "clean_v2"
MODEL_DIR = WORK_DIR / "models"
PREDICTION_DIR = WORK_DIR / "predictions"
RECIPE_PATH = WORK_DIR / "recipe.json"
REPORT_PATH = WORK_DIR / "backtest_report.json"
SOURCE_PATH = WORK_DIR / "test_source_predictions.csv"
SUBMISSION_PATH = ROOT / "submission_clean_v2.csv"

EXPECTED_TRAIN_ROWS = 365_365
EXPECTED_TEST_ROWS = 129_736
DAY_SECONDS = 86_400.0
SEEDS = (42, 2026, 3407)
SOURCE_NAMES = (
    "cat_identity",
    "cat_history",
    "cat_weighted",
    "lgb_giba",
    "lgb_cold",
    "xgb_giba",
)
SEGMENTS = ("strict", "partial", "cold")
HORIZONS = (30, 45, 60, 75)


@dataclass(frozen=True)
class PairSpec:
    name: str
    stage: str
    horizon: int
    train_end_day: int
    valid_start_day: int
    valid_end_day: int


DEV_SPECS = (
    PairSpec("dev_h30_a", "dev", 30, 15, 45, 60),
    PairSpec("dev_h30_b", "dev", 30, 30, 60, 75),
    PairSpec("dev_h30_c", "dev", 30, 45, 75, 90),
    PairSpec("dev_h45_a", "dev", 45, 15, 60, 75),
    PairSpec("dev_h45_b", "dev", 45, 30, 75, 90),
    PairSpec("dev_h60", "dev", 60, 15, 75, 90),
)
LOCK_SPECS = (
    PairSpec("lock_h30", "lock", 30, 60, 90, 106),
    PairSpec("lock_h45", "lock", 45, 45, 90, 106),
    PairSpec("lock_h60", "lock", 60, 30, 90, 106),
    PairSpec("lock_h75", "lock", 75, 15, 90, 106),
)


CAT_PARAMS = {
    "iterations": 1_500,
    "depth": 8,
    "learning_rate": 0.06,
    "l2_leaf_reg": 8,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "rsm": 0.90,
    "border_count": 128,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "one_hot_max_size": 10,
    "max_ctr_complexity": 1,
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": 200,
}

LGB_GIBA_PARAMS = {
    "n_estimators": 1_500,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.03,
    "min_child_samples": 40,
    "subsample": 0.70,
    "subsample_freq": 1,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.5,
    "reg_lambda": 5.0,
    "max_bin": 255,
    "extra_trees": True,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}

LGB_COLD_PARAMS = {
    "n_estimators": 1_000,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 47,
    "learning_rate": 0.035,
    "min_child_samples": 90,
    "subsample": 0.78,
    "subsample_freq": 1,
    "colsample_bytree": 0.78,
    "reg_alpha": 0.75,
    "reg_lambda": 10.0,
    "max_bin": 255,
    "extra_trees": True,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}

XGB_PARAMS = {
    "n_estimators": 1_500,
    "learning_rate": 0.03,
    "max_depth": 7,
    "min_child_weight": 20,
    "subsample": 0.80,
    "colsample_bytree": 0.75,
    "reg_alpha": 0.5,
    "reg_lambda": 10.0,
    "gamma": 0.05,
    "max_bin": 256,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_jobs": -1,
}


def unique(*groups: list[str]) -> list[str]:
    return list(dict.fromkeys(column for group in groups for column in group))


def rank_prediction(values: np.ndarray | pd.Series) -> np.ndarray:
    return pd.Series(np.asarray(values)).rank(method="average", pct=True).to_numpy()


def auc(y: np.ndarray | pd.Series, prediction: np.ndarray) -> float:
    return float(roc_auc_score(np.asarray(y), np.asarray(prediction)))


def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Cannot serialize {type(value)}")


def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, default=json_default), encoding="utf-8"
    )


def read_official_train() -> pd.DataFrame:
    frame = read_and_merge(ROOT, "train").reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS:
        raise ValueError(
            f"Expected {EXPECTED_TRAIN_ROWS:,} train rows, got {len(frame):,}"
        )
    if TARGET not in frame:
        raise ValueError(f"Official train is missing {TARGET}")
    if not frame["TransactionDT"].is_monotonic_increasing:
        raise ValueError("Training rows are not sorted by TransactionDT")
    return frame


def read_official_test() -> pd.DataFrame:
    frame = read_and_merge(ROOT, "test").reset_index(drop=True)
    if len(frame) != EXPECTED_TEST_ROWS:
        raise ValueError(
            f"Expected {EXPECTED_TEST_ROWS:,} test rows, got {len(frame):,}"
        )
    if TARGET in frame:
        raise AssertionError("Target unexpectedly exists in competition test")
    return frame


def normalize_categories(
    history: pd.DataFrame,
    future: pd.DataFrame,
    columns: list[str],
) -> None:
    for column in columns:
        history[column] = (
            history[column].astype("string").fillna("<MISSING>").astype(str)
        )
        future[column] = (
            future[column].astype("string").fillna("<MISSING>").astype(str)
        )


def prepare_pair(
    history_raw: pd.DataFrame,
    future_raw: pd.DataFrame,
) -> dict:
    """Build target-free pair features with no rows inside the time gap."""
    y = history_raw[TARGET].astype("int8").reset_index(drop=True)
    history = history_raw.copy().reset_index(drop=True)
    future = future_raw.drop(columns=TARGET, errors="ignore").copy().reset_index(
        drop=True
    )

    history, future, base_features, base_categorical = build_features(
        history,
        future,
        giba_features=True,
        frequency_mode="selected",
        v307_chain_features=True,
    )
    metadata_history = uid_metadata(history).reset_index(drop=True)
    metadata_future = uid_metadata(future).reset_index(drop=True)

    (
        history,
        future,
        user_features,
        history_components,
        future_components,
        graph_stats,
    ) = add_user_profile_features(history, future, y)
    causal_history = [
        column for column in CAUSAL_HISTORY_COLUMNS if column in history
    ]
    target_free_profile = [
        column for column in user_features if column not in causal_history
    ]

    history, future, vblock_features, vblock_stats = add_vblock_user_features(
        history,
        future,
        history_components,
        future_components,
    )
    history, future, multi_features, multi_categorical, multi_stats = (
        add_multi_counter_features(history, future, max_chain_size=100)
    )
    history, future, velocity_features, velocity_stats = add_velocity_features(
        history, future
    )
    history, future, calendar_features, calendar_stats = (
        add_calendar_amount_features(history, future)
    )
    (
        history,
        future,
        identity_features,
        identity_categorical,
        identity_stats,
    ) = add_identity_features(history, future)
    history, future, behavior_features, behavior_stats = (
        add_behavior_distribution_features(
            history,
            future,
            history_components,
            future_components,
        )
    )

    categorical = unique(
        base_categorical,
        multi_categorical,
        identity_categorical,
    )
    normalize_categories(history, future, categorical)

    identity_view = unique(base_features, multi_features)
    history_view = unique(
        base_features,
        target_free_profile,
        causal_history,
        multi_features,
        velocity_features,
        identity_features,
    )

    generic_core = []
    for column in base_features:
        if column.startswith("uid_"):
            continue
        if re.fullmatch(r"V\d+", column) or re.fullmatch(r"id_\d+", column):
            continue
        generic_core.append(column)
    selected_raw_v = [column for column in TOP_V_COLUMNS if column in history]
    cold_view = unique(
        generic_core,
        selected_raw_v,
        target_free_profile,
        vblock_features,
        velocity_features,
        calendar_features,
        identity_features,
        behavior_features,
    )

    domain_view = [
        column
        for column in unique(
            generic_core,
            selected_raw_v,
            identity_features,
            [name for name in vblock_features if "wide_user_" not in name],
        )
        if column != "TransactionDT"
        and not column.startswith("DT_")
        and not column.startswith("calendar_")
        and not column.endswith("_minus_day")
        and column != "D1_origin_day"
    ]

    views = {
        "identity": identity_view,
        "history": history_view,
        "cold": cold_view,
        "domain": domain_view,
    }
    for name, features in views.items():
        missing = [column for column in features if column not in history]
        if missing:
            raise ValueError(f"{name} view has missing columns: {missing[:5]}")

    return {
        "history": history,
        "future": future,
        "y": y,
        "views": views,
        "categorical": categorical,
        "metadata_history": metadata_history,
        "metadata_future": metadata_future,
        "stats": {
            "rows": {"history": len(history), "future": len(future)},
            "features": {name: len(value) for name, value in views.items()},
            "categorical": len(categorical),
            "graph": graph_stats,
            "vblock": vblock_stats,
            "multi": multi_stats,
            "velocity": velocity_stats,
            "calendar": calendar_stats,
            "identity": identity_stats,
            "behavior": behavior_stats,
        },
    }


def encode_pair(
    history: pd.DataFrame,
    future: pd.DataFrame,
    features: list[str],
    categorical: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [history[features], future[features]], ignore_index=True, copy=False
    ).copy()
    for column in categorical:
        if column in combined:
            codes, _ = pd.factorize(combined[column], sort=False)
            combined[column] = codes.astype("int32", copy=False)
    combined.replace([np.inf, -np.inf], np.nan, inplace=True)
    history_rows = len(history)
    return combined.iloc[:history_rows], combined.iloc[history_rows:]


def domain_weights(prepared: dict, seed: int) -> tuple[np.ndarray, dict]:
    history = prepared["history"]
    future = prepared["future"]
    features = prepared["views"]["domain"]
    categorical = [
        column for column in prepared["categorical"] if column in features
    ]
    X_history, X_future = encode_pair(
        history, future, features, categorical
    )
    rng = np.random.default_rng(seed)
    max_history = min(len(X_history), max(len(X_future) * 2, 20_000))
    history_index = np.sort(
        rng.choice(len(X_history), size=max_history, replace=False)
    )
    X_domain = pd.concat(
        [X_history.iloc[history_index], X_future], ignore_index=True
    )
    y_domain = np.concatenate(
        [np.zeros(len(history_index), dtype="int8"), np.ones(len(X_future), dtype="int8")]
    )
    model = lgb.LGBMClassifier(
        n_estimators=250,
        objective="binary",
        learning_rate=0.04,
        num_leaves=31,
        max_depth=6,
        min_child_samples=120,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.65,
        reg_alpha=1.0,
        reg_lambda=8.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
        force_col_wise=True,
    )
    model.fit(X_domain, y_domain, callbacks=[lgb.log_evaluation(0)])
    domain_train_prediction = model.predict_proba(X_domain)[:, 1]
    history_probability = np.clip(model.predict_proba(X_history)[:, 1], 0.02, 0.98)
    prior_ratio = len(history_index) / max(len(X_future), 1)
    weights = prior_ratio * history_probability / (1.0 - history_probability)
    weights /= max(float(np.mean(weights)), 1e-9)
    weights = np.clip(weights, 0.25, 4.0)
    weights /= float(np.mean(weights))
    report = {
        "features": len(features),
        "sample_rows": len(X_domain),
        "train_auc": auc(y_domain, domain_train_prediction),
        "weight_min": float(weights.min()),
        "weight_max": float(weights.max()),
        "weight_mean": float(weights.mean()),
        "weight_q05": float(np.quantile(weights, 0.05)),
        "weight_q95": float(np.quantile(weights, 0.95)),
    }
    del X_history, X_future, X_domain, model
    gc.collect()
    return weights.astype("float32"), report


def fit_cat_source(
    prepared: dict,
    features: list[str],
    seeds: tuple[int, ...],
    model_stem: str,
    fixed_iterations: int | None,
    sample_weight: np.ndarray | None = None,
) -> tuple[np.ndarray, list[int]]:
    categorical = [
        column for column in prepared["categorical"] if column in features
    ]
    predictions = []
    iterations = []
    for seed in seeds:
        model_path = MODEL_DIR / f"{model_stem}_s{seed}.cbm"
        params = {
            **CAT_PARAMS,
            "random_seed": seed,
            "iterations": fixed_iterations or CAT_PARAMS["iterations"],
        }
        model = CatBoostClassifier(**params)
        fit_kwargs = {
            "X": prepared["history"][features],
            "y": prepared["y"],
            "cat_features": categorical,
        }
        if sample_weight is not None:
            fit_kwargs["sample_weight"] = sample_weight
        if fixed_iterations is None:
            fit_kwargs.update(
                {
                    "eval_set": (
                        prepared["future"][features],
                        prepared["future_y"],
                    ),
                    "early_stopping_rounds": 120,
                    "use_best_model": True,
                }
            )
        model.fit(**fit_kwargs)
        prediction = model.predict_proba(prepared["future"][features])[:, 1]
        predictions.append(prediction)
        best = model.tree_count_
        iterations.append(int(best))
        model.save_model(model_path)
        del model
        gc.collect()
    return np.mean(predictions, axis=0), iterations


def fit_lgb_source(
    prepared: dict,
    features: list[str],
    source: str,
    seed: int,
    model_stem: str,
    fixed_iterations: int | None,
) -> tuple[np.ndarray, int]:
    categorical = [
        column for column in prepared["categorical"] if column in features
    ]
    X_history, X_future = encode_pair(
        prepared["history"], prepared["future"], features, categorical
    )
    base = LGB_GIBA_PARAMS if source == "lgb_giba" else LGB_COLD_PARAMS
    params = {
        **base,
        "random_state": seed,
        "n_estimators": fixed_iterations or base["n_estimators"],
    }
    model = lgb.LGBMClassifier(**params)
    fit_kwargs = {
        "X": X_history,
        "y": prepared["y"],
        "categorical_feature": categorical,
    }
    if fixed_iterations is None:
        fit_kwargs.update(
            {
                "eval_set": [(X_future, prepared["future_y"])],
                "eval_metric": "auc",
                "callbacks": [
                    lgb.early_stopping(120, verbose=False),
                    lgb.log_evaluation(200),
                ],
            }
        )
    else:
        fit_kwargs["callbacks"] = [lgb.log_evaluation(0)]
    model.fit(**fit_kwargs)
    best = int(model.best_iteration_ or params["n_estimators"])
    prediction = model.predict_proba(X_future, num_iteration=best)[:, 1]
    model.booster_.save_model(MODEL_DIR / f"{model_stem}.txt", num_iteration=best)
    del model, X_history, X_future
    gc.collect()
    return prediction, best


def fit_xgb_source(
    prepared: dict,
    features: list[str],
    seed: int,
    model_stem: str,
    fixed_iterations: int | None,
) -> tuple[np.ndarray, int]:
    categorical = [
        column for column in prepared["categorical"] if column in features
    ]
    X_history, X_future = encode_pair(
        prepared["history"], prepared["future"], features, categorical
    )
    params = {
        **XGB_PARAMS,
        "random_state": seed,
        "n_estimators": fixed_iterations or XGB_PARAMS["n_estimators"],
    }
    if fixed_iterations is None:
        params["early_stopping_rounds"] = 120
    model = xgb.XGBClassifier(**params)
    fit_kwargs = {"X": X_history, "y": prepared["y"], "verbose": False}
    if fixed_iterations is None:
        fit_kwargs["eval_set"] = [(X_future, prepared["future_y"])]
    model.fit(**fit_kwargs)
    best = int(
        getattr(model, "best_iteration", params["n_estimators"] - 1) + 1
    )
    prediction = model.predict_proba(X_future)[:, 1]
    model.save_model(MODEL_DIR / f"{model_stem}.json")
    del model, X_history, X_future
    gc.collect()
    return prediction, best


def prediction_cache(spec: PairSpec) -> Path:
    return PREDICTION_DIR / f"{spec.name}.csv"


def cached_fit_report(spec: PairSpec, raw: pd.DataFrame) -> dict:
    """Recover iteration counts from model files after an interrupted run."""
    cat_iterations: dict[str, list[int]] = {}
    cat_layout = {
        "cat_identity": (42, 2026),
        "cat_history": (42,),
        "cat_weighted": (3407,),
    }
    for source, seeds in cat_layout.items():
        values = []
        for seed in seeds:
            path = MODEL_DIR / f"{spec.name}_{source}_s{seed}.cbm"
            if not path.exists():
                continue
            model = CatBoostClassifier()
            model.load_model(path)
            values.append(int(model.tree_count_))
        cat_iterations[source] = values

    lgb_iterations = {}
    for source in ("lgb_giba", "lgb_cold"):
        path = MODEL_DIR / f"{spec.name}_{source}.txt"
        lgb_iterations[source] = (
            [int(lgb.Booster(model_file=str(path)).num_trees())]
            if path.exists()
            else []
        )
    xgb_path = MODEL_DIR / f"{spec.name}_xgb_giba.json"
    xgb_iterations = []
    if xgb_path.exists():
        model = xgb.XGBClassifier()
        model.load_model(xgb_path)
        xgb_iterations = [int(model.get_booster().num_boosted_rounds())]

    day = raw["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS
    return {
        "cached": True,
        "spec": asdict(spec),
        "history_rows": int(np.sum(day < spec.train_end_day)),
        "future_rows": int(
            np.sum((day >= spec.valid_start_day) & (day < spec.valid_end_day))
        ),
        "best_iterations": {
            **cat_iterations,
            **lgb_iterations,
            "xgb_giba": xgb_iterations,
        },
    }


def train_pair(
    raw: pd.DataFrame,
    spec: PairSpec,
    fixed_iterations: dict[str, int] | None,
    force: bool,
) -> tuple[pd.DataFrame, dict]:
    cache = prediction_cache(spec)
    if cache.exists() and not force:
        frame = pd.read_csv(cache)
        required = {"row_index", "TransactionID", TARGET, "segment", *SOURCE_NAMES}
        if required.issubset(frame.columns):
            print(f"Loading cached {spec.name}", flush=True)
            return frame, cached_fit_report(spec, raw)

    day = raw["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS
    history_index = np.flatnonzero(day < spec.train_end_day)
    future_index = np.flatnonzero(
        (day >= spec.valid_start_day) & (day < spec.valid_end_day)
    )
    if not len(history_index) or not len(future_index):
        raise ValueError(f"Empty history/future for {spec.name}")
    skipped = int(
        np.sum((day >= spec.train_end_day) & (day < spec.valid_start_day))
    )
    print(
        f"\n{spec.name}: history={len(history_index):,}, "
        f"future={len(future_index):,}, physically skipped={skipped:,}",
        flush=True,
    )
    started = time.time()
    prepared = prepare_pair(raw.iloc[history_index], raw.iloc[future_index])
    prepared["future_y"] = raw.iloc[future_index][TARGET].astype("int8").reset_index(
        drop=True
    )
    combined_metadata = pd.concat(
        [prepared["metadata_history"], prepared["metadata_future"]],
        ignore_index=True,
    )
    segments = assign_segments(
        combined_metadata,
        np.arange(len(history_index), dtype="int64"),
        np.arange(
            len(history_index), len(history_index) + len(future_index), dtype="int64"
        ),
    )

    tuned = fixed_iterations is None
    identity_iterations = None if tuned else fixed_iterations["cat_identity"]
    identity_prediction, identity_trees = fit_cat_source(
        prepared,
        prepared["views"]["identity"],
        SEEDS[:2],
        f"{spec.name}_cat_identity",
        identity_iterations,
    )
    history_prediction, history_trees = fit_cat_source(
        prepared,
        prepared["views"]["history"],
        (42,),
        f"{spec.name}_cat_history",
        None if tuned else fixed_iterations["cat_history"],
    )
    weights, domain_report = domain_weights(prepared, seed=7300 + spec.train_end_day)
    weighted_prediction, weighted_trees = fit_cat_source(
        prepared,
        prepared["views"]["identity"],
        (3407,),
        f"{spec.name}_cat_weighted",
        None if tuned else fixed_iterations["cat_weighted"],
        sample_weight=weights,
    )
    lgb_giba_prediction, lgb_giba_trees = fit_lgb_source(
        prepared,
        prepared["views"]["identity"],
        "lgb_giba",
        9203,
        f"{spec.name}_lgb_giba",
        None if tuned else fixed_iterations["lgb_giba"],
    )
    lgb_cold_prediction, lgb_cold_trees = fit_lgb_source(
        prepared,
        prepared["views"]["cold"],
        "lgb_cold",
        13303,
        f"{spec.name}_lgb_cold",
        None if tuned else fixed_iterations["lgb_cold"],
    )
    xgb_prediction, xgb_trees = fit_xgb_source(
        prepared,
        prepared["views"]["identity"],
        2026,
        f"{spec.name}_xgb_giba",
        None if tuned else fixed_iterations["xgb_giba"],
    )

    frame = pd.DataFrame(
        {
            "row_index": future_index,
            "TransactionID": raw.iloc[future_index]["TransactionID"].to_numpy(),
            TARGET: prepared["future_y"].to_numpy(),
            "fold": spec.name,
            "stage": spec.stage,
            "horizon": spec.horizon,
            "segment": segments,
            "strict_uid": prepared["metadata_future"]["strict_uid"]
            .astype("string")
            .fillna("<INVALID>"),
            "strict_valid": prepared["metadata_future"]["strict_valid"].to_numpy(),
            "cat_identity": identity_prediction,
            "cat_history": history_prediction,
            "cat_weighted": weighted_prediction,
            "lgb_giba": lgb_giba_prediction,
            "lgb_cold": lgb_cold_prediction,
            "xgb_giba": xgb_prediction,
        }
    )
    PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_csv(cache, index=False)
    source_auc = {source: auc(frame[TARGET], frame[source]) for source in SOURCE_NAMES}
    report = {
        "cached": False,
        "spec": asdict(spec),
        "history_rows": len(history_index),
        "future_rows": len(future_index),
        "physically_skipped_rows": skipped,
        "segments": {segment: int(np.sum(segments == segment)) for segment in SEGMENTS},
        "source_auc": source_auc,
        "best_iterations": {
            "cat_identity": identity_trees,
            "cat_history": history_trees,
            "cat_weighted": weighted_trees,
            "lgb_giba": [lgb_giba_trees],
            "lgb_cold": [lgb_cold_trees],
            "xgb_giba": [xgb_trees],
        },
        "domain": domain_report,
        "feature_stats": prepared["stats"],
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    print(json.dumps({"source_auc": source_auc, "minutes": report["elapsed_minutes"]}, indent=2), flush=True)
    del prepared, combined_metadata, weights
    gc.collect()
    return frame, report


def derive_fixed_iterations(reports: list[dict]) -> dict[str, int]:
    limits = {
        "cat_identity": CAT_PARAMS["iterations"],
        "cat_history": CAT_PARAMS["iterations"],
        "cat_weighted": CAT_PARAMS["iterations"],
        "lgb_giba": LGB_GIBA_PARAMS["n_estimators"],
        "lgb_cold": LGB_COLD_PARAMS["n_estimators"],
        "xgb_giba": XGB_PARAMS["n_estimators"],
    }
    result = {}
    for source in SOURCE_NAMES:
        scaled_values = []
        for report in reports:
            history_rows = max(int(report.get("history_rows", 0)), 1)
            scale = np.sqrt(EXPECTED_TRAIN_ROWS / history_rows)
            scaled_values.extend(
                value * scale
                for value in report.get("best_iterations", {}).get(source, [])
            )
        if not scaled_values:
            result[source] = int(limits[source])
            continue
        # A mild upper quantile compensates for the larger final training set,
        # while several horizons keep one anomalously long fold from dominating.
        estimate = int(np.ceil(np.quantile(scaled_values, 0.65)))
        result[source] = int(np.clip(estimate, 25, limits[source]))
    return result


def derive_spec_iterations(
    reports: list[dict],
    spec: PairSpec,
    raw: pd.DataFrame,
) -> dict[str, int]:
    """Scale development tree counts to a particular untouched lock history."""
    limits = {
        "cat_identity": CAT_PARAMS["iterations"],
        "cat_history": CAT_PARAMS["iterations"],
        "cat_weighted": CAT_PARAMS["iterations"],
        "lgb_giba": LGB_GIBA_PARAMS["n_estimators"],
        "lgb_cold": LGB_COLD_PARAMS["n_estimators"],
        "xgb_giba": XGB_PARAMS["n_estimators"],
    }
    source_horizon = 60 if spec.horizon == 75 else spec.horizon
    selected = [
        report
        for report in reports
        if int(report["spec"]["horizon"]) == source_horizon
    ]
    target_rows = int(
        np.sum(
            raw["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS
            < spec.train_end_day
        )
    )
    output = {}
    for source in SOURCE_NAMES:
        scaled = []
        for report in selected:
            source_rows = max(int(report["history_rows"]), 1)
            factor = np.sqrt(target_rows / source_rows)
            scaled.extend(
                value * factor
                for value in report["best_iterations"].get(source, [])
            )
        if not scaled:
            output[source] = limits[source]
        else:
            output[source] = int(
                np.clip(np.ceil(np.quantile(scaled, 0.55)), 25, limits[source])
            )
    return output


def add_ranks(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for source in SOURCE_NAMES:
        result[f"{source}_rank"] = rank_prediction(result[source])
    return result


def simplex_weights(size: int, denominator: int = 10):
    for values in product(range(denominator + 1), repeat=size):
        if sum(values) == denominator:
            yield np.asarray(values, dtype="float64") / denominator


def score_weights(
    frames: list[pd.DataFrame],
    segment: str,
    weights: np.ndarray,
) -> tuple[float, list[float], list[float]]:
    columns = [f"{source}_rank" for source in SOURCE_NAMES]
    scores = []
    gains = []
    anchor = np.zeros(len(SOURCE_NAMES), dtype="float64")
    anchor[0] = 1.0
    for frame in frames:
        mask = frame["segment"].eq(segment).to_numpy()
        if np.unique(frame.loc[mask, TARGET]).size < 2:
            continue
        matrix = frame.loc[mask, columns].to_numpy()
        y = frame.loc[mask, TARGET].to_numpy()
        score = auc(y, matrix @ weights)
        base = auc(y, matrix @ anchor)
        scores.append(score)
        gains.append(score - base)
    return float(np.mean(scores)), scores, gains


def learn_weights(dev_frames: list[pd.DataFrame]) -> tuple[dict, dict]:
    ranked = [add_ranks(frame) for frame in dev_frames]
    global_weights = {}
    report = {"global": {}, "local": {}}
    anchor = np.zeros(len(SOURCE_NAMES), dtype="float64")
    anchor[0] = 1.0

    for segment in SEGMENTS:
        candidates = []
        for weights in simplex_weights(len(SOURCE_NAMES), denominator=5):
            mean_score, scores, gains = score_weights(ranked, segment, weights)
            positive = int(np.sum(np.asarray(gains) > 0))
            candidates.append((mean_score, positive, float(np.mean(gains)), weights, scores, gains))
        candidates.sort(key=lambda row: (row[1], row[2], row[0]), reverse=True)
        best = candidates[0]
        required = max(1, int(np.ceil(len(best[5]) * 0.60)))
        weights = best[3] if best[1] >= required and best[2] > 0 else anchor.copy()
        global_weights[segment] = weights
        report["global"][segment] = {
            "weights": dict(zip(SOURCE_NAMES, weights.tolist())),
            "mean_auc": best[0],
            "positive_folds": best[1],
            "mean_gain": best[2],
            "fold_auc": best[4],
            "fold_gain": best[5],
        }

    horizon_weights = {}
    for horizon in (30, 45, 60):
        local_frames = [
            frame for frame in ranked if int(frame["horizon"].iloc[0]) == horizon
        ]
        horizon_weights[str(horizon)] = {}
        report["local"][str(horizon)] = {}
        for segment in SEGMENTS:
            best = None
            for weights in simplex_weights(len(SOURCE_NAMES), denominator=5):
                values = score_weights(local_frames, segment, weights)
                row = (values[0], float(np.mean(values[2])), weights, values)
                if best is None or row[:2] > best[:2]:
                    best = row
            assert best is not None
            alpha = len(local_frames) / (len(local_frames) + 3.0)
            shrunk = alpha * best[2] + (1.0 - alpha) * global_weights[segment]
            shrunk /= shrunk.sum()
            horizon_weights[str(horizon)][segment] = dict(
                zip(SOURCE_NAMES, shrunk.tolist())
            )
            report["local"][str(horizon)][segment] = {
                "raw_weights": dict(zip(SOURCE_NAMES, best[2].tolist())),
                "weights": horizon_weights[str(horizon)][segment],
                "alpha": alpha,
                "mean_auc": best[0],
                "mean_gain": best[1],
                "fold_auc": best[3][1],
                "fold_gain": best[3][2],
            }
    horizon_weights["75"] = {
        segment: dict(horizon_weights["60"][segment]) for segment in SEGMENTS
    }
    report["local"]["75"] = {"source": "h60"}
    return horizon_weights, report


def blend_frame(frame: pd.DataFrame, horizon_weights: dict) -> np.ndarray:
    ranked = add_ranks(frame)
    prediction = np.zeros(len(frame), dtype="float64")
    columns = [f"{source}_rank" for source in SOURCE_NAMES]
    weights = horizon_weights[str(int(frame["horizon"].iloc[0]))]
    for segment in SEGMENTS:
        mask = ranked["segment"].eq(segment).to_numpy()
        vector = np.asarray([weights[segment][source] for source in SOURCE_NAMES])
        prediction[mask] = ranked.loc[mask, columns].to_numpy() @ vector
    return prediction


def aggregate_group(
    prediction: np.ndarray,
    group: pd.Series,
    valid: np.ndarray,
    method: str,
) -> np.ndarray:
    result = prediction.copy()
    values = pd.DataFrame(
        {"group": group.loc[valid].to_numpy(), "prediction": prediction[valid]}
    )
    grouped = values.groupby("group", sort=False)["prediction"]
    if method == "mean":
        aggregate = grouped.mean()
    elif method == "q75":
        aggregate = grouped.quantile(0.75)
    elif method == "max":
        aggregate = grouped.max()
    else:
        raise ValueError(method)
    result[valid] = values["group"].map(aggregate).to_numpy()
    return result


def choose_uid_recipe(dev_frames: list[pd.DataFrame], horizon_weights: dict) -> tuple[dict, list[dict]]:
    rows = []
    candidates = [("none", 0.0)] + [
        (method, weight)
        for method in ("mean", "q75", "max")
        for weight in (0.05, 0.10, 0.20, 0.30)
    ]
    for method, weight in candidates:
        gains = []
        scores = []
        for frame in dev_frames:
            base = blend_frame(frame, horizon_weights)
            if method == "none":
                prediction = base
            else:
                valid = frame["strict_valid"].to_numpy(dtype=bool)
                grouped = aggregate_group(base, frame["strict_uid"], valid, method)
                prediction = (1.0 - weight) * base + weight * grouped
            scores.append(auc(frame[TARGET], prediction))
            gains.append(scores[-1] - auc(frame[TARGET], base))
        rows.append(
            {
                "method": method,
                "weight": weight,
                "mean_auc": float(np.mean(scores)),
                "mean_gain": float(np.mean(gains)),
                "min_gain": float(np.min(gains)),
                "positive_folds": int(np.sum(np.asarray(gains) > 0)),
                "fold_auc": scores,
                "fold_gain": gains,
            }
        )
    rows.sort(
        key=lambda row: (row["positive_folds"], row["min_gain"], row["mean_gain"]),
        reverse=True,
    )
    best = rows[0]
    if best["mean_gain"] <= 0 or best["positive_folds"] < 4:
        best = next(row for row in rows if row["method"] == "none")
    return {"method": best["method"], "weight": best["weight"]}, rows


def apply_uid_recipe(frame: pd.DataFrame, prediction: np.ndarray, recipe: dict) -> np.ndarray:
    if recipe["method"] == "none" or recipe["weight"] == 0:
        return prediction
    valid = frame["strict_valid"].to_numpy(dtype=bool)
    grouped = aggregate_group(
        prediction,
        frame["strict_uid"],
        valid,
        recipe["method"],
    )
    return (1.0 - recipe["weight"]) * prediction + recipe["weight"] * grouped


def evaluate_frame(frame: pd.DataFrame, prediction: np.ndarray) -> dict:
    result = {
        "overall": auc(frame[TARGET], prediction),
        "anchor": auc(frame[TARGET], frame["cat_identity"]),
        "gain": auc(frame[TARGET], prediction) - auc(frame[TARGET], frame["cat_identity"]),
        "segments": {},
    }
    for segment in SEGMENTS:
        mask = frame["segment"].eq(segment).to_numpy()
        result["segments"][segment] = {
            "rows": int(mask.sum()),
            "auc": auc(frame.loc[mask, TARGET], prediction[mask]),
            "anchor_auc": auc(frame.loc[mask, TARGET], frame.loc[mask, "cat_identity"]),
        }
    return result


def apply_lock_fallbacks(
    lock_frames: list[pd.DataFrame],
    horizon_weights: dict,
) -> tuple[dict, dict]:
    """Use train-only lock folds for conservative segment fallback decisions."""
    adjusted = copy.deepcopy(horizon_weights)
    anchor = {source: float(source == "cat_identity") for source in SOURCE_NAMES}
    segment_gains: dict[str, dict[str, float]] = {segment: {} for segment in SEGMENTS}
    for frame in lock_frames:
        horizon = str(int(frame["horizon"].iloc[0]))
        candidate = blend_frame(frame, horizon_weights)
        for segment in SEGMENTS:
            mask = frame["segment"].eq(segment).to_numpy()
            segment_gains[segment][horizon] = (
                auc(frame.loc[mask, TARGET], candidate[mask])
                - auc(frame.loc[mask, TARGET], frame.loc[mask, "cat_identity"])
            )

    fallbacks = []
    for segment in SEGMENTS:
        gains = segment_gains[segment]
        positive = sum(value > 0 for value in gains.values())
        if positive < 3 or float(np.mean(list(gains.values()))) <= 0:
            for horizon in HORIZONS:
                adjusted[str(horizon)][segment] = dict(anchor)
            fallbacks.append(
                {
                    "segment": segment,
                    "horizons": list(HORIZONS),
                    "reason": "fewer than three positive lock horizons",
                }
            )
            continue
        # Horizon 75 has no development fold. A non-positive lock result there
        # therefore falls back instead of extrapolating the h60 recipe blindly.
        if gains.get("75", 0.0) <= 0:
            adjusted["75"][segment] = dict(anchor)
            fallbacks.append(
                {
                    "segment": segment,
                    "horizons": [75],
                    "reason": "no h75 development fold and non-positive h75 lock gain",
                }
            )
    return adjusted, {"segment_lock_gain": segment_gains, "fallbacks": fallbacks}


def run_backtest(force: bool, only_spec: str | None) -> dict:
    WORK_DIR.mkdir(exist_ok=True)
    MODEL_DIR.mkdir(exist_ok=True)
    PREDICTION_DIR.mkdir(exist_ok=True)
    raw = read_official_train()
    dev_frames = []
    dev_reports = []
    for spec in DEV_SPECS:
        if only_spec and spec.name != only_spec:
            continue
        frame, report = train_pair(raw, spec, None, force)
        dev_frames.append(frame)
        dev_reports.append(report)
    if only_spec:
        payload = {"spec": only_spec, "reports": dev_reports}
        save_json(WORK_DIR / f"{only_spec}_report.json", payload)
        return payload
    if len(dev_frames) != len(DEV_SPECS):
        raise RuntimeError("Development fold set is incomplete")

    fixed_iterations = derive_fixed_iterations(dev_reports)
    horizon_weights, weight_report = learn_weights(dev_frames)
    uid_recipe, uid_search = choose_uid_recipe(dev_frames, horizon_weights)
    recipe = {
        "data_policy": "official train/test only; embargo rows excluded from feature building",
        "sources": list(SOURCE_NAMES),
        "segments": list(SEGMENTS),
        "horizons": list(HORIZONS),
        "fixed_iterations": fixed_iterations,
        "horizon_weights": horizon_weights,
        "uid_postprocess": uid_recipe,
        "seeds": list(SEEDS),
    }
    save_json(RECIPE_PATH, recipe)

    lock_frames = []
    lock_reports = []
    lock_iterations = {}
    for spec in LOCK_SPECS:
        spec_iterations = derive_spec_iterations(dev_reports, spec, raw)
        lock_iterations[spec.name] = spec_iterations
        frame, report = train_pair(raw, spec, spec_iterations, force)
        lock_frames.append(frame)
        lock_reports.append(report)

    adjusted_weights, confirmation = apply_lock_fallbacks(
        lock_frames, horizon_weights
    )
    recipe["horizon_weights_before_lock_fallback"] = horizon_weights
    recipe["horizon_weights"] = adjusted_weights
    recipe["lock_fallback_policy"] = confirmation
    save_json(RECIPE_PATH, recipe)

    dev_evaluation = {}
    for frame in dev_frames:
        prediction = apply_uid_recipe(
            frame, blend_frame(frame, adjusted_weights), uid_recipe
        )
        dev_evaluation[str(frame["fold"].iloc[0])] = evaluate_frame(frame, prediction)
    lock_evaluation = {}
    for frame in lock_frames:
        prediction = apply_uid_recipe(
            frame, blend_frame(frame, adjusted_weights), uid_recipe
        )
        lock_evaluation[str(frame["fold"].iloc[0])] = evaluate_frame(frame, prediction)

    mean_lock_gain = float(
        np.mean([value["gain"] for value in lock_evaluation.values()])
    )
    report = {
        "data_policy": recipe["data_policy"],
        "development_specs": [asdict(spec) for spec in DEV_SPECS],
        "lock_specs": [asdict(spec) for spec in LOCK_SPECS],
        "development_fit": dev_reports,
        "fixed_iterations": fixed_iterations,
        "lock_iterations": lock_iterations,
        "weight_search": weight_report,
        "uid_search": uid_search,
        "lock_fallback_confirmation": confirmation,
        "recipe": recipe,
        "development_evaluation": dev_evaluation,
        "lock_fit": lock_reports,
        "lock_evaluation": lock_evaluation,
        "mean_lock_gain_over_multiseed_cat": mean_lock_gain,
        "accepted": bool(mean_lock_gain > 0),
    }
    save_json(REPORT_PATH, report)
    print("\nLOCK RESULTS", flush=True)
    print(json.dumps(lock_evaluation, indent=2), flush=True)
    print(f"Mean lock gain: {mean_lock_gain:+.9f}", flush=True)
    return report


def interpolated_source_weights(
    horizon: np.ndarray,
    segments: np.ndarray,
    recipe: dict,
) -> np.ndarray:
    grid = np.asarray(HORIZONS, dtype="float64")
    output = np.zeros((len(horizon), len(SOURCE_NAMES)), dtype="float64")
    for segment in SEGMENTS:
        mask = segments == segment
        if not mask.any():
            continue
        for source_number, source in enumerate(SOURCE_NAMES):
            values = [
                recipe["horizon_weights"][str(value)][segment][source]
                for value in HORIZONS
            ]
            output[mask, source_number] = np.interp(
                horizon[mask], grid, values
            )
    output /= output.sum(axis=1, keepdims=True)
    return output


def train_final_source_models(prepared: dict, recipe: dict, force: bool) -> pd.DataFrame:
    fixed = recipe["fixed_iterations"]
    output_cache = WORK_DIR / "raw_test_sources.csv"
    if output_cache.exists() and not force:
        frame = pd.read_csv(output_cache)
        if set(SOURCE_NAMES).issubset(frame.columns) and len(frame) == len(prepared["future"]):
            return frame

    prepared["future_y"] = None
    active_sources = {
        source
        for horizon in recipe["horizon_weights"].values()
        for segment in horizon.values()
        for source, weight in segment.items()
        if float(weight) > 0
    }
    weights, domain_report = domain_weights(prepared, seed=9307)
    identity, _ = fit_cat_source(
        prepared,
        prepared["views"]["identity"],
        SEEDS,
        "final_cat_identity",
        fixed["cat_identity"],
    )
    if "cat_history" in active_sources:
        history, _ = fit_cat_source(
            prepared,
            prepared["views"]["history"],
            SEEDS[:2],
            "final_cat_history",
            fixed["cat_history"],
        )
    else:
        print("Skipping inactive final source: cat_history", flush=True)
        history = identity.copy()
    weighted, _ = fit_cat_source(
        prepared,
        prepared["views"]["identity"],
        (3407,),
        "final_cat_weighted",
        fixed["cat_weighted"],
        sample_weight=weights,
    )
    lgb_giba_predictions = []
    lgb_cold_predictions = []
    for seed in (9203, 13303):
        value, _ = fit_lgb_source(
            prepared,
            prepared["views"]["identity"],
            "lgb_giba",
            seed,
            f"final_lgb_giba_s{seed}",
            fixed["lgb_giba"],
        )
        lgb_giba_predictions.append(value)
        value, _ = fit_lgb_source(
            prepared,
            prepared["views"]["cold"],
            "lgb_cold",
            seed,
            f"final_lgb_cold_s{seed}",
            fixed["lgb_cold"],
        )
        lgb_cold_predictions.append(value)
    xgb_prediction, _ = fit_xgb_source(
        prepared,
        prepared["views"]["identity"],
        2026,
        "final_xgb_giba",
        fixed["xgb_giba"],
    )
    frame = pd.DataFrame(
        {
            "TransactionID": prepared["future"]["TransactionID"].to_numpy(),
            "cat_identity": identity,
            "cat_history": history,
            "cat_weighted": weighted,
            "lgb_giba": np.mean(lgb_giba_predictions, axis=0),
            "lgb_cold": np.mean(lgb_cold_predictions, axis=0),
            "xgb_giba": xgb_prediction,
        }
    )
    frame.to_csv(output_cache, index=False)
    save_json(
        WORK_DIR / "final_domain_report.json",
        {"active_sources": sorted(active_sources), **domain_report},
    )
    return frame


def run_final(force: bool) -> dict:
    if not RECIPE_PATH.exists():
        raise FileNotFoundError("Run backtest before final training")
    recipe = json.loads(RECIPE_PATH.read_text(encoding="utf-8"))
    train_raw = read_official_train()
    test_raw = read_official_test()
    y = train_raw[TARGET].astype("int8").reset_index(drop=True)
    prepared = prepare_pair(train_raw, test_raw)
    if not np.array_equal(
        prepared["future"]["TransactionID"].to_numpy(),
        test_raw["TransactionID"].to_numpy(),
    ):
        raise ValueError("Test rows moved during feature preparation")
    sources = train_final_source_models(prepared, recipe, force)

    combined_metadata = pd.concat(
        [prepared["metadata_history"], prepared["metadata_future"]],
        ignore_index=True,
    )
    segments = assign_segments(
        combined_metadata,
        np.arange(len(train_raw), dtype="int64"),
        np.arange(len(train_raw), len(train_raw) + len(test_raw), dtype="int64"),
    )
    train_end = float(train_raw["TransactionDT"].max() / DAY_SECONDS)
    horizon = test_raw["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS - train_end
    rank_matrix = np.column_stack(
        [rank_prediction(sources[source]) for source in SOURCE_NAMES]
    )
    row_weights = interpolated_source_weights(horizon, segments, recipe)
    prediction = np.sum(rank_matrix * row_weights, axis=1)
    post_frame = pd.DataFrame(
        {
            "strict_uid": prepared["metadata_future"]["strict_uid"]
            .astype("string")
            .fillna("<INVALID>"),
            "strict_valid": prepared["metadata_future"]["strict_valid"].to_numpy(),
        }
    )
    prediction = apply_uid_recipe(post_frame, prediction, recipe["uid_postprocess"])
    prediction = rank_prediction(prediction)

    source_output = sources.copy()
    source_output["segment"] = segments
    source_output["forecast_horizon"] = horizon
    source_output["prediction"] = prediction
    source_output.to_csv(SOURCE_PATH, index=False)

    sample = pd.read_csv(ROOT / "sample_submission.csv").drop(
        columns="Unnamed: 0", errors="ignore"
    )
    if not np.array_equal(
        sample["TransactionID"].to_numpy(), test_raw["TransactionID"].to_numpy()
    ):
        raise ValueError("Sample submission IDs differ from official test")
    submission = sample[["TransactionID"]].copy()
    submission[TARGET] = prediction
    if not np.isfinite(submission[TARGET]).all():
        raise ValueError("Non-finite final predictions")
    submission.to_csv(SUBMISSION_PATH, index=False)
    report = {
        "data_policy": recipe["data_policy"],
        "train_rows": len(train_raw),
        "test_rows": len(test_raw),
        "sources": list(SOURCE_NAMES),
        "segments": {segment: int(np.sum(segments == segment)) for segment in SEGMENTS},
        "forecast_horizon": {"min": float(horizon.min()), "max": float(horizon.max())},
        "submission": SUBMISSION_PATH.name,
        "prediction": {
            "min": float(prediction.min()),
            "max": float(prediction.max()),
            "mean": float(prediction.mean()),
        },
    }
    save_json(WORK_DIR / "final_report.json", report)
    print(json.dumps(report, indent=2), flush=True)
    del prepared, combined_metadata, y
    gc.collect()
    return report


def self_test() -> None:
    values = list(simplex_weights(3, denominator=5))
    assert len(values) == 21
    assert all(np.isclose(value.sum(), 1.0) for value in values)
    horizon = np.asarray([30.0, 37.5, 75.0])
    segments = np.asarray(["strict", "partial", "cold"])
    recipe = {
        "horizon_weights": {
            str(h): {
                segment: {
                    source: float(source == "cat_identity")
                    for source in SOURCE_NAMES
                }
                for segment in SEGMENTS
            }
            for h in HORIZONS
        }
    }
    weights = interpolated_source_weights(horizon, segments, recipe)
    assert np.allclose(weights[:, 0], 1.0)
    assert np.allclose(weights.sum(axis=1), 1.0)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--mode", choices=("backtest", "final", "all", "inspect"), default="all"
    )
    parser.add_argument("--spec", choices=[spec.name for spec in DEV_SPECS])
    parser.add_argument("--force", action="store_true")
    parser.add_argument("--self-test", action="store_true")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.self_test:
        self_test()
        print("Self-test: OK")
        return
    if args.mode == "inspect":
        train = read_official_train()
        test = read_official_test()
        print(
            json.dumps(
                {
                    "train_rows": len(train),
                    "test_rows": len(test),
                    "train_day_max": float(train["TransactionDT"].max() / DAY_SECONDS),
                    "test_day_min": float(test["TransactionDT"].min() / DAY_SECONDS),
                    "test_day_max": float(test["TransactionDT"].max() / DAY_SECONDS),
                    "development": [asdict(spec) for spec in DEV_SPECS],
                    "lock": [asdict(spec) for spec in LOCK_SPECS],
                },
                indent=2,
            )
        )
        return

    started = time.time()
    if args.mode in {"backtest", "all"}:
        run_backtest(args.force, args.spec)
    if args.mode in {"final", "all"} and not args.spec:
        run_final(args.force)
    print(f"Total elapsed: {(time.time() - started) / 60.0:.2f} min", flush=True)


if __name__ == "__main__":
    main()


Overwriting clean_v2_pipeline.py


In [6]:
%%writefile finalize_honest_featureview_meta.py
"""Finalize the train-only selected feature-view client stack."""

from __future__ import annotations

import gc
import hashlib
import json
from pathlib import Path
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

import refine_honest_client_segments as segments
import honest_featureview_sources as next_features
import search_honest_featureview_meta as search
import train_honest_client_meta as client
import train_honest_heavy_temporal_client as heavy


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_featureview_meta"
SOURCE_CACHE = WORK_DIR / "test_featureview_sources.csv"
OUTPUT_PATH = ROOT / "submission_honest_featureview_client.csv"
FINAL_REPORT_PATH = WORK_DIR / "final_report.json"

MODEL_PATHS = {
    "fv_vblock_dynamics": ROOT
    / "advanced_feature_ablation_models/vblock_dynamics_final.txt",
    "fv_vblock_cd_dynamics": ROOT
    / "advanced_feature_ablation_models/vblock_cd_dynamics_final.txt",
    "fv_vblock_structured": ROOT
    / "next_feature_ablation_models_v1/vblock_structured_final.txt",
}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def predict_clean_sources() -> pd.DataFrame:
    sample = pd.read_csv(ROOT / "sample_submission.csv", usecols=["TransactionID"])
    if SOURCE_CACHE.exists():
        cached = pd.read_csv(SOURCE_CACHE)
        required = {"TransactionID", *MODEL_PATHS}
        if required.issubset(cached.columns) and np.array_equal(
            cached["TransactionID"].to_numpy(), sample["TransactionID"].to_numpy()
        ):
            print("Loading cached clean feature-view predictions", flush=True)
            return cached

    print("Preparing official train/test feature views...", flush=True)
    prepared, _, _ = next_features.prepare_next_views()
    if client.TARGET in prepared["inference"]:
        raise RuntimeError("Target reached feature-view inference frame")
    if not np.array_equal(
        prepared["inference_ids"].to_numpy(), sample["TransactionID"].to_numpy()
    ):
        raise RuntimeError("Prepared test rows differ from sample_submission")

    output = sample.copy()
    for source, model_path in MODEL_PATHS.items():
        if not model_path.exists():
            raise FileNotFoundError(model_path)
        model = lgb.Booster(model_file=str(model_path))
        features = model.feature_name()
        missing = sorted(set(features).difference(prepared["inference"].columns))
        if missing:
            raise RuntimeError(f"Missing {source} features: {missing[:10]}")
        print(f"Predicting {source}: {len(features)} features", flush=True)
        output[source] = model.predict(prepared["inference"][features])
        del model
        gc.collect()
    output.to_csv(SOURCE_CACHE, index=False)
    del prepared
    gc.collect()
    return output


def main() -> None:
    started = time.time()
    recipe = json.loads(
        (WORK_DIR / "search_report.json").read_text(encoding="utf-8")
    )
    if not recipe["accepted"]:
        raise RuntimeError("Feature-view candidate did not pass temporal lock")
    selected = recipe["selected_model"]
    selected_sources, _ = search.VIEWS[selected["view"]]
    needed_feature_sources = [
        source for source in selected_sources if source.startswith("fv_")
    ]
    if set(needed_feature_sources).difference(MODEL_PATHS):
        raise RuntimeError("Selected source has no clean final model")

    clean_sources = predict_clean_sources()
    oof = search.build_oof()
    test_predictions = client.build_test_sources()
    for source in needed_feature_sources:
        test_predictions[source] = clean_sources[source].to_numpy(dtype="float64")
    membership_oof = pd.read_csv(ROOT / "honest_client_meta/oof_client_features.csv")
    membership_test = pd.read_csv(ROOT / "honest_client_meta/test_client_features.csv")

    original_views = heavy.VIEWS
    heavy.VIEWS = search.VIEWS
    try:
        train_features = heavy.make_view(
            oof, membership_oof, selected["view"], fold=oof["fold"]
        )
        test_features = heavy.make_view(
            test_predictions, membership_test, selected["view"], fold=None
        )
    finally:
        heavy.VIEWS = original_views
    final_iterations = max(
        30, int(np.ceil(float(recipe["lock_iterations"]) * 1.15))
    )
    model = lgb.LGBMClassifier(
        **{
            **heavy.LGB_PARAMS,
            **heavy.LGB_CONFIGS[int(selected["config_index"])],
            "n_estimators": final_iterations,
        }
    )
    model.fit(
        train_features,
        oof[client.TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    model_path = WORK_DIR / "final_lgb.txt"
    model.booster_.save_model(model_path)
    feature_prediction = model.predict_proba(test_features)[:, 1]

    raw_train = client.prepare_raw(
        pd.read_csv(ROOT / "train_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    raw_test = client.prepare_raw(
        pd.read_csv(ROOT / "test_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    _, _, _, test_groups = client.build_client_features(
        oof, raw_train, raw_test
    )
    current = pd.read_csv(ROOT / "submission_honest_user_means.csv")
    if not np.array_equal(
        current["TransactionID"].to_numpy(), clean_sources["TransactionID"].to_numpy()
    ):
        raise RuntimeError("Current baseline and feature sources are misaligned")
    prediction = segments.segmented_blend(
        current[client.TARGET].to_numpy(dtype="float64"),
        feature_prediction,
        segments.segment_labels(membership_test),
        recipe["selected_segment_weights"]["weights"],
    )
    prediction = client.apply_postprocess(
        prediction, test_groups, recipe["postprocess_locked_from_dev"]
    )
    output = current[["TransactionID"]].copy()
    output[client.TARGET] = prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test only",
        "selection_report": str(WORK_DIR / "search_report.json"),
        "selected_sources": list(selected_sources),
        "clean_feature_sources": needed_feature_sources,
        "source_models": {
            source: {"path": str(path), "sha256": file_sha256(path)}
            for source, path in MODEL_PATHS.items()
            if source in needed_feature_sources
        },
        "meta_model": {
            "path": str(model_path),
            "iterations": final_iterations,
            "features": int(train_features.shape[1]),
            "seed": heavy.LGB_PARAMS["random_state"],
        },
        "output": OUTPUT_PATH.name,
        "output_sha256": file_sha256(OUTPUT_PATH),
        "rows": int(len(output)),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    FINAL_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting finalize_honest_featureview_meta.py


In [7]:
%%writefile finalize_honest_xgb_magic_blend.py
"""Add the published XGB-magic source above the locked clean-v2 blend."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import refine_honest_client_segments as segments
import search_honest_cleanv2_blend as clean_blend
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client
import train_honest_xgb_magic as magic


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_xgb_magic"
REPORT_PATH = WORK_DIR / "blend_report.json"
OUTPUT_PATH = ROOT / "submission_honest_xgb_magic_blend.csv"


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def reconstruct_clean_oof(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    reference: dict[int, np.ndarray],
    fold_groups: dict,
    reference_report: dict,
) -> tuple[dict[int, np.ndarray], dict]:
    clean_recipe = json.loads(
        (ROOT / "clean_v2/recipe.json").read_text(encoding="utf-8")
    )
    blend_report = json.loads(
        (ROOT / "honest_cleanv2_blend/report.json").read_text(encoding="utf-8")
    )
    selected = blend_report["selected"]
    fullrow_report = json.loads(
        (ROOT / "honest_fullrow_lgb/report.json").read_text(encoding="utf-8")
    )
    postprocess = reference_report["recipe"]["postprocess_locked_from_dev"]
    specs = {
        client.META_DEV_FOLD: (
            ROOT / "clean_v2/predictions/dev_h30_c.csv",
            ROOT / "honest_fullrow_lgb/dev_raw.npy",
        ),
        client.META_LOCK_FOLD: (
            ROOT / "clean_v2/predictions/lock_h30.csv",
            ROOT / "honest_fullrow_lgb/lock_raw.npy",
        ),
    }
    predictions = {}
    metrics = {}
    for fold, (clean_path, lgb_path) in specs.items():
        fold_mask = oof["fold"].eq(fold).to_numpy()
        frame, clean_prediction = clean_blend.load_clean_prediction(
            clean_path, clean_recipe
        )
        expected_rows = oof.loc[fold_mask, "row_index"].to_numpy(dtype="int64")
        if not np.array_equal(frame["row_index"].to_numpy(), expected_rows):
            raise RuntimeError(f"clean_v2 rows differ on fold {fold}")
        lgb_prediction = fullrow.transform_variant(
            np.load(lgb_path),
            fullrow_report["selected_blend"]["variant"],
            fold_groups[fold],
            postprocess,
        )
        values = clean_blend.transform_sources(
            {
                "featureview": reference[fold],
                "clean_v2": clean_prediction,
                "fullrow_lgb": lgb_prediction,
            },
            selected["mode"],
        )
        prediction = clean_blend.apply_segment_recipe(
            values,
            segments.segment_labels(
                membership.loc[fold_mask].reset_index(drop=True)
            ),
            selected["weights"],
        )
        y = oof.loc[fold_mask, client.TARGET].to_numpy(dtype="int8")
        predictions[fold] = prediction
        metrics[fold] = float(roc_auc_score(y, prediction))
    expected = float(blend_report["candidate_lock_auc"])
    if abs(metrics[client.META_LOCK_FOLD] - expected) > 1e-12:
        raise RuntimeError("The locked clean-v2 blend was not reproduced")
    return predictions, {
        "recipe": selected,
        "metrics": metrics,
        "fullrow_report": fullrow_report,
        "postprocess": postprocess,
    }


def main() -> None:
    started = time.time()
    manifest, arrays = magic.load_matrix()
    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    clean_oof, clean_report = reconstruct_clean_oof(
        oof, membership_oof, reference, fold_groups, reference_report
    )

    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    dev_magic = np.load(WORK_DIR / "dev_prediction.npy")
    lock_magic = np.load(WORK_DIR / "lock_prediction.npy")
    dev_variants = fullrow.prediction_variants(
        dev_magic,
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
    )
    selected, search_rows = fullrow.search_blend(
        np.asarray(arrays["target"][dev_index], dtype="int8"),
        clean_oof[client.META_DEV_FOLD],
        dev_variants,
        segments.segment_labels(
            membership_oof.loc[dev_mask].reset_index(drop=True)
        ),
        np.asarray(arrays["day"][dev_index]),
    )
    lock_variant = fullrow.transform_variant(
        lock_magic,
        selected["variant"],
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
    )
    lock_prediction = segments.segmented_blend(
        clean_oof[client.META_LOCK_FOLD],
        lock_variant,
        segments.segment_labels(
            membership_oof.loc[lock_mask].reset_index(drop=True)
        ),
        selected["weights"],
    )
    y_lock = np.asarray(arrays["target"][lock_index], dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    previous_lock = clean_report["metrics"][client.META_LOCK_FOLD]
    accepted = bool(selected["gain"] > 0.0 and lock_auc > previous_lock)

    group_models = []
    if accepted:
        source_report = json.loads(
            (WORK_DIR / "report.json").read_text(encoding="utf-8")
        )
        final_iterations = max(
            50, int(np.ceil(source_report["lock_iterations"] * 1.15))
        )
        test_magic, group_models = magic.train_group_models(
            arrays["train"],
            arrays["test"],
            arrays["target"],
            arrays["month"],
            fixed_iterations=final_iterations,
        )
        np.save(WORK_DIR / "test_prediction.npy", test_magic.astype("float32"))
        test_variant = fullrow.transform_variant(
            test_magic,
            selected["variant"],
            reference_report["test_groups"],
            clean_report["postprocess"],
        )
        baseline = pd.read_csv(ROOT / "submission_honest_cleanv2_blend.csv")
        if not np.array_equal(
            baseline["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("Magic test rows differ from the clean baseline")
        prediction = segments.segmented_blend(
            baseline[client.TARGET].to_numpy(dtype="float64"),
            test_variant,
            segments.segment_labels(membership_test),
            selected["weights"],
        )
        output = baseline[["TransactionID"]].copy()
        output[client.TARGET] = prediction
        output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test covariates; official train labels only",
        "feature_recipe": manifest["source"],
        "selection": "blend on days 75-90; days 90-105 one-time lock",
        "clean_baseline": clean_report["metrics"],
        "selected": selected,
        "search_top": search_rows[:30],
        "previous_lock_auc": previous_lock,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - previous_lock,
        "accepted": accepted,
        "final_group_iterations": (
            final_iterations if accepted else None
        ),
        "group_models": group_models,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting finalize_honest_xgb_magic_blend.py


In [8]:
%%writefile fraud_advanced_user_features.py
from __future__ import annotations

import numpy as np
import pandas as pd


GENERIC_EMAIL_DOMAINS = {
    "<MISSING>",
    "anonymous.com",
    "mail.com",
}
ROLLING_WINDOWS = (2, 3, 4, 5, 10, 20)
BEHAVIOR_COLUMNS = (
    "addr1",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceInfo",
    "ProductCD",
    "card2",
    "card5",
)
ADVANCED_UID_COLUMNS = (
    "uid_adv_component",
    "uid_adv_clean_email",
    "uid_adv_product",
)


def _tokens(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("<MISSING>")


def _integer_tokens(series: pd.Series) -> pd.Series:
    return series.round().astype("Int32").astype("string").fillna("<MISSING>")


def _join_tokens(*values: pd.Series) -> pd.Series:
    result = _tokens(values[0])
    for value in values[1:]:
        result = result.str.cat(_tokens(value), sep="|")
    return result


def _clean_email(series: pd.Series) -> pd.Series:
    result = _tokens(series).str.lower()
    return result.mask(result.isin(GENERIC_EMAIL_DOMAINS), "<MISSING>")


def _find(parent: np.ndarray, node: int) -> int:
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = int(parent[node])
    return node


def _union(
    parent: np.ndarray,
    sizes: np.ndarray,
    first: int,
    second: int,
    max_component_size: int,
) -> bool:
    first_root = _find(parent, first)
    second_root = _find(parent, second)
    if first_root == second_root:
        return False
    if int(sizes[first_root]) + int(sizes[second_root]) > max_component_size:
        return False
    if sizes[first_root] < sizes[second_root]:
        first_root, second_root = second_root, first_root
    parent[second_root] = first_root
    sizes[first_root] += sizes[second_root]
    return True


def _union_hash_groups(
    frame: pd.DataFrame,
    columns: list[str],
    valid: np.ndarray,
    parent: np.ndarray,
    sizes: np.ndarray,
    max_group_size: int,
    max_component_size: int,
) -> dict:
    rows = np.flatnonzero(valid).astype("int32", copy=False)
    if len(rows) == 0:
        return {
            "valid_rows": 0,
            "linked_groups": 0,
            "oversized_groups": 0,
            "union_edges": 0,
        }
    hashes = pd.util.hash_pandas_object(
        frame.loc[valid, columns],
        index=False,
        categorize=True,
    ).to_numpy(dtype="uint64", copy=False)
    order = np.argsort(hashes, kind="stable")
    sorted_hashes = hashes[order]
    boundaries = np.flatnonzero(
        np.r_[True, sorted_hashes[1:] != sorted_hashes[:-1], True]
    )
    linked_groups = 0
    oversized_groups = 0
    edges = 0
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        group_size = int(end - start)
        if group_size < 2:
            continue
        if group_size > max_group_size:
            oversized_groups += 1
            continue
        members = rows[order[start:end]]
        anchor = int(members[0])
        linked_groups += 1
        for member in members[1:]:
            edges += int(
                _union(
                    parent,
                    sizes,
                    anchor,
                    int(member),
                    max_component_size,
                )
            )
    return {
        "valid_rows": int(len(rows)),
        "linked_groups": linked_groups,
        "oversized_groups": oversized_groups,
        "union_edges": edges,
    }


def build_advanced_components(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    max_component_size: int = 500,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, dict]:
    columns = [
        "TransactionDT",
        "card1",
        "card2",
        "card3",
        "card5",
        "addr1",
        "D1_origin_day",
        "D3",
        "C13",
        "P_emaildomain",
        "ProductCD",
    ]
    missing = sorted(set(columns).difference(train.columns))
    if missing:
        raise ValueError(f"Missing columns for advanced UID: {missing}")
    combined = pd.concat(
        [train[columns], inference[columns]],
        ignore_index=True,
        copy=False,
    )
    combined["_origin"] = combined["D1_origin_day"].round()
    combined["_clean_email"] = _clean_email(combined["P_emaildomain"])
    row_count = len(combined)
    parent = np.arange(row_count, dtype="int32")
    sizes = np.ones(row_count, dtype="int32")
    key_metrics = {}

    exact_keys = {
        "card_addr_origin": ["card1", "addr1", "_origin"],
        "card_origin_clean_email": [
            "card1",
            "_origin",
            "_clean_email",
        ],
        "card_full_origin": [
            "card1",
            "card2",
            "card3",
            "card5",
            "_origin",
        ],
    }
    max_group_sizes = {
        "card_addr_origin": 200,
        "card_origin_clean_email": 150,
        "card_full_origin": 150,
    }
    for name, key_columns in exact_keys.items():
        valid = combined[key_columns].notna().all(axis=1).to_numpy()
        if "_clean_email" in key_columns:
            valid &= combined["_clean_email"].ne("<MISSING>").to_numpy()
        key_metrics[name] = _union_hash_groups(
            combined,
            key_columns,
            valid,
            parent,
            sizes,
            max_group_sizes[name],
            max_component_size,
        )

    ordered = combined.reset_index(names="_row").sort_values(
        ["card1", "addr1", "TransactionDT", "_row"],
        kind="stable",
    )
    pair_group = ordered.groupby(
        ["card1", "addr1"],
        sort=False,
        observed=True,
        dropna=False,
    )
    previous_row = pair_group["_row"].shift(1)
    gap_days = pair_group["TransactionDT"].diff() / 86400.0
    d3_error = (gap_days - ordered["D3"]).abs()
    same_product = _tokens(ordered["ProductCD"]).eq(
        _tokens(pair_group["ProductCD"].shift(1))
    )
    current_email = ordered["_clean_email"]
    previous_email = pair_group["_clean_email"].shift(1)
    same_email = (
        current_email.eq(previous_email)
        & current_email.ne("<MISSING>")
    )
    c13_diff = ordered["C13"] - pair_group["C13"].shift(1)
    c13_consistent = c13_diff.between(-1, 10, inclusive="both")
    origin_diff = (
        ordered["_origin"] - pair_group["_origin"].shift(1)
    ).abs()
    fuzzy_valid = (
        previous_row.notna()
        & gap_days.between(0, 120, inclusive="both")
        & d3_error.le(1.5)
        & origin_diff.le(35)
        & (same_product | same_email | c13_consistent)
    )
    fuzzy_edges = 0
    fuzzy_rejected_by_cap = 0
    for current, previous in zip(
        ordered.loc[fuzzy_valid, "_row"].to_numpy(dtype="int32"),
        previous_row.loc[fuzzy_valid].to_numpy(dtype="int32"),
    ):
        if _union(
            parent,
            sizes,
            int(current),
            int(previous),
            max_component_size,
        ):
            fuzzy_edges += 1
        else:
            fuzzy_rejected_by_cap += 1

    roots = np.empty(row_count, dtype="int32")
    for row in range(row_count):
        roots[row] = _find(parent, row)
    components, _ = pd.factorize(roots, sort=False)
    components = components.astype("int32", copy=False)
    component_counts = np.bincount(components)

    origin_token = _integer_tokens(combined["_origin"])
    clean_uid = _join_tokens(
        combined["card1"],
        combined["addr1"],
        origin_token,
        combined["_clean_email"],
    )
    product_uid = _join_tokens(
        combined["card1"],
        combined["addr1"],
        origin_token,
        combined["ProductCD"],
    )
    uid_frame = pd.DataFrame(
        {
            "uid_adv_component": pd.Series(components).map(
                lambda value: f"u{value}"
            ).astype("string"),
            "uid_adv_clean_email": clean_uid,
            "uid_adv_product": product_uid,
        }
    )
    train_rows = len(train)
    stats = {
        "rows": row_count,
        "components": int(len(component_counts)),
        "singleton_components": int(np.sum(component_counts == 1)),
        "rows_in_multirow_components": int(
            component_counts[component_counts > 1].sum()
        ),
        "max_component_size": int(component_counts.max()),
        "exact_keys": key_metrics,
        "fuzzy_d3_candidates": int(fuzzy_valid.sum()),
        "fuzzy_d3_union_edges": fuzzy_edges,
        "fuzzy_d3_rejected_or_existing": fuzzy_rejected_by_cap,
    }
    return (
        components[:train_rows],
        components[train_rows:],
        uid_frame,
        stats,
    )


def _rolling_group_feature(
    sequence: pd.DataFrame,
    value_column: str,
    window: int,
    statistic: str,
) -> pd.Series:
    rolled = (
        sequence.groupby("_component", sort=False, observed=True)[value_column]
        .rolling(window=window, min_periods=1)
        .agg(statistic)
        .reset_index(level=0, drop=True)
    )
    return rolled.reindex(sequence.index).astype("float32")


def add_advanced_user_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    list[str],
    list[str],
    np.ndarray,
    np.ndarray,
    dict,
]:
    (
        train_components,
        inference_components,
        uid_frame,
        stats,
    ) = build_advanced_components(train, inference)
    source_columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "D3",
        "C13",
        *[column for column in BEHAVIOR_COLUMNS if column in train],
    ]
    source_columns = list(dict.fromkeys(source_columns))
    sequence = pd.concat(
        [train[source_columns], inference[source_columns]],
        ignore_index=True,
        copy=False,
    )
    sequence["_component"] = np.concatenate(
        [train_components, inference_components]
    )
    sequence["_row_order"] = np.arange(len(sequence), dtype="int32")
    for column in ADVANCED_UID_COLUMNS:
        sequence[column] = uid_frame[column]
    sequence = sequence.sort_values(
        ["TransactionDT", "TransactionID"],
        kind="stable",
    ).reset_index(drop=True)
    group = sequence.groupby("_component", sort=False, observed=True)

    feature_values: dict[str, pd.Series | np.ndarray] = {}
    count = group["TransactionID"].transform("size").astype("int32")
    order = group.cumcount().astype("int32")
    feature_values["adv_user_total_count"] = count
    feature_values["adv_user_is_singleton"] = count.eq(1).astype("int8")
    feature_values["adv_user_prior_count"] = order

    day = (sequence["TransactionDT"] // 86400).astype("int32")
    sequence["_day"] = day
    previous_day = group["_day"].shift(1)
    sequence["_new_active_day"] = day.ne(previous_day).astype("int8")
    active_days = group["_new_active_day"].cumsum().astype("int16")
    feature_values["adv_user_active_days_so_far"] = active_days
    feature_values["adv_user_transactions_per_active_day"] = (
        (order + 1) / active_days.replace(0, np.nan)
    ).astype("float32")

    sequence["_dt_gap"] = group["TransactionDT"].diff().astype("float32")
    for window in ROLLING_WINDOWS:
        feature_values[f"adv_user_dt_gap_mean_last_{window}"] = (
            _rolling_group_feature(sequence, "_dt_gap", window, "mean")
        )
        feature_values[f"adv_user_dt_gap_std_last_{window}"] = (
            _rolling_group_feature(sequence, "_dt_gap", window, "std")
        )

    previous_c13 = group["C13"].shift(1)
    feature_values["adv_user_C13_diff_previous"] = (
        sequence["C13"] - previous_c13
    ).astype("float32")
    gap_days = sequence["_dt_gap"] / 86400.0
    feature_values["adv_user_D3_gap_error"] = (
        sequence["D3"] - gap_days
    ).astype("float32")
    feature_values["adv_user_D3_gap_match"] = (
        (sequence["D3"] - gap_days).abs().le(1.5)
        & sequence["D3"].notna()
        & gap_days.notna()
    ).astype("int8")

    for column in BEHAVIOR_COLUMNS:
        if column not in sequence:
            continue
        value = _tokens(sequence[column])
        behavior = pd.DataFrame(
            {
                "component": sequence["_component"],
                "value": value,
                "dt": sequence["TransactionDT"],
            }
        )
        pair_group = behavior.groupby(
            ["component", "value"],
            sort=False,
            observed=True,
            dropna=False,
        )
        current_value_count = pair_group.cumcount().add(1).astype("int32")
        prior_same_count = current_value_count - 1
        running_mode_count = current_value_count.groupby(
            behavior["component"], sort=False
        ).cummax()
        prior_mode_count = running_mode_count.groupby(
            behavior["component"], sort=False
        ).shift(1).fillna(0)
        seconds_since_same = pair_group["dt"].diff()
        previous_value = value.groupby(
            sequence["_component"], sort=False
        ).shift(1)
        prefix = f"adv_user_{column}"
        feature_values[f"{prefix}_prior_count"] = prior_same_count
        feature_values[f"{prefix}_prior_fraction"] = (
            prior_same_count / order.replace(0, np.nan)
        ).astype("float32")
        feature_values[f"{prefix}_is_prior_mode"] = (
            prior_same_count.ge(prior_mode_count) & order.gt(0)
        ).astype("int8")
        feature_values[f"{prefix}_seen_before"] = prior_same_count.gt(0).astype(
            "int8"
        )
        feature_values[f"{prefix}_seconds_since_same"] = (
            seconds_since_same.astype("float32")
        )
        feature_values[f"{prefix}_seen_previous_30d"] = (
            seconds_since_same.between(0, 30 * 86400, inclusive="both")
        ).astype("int8")
        feature_values[f"{prefix}_changed_from_previous"] = (
            previous_value.notna() & value.ne(previous_value)
        ).astype("int8")

    for uid in ADVANCED_UID_COLUMNS:
        feature_values[uid] = sequence[uid].astype("string")
        train_uid = uid_frame[uid].iloc[: len(train)]
        frequencies = train_uid.value_counts(dropna=False) / len(train_uid)
        feature_values[f"{uid}_freq"] = sequence[uid].map(frequencies).fillna(
            0
        ).astype("float32")

    features = pd.DataFrame(feature_values)
    features["_row_order"] = sequence["_row_order"].to_numpy()
    features = features.sort_values("_row_order", kind="stable").drop(
        columns="_row_order"
    )
    features.reset_index(drop=True, inplace=True)
    train_rows = len(train)
    train_features = features.iloc[:train_rows].copy()
    inference_features = features.iloc[train_rows:].reset_index(drop=True).copy()
    numeric_columns = [
        column
        for column in features.columns
        if column not in ADVANCED_UID_COLUMNS
    ]
    for frame in (train_features, inference_features):
        for column in numeric_columns:
            if pd.api.types.is_float_dtype(frame[column]):
                frame[column] = frame[column].astype("float32")
    return (
        pd.concat([train.reset_index(drop=True), train_features], axis=1),
        pd.concat(
            [inference.reset_index(drop=True), inference_features], axis=1
        ),
        features.columns.tolist(),
        list(ADVANCED_UID_COLUMNS),
        train_components,
        inference_components,
        stats,
    )


Overwriting fraud_advanced_user_features.py


In [9]:
%%writefile fraud_features.py
from pathlib import Path

import numpy as np
import pandas as pd


TARGET = "isFraud"
GIBA_UID_COLUMNS = [
    "uid_d1_email",
    "uid_card_addr_d1",
    "uid_card_addr_d1_email",
]
GIBA_SEQUENCE_UID_COLUMNS = [
    "uid_d1_email",
    "uid_card_addr_d1_email",
]
SELECTED_FREQUENCY_COLUMNS = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceInfo",
    "DeviceInfo_family",
    "uid_card_addr",
    "uid_card_email",
    "uid_card_full",
    "uid_card_device",
    "uid_email_pair",
    *GIBA_UID_COLUMNS,
]


def read_and_merge(data_dir, split):
    data_dir = Path(data_dir)
    transaction = pd.read_csv(data_dir / f"{split}_transaction.csv")
    identity = pd.read_csv(data_dir / f"{split}_identity.csv")

    transaction = transaction.drop(columns=["Unnamed: 0"], errors="ignore")
    identity = identity.drop(columns=["Unnamed: 0"], errors="ignore")

    return transaction.merge(identity, on="TransactionID", how="left")


def _as_token(series):
    return series.astype("string").fillna("<MISSING>")


def _as_integer_token(series):
    return series.round().astype("Int32").astype("string").fillna("<MISSING>")


def _join_tokens(*series):
    result = _as_token(series[0])
    for value in series[1:]:
        result = result.str.cat(_as_token(value), sep="|")
    return result


def _add_group_aggregates(frame, prefix, columns):
    if not columns:
        return

    values = frame[columns]
    frame[f"{prefix}_missing_count"] = values.isna().sum(axis=1).astype("int16")
    frame[f"{prefix}_mean"] = values.mean(axis=1).astype("float32")
    frame[f"{prefix}_std"] = values.std(axis=1).astype("float32")
    frame[f"{prefix}_min"] = values.min(axis=1).astype("float32")
    frame[f"{prefix}_max"] = values.max(axis=1).astype("float32")


def _add_row_features(frame):
    transaction_dt = frame["TransactionDT"]
    transaction_day = transaction_dt / 86400
    amount = frame["TransactionAmt"]

    frame["DT_day"] = np.floor(transaction_day).astype("int16")
    frame["DT_week"] = np.floor(transaction_day / 7).astype("int16")
    frame["DT_hour"] = ((transaction_dt // 3600) % 24).astype("int8")
    frame["DT_dayofweek"] = (
        (transaction_dt // 86400) % 7
    ).astype("int8")

    frame["TransactionAmt_log1p"] = np.log1p(amount).astype("float32")
    frame["TransactionAmt_cents"] = (
        np.round((amount - np.floor(amount)) * 100) % 100
    ).astype("int8")
    frame["TransactionAmt_is_integer"] = (
        np.isclose(amount % 1, 0)
    ).astype("int8")
    frame["TransactionAmt_is_round_10"] = (
        np.isclose(amount % 10, 0)
    ).astype("int8")

    p_email = _as_token(frame["P_emaildomain"])
    r_email = _as_token(frame["R_emaildomain"])
    frame["P_R_email_match"] = (p_email == r_email).astype("int8")
    frame["P_email_suffix"] = p_email.str.rsplit(".", n=1).str[-1]
    frame["R_email_suffix"] = r_email.str.rsplit(".", n=1).str[-1]

    if "DeviceInfo" in frame:
        device_info = _as_token(frame["DeviceInfo"])
        frame["DeviceInfo_family"] = device_info.str.split("/", n=1).str[0]

    if "id_31" in frame:
        browser = _as_token(frame["id_31"])
        frame["browser_family"] = browser.str.replace(
            r"[\d._-]+$", "", regex=True
        ).str.strip()

    combo_columns = {
        "uid_card_addr": ["card1", "addr1"],
        "uid_card_email": ["card1", "addr1", "P_emaildomain"],
        "uid_card_full": ["card1", "card2", "card3", "card5"],
        "uid_card_device": ["card1", "addr1", "DeviceInfo"],
        "uid_email_pair": ["P_emaildomain", "R_emaildomain"],
    }
    for name, columns in combo_columns.items():
        frame[name] = _as_token(frame[columns[0]])
        for column in columns[1:]:
            frame[name] = frame[name].str.cat(
                _as_token(frame[column]),
                sep="|",
            )

    d_columns = [f"D{i}" for i in range(1, 16) if f"D{i}" in frame]
    for column in d_columns:
        frame[f"{column}_minus_day"] = (
            frame[column] - transaction_day
        ).astype("float32")

    c_columns = [f"C{i}" for i in range(1, 15) if f"C{i}" in frame]
    v_columns = [f"V{i}" for i in range(1, 340) if f"V{i}" in frame]
    id_numeric = [
        column
        for column in frame.columns
        if column.startswith("id_") and pd.api.types.is_numeric_dtype(frame[column])
    ]

    _add_group_aggregates(frame, "C", c_columns)
    _add_group_aggregates(frame, "D", d_columns)
    _add_group_aggregates(frame, "V", v_columns)
    _add_group_aggregates(frame, "id_numeric", id_numeric)

    frame["row_missing_count"] = frame.isna().sum(axis=1).astype("int16")


def _add_giba_uid_features(frame):
    d1_origin_day = frame["DT_day"].astype("float32") - frame["D1"]
    frame["D1_origin_day"] = d1_origin_day.astype("float32")

    origin_token = _as_integer_token(d1_origin_day)
    p_email = _as_token(frame["P_emaildomain"])
    frame["uid_d1_email"] = origin_token.str.cat(p_email, sep="|")
    frame["uid_card_addr_d1"] = _join_tokens(
        frame["card1"],
        frame["addr1"],
        origin_token,
    )
    frame["uid_card_addr_d1_email"] = frame["uid_card_addr_d1"].str.cat(
        p_email,
        sep="|",
    )


def _add_uid_sequence_features(
    train,
    test,
    add_v307_chain=False,
):
    columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        *GIBA_SEQUENCE_UID_COLUMNS,
    ]
    if add_v307_chain and "V307" in train:
        columns.append("V307")
    sequence = pd.concat(
        [train[columns], test[columns]],
        ignore_index=True,
        copy=False,
    )
    sequence["_row_order"] = np.arange(len(sequence), dtype="int32")
    sequence = sequence.sort_values(
        ["TransactionDT", "TransactionID"],
        kind="stable",
    ).reset_index(drop=True)

    feature_names = []
    for uid in GIBA_SEQUENCE_UID_COLUMNS:
        group = sequence.groupby(uid, sort=False, observed=True, dropna=False)
        prefix = f"{uid}_seq"

        sequence[f"{prefix}_count"] = group["TransactionID"].transform(
            "size"
        ).astype("int32")
        sequence[f"{prefix}_order"] = group.cumcount().astype("int32")
        sequence[f"{prefix}_order_from_end"] = (
            sequence[f"{prefix}_count"] - sequence[f"{prefix}_order"] - 1
        ).astype("int32")

        previous_dt = group["TransactionDT"].diff()
        next_dt = group["TransactionDT"].shift(-1) - sequence["TransactionDT"]
        sequence[f"{prefix}_previous_dt"] = previous_dt.astype("float32")
        sequence[f"{prefix}_next_dt"] = next_dt.astype("float32")

        gap_column = f"_{uid}_gap"
        sequence[gap_column] = previous_dt
        gap_group = sequence.groupby(
            uid,
            sort=False,
            observed=True,
            dropna=False,
        )[gap_column]
        sequence[f"{prefix}_mean_dt"] = gap_group.transform("mean").astype(
            "float32"
        )
        sequence[f"{prefix}_std_dt"] = gap_group.transform("std").astype(
            "float32"
        )
        sequence[f"{prefix}_median_dt"] = gap_group.transform("median").astype(
            "float32"
        )
        sequence.drop(columns=[gap_column], inplace=True)

        first_dt = group["TransactionDT"].transform("min")
        last_dt = group["TransactionDT"].transform("max")
        sequence[f"{prefix}_time_span"] = (last_dt - first_dt).astype("float32")

        mean_amount = group["TransactionAmt"].transform("mean")
        std_amount = group["TransactionAmt"].transform("std")
        sequence[f"{prefix}_mean_amt"] = mean_amount.astype("float32")
        sequence[f"{prefix}_std_amt"] = std_amount.astype("float32")
        sequence[f"{prefix}_median_amt"] = group[
            "TransactionAmt"
        ].transform("median").astype("float32")
        sequence[f"{prefix}_previous_amt"] = group[
            "TransactionAmt"
        ].shift(1).astype("float32")
        sequence[f"{prefix}_next_amt"] = group[
            "TransactionAmt"
        ].shift(-1).astype("float32")
        sequence[f"{prefix}_amt_to_mean"] = (
            sequence["TransactionAmt"] / mean_amount.replace(0, np.nan)
        ).astype("float32")
        sequence[f"{prefix}_amt_zscore"] = (
            (sequence["TransactionAmt"] - mean_amount)
            / std_amount.replace(0, np.nan)
        ).astype("float32")

        chain_feature_names = []
        if add_v307_chain and "V307" in sequence:
            previous_v307 = group["V307"].shift(1)
            next_v307 = group["V307"].shift(-1)
            previous_amount = group["TransactionAmt"].shift(1)
            previous_delta = sequence["V307"] - previous_v307
            next_delta = next_v307 - sequence["V307"]
            previous_error = previous_delta - previous_amount
            next_error = next_delta - sequence["TransactionAmt"]
            chain_feature_names = [
                f"{prefix}_V307_previous_delta",
                f"{prefix}_V307_previous_amount_error",
                f"{prefix}_V307_previous_amount_match",
                f"{prefix}_V307_next_delta",
                f"{prefix}_V307_next_amount_error",
                f"{prefix}_V307_next_amount_match",
            ]
            sequence[chain_feature_names[0]] = previous_delta.astype("float32")
            sequence[chain_feature_names[1]] = previous_error.astype("float32")
            sequence[chain_feature_names[2]] = (
                previous_error.abs().le(0.011)
                & previous_error.notna()
            ).astype("int8")
            sequence[chain_feature_names[3]] = next_delta.astype("float32")
            sequence[chain_feature_names[4]] = next_error.astype("float32")
            sequence[chain_feature_names[5]] = (
                next_error.abs().le(0.011) & next_error.notna()
            ).astype("int8")

        feature_names.extend(
            [
                f"{prefix}_count",
                f"{prefix}_order",
                f"{prefix}_order_from_end",
                f"{prefix}_previous_dt",
                f"{prefix}_next_dt",
                f"{prefix}_mean_dt",
                f"{prefix}_std_dt",
                f"{prefix}_median_dt",
                f"{prefix}_time_span",
                f"{prefix}_mean_amt",
                f"{prefix}_std_amt",
                f"{prefix}_median_amt",
                f"{prefix}_previous_amt",
                f"{prefix}_next_amt",
                f"{prefix}_amt_to_mean",
                f"{prefix}_amt_zscore",
                *chain_feature_names,
            ]
        )

    sequence = sequence.sort_values("_row_order", kind="stable")
    train_rows = len(train)
    for feature_name in feature_names:
        values = sequence[feature_name].to_numpy()
        train[feature_name] = values[:train_rows]
        test[feature_name] = values[train_rows:]


def _add_frequency_features(
    train,
    test,
    mode="selected",
    raw_columns=None,
):
    if mode == "selected":
        frequency_columns = SELECTED_FREQUENCY_COLUMNS
    elif mode in {"all-train", "all-joint"}:
        raw_frequency_columns = [
            column for column in (raw_columns or [])
            if column not in {TARGET, "TransactionID"}
        ]
        frequency_columns = list(
            dict.fromkeys(
                [*SELECTED_FREQUENCY_COLUMNS, *raw_frequency_columns]
            )
        )
    else:
        raise ValueError(f"Unknown frequency mode: {mode}")

    train_features = {}
    test_features = {}
    for column in frequency_columns:
        if column not in train:
            continue
        feature_name = f"{column}_freq"
        if mode == "selected":
            train_tokens = _as_token(train[column])
            counts = train_tokens.value_counts(dropna=False)
            frequencies = counts / len(train)
            train_frequency = train_tokens.map(frequencies).fillna(0)
            test_frequency = _as_token(test[column]).map(frequencies).fillna(0)
        else:
            combined = pd.concat(
                [train[column], test[column]],
                ignore_index=True,
                copy=False,
            )
            codes, _ = pd.factorize(combined, sort=False)
            # Reserve code zero for missing values, which factorize marks as -1.
            codes = codes.astype("int32", copy=False) + 1
            train_codes = codes[: len(train)]
            if mode == "all-joint":
                counts = np.bincount(codes)
                denominator = len(combined)
            else:
                counts = np.bincount(train_codes, minlength=codes.max() + 1)
                denominator = len(train)
            train_frequency = counts[train_codes] / denominator
            test_frequency = counts[codes[len(train) :]] / denominator

            # A constant train feature cannot be used by a supervised model.
            if np.unique(train_frequency).size < 2:
                continue

        train_features[feature_name] = np.asarray(
            train_frequency,
            dtype="float32",
        )
        test_features[feature_name] = np.asarray(
            test_frequency,
            dtype="float32",
        )

    return (
        pd.concat([train, pd.DataFrame(train_features, index=train.index)], axis=1),
        pd.concat([test, pd.DataFrame(test_features, index=test.index)], axis=1),
    )


def _downcast(frame):
    for column in frame.select_dtypes(include=["float64"]).columns:
        frame[column] = frame[column].astype("float32")

    for column in frame.select_dtypes(include=["int64"]).columns:
        if column == "TransactionID":
            frame[column] = frame[column].astype("int32")
        elif column == TARGET:
            frame[column] = frame[column].astype("int8")
        else:
            frame[column] = pd.to_numeric(frame[column], downcast="integer")


def build_features(
    train,
    test,
    giba_features=False,
    frequency_mode="selected",
    v307_chain_features=False,
):
    raw_frequency_columns = train.columns.tolist()
    _add_row_features(train)
    _add_row_features(test)
    if giba_features:
        _add_giba_uid_features(train)
        _add_giba_uid_features(test)
        _add_uid_sequence_features(
            train,
            test,
            add_v307_chain=v307_chain_features,
        )
    train, test = _add_frequency_features(
        train,
        test,
        mode=frequency_mode,
        raw_columns=raw_frequency_columns,
    )
    _downcast(train)
    _downcast(test)

    feature_columns = [
        column
        for column in train.columns
        if column not in {TARGET, "TransactionID"}
    ]
    test = test.reindex(columns=["TransactionID", *feature_columns])

    categorical_columns = train[feature_columns].select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
    for column in categorical_columns:
        train[column] = _as_token(train[column])
        test[column] = _as_token(test[column])

    return train, test, feature_columns, categorical_columns


Overwriting fraud_features.py


In [10]:
%%writefile fraud_honest_advanced_data.py
"""Clean official-data preparation shared by the honest CatBoost views."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from fraud_advanced_user_features import add_advanced_user_features
from fraud_features import TARGET, build_features, read_and_merge
from fraud_user_features import add_target_free_user_profile_features


ADVANCED_CAT_PARAMS = {
    "iterations": 2_100,
    "depth": 8,
    "learning_rate": 0.055,
    "l2_leaf_reg": 9,
    "random_strength": 0.45,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.82,
    "rsm": 0.90,
    "border_count": 128,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "one_hot_max_size": 10,
    "max_ctr_complexity": 1,
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": 100,
}


def add_amount_patterns(frame: pd.DataFrame) -> list[str]:
    amount = frame["TransactionAmt"].astype("float64")
    values: dict[str, np.ndarray] = {}
    for modulus in (50.0, 100.0, 200.0):
        suffix = int(modulus)
        remainder = np.mod(amount, modulus)
        distance = np.minimum(remainder, modulus - remainder)
        values[f"TransactionAmt_mod_{suffix}_is_zero"] = np.isclose(
            remainder, 0.0, atol=0.011
        ).astype("int8")
        values[f"TransactionAmt_mod_{suffix}_distance"] = distance.astype(
            "float32"
        )
    names = list(values)
    frame[names] = pd.DataFrame(values, index=frame.index)
    return names


def prepare_advanced_catboost_data(data_dir: Path) -> tuple[dict, list[str], list[str]]:
    data_dir = Path(data_dir)
    print("Reading official train/test...", flush=True)
    train = read_and_merge(data_dir, "train")
    inference = read_and_merge(data_dir, "test")
    if TARGET in inference:
        raise AssertionError("Competition test unexpectedly contains target")
    y = train[TARGET].astype("int8").reset_index(drop=True)

    print("Building Giba and target-free graph profiles...", flush=True)
    train, inference, base_features, base_categorical = build_features(
        train,
        inference,
        giba_features=True,
    )
    (
        train,
        inference,
        profile_features,
        _,
        _,
        graph_stats,
    ) = add_target_free_user_profile_features(train, inference)

    print("Building advanced UID, rolling and behavior features...", flush=True)
    (
        train,
        inference,
        advanced_features,
        advanced_categorical,
        train_components,
        inference_components,
        advanced_stats,
    ) = add_advanced_user_features(train, inference)
    train_amount = add_amount_patterns(train)
    test_amount = add_amount_patterns(inference)
    if train_amount != test_amount:
        raise ValueError("Train/test amount feature names differ")

    features = list(
        dict.fromkeys(
            [
                *base_features,
                *profile_features,
                *advanced_features,
                *train_amount,
            ]
        )
    )
    categorical = list(
        dict.fromkeys([*base_categorical, *advanced_categorical])
    )
    train_ids = train.pop("TransactionID").reset_index(drop=True)
    inference_ids = inference.pop("TransactionID").reset_index(drop=True)
    train.pop(TARGET)

    # Match the original LightGBM-compatible preparation before converting
    # categories to CatBoost-safe strings. Unseen test categories become missing.
    for column in categorical:
        categories = pd.Index(train[column].dropna().unique())
        dtype = pd.CategoricalDtype(categories=categories)
        train[column] = train[column].astype(dtype).astype("string").fillna(
            "<MISSING>"
        )
        inference[column] = (
            inference[column]
            .astype(dtype)
            .astype("string")
            .fillna("<MISSING>")
        )

    prepared = {
        "train": train,
        "inference": inference,
        "y": y,
        "train_ids": train_ids,
        "inference_ids": inference_ids,
        "train_components": train_components,
        "inference_components": inference_components,
        "graph_stats": graph_stats,
        "advanced_stats": advanced_stats,
    }
    print(
        f"Advanced CatBoost data: {len(features)} features, "
        f"{len(categorical)} categorical",
        flush=True,
    )
    return prepared, features, categorical


Overwriting fraud_honest_advanced_data.py


In [11]:
%%writefile fraud_multicounter_features.py
from __future__ import annotations

import numpy as np
import pandas as pd


COUNTER_COLUMNS = ("V126", "V127", "V128", "V306", "V307", "V308")
CHAIN_COLUMN = "multi_counter_transaction_chain"


def _build_chain_ids(
    sequence: pd.DataFrame,
    max_chain_size: int,
) -> tuple[np.ndarray, np.ndarray, dict]:
    uid_codes, _ = pd.factorize(
        sequence["uid_card_addr_d1_email"],
        sort=False,
    )
    transaction_dt = sequence["TransactionDT"].to_numpy()
    transaction_id = sequence["TransactionID"].to_numpy()
    amount = sequence["TransactionAmt"].to_numpy(dtype="float64", copy=False)
    counter_values = {
        column: sequence[column].to_numpy(dtype="float64", copy=False)
        for column in COUNTER_COLUMNS
    }
    order = np.lexsort((transaction_id, transaction_dt, uid_codes))

    chain_ids = np.empty(len(sequence), dtype="int32")
    match_votes = np.zeros(len(sequence), dtype="int8")
    chain_sizes = np.zeros(len(sequence), dtype="int32")
    next_chain_id = 0
    current_uid = None
    endpoint_maps: dict[str, dict[int, tuple[int, int]]] = {}
    capped_candidates = 0

    for sequence_position, row in enumerate(order):
        uid = int(uid_codes[row])
        if uid != current_uid:
            endpoint_maps = {column: {} for column in COUNTER_COLUMNS}
            current_uid = uid

        candidates: dict[int, tuple[int, int]] = {}
        if np.isfinite(amount[row]):
            for column in COUNTER_COLUMNS:
                value = counter_values[column][row]
                if not np.isfinite(value):
                    continue
                start = int(np.rint(value * 1000.0))
                matches = [
                    endpoint_maps[column][key]
                    for key in (start - 1, start, start + 1)
                    if key in endpoint_maps[column]
                ]
                if not matches:
                    continue
                chain_id, last_position = max(
                    matches,
                    key=lambda candidate: candidate[1],
                )
                if chain_sizes[chain_id] >= max_chain_size:
                    capped_candidates += 1
                    continue
                votes, latest = candidates.get(chain_id, (0, -1))
                candidates[chain_id] = (
                    votes + 1,
                    max(latest, last_position),
                )

        if candidates:
            chain_id, (votes, _) = max(
                candidates.items(),
                key=lambda item: (item[1][0], item[1][1]),
            )
            match_votes[row] = votes
        else:
            chain_id = next_chain_id
            next_chain_id += 1
        chain_ids[row] = chain_id
        chain_sizes[chain_id] += 1

        if np.isfinite(amount[row]):
            for column in COUNTER_COLUMNS:
                value = counter_values[column][row]
                if not np.isfinite(value):
                    continue
                endpoint = int(np.rint((value + amount[row]) * 1000.0))
                endpoint_maps[column][endpoint] = (
                    chain_id,
                    sequence_position,
                )

    counts = np.bincount(chain_ids)
    stats = {
        "chains": int(len(counts)),
        "linked_rows": int(np.sum(match_votes > 0)),
        "rows_with_two_or_more_votes": int(np.sum(match_votes >= 2)),
        "multirow_chains": int(np.sum(counts > 1)),
        "rows_in_multirow_chains": int(counts[counts > 1].sum()),
        "max_chain_size": int(counts.max()),
        "chain_size_cap": int(max_chain_size),
        "capped_counter_matches": int(capped_candidates),
    }
    return chain_ids, match_votes, stats


def add_multi_counter_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    max_chain_size: int = 100,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], dict]:
    source_columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "uid_card_addr_d1_email",
        *COUNTER_COLUMNS,
    ]
    missing = sorted(set(source_columns).difference(train.columns))
    if missing:
        raise ValueError(f"Missing multi-counter columns: {missing}")
    sequence = pd.concat(
        [train[source_columns], inference[source_columns]],
        ignore_index=True,
        copy=False,
    )
    train_rows = len(train)
    chain_ids, match_votes, stats = _build_chain_ids(
        sequence,
        max_chain_size=max_chain_size,
    )

    sequence["_chain"] = chain_ids
    sequence["_match_votes"] = match_votes
    sequence["_row_order"] = np.arange(len(sequence), dtype="int32")
    sequence = sequence.sort_values(
        ["TransactionDT", "TransactionID"],
        kind="stable",
    ).reset_index(drop=True)
    uid_group = sequence.groupby(
        "uid_card_addr_d1_email",
        sort=False,
        observed=True,
        dropna=False,
    )
    chain_group = sequence.groupby("_chain", sort=False, observed=True)

    feature_values: dict[str, pd.Series | np.ndarray] = {}
    previous_matches = []
    next_matches = []
    previous_unchanged = []
    next_unchanged = []
    previous_amount = uid_group["TransactionAmt"].shift(1)
    for column in COUNTER_COLUMNS:
        previous_value = uid_group[column].shift(1)
        next_value = uid_group[column].shift(-1)
        previous_delta = sequence[column] - previous_value
        next_delta = next_value - sequence[column]
        previous_error = previous_delta - previous_amount
        next_error = next_delta - sequence["TransactionAmt"]
        prefix = f"multi_{column}"
        feature_values[f"{prefix}_previous_delta"] = previous_delta.astype(
            "float32"
        )
        feature_values[f"{prefix}_previous_amount_error"] = (
            previous_error.astype("float32")
        )
        feature_values[f"{prefix}_next_delta"] = next_delta.astype("float32")
        feature_values[f"{prefix}_next_amount_error"] = next_error.astype(
            "float32"
        )
        previous_match = previous_error.abs().le(0.011) & previous_error.notna()
        next_match = next_error.abs().le(0.011) & next_error.notna()
        previous_zero = previous_delta.abs().le(0.011) & previous_delta.notna()
        next_zero = next_delta.abs().le(0.011) & next_delta.notna()
        feature_values[f"{prefix}_previous_amount_match"] = previous_match.astype(
            "int8"
        )
        feature_values[f"{prefix}_next_amount_match"] = next_match.astype("int8")
        feature_values[f"{prefix}_previous_unchanged"] = previous_zero.astype(
            "int8"
        )
        feature_values[f"{prefix}_next_unchanged"] = next_zero.astype("int8")
        previous_matches.append(previous_match.to_numpy(dtype="int8"))
        next_matches.append(next_match.to_numpy(dtype="int8"))
        previous_unchanged.append(previous_zero.to_numpy(dtype="int8"))
        next_unchanged.append(next_zero.to_numpy(dtype="int8"))

    feature_values["multi_counter_previous_match_count"] = np.sum(
        previous_matches,
        axis=0,
        dtype="int8",
    )
    feature_values["multi_counter_next_match_count"] = np.sum(
        next_matches,
        axis=0,
        dtype="int8",
    )
    feature_values["multi_counter_previous_unchanged_count"] = np.sum(
        previous_unchanged,
        axis=0,
        dtype="int8",
    )
    feature_values["multi_counter_next_unchanged_count"] = np.sum(
        next_unchanged,
        axis=0,
        dtype="int8",
    )

    count = chain_group["TransactionID"].transform("size").astype("int32")
    position = chain_group.cumcount().astype("int32")
    first_dt = chain_group["TransactionDT"].transform("min")
    last_dt = chain_group["TransactionDT"].transform("max")
    feature_values[CHAIN_COLUMN] = (
        "mc" + sequence["_chain"].astype("string")
    )
    feature_values["multi_chain_total_count"] = count
    feature_values["multi_chain_position"] = position
    feature_values["multi_chain_position_from_end"] = count - position - 1
    feature_values["multi_chain_previous_dt"] = chain_group[
        "TransactionDT"
    ].diff().astype("float32")
    feature_values["multi_chain_next_dt"] = (
        chain_group["TransactionDT"].shift(-1) - sequence["TransactionDT"]
    ).astype("float32")
    feature_values["multi_chain_time_span"] = (last_dt - first_dt).astype(
        "float32"
    )
    feature_values["multi_chain_previous_amount"] = chain_group[
        "TransactionAmt"
    ].shift(1).astype("float32")
    feature_values["multi_chain_next_amount"] = chain_group[
        "TransactionAmt"
    ].shift(-1).astype("float32")
    feature_values["multi_chain_match_votes"] = sequence["_match_votes"].astype(
        "int8"
    )

    features = pd.DataFrame(feature_values)
    features["_row_order"] = sequence["_row_order"].to_numpy()
    features = features.sort_values("_row_order", kind="stable").drop(
        columns="_row_order"
    )
    features.reset_index(drop=True, inplace=True)
    feature_names = features.columns.tolist()
    stats["rows"] = int(len(sequence))
    stats["features"] = int(len(feature_names))

    train_features = features.iloc[:train_rows].reset_index(drop=True)
    inference_features = features.iloc[train_rows:].reset_index(drop=True)
    return (
        pd.concat([train.reset_index(drop=True), train_features], axis=1),
        pd.concat(
            [inference.reset_index(drop=True), inference_features],
            axis=1,
        ),
        feature_names,
        [CHAIN_COLUMN],
        stats,
    )


Overwriting fraud_multicounter_features.py


In [12]:
%%writefile fraud_next_features.py
"""Additional target-free feature families for the fraud temporal ablation."""

from __future__ import annotations

import numpy as np
import pandas as pd


START_DATE = pd.Timestamp("2017-12-01")
VELOCITY_WINDOWS = (
    ("1h", 3_600.0),
    ("6h", 6 * 3_600.0),
    ("1d", 86_400.0),
    ("7d", 7 * 86_400.0),
    ("30d", 30 * 86_400.0),
)
VELOCITY_KEYS = (
    "uid_card_addr_d1_email",
    "uid_d1_email",
    "uid_card_addr",
)
BEHAVIOR_COLUMNS = (
    "ProductCD",
    "card2",
    "card5",
    "addr1",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceInfo",
    "id_30",
    "id_31",
    "id_33",
)
COOCCURRENCE_PAIRS = (
    ("card1", "addr1"),
    ("card1", "P_emaildomain"),
    ("card1", "DeviceInfo"),
    ("P_emaildomain", "DeviceInfo"),
    ("id_30", "id_31"),
    ("card4", "card6"),
)


def _tokens(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("<MISSING>")


def _joined_tokens(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.Series:
    result = _tokens(frame[columns[0]])
    for column in columns[1:]:
        values = frame[column]
        if column.endswith("origin_day"):
            values = values.round()
        result = result.str.cat(_tokens(values), sep="|")
    return result


def _attach_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    features: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    features = features.reset_index(drop=True)
    train_rows = len(train)
    names = features.columns.tolist()
    return (
        pd.concat(
            [train.reset_index(drop=True), features.iloc[:train_rows]],
            axis=1,
        ),
        pd.concat(
            [
                inference.reset_index(drop=True),
                features.iloc[train_rows:].reset_index(drop=True),
            ],
            axis=1,
        ),
        names,
    )


def _velocity_arrays(
    group_codes: np.ndarray,
    transaction_dt: np.ndarray,
    transaction_id: np.ndarray,
    amount: np.ndarray,
) -> dict[str, np.ndarray]:
    row_count = len(group_codes)
    result: dict[str, np.ndarray] = {
        "prior_count": np.zeros(row_count, dtype="int32"),
        "seconds_since_previous": np.full(row_count, np.nan, dtype="float32"),
        "seconds_since_first": np.full(row_count, np.nan, dtype="float32"),
        "prior_amount_mean": np.full(row_count, np.nan, dtype="float32"),
        "prior_amount_std": np.full(row_count, np.nan, dtype="float32"),
        "amount_zscore_prior": np.full(row_count, np.nan, dtype="float32"),
        "amount_diff_previous": np.full(row_count, np.nan, dtype="float32"),
    }
    for name, _ in VELOCITY_WINDOWS:
        result[f"count_last_{name}"] = np.zeros(row_count, dtype="int32")
    for name in ("1d", "7d"):
        result[f"amount_sum_last_{name}"] = np.zeros(
            row_count, dtype="float32"
        )
        result[f"amount_mean_last_{name}"] = np.full(
            row_count, np.nan, dtype="float32"
        )

    order = np.lexsort((transaction_id, transaction_dt, group_codes))
    sorted_codes = group_codes[order]
    boundaries = np.flatnonzero(
        np.r_[True, sorted_codes[1:] != sorted_codes[:-1], True]
    )

    for group_start, group_end in zip(boundaries[:-1], boundaries[1:]):
        rows = order[group_start:group_end]
        dt = transaction_dt[rows]
        amt = amount[rows]
        size = len(rows)
        amount_valid = np.isfinite(amt)
        amount_zeroed = np.where(amount_valid, amt, 0.0)
        cumulative_amount = np.r_[0.0, np.cumsum(amount_zeroed)]
        cumulative_square = np.r_[0.0, np.cumsum(amount_zeroed**2)]
        cumulative_valid = np.r_[0, np.cumsum(amount_valid.astype("int32"))]
        left = np.zeros(len(VELOCITY_WINDOWS), dtype="int32")

        bucket_start = 0
        while bucket_start < size:
            bucket_end = bucket_start + 1
            while bucket_end < size and dt[bucket_end] == dt[bucket_start]:
                bucket_end += 1
            current_rows = rows[bucket_start:bucket_end]
            prior_count = bucket_start
            result["prior_count"][current_rows] = prior_count

            if prior_count:
                result["seconds_since_previous"][current_rows] = (
                    dt[bucket_start] - dt[bucket_start - 1]
                )
                result["seconds_since_first"][current_rows] = (
                    dt[bucket_start] - dt[0]
                )
                valid_count = cumulative_valid[bucket_start]
                if valid_count:
                    prior_sum = cumulative_amount[bucket_start]
                    prior_mean = prior_sum / valid_count
                    prior_variance = max(
                        cumulative_square[bucket_start] / valid_count
                        - prior_mean**2,
                        0.0,
                    )
                    prior_std = np.sqrt(prior_variance)
                    result["prior_amount_mean"][current_rows] = prior_mean
                    result["prior_amount_std"][current_rows] = prior_std
                    if prior_std > 1e-9:
                        result["amount_zscore_prior"][current_rows] = (
                            amt[bucket_start:bucket_end] - prior_mean
                        ) / prior_std
                if amount_valid[bucket_start - 1]:
                    result["amount_diff_previous"][current_rows] = (
                        amt[bucket_start:bucket_end] - amt[bucket_start - 1]
                    )

            for window_index, (name, seconds) in enumerate(VELOCITY_WINDOWS):
                cutoff = dt[bucket_start] - seconds
                while left[window_index] < bucket_start and (
                    dt[left[window_index]] < cutoff
                ):
                    left[window_index] += 1
                window_start = int(left[window_index])
                count = bucket_start - window_start
                result[f"count_last_{name}"][current_rows] = count
                if name in {"1d", "7d"}:
                    valid_count = (
                        cumulative_valid[bucket_start]
                        - cumulative_valid[window_start]
                    )
                    amount_sum = (
                        cumulative_amount[bucket_start]
                        - cumulative_amount[window_start]
                    )
                    result[f"amount_sum_last_{name}"][current_rows] = amount_sum
                    if valid_count:
                        result[f"amount_mean_last_{name}"][current_rows] = (
                            amount_sum / valid_count
                        )
            bucket_start = bucket_end
    return result


def add_velocity_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    required = {
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        *VELOCITY_KEYS,
    }
    missing = sorted(required.difference(train.columns))
    if missing:
        raise ValueError(f"Missing velocity columns: {missing}")
    columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        *VELOCITY_KEYS,
    ]
    combined = pd.concat(
        [train[columns], inference[columns]], ignore_index=True, copy=False
    )
    transaction_dt = combined["TransactionDT"].to_numpy(dtype="float64")
    transaction_id = combined["TransactionID"].to_numpy(dtype="int64")
    amount = combined["TransactionAmt"].to_numpy(dtype="float64")
    feature_values: dict[str, np.ndarray] = {}
    for key in VELOCITY_KEYS:
        print(f"Building causal velocity for {key}...", flush=True)
        codes, _ = pd.factorize(_tokens(combined[key]), sort=False)
        values = _velocity_arrays(
            codes.astype("int32", copy=False),
            transaction_dt,
            transaction_id,
            amount,
        )
        feature_values.update(
            {f"velocity_{key}_{name}": value for name, value in values.items()}
        )
    features = pd.DataFrame(feature_values)
    train, inference, names = _attach_features(train, inference, features)
    return train, inference, names, {
        "keys": list(VELOCITY_KEYS),
        "windows": [name for name, _ in VELOCITY_WINDOWS],
        "features": len(names),
    }


def _calendar_amount_rows(frame: pd.DataFrame) -> pd.DataFrame:
    dt = frame["TransactionDT"].astype("float64")
    day = dt / 86_400.0
    dates = START_DATE + pd.to_timedelta(dt, unit="s")
    amount = frame["TransactionAmt"].astype("float64")
    values: dict[str, pd.Series | np.ndarray] = {
        "calendar_month": dates.dt.month.astype("int8"),
        "calendar_dayofmonth": dates.dt.day.astype("int8"),
        "calendar_weekofyear": dates.dt.isocalendar().week.astype("int16"),
        "calendar_is_weekend": dates.dt.dayofweek.ge(5).astype("int8"),
        "calendar_minute": dates.dt.minute.astype("int8"),
        "calendar_seconds_into_day": np.mod(dt, 86_400).astype("float32"),
        "calendar_hour_sin": np.sin(2 * np.pi * dt / 86_400).astype("float32"),
        "calendar_hour_cos": np.cos(2 * np.pi * dt / 86_400).astype("float32"),
        "calendar_week_sin": np.sin(2 * np.pi * day / 7).astype("float32"),
        "calendar_week_cos": np.cos(2 * np.pi * day / 7).astype("float32"),
        "amount_log10": np.log10(amount.clip(lower=0) + 0.01).astype("float32"),
        "amount_magnitude": np.floor(
            np.log10(amount.clip(lower=0) + 0.01)
        ).astype("int8"),
        "amount_fraction_1000": np.mod(np.rint(amount * 1000), 1000).astype(
            "int16"
        ),
        "amount_nearest_integer_distance": np.abs(amount - np.rint(amount)).astype(
            "float32"
        ),
    }
    cents = np.mod(np.rint(amount * 100), 100).astype("int16")
    values["amount_cent_ending"] = cents
    for ending in (0, 25, 50, 95, 99):
        values[f"amount_cent_is_{ending:02d}"] = cents.eq(ending).astype("int8")
    for modulus in (1.0, 5.0, 10.0, 25.0, 500.0):
        suffix = str(modulus).replace(".", "p")
        remainder = np.mod(amount, modulus)
        values[f"amount_mod_{suffix}_distance"] = np.minimum(
            remainder, modulus - remainder
        ).astype("float32")

    origins = {}
    for number in range(1, 16):
        column = f"D{number}"
        if column not in frame:
            continue
        origin = day - frame[column].astype("float64")
        rounded = np.rint(origin)
        origins[column] = rounded
        values[f"{column}_origin_round"] = rounded.astype("float32")
        values[f"{column}_origin_week"] = np.floor(rounded / 7).astype("float32")
        values[f"{column}_origin_round_distance"] = np.abs(
            origin - rounded
        ).astype("float32")
    if origins:
        origin_frame = pd.DataFrame(origins, index=frame.index)
        values["D_origin_mean"] = origin_frame.mean(axis=1).astype("float32")
        values["D_origin_std"] = origin_frame.std(axis=1).astype("float32")
        values["D_origin_range"] = (
            origin_frame.max(axis=1) - origin_frame.min(axis=1)
        ).astype("float32")
        values["D_origin_nunique"] = origin_frame.nunique(axis=1).astype("int8")
        if "D1" in origin_frame:
            values["D_origin_matches_D1"] = origin_frame.sub(
                origin_frame["D1"], axis=0
            ).abs().le(1).sum(axis=1).astype("int8")

    c_columns = [f"C{i}" for i in range(1, 15) if f"C{i}" in frame]
    for column in c_columns:
        values[f"{column}_log1p"] = np.log1p(
            frame[column].clip(lower=0)
        ).astype("float32")
    for first, second in (
        ("C1", "C2"),
        ("C1", "C14"),
        ("C2", "C14"),
        ("C5", "C9"),
        ("C6", "C8"),
        ("C10", "C13"),
        ("C11", "C13"),
        ("C12", "C13"),
    ):
        if first in frame and second in frame:
            values[f"{first}_to_{second}"] = (
                frame[first] / frame[second].replace(0, np.nan)
            ).astype("float32")
    return pd.DataFrame(values, index=frame.index)


def _amount_group_features(combined: pd.DataFrame) -> pd.DataFrame:
    amount = combined["TransactionAmt"].astype("float64")
    amount_cents = np.rint(amount * 100).astype("int64")
    key_specs = {
        "card1": ("card1",),
        "card1_addr1": ("card1", "addr1"),
        "card1_product": ("card1", "ProductCD"),
        "card1_card5": ("card1", "card5"),
        "card1_register": ("card1", "D1_origin_day"),
        "product_card4": ("ProductCD", "card4"),
    }
    values: dict[str, pd.Series] = {}
    for name, columns in key_specs.items():
        key = _joined_tokens(combined, columns)
        helper = pd.DataFrame(
            {"key": key, "amount": amount, "amount_cents": amount_cents}
        )
        group = helper.groupby("key", sort=False, observed=True)
        count = group["amount"].transform("size")
        mean = group["amount"].transform("mean")
        std = group["amount"].transform("std")
        median = group["amount"].transform("median")
        same_count = helper.groupby(
            ["key", "amount_cents"], sort=False, observed=True
        )["amount"].transform("size")
        prefix = f"amount_group_{name}"
        values[f"{prefix}_log_count"] = np.log1p(count).astype("float32")
        values[f"{prefix}_mean"] = mean.astype("float32")
        values[f"{prefix}_std"] = std.astype("float32")
        values[f"{prefix}_median"] = median.astype("float32")
        values[f"{prefix}_diff_mean"] = (amount - mean).astype("float32")
        values[f"{prefix}_zscore"] = (
            (amount - mean) / std.replace(0, np.nan)
        ).astype("float32")
        values[f"{prefix}_same_amount_fraction"] = (
            same_count / count
        ).astype("float32")
    return pd.DataFrame(values, index=combined.index)


def add_calendar_amount_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    print("Building calendar, D-origin and amount fingerprints...", flush=True)
    train_rows = _calendar_amount_rows(train)
    inference_rows = _calendar_amount_rows(inference)
    combined = pd.concat([train, inference], ignore_index=True, copy=False)
    print("Building amount aggregates for stable entity keys...", flush=True)
    group_features = _amount_group_features(combined)
    features = pd.concat(
        [
            pd.concat([train_rows, inference_rows], ignore_index=True),
            group_features.reset_index(drop=True),
        ],
        axis=1,
    )
    train, inference, names = _attach_features(train, inference, features)
    return train, inference, names, {
        "features": len(names),
        "amount_group_keys": 6,
        "start_date": str(START_DATE.date()),
    }


def _device_brand(device: pd.Series) -> pd.Series:
    lower = _tokens(device).str.lower()
    brand = lower.str.split(r"[/ ]", n=1, regex=True).str[0]
    mappings = (
        (r"windows|trident|^rv:", "windows"),
        (r"ios|iphone|ipad", "apple_ios"),
        (r"macos|mac os", "apple_mac"),
        (r"^sm-|^gt-|^sch-|^sgh-", "samsung"),
        (r"huawei|^ale-|^ane-|^bla-|^vns-", "huawei"),
        (r"redmi|xiaomi|^mi ", "xiaomi"),
        (r"^lg-|^lg$", "lg"),
        (r"moto|motorola", "motorola"),
        (r"pixel|nexus", "google"),
        (r"htc", "htc"),
        (r"zte", "zte"),
    )
    for pattern, replacement in mappings:
        brand = brand.mask(lower.str.contains(pattern, regex=True, na=False), replacement)
    return brand.astype("string")


def _identity_rows(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    values: dict[str, pd.Series | np.ndarray] = {}
    categorical = []
    device = _tokens(frame.get("DeviceInfo", pd.Series(index=frame.index))).str.lower()
    device_brand = _device_brand(device)
    values["identity_device_brand"] = device_brand
    categorical.append("identity_device_brand")
    values["identity_device_is_mobile"] = device.str.contains(
        r"android|ios|iphone|ipad|^sm-|moto|huawei|redmi|xiaomi",
        regex=True,
        na=False,
    ).astype("int8")
    values["identity_device_has_build"] = device.str.contains(
        "build", regex=False, na=False
    ).astype("int8")

    os_value = _tokens(frame.get("id_30", pd.Series(index=frame.index))).str.lower()
    os_family = os_value.str.replace(r"[\d._-]+.*$", "", regex=True).str.strip()
    values["identity_os_family"] = os_family
    categorical.append("identity_os_family")
    values["identity_os_major"] = pd.to_numeric(
        os_value.str.extract(r"(\d+)", expand=False), errors="coerce"
    ).astype("float32")

    browser = _tokens(frame.get("id_31", pd.Series(index=frame.index))).str.lower()
    browser_family = browser.str.replace(r"[\d._-]+.*$", "", regex=True).str.strip()
    values["identity_browser_family"] = browser_family
    categorical.append("identity_browser_family")
    values["identity_browser_major"] = pd.to_numeric(
        browser.str.extract(r"(\d+)", expand=False), errors="coerce"
    ).astype("float32")
    values["identity_browser_is_mobile"] = browser.str.contains(
        r"mobile|android|ios", regex=True, na=False
    ).astype("int8")
    values["identity_browser_is_generic"] = browser.str.contains(
        "generic", regex=False, na=False
    ).astype("int8")

    screen = _tokens(frame.get("id_33", pd.Series(index=frame.index))).str.lower()
    dimensions = screen.str.extract(r"^(\d+)x(\d+)$")
    width = pd.to_numeric(dimensions[0], errors="coerce")
    height = pd.to_numeric(dimensions[1], errors="coerce")
    values["identity_screen_width"] = width.astype("float32")
    values["identity_screen_height"] = height.astype("float32")
    values["identity_screen_pixels"] = (width * height).astype("float32")
    values["identity_screen_aspect"] = (
        width / height.replace(0, np.nan)
    ).astype("float32")
    screen_known = width.notna() & height.notna()
    values["identity_screen_known"] = screen_known.astype("int8")
    values["identity_screen_portrait"] = (
        width.lt(height) & screen_known
    ).fillna(False).astype("int8")

    match_status = _tokens(
        frame.get("id_34", pd.Series(index=frame.index))
    ).str.extract(r"(-?\d+)$", expand=False)
    values["identity_match_status"] = pd.to_numeric(
        match_status, errors="coerce"
    ).astype("float32")

    m_columns = [f"M{i}" for i in range(1, 10) if f"M{i}" in frame]
    if m_columns:
        m_values = frame[m_columns].astype("string")
        values["identity_M_true_count"] = m_values.eq("T").sum(axis=1).astype("int8")
        values["identity_M_false_count"] = m_values.eq("F").sum(axis=1).astype("int8")
        values["identity_M_missing_count"] = m_values.isna().sum(axis=1).astype("int8")
        values["identity_M_pattern"] = m_values.fillna("N").agg("".join, axis=1)
        categorical.append("identity_M_pattern")

    id_columns = [f"id_{i:02d}" for i in range(1, 39) if f"id_{i:02d}" in frame]
    for block_number, start in enumerate((0, 19)):
        block = id_columns[start : start + 19]
        if not block:
            continue
        present = frame[block].notna().to_numpy(dtype="uint32")
        weights = np.left_shift(np.uint32(1), np.arange(len(block), dtype="uint32"))
        values[f"identity_present_mask_{block_number}"] = (
            present @ weights
        ).astype("uint32")
    if id_columns:
        values["identity_present_count"] = frame[id_columns].notna().sum(axis=1).astype(
            "int8"
        )

    d_columns = [f"D{i}" for i in range(1, 16) if f"D{i}" in frame]
    if d_columns:
        present = frame[d_columns].notna().to_numpy(dtype="uint16")
        weights = np.left_shift(np.uint16(1), np.arange(len(d_columns), dtype="uint16"))
        values["identity_D_present_mask"] = (present @ weights).astype("uint16")

    p_email = _tokens(frame["P_emaildomain"])
    r_email = _tokens(frame["R_emaildomain"])
    email_state = pd.Series("different", index=frame.index, dtype="string")
    both_missing = p_email.eq("<MISSING>") & r_email.eq("<MISSING>")
    one_missing = p_email.eq("<MISSING>") ^ r_email.eq("<MISSING>")
    email_state = email_state.mask(p_email.eq(r_email) & ~both_missing, "same")
    email_state = email_state.mask(one_missing, "one_missing")
    email_state = email_state.mask(both_missing, "both_missing")
    values["identity_email_state"] = email_state
    categorical.append("identity_email_state")
    return pd.DataFrame(values, index=frame.index), categorical


def add_identity_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], dict]:
    print("Parsing device, OS, browser and missingness signatures...", flush=True)
    train_features, train_categorical = _identity_rows(train)
    inference_features, inference_categorical = _identity_rows(inference)
    if train_categorical != inference_categorical:
        raise ValueError("Identity categorical columns differ")
    features = pd.concat([train_features, inference_features], ignore_index=True)
    train, inference, names = _attach_features(train, inference, features)
    return train, inference, names, train_categorical, {"features": len(names)}


def _distribution_features(
    group_codes: np.ndarray,
    values: pd.Series,
    prefix: str,
) -> dict[str, np.ndarray]:
    value_codes, _ = pd.factorize(_tokens(values), sort=False)
    helper = pd.DataFrame(
        {
            "group": group_codes,
            "value": value_codes.astype("int32", copy=False),
        }
    )
    group_count = helper.groupby("group", sort=False)["value"].transform("size")
    pair_count = helper.groupby(["group", "value"], sort=False)["value"].transform(
        "size"
    )
    unique_count = helper.groupby("group", sort=False)["value"].transform("nunique")
    mode_count = pair_count.groupby(helper["group"], sort=False).transform("max")
    share = pair_count / group_count

    pair_table = helper.groupby(["group", "value"], sort=False).size().rename("count")
    pair_frame = pair_table.reset_index()
    totals = pair_frame.groupby("group", sort=False)["count"].transform("sum")
    probability = pair_frame["count"] / totals
    pair_frame["entropy_part"] = -probability * np.log(probability)
    entropy = pair_frame.groupby("group", sort=False)["entropy_part"].sum()
    entropy_rows = helper["group"].map(entropy)
    normalized_entropy = entropy_rows / np.log(unique_count.clip(lower=2))
    return {
        f"{prefix}_nunique": unique_count.to_numpy(dtype="int16"),
        f"{prefix}_current_share": share.to_numpy(dtype="float32"),
        f"{prefix}_surprise": (-np.log(share.clip(lower=1e-8))).to_numpy(
            dtype="float32"
        ),
        f"{prefix}_mode_share": (mode_count / group_count).to_numpy(
            dtype="float32"
        ),
        f"{prefix}_is_mode": pair_count.eq(mode_count).to_numpy(dtype="int8"),
        f"{prefix}_entropy": entropy_rows.to_numpy(dtype="float32"),
        f"{prefix}_normalized_entropy": normalized_entropy.to_numpy(
            dtype="float32"
        ),
    }


def _cooccurrence_features(
    combined: pd.DataFrame,
    first: str,
    second: str,
) -> dict[str, np.ndarray]:
    first_codes, _ = pd.factorize(_tokens(combined[first]), sort=False)
    second_codes, _ = pd.factorize(_tokens(combined[second]), sort=False)
    helper = pd.DataFrame(
        {
            "first": first_codes.astype("int32", copy=False),
            "second": second_codes.astype("int32", copy=False),
        }
    )
    first_count = helper.groupby("first", sort=False)["second"].transform("size")
    second_count = helper.groupby("second", sort=False)["first"].transform("size")
    pair_count = helper.groupby(["first", "second"], sort=False)["first"].transform(
        "size"
    )
    first_nunique = helper.groupby("first", sort=False)["second"].transform("nunique")
    second_nunique = helper.groupby("second", sort=False)["first"].transform("nunique")
    prefix = f"cooc_{first}_{second}"
    return {
        f"{prefix}_log_pair_count": np.log1p(pair_count).to_numpy(dtype="float32"),
        f"{prefix}_given_first": (pair_count / first_count).to_numpy(
            dtype="float32"
        ),
        f"{prefix}_given_second": (pair_count / second_count).to_numpy(
            dtype="float32"
        ),
        f"{prefix}_first_nunique": first_nunique.to_numpy(dtype="int32"),
        f"{prefix}_second_nunique": second_nunique.to_numpy(dtype="int32"),
    }


def add_behavior_distribution_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    train_components: np.ndarray,
    inference_components: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    columns = sorted(
        set(BEHAVIOR_COLUMNS)
        | {column for pair in COOCCURRENCE_PAIRS for column in pair}
        | {"uid_card_addr_d1_email"}
    )
    missing = sorted(set(columns).difference(train.columns))
    if missing:
        raise ValueError(f"Missing behavior columns: {missing}")
    combined = pd.concat(
        [train[columns], inference[columns]], ignore_index=True, copy=False
    )
    strict_codes, _ = pd.factorize(
        _tokens(combined["uid_card_addr_d1_email"]), sort=False
    )
    component_codes = np.concatenate([train_components, inference_components])
    group_specs = {
        "component": component_codes.astype("int32", copy=False),
        "strict_uid": strict_codes.astype("int32", copy=False),
    }
    feature_values: dict[str, np.ndarray] = {}
    for group_name, group_codes in group_specs.items():
        print(f"Building behavior distributions for {group_name}...", flush=True)
        for column in BEHAVIOR_COLUMNS:
            feature_values.update(
                _distribution_features(
                    group_codes,
                    combined[column],
                    f"behavior_{group_name}_{column}",
                )
            )
    print("Building categorical co-occurrence features...", flush=True)
    for first, second in COOCCURRENCE_PAIRS:
        feature_values.update(_cooccurrence_features(combined, first, second))
    features = pd.DataFrame(feature_values)
    train, inference, names = _attach_features(train, inference, features)
    return train, inference, names, {
        "features": len(names),
        "behavior_columns": list(BEHAVIOR_COLUMNS),
        "cooccurrence_pairs": [list(pair) for pair in COOCCURRENCE_PAIRS],
    }


Overwriting fraud_next_features.py


In [13]:
%%writefile fraud_overlap_recipe.py
from __future__ import annotations

from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from fraud_features import TARGET


SOURCE_NAMES = (
    "identity_cat",
    "repeat_cat",
    "generic_cat",
    "generic_lgb",
)
SEGMENT_NAMES = ("strict", "partial", "cold")
HORIZONS = (30, 45, 60, 75)


@dataclass(frozen=True)
class FoldSpec:
    name: str
    stage: str
    horizon: int
    train_end_day: int
    valid_start_day: int
    valid_end_day: int

    def to_dict(self) -> dict:
        return asdict(self)


DEV_FOLDS = (
    FoldSpec("dev_h30_a", "dev", 30, 15, 45, 60),
    FoldSpec("dev_h30_b", "dev", 30, 30, 60, 75),
    FoldSpec("dev_h45_a", "dev", 45, 15, 60, 75),
    FoldSpec("dev_h45_b", "dev", 45, 30, 75, 90),
    FoldSpec("dev_h60", "dev", 60, 15, 75, 90),
)

LOCK_FOLDS = (
    FoldSpec("lock_h30", "lock", 30, 60, 90, 106),
    FoldSpec("lock_h45", "lock", 45, 45, 90, 106),
    FoldSpec("lock_h60", "lock", 60, 30, 90, 106),
    FoldSpec("lock_h75", "lock", 75, 15, 90, 106),
)


def read_labeled_split(data_dir: Path, split: str = "train") -> pd.DataFrame:
    transaction = pd.read_csv(data_dir / f"{split}_transaction.csv")
    identity = pd.read_csv(data_dir / f"{split}_identity.csv")
    transaction.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)
    identity.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)
    frame = transaction.merge(identity, on="TransactionID", how="left")
    if TARGET not in frame:
        raise ValueError(f"{split} does not contain {TARGET}")
    return frame


def read_unlabeled_split(data_dir: Path, split: str) -> pd.DataFrame:
    excluded = {TARGET, "Unnamed: 0"}
    transaction = pd.read_csv(
        data_dir / f"{split}_transaction.csv",
        usecols=lambda column: column not in excluded,
    )
    identity = pd.read_csv(
        data_dir / f"{split}_identity.csv",
        usecols=lambda column: column not in excluded,
    )
    frame = transaction.merge(identity, on="TransactionID", how="left")
    if TARGET in frame:
        raise AssertionError(f"Unexpected {TARGET} in unlabeled split")
    return frame


def make_fold_indices(day: pd.Series, spec: FoldSpec) -> tuple[np.ndarray, np.ndarray]:
    values = day.to_numpy()
    train_index = np.flatnonzero(values < spec.train_end_day)
    valid_index = np.flatnonzero(
        (values >= spec.valid_start_day) & (values < spec.valid_end_day)
    )
    if not len(train_index) or not len(valid_index):
        raise ValueError(f"Empty train or validation window for {spec.name}")
    return train_index, valid_index


def uid_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    required = {
        "uid_card_addr_d1_email",
        "uid_card_addr_d1",
        "uid_d1_email",
        "card1",
        "addr1",
        "D1_origin_day",
        "P_emaildomain",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing UID columns: {sorted(missing)}")

    strict_valid = (
        frame["card1"].notna()
        & frame["addr1"].notna()
        & frame["D1_origin_day"].notna()
    )
    d1_email_valid = (
        frame["D1_origin_day"].notna()
        & frame["P_emaildomain"].notna()
    )
    return pd.DataFrame(
        {
            "strict_uid": frame["uid_card_addr_d1_email"].astype("string"),
            "card_d1_uid": frame["uid_card_addr_d1"].astype("string"),
            "d1_email_uid": frame["uid_d1_email"].astype("string"),
            "strict_valid": strict_valid.to_numpy(dtype=bool),
            "card_d1_valid": strict_valid.to_numpy(dtype=bool),
            "d1_email_valid": d1_email_valid.to_numpy(dtype=bool),
        },
        index=frame.index,
    )


def _seen_mask(
    train_values: pd.Series,
    train_valid: pd.Series,
    query_values: pd.Series,
    query_valid: pd.Series,
) -> np.ndarray:
    seen = pd.Index(train_values.loc[train_valid].dropna().unique())
    return query_valid.to_numpy(dtype=bool) & query_values.isin(seen).to_numpy()


def assign_segments(
    metadata: pd.DataFrame,
    train_index: np.ndarray,
    query_index: np.ndarray,
) -> np.ndarray:
    train = metadata.iloc[train_index]
    query = metadata.iloc[query_index]
    strict = _seen_mask(
        train["strict_uid"],
        train["strict_valid"],
        query["strict_uid"],
        query["strict_valid"],
    )
    card_d1 = _seen_mask(
        train["card_d1_uid"],
        train["card_d1_valid"],
        query["card_d1_uid"],
        query["card_d1_valid"],
    )
    d1_email = _seen_mask(
        train["d1_email_uid"],
        train["d1_email_valid"],
        query["d1_email_uid"],
        query["d1_email_valid"],
    )
    result = np.full(len(query), "cold", dtype=object)
    result[~strict & (card_d1 | d1_email)] = "partial"
    result[strict] = "strict"
    return result


def repeated_training_indices(
    metadata: pd.DataFrame,
    train_index: np.ndarray,
    minimum_count: int = 2,
) -> np.ndarray:
    train = metadata.iloc[train_index]
    valid_uid = train.loc[train["strict_valid"], "strict_uid"]
    counts = valid_uid.value_counts(dropna=False)
    repeated = pd.Index(counts.index[counts >= minimum_count])
    keep = train["strict_valid"].to_numpy(dtype=bool) & train[
        "strict_uid"
    ].isin(repeated).to_numpy()
    return train_index[keep]


def rank_prediction(values: Iterable[float]) -> np.ndarray:
    return pd.Series(np.asarray(values)).rank(method="average", pct=True).to_numpy()


def add_source_ranks(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for source in SOURCE_NAMES:
        result[f"{source}_rank"] = rank_prediction(result[source].to_numpy())
    return result


def safe_auc(y_true: Iterable[int], prediction: Iterable[float]) -> float | None:
    y = np.asarray(y_true)
    if len(y) == 0 or np.unique(y).size < 2:
        return None
    return float(roc_auc_score(y, np.asarray(prediction)))


def simplex_weights(n_sources: int, denominator: int = 10) -> Iterable[np.ndarray]:
    for values in product(range(denominator + 1), repeat=n_sources):
        if sum(values) == denominator:
            yield np.asarray(values, dtype="float64") / denominator


def _mean_fold_auc(
    fold_values: list[tuple[np.ndarray, np.ndarray]],
    weights: np.ndarray,
) -> float:
    scores = []
    for y_true, source_values in fold_values:
        score = safe_auc(y_true, source_values @ weights)
        if score is not None:
            scores.append(score)
    return float(np.mean(scores)) if scores else float("-inf")


def search_segment_weights(
    frames: list[pd.DataFrame],
    segment: str,
    denominator: int = 10,
) -> tuple[np.ndarray, float]:
    columns = [f"{source}_rank" for source in SOURCE_NAMES]
    fold_values = []
    for frame in frames:
        mask = frame["segment"].eq(segment).to_numpy()
        if mask.any():
            fold_values.append(
                (
                    frame.loc[mask, TARGET].to_numpy(),
                    frame.loc[mask, columns].to_numpy(),
                )
            )
    best_weights = np.full(len(SOURCE_NAMES), 1 / len(SOURCE_NAMES))
    best_score = float("-inf")
    coarse_denominator = min(5, denominator)
    for weights in simplex_weights(len(SOURCE_NAMES), coarse_denominator):
        score = _mean_fold_auc(fold_values, weights)
        if score > best_score + 1e-12:
            best_score = score
            best_weights = weights
    if denominator > coarse_denominator:
        radius = 2.0 / coarse_denominator
        for weights in simplex_weights(len(SOURCE_NAMES), denominator):
            if np.abs(weights - best_weights).sum() > radius + 1e-12:
                continue
            score = _mean_fold_auc(fold_values, weights)
            if score > best_score + 1e-12:
                best_score = score
                best_weights = weights
    return best_weights, best_score


def learn_horizon_weights(
    dev_frames: list[pd.DataFrame],
    denominator: int = 10,
) -> tuple[dict[str, dict[str, dict[str, float]]], dict]:
    global_weights = {}
    search_report = {"global": {}, "local": {}}
    equal = np.full(len(SOURCE_NAMES), 1 / len(SOURCE_NAMES))
    for segment in SEGMENT_NAMES:
        weights, score = search_segment_weights(dev_frames, segment, denominator)
        if not np.isfinite(score):
            weights = equal.copy()
        global_weights[segment] = weights
        search_report["global"][segment] = {
            "auc": None if not np.isfinite(score) else score,
            "weights": dict(zip(SOURCE_NAMES, weights.tolist())),
        }

    result: dict[str, dict[str, dict[str, float]]] = {}
    for horizon in (30, 45, 60):
        local_frames = [
            frame for frame in dev_frames if int(frame["horizon"].iloc[0]) == horizon
        ]
        result[str(horizon)] = {}
        search_report["local"][str(horizon)] = {}
        for segment in SEGMENT_NAMES:
            local, score = search_segment_weights(
                local_frames, segment, denominator
            )
            if not np.isfinite(score):
                local = global_weights[segment]
            alpha = len(local_frames) / (len(local_frames) + 2.0)
            shrunk = alpha * local + (1.0 - alpha) * global_weights[segment]
            shrunk /= shrunk.sum()
            result[str(horizon)][segment] = dict(
                zip(SOURCE_NAMES, shrunk.tolist())
            )
            search_report["local"][str(horizon)][segment] = {
                "auc": None if not np.isfinite(score) else score,
                "folds": len(local_frames),
                "shrinkage": alpha,
                "raw_weights": dict(zip(SOURCE_NAMES, local.tolist())),
                "weights": dict(zip(SOURCE_NAMES, shrunk.tolist())),
            }

    result["75"] = {
        segment: dict(result["60"][segment]) for segment in SEGMENT_NAMES
    }
    search_report["local"]["75"] = {
        segment: {
            "auc": None,
            "folds": 0,
            "shrinkage": 0.0,
            "source": "h60",
            "weights": dict(result["75"][segment]),
        }
        for segment in SEGMENT_NAMES
    }
    return result, search_report


def blend_with_weights(
    frame: pd.DataFrame,
    weights: dict[str, dict[str, float]],
) -> np.ndarray:
    prediction = np.zeros(len(frame), dtype="float64")
    for segment in SEGMENT_NAMES:
        mask = frame["segment"].eq(segment).to_numpy()
        if not mask.any():
            continue
        columns = [f"{source}_rank" for source in SOURCE_NAMES]
        vector = np.asarray([weights[segment][source] for source in SOURCE_NAMES])
        prediction[mask] = frame.loc[mask, columns].to_numpy() @ vector
    return prediction


def interpolated_blend(
    frame: pd.DataFrame,
    horizon_weights: dict[str, dict[str, dict[str, float]]],
) -> np.ndarray:
    horizons = np.asarray(HORIZONS, dtype="float64")
    prediction = np.zeros(len(frame), dtype="float64")
    source_values = frame[
        [f"{source}_rank" for source in SOURCE_NAMES]
    ].to_numpy()
    row_horizon = frame["forecast_horizon"].to_numpy(dtype="float64")
    for segment in SEGMENT_NAMES:
        mask = frame["segment"].eq(segment).to_numpy()
        if not mask.any():
            continue
        row_weights = np.column_stack(
            [
                np.interp(
                    row_horizon[mask],
                    horizons,
                    [
                        horizon_weights[str(h)][segment][source]
                        for h in HORIZONS
                    ],
                )
                for source in SOURCE_NAMES
            ]
        )
        row_weights /= row_weights.sum(axis=1, keepdims=True)
        prediction[mask] = np.sum(source_values[mask] * row_weights, axis=1)
    return prediction


def uid_aggregate(
    prediction: np.ndarray,
    uid: pd.Series,
    valid: pd.Series | np.ndarray,
    method: str,
) -> np.ndarray:
    if method == "none":
        return np.asarray(prediction, dtype="float64").copy()
    valid_array = np.asarray(valid, dtype=bool)
    result = np.asarray(prediction, dtype="float64").copy()
    values = pd.DataFrame(
        {
            "prediction": result[valid_array],
            "uid": uid.iloc[np.flatnonzero(valid_array)].to_numpy(),
        }
    )
    grouped = values.groupby("uid", sort=False, dropna=False)["prediction"]
    if method == "mean":
        aggregate_by_uid = grouped.mean()
    elif method == "max":
        aggregate_by_uid = grouped.max()
    elif method == "q75":
        aggregate_by_uid = grouped.quantile(0.75)
    elif method == "q90":
        aggregate_by_uid = grouped.quantile(0.90)
    else:
        raise ValueError(f"Unknown UID aggregate: {method}")
    result[valid_array] = values["uid"].map(aggregate_by_uid).to_numpy()
    return result


def apply_uid_postprocess(
    prediction: np.ndarray,
    uid: pd.Series,
    valid: pd.Series | np.ndarray,
    method: str,
    weight: float,
) -> np.ndarray:
    if method == "none" or weight == 0:
        return np.asarray(prediction, dtype="float64").copy()
    aggregate = uid_aggregate(prediction, uid, valid, method)
    return (1.0 - weight) * np.asarray(prediction) + weight * aggregate


def choose_uid_postprocess(
    dev_frames: list[pd.DataFrame],
) -> tuple[dict, list[dict]]:
    candidates = [("none", 0.0)]
    for method in ("mean", "q75", "q90", "max"):
        for weight in (0.25, 0.50, 0.75, 1.0):
            candidates.append((method, weight))

    aggregates = []
    for frame in dev_frames:
        fold_aggregates = {"none": frame["gated"].to_numpy()}
        for method in ("mean", "q75", "q90", "max"):
            fold_aggregates[method] = uid_aggregate(
                frame["gated"].to_numpy(),
                frame["strict_uid"],
                frame["strict_valid"],
                method,
            )
        aggregates.append(fold_aggregates)

    rows = []
    for method, weight in candidates:
        fold_scores = []
        for frame, fold_aggregates in zip(dev_frames, aggregates):
            base = frame["gated"].to_numpy()
            prediction = (
                (1.0 - weight) * base + weight * fold_aggregates[method]
            )
            score = safe_auc(frame[TARGET], prediction)
            if score is not None:
                fold_scores.append(score)
        rows.append(
            {
                "method": method,
                "weight": weight,
                "mean_auc": float(np.mean(fold_scores)),
                "fold_auc": fold_scores,
            }
        )
    best = max(rows, key=lambda row: (row["mean_auc"], -row["weight"]))
    return dict(best), rows


def evaluate_prediction(
    frame: pd.DataFrame,
    prediction: np.ndarray,
) -> dict:
    result = {"overall": safe_auc(frame[TARGET], prediction), "segments": {}}
    for segment in SEGMENT_NAMES:
        mask = frame["segment"].eq(segment).to_numpy()
        result["segments"][segment] = {
            "rows": int(mask.sum()),
            "rate": float(mask.mean()),
            "auc": safe_auc(frame.loc[mask, TARGET], prediction[mask]),
        }
    return result


def self_test() -> None:
    frame = pd.DataFrame(
        {
            "uid_card_addr_d1_email": ["a", "a", "b", "c", "a", "x"],
            "uid_card_addr_d1": ["aa", "aa", "bb", "cc", "aa", "xx"],
            "uid_d1_email": ["da", "da", "db", "dc", "da", "dx"],
            "card1": [1, 1, 2, 3, 1, np.nan],
            "addr1": [1, 1, 2, 3, 1, np.nan],
            "D1_origin_day": [1, 1, 2, 3, 1, np.nan],
            "P_emaildomain": ["e", "e", "e", "e", "e", None],
        }
    )
    metadata = uid_metadata(frame)
    train_index = np.asarray([0, 1, 2])
    query_index = np.asarray([3, 4, 5])
    assert assign_segments(metadata, train_index, query_index).tolist() == [
        "cold",
        "strict",
        "cold",
    ]
    assert repeated_training_indices(metadata, train_index).tolist() == [0, 1]
    prediction = np.asarray([0.1, 0.9, 0.4])
    grouped = uid_aggregate(
        prediction,
        pd.Series(["u", "u", "v"]),
        np.asarray([True, True, True]),
        "q90",
    )
    assert np.allclose(grouped, [0.82, 0.82, 0.4])
    weights = list(simplex_weights(4, 10))
    assert len(weights) == 286
    assert all(np.isclose(value.sum(), 1.0) for value in weights)


Overwriting fraud_overlap_recipe.py


In [14]:
%%writefile fraud_temporal_validation.py
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class PurgedFold:
    number: int
    train_end_day: float
    validation_start_day: float
    validation_end_day: float
    train_index: np.ndarray
    validation_index: np.ndarray


FOLD_WINDOWS = (
    (15.0, 45.0, 60.0),
    (30.0, 60.0, 75.0),
    (45.0, 75.0, 90.0),
    (60.0, 90.0, 106.0),
)


def make_four_long_gap_folds(
    transaction_dt: pd.Series,
) -> list[PurgedFold]:
    day = transaction_dt.to_numpy(dtype="float64") / 86400.0
    folds = []
    for number, (train_end, validation_start, validation_end) in enumerate(
        FOLD_WINDOWS
    ):
        train_index = np.flatnonzero(day < train_end)
        validation_index = np.flatnonzero(
            (day >= validation_start) & (day < validation_end)
        )
        if len(train_index) == 0 or len(validation_index) == 0:
            raise ValueError(f"Temporal fold {number} is empty")
        folds.append(
            PurgedFold(
                number=number,
                train_end_day=train_end,
                validation_start_day=validation_start,
                validation_end_day=validation_end,
                train_index=train_index,
                validation_index=validation_index,
            )
        )
    return folds


def fold_metadata(
    folds: list[PurgedFold],
    y: pd.Series,
) -> list[dict]:
    target = y.to_numpy(dtype="int8")
    rows = []
    for fold in folds:
        rows.append(
            {
                "fold": fold.number,
                "train_end_day": fold.train_end_day,
                "validation_start_day": fold.validation_start_day,
                "validation_end_day": fold.validation_end_day,
                "embargo_days": (
                    fold.validation_start_day - fold.train_end_day
                ),
                "train_rows": int(len(fold.train_index)),
                "validation_rows": int(len(fold.validation_index)),
                "train_fraud_rate": float(target[fold.train_index].mean()),
                "validation_fraud_rate": float(
                    target[fold.validation_index].mean()
                ),
            }
        )
    return rows


Overwriting fraud_temporal_validation.py


In [15]:
%%writefile fraud_transaction_chain_features.py
from __future__ import annotations

import numpy as np
import pandas as pd


CHAIN_COLUMN = "v307_transaction_chain"
CHAIN_TARGET_COLUMNS = (
    "v307_chain_known_txn_count",
    "v307_chain_known_fraud_count",
    "v307_chain_known_fraud_rate_smoothed_5",
    "v307_chain_known_fraud_rate_smoothed_20",
    "v307_chain_known_any_fraud",
    "v307_chain_seconds_since_known_fraud",
)


def _build_chain_ids(
    sequence: pd.DataFrame,
    max_chain_size: int | None = None,
) -> tuple[np.ndarray, dict]:
    uid_codes, _ = pd.factorize(
        sequence["uid_card_addr_d1_email"],
        sort=False,
    )
    transaction_dt = sequence["TransactionDT"].to_numpy()
    transaction_id = sequence["TransactionID"].to_numpy()
    v307 = sequence["V307"].to_numpy(dtype="float64", copy=False)
    amount = sequence["TransactionAmt"].to_numpy(dtype="float64", copy=False)
    valid = np.isfinite(v307) & np.isfinite(amount)
    order = np.lexsort((transaction_id, transaction_dt, uid_codes))

    chain_ids = np.empty(len(sequence), dtype="int32")
    chain_sizes = np.zeros(len(sequence), dtype="int32")
    next_chain_id = 0
    current_uid = None
    endpoints: dict[int, tuple[int, int]] = {}
    linked_rows = 0
    capped_candidates = 0
    for sequence_position, row in enumerate(order):
        uid = int(uid_codes[row])
        if uid != current_uid:
            endpoints = {}
            current_uid = uid

        chain_id = -1
        if valid[row]:
            start = int(np.rint(v307[row] * 1000.0))
            candidates = [
                endpoints[key]
                for key in (start - 1, start, start + 1)
                if key in endpoints
            ]
            if max_chain_size is not None and candidates:
                uncapped_candidates = candidates
                candidates = [
                    candidate
                    for candidate in candidates
                    if chain_sizes[candidate[0]] < max_chain_size
                ]
                capped_candidates += int(
                    bool(uncapped_candidates) and not candidates
                )
            if candidates:
                chain_id, _ = max(candidates, key=lambda value: value[1])
                linked_rows += 1

        if chain_id < 0:
            chain_id = next_chain_id
            next_chain_id += 1
        chain_ids[row] = chain_id
        chain_sizes[chain_id] += 1

        if valid[row]:
            endpoint = int(np.rint((v307[row] + amount[row]) * 1000.0))
            endpoints[endpoint] = (chain_id, sequence_position)

    counts = np.bincount(chain_ids)
    return chain_ids, {
        "chains": int(len(counts)),
        "linked_rows": int(linked_rows),
        "multirow_chains": int(np.sum(counts > 1)),
        "rows_in_multirow_chains": int(counts[counts > 1].sum()),
        "max_chain_size": int(counts.max()),
        "chain_size_cap": max_chain_size,
        "capped_link_candidates": int(capped_candidates),
    }


def _target_free_features(
    sequence: pd.DataFrame,
    chain_ids: np.ndarray,
    train_rows: int,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    ordered = sequence.copy()
    ordered["_chain"] = chain_ids
    ordered["_row_order"] = np.arange(len(ordered), dtype="int32")
    ordered = ordered.sort_values(
        ["TransactionDT", "TransactionID"],
        kind="stable",
    ).reset_index(drop=True)
    group = ordered.groupby("_chain", sort=False, observed=True)
    count = group["TransactionID"].transform("size").astype("int32")
    position = group.cumcount().astype("int32")
    first_dt = group["TransactionDT"].transform("min")
    last_dt = group["TransactionDT"].transform("max")

    feature_names = [
        CHAIN_COLUMN,
        "v307_chain_total_count",
        "v307_chain_position",
        "v307_chain_position_from_end",
        "v307_chain_previous_dt",
        "v307_chain_next_dt",
        "v307_chain_time_span",
        "v307_chain_previous_amount",
        "v307_chain_next_amount",
        "v307_chain_has_previous",
        "v307_chain_has_next",
    ]
    features = pd.DataFrame(
        {
            CHAIN_COLUMN: (
                "vc" + ordered["_chain"].astype("string")
            ),
            "v307_chain_total_count": count,
            "v307_chain_position": position,
            "v307_chain_position_from_end": count - position - 1,
            "v307_chain_previous_dt": group["TransactionDT"].diff().astype(
                "float32"
            ),
            "v307_chain_next_dt": (
                group["TransactionDT"].shift(-1) - ordered["TransactionDT"]
            ).astype("float32"),
            "v307_chain_time_span": (last_dt - first_dt).astype("float32"),
            "v307_chain_previous_amount": group["TransactionAmt"].shift(1).astype(
                "float32"
            ),
            "v307_chain_next_amount": group["TransactionAmt"].shift(-1).astype(
                "float32"
            ),
            "v307_chain_has_previous": position.gt(0).astype("int8"),
            "v307_chain_has_next": position.lt(count - 1).astype("int8"),
            "_row_order": ordered["_row_order"],
        }
    )
    features = features.sort_values("_row_order", kind="stable").drop(
        columns="_row_order"
    )
    features.reset_index(drop=True, inplace=True)
    return features, feature_names, [CHAIN_COLUMN]


def _causal_target_features(
    sequence: pd.DataFrame,
    chain_ids: np.ndarray,
    y: pd.Series,
    train_rows: int,
) -> pd.DataFrame:
    global_rate = float(np.asarray(y).mean())
    history = pd.DataFrame(
        {
            "_chain": chain_ids[:train_rows],
            "_dt": sequence["TransactionDT"].to_numpy()[:train_rows],
            "_target": np.asarray(y, dtype="int8"),
        }
    )
    buckets = (
        history.groupby(["_chain", "_dt"], sort=False, observed=True)
        .agg(
            _bucket_txn_count=("_target", "size"),
            _bucket_fraud_count=("_target", "sum"),
        )
        .reset_index()
        .sort_values(["_chain", "_dt"], kind="stable")
    )
    group = buckets.groupby("_chain", sort=False, observed=True)
    buckets["_known_txn"] = (
        group["_bucket_txn_count"].cumsum() - buckets["_bucket_txn_count"]
    ).astype("int32")
    buckets["_known_fraud"] = (
        group["_bucket_fraud_count"].cumsum()
        - buckets["_bucket_fraud_count"]
    ).astype("int32")
    buckets["_fraud_dt"] = buckets["_dt"].where(
        buckets["_bucket_fraud_count"].gt(0)
    )
    buckets["_last_fraud"] = group["_fraud_dt"].ffill()
    buckets["_known_last_fraud"] = buckets.groupby(
        "_chain", sort=False, observed=True
    )["_last_fraud"].shift(1)
    train_history = history.merge(
        buckets[
            [
                "_chain",
                "_dt",
                "_known_txn",
                "_known_fraud",
                "_known_last_fraud",
            ]
        ],
        on=["_chain", "_dt"],
        how="left",
        sort=False,
        validate="many_to_one",
    )

    totals = (
        history.groupby("_chain", sort=False, observed=True)
        .agg(
            _known_txn=("_target", "size"),
            _known_fraud=("_target", "sum"),
        )
        .reset_index()
    )
    last_fraud = history.loc[history["_target"].eq(1)].groupby(
        "_chain", sort=False, observed=True
    )["_dt"].max()
    inference_history = pd.DataFrame(
        {
            "_chain": chain_ids[train_rows:],
            "_dt": sequence["TransactionDT"].to_numpy()[train_rows:],
        }
    ).merge(
        totals,
        on="_chain",
        how="left",
        sort=False,
        validate="many_to_one",
    )
    inference_history["_known_txn"] = inference_history["_known_txn"].fillna(0)
    inference_history["_known_fraud"] = inference_history["_known_fraud"].fillna(0)
    inference_history["_known_last_fraud"] = inference_history["_chain"].map(
        last_fraud
    )

    def finish(frame: pd.DataFrame) -> pd.DataFrame:
        known_txn = frame["_known_txn"].astype("int32")
        known_fraud = frame["_known_fraud"].astype("int32")
        return pd.DataFrame(
            {
                "v307_chain_known_txn_count": known_txn,
                "v307_chain_known_fraud_count": known_fraud,
                "v307_chain_known_fraud_rate_smoothed_5": (
                    (known_fraud + 5.0 * global_rate) / (known_txn + 5.0)
                ).astype("float32"),
                "v307_chain_known_fraud_rate_smoothed_20": (
                    (known_fraud + 20.0 * global_rate) / (known_txn + 20.0)
                ).astype("float32"),
                "v307_chain_known_any_fraud": known_fraud.gt(0).astype("int8"),
                "v307_chain_seconds_since_known_fraud": (
                    frame["_dt"] - frame["_known_last_fraud"]
                ).astype("float32"),
            }
        )[list(CHAIN_TARGET_COLUMNS)]

    return pd.concat(
        [finish(train_history), finish(inference_history)],
        ignore_index=True,
    )


def add_v307_transaction_chain_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    y: pd.Series,
    include_target_history: bool = False,
    max_chain_size: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], dict]:
    required = {
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "V307",
        "uid_card_addr_d1_email",
    }
    missing = sorted(required.difference(train.columns))
    if missing:
        raise ValueError(f"Missing V307 chain columns: {missing}")

    source_columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "V307",
        "uid_card_addr_d1_email",
    ]
    sequence = pd.concat(
        [train[source_columns], inference[source_columns]],
        ignore_index=True,
        copy=False,
    )
    train_rows = len(train)
    chain_ids, stats = _build_chain_ids(
        sequence,
        max_chain_size=max_chain_size,
    )
    features, feature_names, categorical = _target_free_features(
        sequence,
        chain_ids,
        train_rows,
    )
    if include_target_history:
        target_features = _causal_target_features(
            sequence,
            chain_ids,
            y,
            train_rows,
        )
        features = pd.concat([features, target_features], axis=1)
        feature_names.extend(CHAIN_TARGET_COLUMNS)

    train_features = features.iloc[:train_rows].reset_index(drop=True)
    inference_features = features.iloc[train_rows:].reset_index(drop=True)
    reference_counts = np.bincount(
        chain_ids[:train_rows],
        minlength=int(chain_ids.max()) + 1,
    )
    linked_inference = reference_counts[chain_ids[train_rows:]] > 0
    stats.update(
        {
            "rows": int(len(sequence)),
            "inference_rows_linked_to_reference": int(linked_inference.sum()),
            "inference_link_rate": float(linked_inference.mean()),
            "target_history_features": bool(include_target_history),
        }
    )
    return (
        pd.concat([train.reset_index(drop=True), train_features], axis=1),
        pd.concat(
            [inference.reset_index(drop=True), inference_features],
            axis=1,
        ),
        feature_names,
        categorical,
        stats,
    )


Overwriting fraud_transaction_chain_features.py


In [16]:
%%writefile fraud_user_features.py
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd


RAW_UID_COLUMNS = {
    "uid_card_addr",
    "uid_card_email",
    "uid_card_full",
    "uid_card_device",
    "uid_email_pair",
    "uid_d1_email",
    "uid_card_addr_d1",
    "uid_card_addr_d1_email",
}


@dataclass(frozen=True)
class GraphKey:
    name: str
    columns: tuple[str, ...]
    max_group_size: int


GRAPH_KEYS = (
    GraphKey(
        "card_addr_origin",
        ("card1", "addr1", "D1_origin_day"),
        200,
    ),
    GraphKey(
        "card_origin_email",
        ("card1", "D1_origin_day", "P_emaildomain"),
        200,
    ),
    GraphKey(
        "card_full_origin",
        ("card1", "card2", "card3", "card5", "D1_origin_day"),
        150,
    ),
    GraphKey(
        "card_addr_origin_device",
        ("card1", "addr1", "D1_origin_day", "DeviceInfo"),
        100,
    ),
)

PROFILE_NUMERIC_COLUMNS = (
    "dist1",
    "dist2",
    "D1",
    "D3",
    "C1",
    "C2",
    "C13",
    "C14",
    "V279",
    "V280",
    "V306",
    "V307",
    "V308",
)

PROFILE_ID_COLUMNS = (
    "card1",
    "card2",
    "card5",
    "addr1",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceInfo",
    "ProductCD",
)

CAUSAL_HISTORY_COLUMNS = (
    "user_known_txn_count",
    "user_known_fraud_count",
    "user_known_nonfraud_count",
    "user_known_fraud_rate",
    "user_known_fraud_rate_smoothed",
    "user_known_any_fraud",
    "user_seconds_since_known_fraud",
)


def is_raw_uid_feature(column: str) -> bool:
    """Return True only for direct identifiers, not their numeric aggregates."""
    return column in RAW_UID_COLUMNS


def _valid_key_value(series: pd.Series) -> pd.Series:
    valid = series.notna()
    if isinstance(series.dtype, pd.StringDtype) or pd.api.types.is_object_dtype(
        series.dtype
    ):
        valid &= series.astype("string").ne("<MISSING>")
    return valid


def _normalized_key_frame(frame: pd.DataFrame, key: GraphKey) -> pd.DataFrame:
    values = frame.loc[:, key.columns].copy()
    if "D1_origin_day" in values:
        values["D1_origin_day"] = values["D1_origin_day"].round()
    return values


def _find_root(parent: np.ndarray, node: int) -> int:
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = int(parent[node])
    return node


def _union_nodes(
    parent: np.ndarray,
    component_size: np.ndarray,
    first: int,
    second: int,
    max_component_size: int,
) -> bool:
    first_root = _find_root(parent, first)
    second_root = _find_root(parent, second)
    if first_root == second_root:
        return False
    if (
        int(component_size[first_root]) + int(component_size[second_root])
        > max_component_size
    ):
        return False
    if component_size[first_root] < component_size[second_root]:
        first_root, second_root = second_root, first_root
    parent[second_root] = first_root
    component_size[first_root] += component_size[second_root]
    return True


def build_user_components(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    max_component_size: int = 500,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict]:
    """Join strict UID variants into graph components without exposing their IDs."""
    required = {column for key in GRAPH_KEYS for column in key.columns}
    missing = sorted(required.difference(train.columns))
    if missing:
        raise ValueError(f"Cannot build user graph; missing columns: {missing}")

    combined = pd.concat(
        [train[list(required)], inference[list(required)]],
        ignore_index=True,
        copy=False,
    )
    row_count = len(combined)
    parent = np.arange(row_count, dtype="int32")
    component_size = np.ones(row_count, dtype="int32")
    key_support = np.zeros(row_count, dtype="int8")
    key_metrics = []

    for key in GRAPH_KEYS:
        key_values = _normalized_key_frame(combined, key)
        valid = np.ones(row_count, dtype=bool)
        for column in key.columns:
            valid &= _valid_key_value(key_values[column]).to_numpy()
        valid_rows = np.flatnonzero(valid).astype("int32", copy=False)
        key_support[valid_rows] += 1

        if len(valid_rows) == 0:
            key_metrics.append(
                {
                    "key": key.name,
                    "valid_rows": 0,
                    "linked_groups": 0,
                    "oversized_groups": 0,
                    "union_edges": 0,
                }
            )
            continue

        hashes = pd.util.hash_pandas_object(
            key_values.loc[valid, list(key.columns)],
            index=False,
            categorize=True,
        ).to_numpy(dtype="uint64", copy=False)
        order = np.argsort(hashes, kind="stable")
        sorted_hashes = hashes[order]
        boundaries = np.flatnonzero(
            np.r_[True, sorted_hashes[1:] != sorted_hashes[:-1], True]
        )

        linked_groups = 0
        oversized_groups = 0
        union_edges = 0
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            group_size = int(end - start)
            if group_size < 2:
                continue
            if group_size > key.max_group_size:
                oversized_groups += 1
                continue
            members = valid_rows[order[start:end]]
            anchor = int(members[0])
            linked_groups += 1
            for member in members[1:]:
                union_edges += int(
                    _union_nodes(
                        parent,
                        component_size,
                        anchor,
                        int(member),
                        max_component_size,
                    )
                )

        key_metrics.append(
            {
                "key": key.name,
                "valid_rows": int(len(valid_rows)),
                "linked_groups": linked_groups,
                "oversized_groups": oversized_groups,
                "union_edges": union_edges,
            }
        )

    roots = np.empty(row_count, dtype="int32")
    for row in range(row_count):
        roots[row] = _find_root(parent, row)
    components, _ = pd.factorize(roots, sort=False)
    components = components.astype("int32", copy=False)
    component_counts = np.bincount(components)
    train_rows = len(train)
    stats = {
        "rows": row_count,
        "components": int(len(component_counts)),
        "singleton_components": int(np.sum(component_counts == 1)),
        "multirow_components": int(np.sum(component_counts > 1)),
        "rows_in_multirow_components": int(
            component_counts[component_counts > 1].sum()
        ),
        "max_component_size": int(component_counts.max()),
        "keys": key_metrics,
    }
    return (
        components[:train_rows],
        components[train_rows:],
        key_support,
        stats,
    )


def _profile_aggregates(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    train_components: np.ndarray,
    inference_components: np.ndarray,
    key_support: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    source_columns = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        *[column for column in PROFILE_NUMERIC_COLUMNS if column in train],
        *[column for column in PROFILE_ID_COLUMNS if column in train],
    ]
    source_columns = list(dict.fromkeys(source_columns))
    sequence = pd.concat(
        [train[source_columns], inference[source_columns]],
        ignore_index=True,
        copy=False,
    )
    sequence["_component"] = np.concatenate(
        [train_components, inference_components]
    )
    sequence["_row_order"] = np.arange(len(sequence), dtype="int32")
    sequence = sequence.sort_values(
        ["TransactionDT", "TransactionID"],
        kind="stable",
    ).reset_index(drop=True)
    group = sequence.groupby("_component", sort=False, observed=True)

    feature_values: dict[str, pd.Series | np.ndarray] = {}
    count = group["TransactionID"].transform("size").astype("int32")
    order = group.cumcount().astype("int32")
    feature_values["user_txn_count"] = count
    feature_values["user_order"] = order
    feature_values["user_order_from_end"] = (count - order - 1).astype("int32")
    feature_values["user_order_fraction"] = (
        order / (count - 1).replace(0, np.nan)
    ).astype("float32")

    previous_dt = group["TransactionDT"].diff()
    next_dt = group["TransactionDT"].shift(-1) - sequence["TransactionDT"]
    first_dt = group["TransactionDT"].transform("min")
    last_dt = group["TransactionDT"].transform("max")
    feature_values["user_previous_dt"] = previous_dt.astype("float32")
    feature_values["user_next_dt"] = next_dt.astype("float32")
    feature_values["user_time_span"] = (last_dt - first_dt).astype("float32")
    feature_values["user_time_since_first"] = (
        sequence["TransactionDT"] - first_dt
    ).astype("float32")
    feature_values["user_time_to_last"] = (
        last_dt - sequence["TransactionDT"]
    ).astype("float32")

    sequence["_previous_dt"] = previous_dt
    gap_group = sequence.groupby(
        "_component", sort=False, observed=True
    )["_previous_dt"]
    feature_values["user_mean_dt"] = gap_group.transform("mean").astype(
        "float32"
    )
    feature_values["user_std_dt"] = gap_group.transform("std").astype(
        "float32"
    )
    feature_values["user_median_dt"] = gap_group.transform("median").astype(
        "float32"
    )

    amount = sequence["TransactionAmt"]
    amount_mean = group["TransactionAmt"].transform("mean")
    amount_std = group["TransactionAmt"].transform("std")
    feature_values["user_amount_mean"] = amount_mean.astype("float32")
    feature_values["user_amount_std"] = amount_std.astype("float32")
    feature_values["user_amount_median"] = group["TransactionAmt"].transform(
        "median"
    ).astype("float32")
    feature_values["user_amount_min"] = group["TransactionAmt"].transform(
        "min"
    ).astype("float32")
    feature_values["user_amount_max"] = group["TransactionAmt"].transform(
        "max"
    ).astype("float32")
    feature_values["user_previous_amount"] = group["TransactionAmt"].shift(
        1
    ).astype("float32")
    feature_values["user_next_amount"] = group["TransactionAmt"].shift(-1).astype(
        "float32"
    )
    feature_values["user_amount_to_mean"] = (
        amount / amount_mean.replace(0, np.nan)
    ).astype("float32")
    feature_values["user_amount_zscore"] = (
        (amount - amount_mean) / amount_std.replace(0, np.nan)
    ).astype("float32")

    amount_pair = sequence.groupby(
        ["_component", "TransactionAmt"],
        sort=False,
        observed=True,
        dropna=False,
    )["TransactionID"].transform("size")
    feature_values["user_same_amount_count"] = amount_pair.astype("int32")
    feature_values["user_same_amount_fraction"] = (amount_pair / count).astype(
        "float32"
    )

    day = (sequence["TransactionDT"] // 86400).astype("int32")
    day_frame = pd.DataFrame(
        {"component": sequence["_component"], "day": day}
    )
    feature_values["user_active_day_count"] = day_frame.groupby(
        "component", sort=False, observed=True
    )["day"].transform("nunique").astype("int16")

    for column in PROFILE_ID_COLUMNS:
        if column not in sequence:
            continue
        value_frame = pd.DataFrame(
            {
                "component": sequence["_component"],
                "value": sequence[column],
            }
        )
        unique_count = value_frame.groupby(
            "component", sort=False, observed=True
        )["value"].transform("nunique")
        same_count = value_frame.groupby(
            ["component", "value"],
            sort=False,
            observed=True,
            dropna=False,
        )["value"].transform("size")
        prefix = f"user_{column}"
        feature_values[f"{prefix}_nunique"] = unique_count.astype("int16")
        feature_values[f"{prefix}_same_fraction"] = (same_count / count).astype(
            "float32"
        )

    numeric_columns = [
        column for column in PROFILE_NUMERIC_COLUMNS if column in sequence
    ]
    for column in numeric_columns:
        column_group = group[column]
        prefix = f"user_{column}"
        feature_values[f"{prefix}_mean"] = column_group.transform("mean").astype(
            "float32"
        )
        feature_values[f"{prefix}_std"] = column_group.transform("std").astype(
            "float32"
        )
        if column in {"V279", "V280", "V306", "V307", "V308"}:
            feature_values[f"{prefix}_min"] = column_group.transform("min").astype(
                "float32"
            )
            feature_values[f"{prefix}_max"] = column_group.transform("max").astype(
                "float32"
            )

    sorted_support = key_support[sequence["_row_order"].to_numpy()]
    feature_values["user_valid_link_key_count"] = sorted_support.astype("int8")
    feature_values["user_is_multirow"] = count.gt(1).astype("int8")

    features = pd.DataFrame(feature_values)
    features["_row_order"] = sequence["_row_order"].to_numpy()
    features = features.sort_values("_row_order", kind="stable").drop(
        columns="_row_order"
    )
    features.reset_index(drop=True, inplace=True)
    train_rows = len(train)
    return (
        features.iloc[:train_rows].copy(),
        features.iloc[train_rows:].reset_index(drop=True).copy(),
        features.columns.tolist(),
    )


def _strictly_causal_history(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    y: pd.Series,
    train_components: np.ndarray,
    inference_components: np.ndarray,
    smoothing: float,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    global_rate = float(np.asarray(y).mean())
    history = pd.DataFrame(
        {
            "_component": train_components,
            "_dt": train["TransactionDT"].to_numpy(),
            "_target": np.asarray(y, dtype="int8"),
        }
    )

    buckets = (
        history.groupby(["_component", "_dt"], sort=False, observed=True)
        .agg(
            _bucket_txn_count=("_target", "size"),
            _bucket_fraud_count=("_target", "sum"),
        )
        .reset_index()
        .sort_values(["_component", "_dt"], kind="stable")
    )
    bucket_group = buckets.groupby("_component", sort=False, observed=True)
    buckets["user_known_txn_count"] = (
        bucket_group["_bucket_txn_count"].cumsum()
        - buckets["_bucket_txn_count"]
    ).astype("int32")
    buckets["user_known_fraud_count"] = (
        bucket_group["_bucket_fraud_count"].cumsum()
        - buckets["_bucket_fraud_count"]
    ).astype("int32")
    buckets["_fraud_dt"] = buckets["_dt"].where(
        buckets["_bucket_fraud_count"].gt(0)
    )
    buckets["_last_fraud_including_bucket"] = bucket_group["_fraud_dt"].ffill()
    buckets["_last_known_fraud_dt"] = buckets.groupby(
        "_component", sort=False, observed=True
    )["_last_fraud_including_bucket"].shift(1)

    history_columns = [
        "user_known_txn_count",
        "user_known_fraud_count",
        "_last_known_fraud_dt",
    ]
    train_history = history.merge(
        buckets[["_component", "_dt", *history_columns]],
        on=["_component", "_dt"],
        how="left",
        sort=False,
        validate="many_to_one",
    )

    component_totals = (
        history.groupby("_component", sort=False, observed=True)
        .agg(
            user_known_txn_count=("_target", "size"),
            user_known_fraud_count=("_target", "sum"),
        )
        .reset_index()
    )
    fraud_rows = history.loc[history["_target"].eq(1)]
    last_fraud = fraud_rows.groupby(
        "_component", sort=False, observed=True
    )["_dt"].max()

    inference_history = pd.DataFrame(
        {
            "_component": inference_components,
            "_dt": inference["TransactionDT"].to_numpy(),
        }
    ).merge(
        component_totals,
        on="_component",
        how="left",
        sort=False,
        validate="many_to_one",
    )
    inference_history["_last_known_fraud_dt"] = inference_history[
        "_component"
    ].map(last_fraud)
    inference_history["user_known_txn_count"] = inference_history[
        "user_known_txn_count"
    ].fillna(0).astype("int32")
    inference_history["user_known_fraud_count"] = inference_history[
        "user_known_fraud_count"
    ].fillna(0).astype("int32")

    feature_names = list(CAUSAL_HISTORY_COLUMNS)

    def finish(frame: pd.DataFrame) -> pd.DataFrame:
        result = frame[
            ["user_known_txn_count", "user_known_fraud_count"]
        ].copy()
        result["user_known_nonfraud_count"] = (
            result["user_known_txn_count"]
            - result["user_known_fraud_count"]
        ).astype("int32")
        result["user_known_fraud_rate"] = (
            result["user_known_fraud_count"]
            / result["user_known_txn_count"].replace(0, np.nan)
        ).astype("float32")
        result["user_known_fraud_rate_smoothed"] = (
            (result["user_known_fraud_count"] + smoothing * global_rate)
            / (result["user_known_txn_count"] + smoothing)
        ).astype("float32")
        result["user_known_any_fraud"] = result[
            "user_known_fraud_count"
        ].gt(0).astype("int8")
        result["user_seconds_since_known_fraud"] = (
            frame["_dt"] - frame["_last_known_fraud_dt"]
        ).astype("float32")
        return result[feature_names]

    return finish(train_history), finish(inference_history), feature_names


def add_user_profile_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    y: pd.Series,
    smoothing: float = 20.0,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    list[str],
    np.ndarray,
    np.ndarray,
    dict,
]:
    """Add graph-user aggregates and causal history, but never the UID itself."""
    (
        train_components,
        inference_components,
        key_support,
        graph_stats,
    ) = build_user_components(train, inference)
    train_profile, inference_profile, profile_names = _profile_aggregates(
        train,
        inference,
        train_components,
        inference_components,
        key_support,
    )
    train_history, inference_history, history_names = _strictly_causal_history(
        train,
        inference,
        y,
        train_components,
        inference_components,
        smoothing,
    )

    train_result = pd.concat(
        [
            train.reset_index(drop=True),
            train_profile,
            train_history.reset_index(drop=True),
        ],
        axis=1,
    )
    inference_result = pd.concat(
        [
            inference.reset_index(drop=True),
            inference_profile,
            inference_history.reset_index(drop=True),
        ],
        axis=1,
    )
    graph_stats["profile_features"] = len(profile_names)
    graph_stats["causal_history_features"] = len(history_names)
    graph_stats["history_smoothing"] = smoothing
    graph_stats["train_target_rate"] = float(np.asarray(y).mean())
    return (
        train_result,
        inference_result,
        [*profile_names, *history_names],
        train_components,
        inference_components,
        graph_stats,
    )


def add_target_free_user_profile_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    list[str],
    np.ndarray,
    np.ndarray,
    dict,
]:
    """Add graph-user aggregates without accepting or deriving target data."""
    (
        train_components,
        inference_components,
        key_support,
        graph_stats,
    ) = build_user_components(train, inference)
    train_profile, inference_profile, profile_names = _profile_aggregates(
        train,
        inference,
        train_components,
        inference_components,
        key_support,
    )
    graph_stats["profile_features"] = len(profile_names)
    graph_stats["causal_history_features"] = 0
    return (
        pd.concat(
            [train.reset_index(drop=True), train_profile],
            axis=1,
        ),
        pd.concat(
            [inference.reset_index(drop=True), inference_profile],
            axis=1,
        ),
        profile_names,
        train_components,
        inference_components,
        graph_stats,
    )


Overwriting fraud_user_features.py


In [17]:
%%writefile fraud_vblock_features.py
from __future__ import annotations

import numpy as np
import pandas as pd


V_BLOCKS = (
    (1, 11),
    (12, 34),
    (35, 52),
    (53, 74),
    (75, 94),
    (95, 137),
    (138, 166),
    (167, 216),
    (217, 278),
    (279, 321),
    (322, 339),
)

TOP_V_COLUMNS = (
    "V242",
    "V243",
    "V258",
    "V265",
    "V199",
    "V201",
    "V257",
    "V274",
    "V86",
    "V156",
    "V44",
    "V87",
    "V62",
    "V217",
    "V91",
    "V67",
    "V275",
    "V189",
    "V212",
    "V70",
    "V61",
    "V69",
    "V45",
    "V149",
    "V256",
    "V90",
    "V266",
    "V94",
    "V308",
    "V200",
)


def _row_vblock_features(frame: pd.DataFrame) -> pd.DataFrame:
    features: dict[str, pd.Series] = {}
    for start, end in V_BLOCKS:
        columns = [
            f"V{number}"
            for number in range(start, end + 1)
            if f"V{number}" in frame
        ]
        if not columns:
            continue
        values = frame[columns]
        prefix = f"vblock_{start}_{end}"
        features[f"{prefix}_missing_rate"] = values.isna().mean(axis=1).astype(
            "float32"
        )
        features[f"{prefix}_mean"] = values.mean(axis=1).astype("float32")
        features[f"{prefix}_std"] = values.std(axis=1).astype("float32")
        block_min = values.min(axis=1)
        block_max = values.max(axis=1)
        features[f"{prefix}_min"] = block_min.astype("float32")
        features[f"{prefix}_max"] = block_max.astype("float32")
        features[f"{prefix}_range"] = (block_max - block_min).astype("float32")
        features[f"{prefix}_nunique"] = values.nunique(axis=1).astype("int16")
        features[f"{prefix}_zero_count"] = values.eq(0).sum(axis=1).astype(
            "int16"
        )
        features[f"{prefix}_one_count"] = values.eq(1).sum(axis=1).astype(
            "int16"
        )
        features[f"{prefix}_two_count"] = values.eq(2).sum(axis=1).astype(
            "int16"
        )
    return pd.DataFrame(features, index=frame.index)


def add_vblock_user_features(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    train_components: np.ndarray,
    inference_components: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    print("Building row-level V-block encodings...", flush=True)
    train_blocks = _row_vblock_features(train).reset_index(drop=True)
    inference_blocks = _row_vblock_features(inference).reset_index(drop=True)
    block_features = train_blocks.columns.tolist()

    c_columns = [f"C{i}" for i in range(1, 15) if f"C{i}" in train]
    d_columns = [f"D{i}" for i in range(1, 16) if f"D{i}" in train]
    v_columns = [column for column in TOP_V_COLUMNS if column in train]
    aggregate_columns = [*c_columns, *d_columns, *v_columns]
    combined = pd.concat(
        [train[aggregate_columns], inference[aggregate_columns]],
        ignore_index=True,
        copy=False,
    )
    combined["_component"] = np.concatenate(
        [train_components, inference_components]
    )
    group = combined.groupby("_component", sort=False, observed=True)
    component_count = group["_component"].transform("size")
    aggregate_features: dict[str, pd.Series] = {}

    print("Building broad user C/D aggregates...", flush=True)
    for column in [*c_columns, *d_columns]:
        values = combined[column]
        column_group = group[column]
        mean = column_group.transform("mean")
        std = column_group.transform("std")
        minimum = column_group.transform("min")
        maximum = column_group.transform("max")
        nunique = column_group.transform("nunique")
        prefix = f"wide_user_{column}"
        aggregate_features[f"{prefix}_mean"] = mean.astype("float32")
        aggregate_features[f"{prefix}_std"] = std.astype("float32")
        aggregate_features[f"{prefix}_range"] = (maximum - minimum).astype(
            "float32"
        )
        aggregate_features[f"{prefix}_nunique_rate"] = (
            nunique / component_count
        ).astype("float32")
        aggregate_features[f"{prefix}_diff_mean"] = (values - mean).astype(
            "float32"
        )
        aggregate_features[f"{prefix}_zscore"] = (
            (values - mean) / std.replace(0, np.nan)
        ).astype("float32")

    print("Building selected user V aggregates...", flush=True)
    for column in v_columns:
        values = combined[column]
        column_group = group[column]
        mean = column_group.transform("mean")
        std = column_group.transform("std")
        prefix = f"wide_user_{column}"
        aggregate_features[f"{prefix}_mean"] = mean.astype("float32")
        aggregate_features[f"{prefix}_std"] = std.astype("float32")
        aggregate_features[f"{prefix}_zscore"] = (
            (values - mean) / std.replace(0, np.nan)
        ).astype("float32")

    aggregate_frame = pd.DataFrame(aggregate_features)
    train_rows = len(train)
    train_aggregate = aggregate_frame.iloc[:train_rows].reset_index(drop=True)
    inference_aggregate = aggregate_frame.iloc[train_rows:].reset_index(
        drop=True
    )

    print("Building user-level V-block missingness profiles...", flush=True)
    combined_blocks = pd.concat(
        [train_blocks, inference_blocks], ignore_index=True, copy=False
    )
    combined_blocks["_component"] = np.concatenate(
        [train_components, inference_components]
    )
    missing_columns = [
        column for column in block_features if column.endswith("_missing_rate")
    ]
    missing_features: dict[str, pd.Series] = {}
    missing_group = combined_blocks.groupby(
        "_component", sort=False, observed=True
    )
    for column in missing_columns:
        mean = missing_group[column].transform("mean")
        missing_features[f"wide_user_{column}_mean"] = mean.astype("float32")
        missing_features[f"wide_user_{column}_diff"] = (
            combined_blocks[column] - mean
        ).astype("float32")
    missing_frame = pd.DataFrame(missing_features)
    train_missing = missing_frame.iloc[:train_rows].reset_index(drop=True)
    inference_missing = missing_frame.iloc[train_rows:].reset_index(drop=True)

    train_new = pd.concat(
        [train_blocks, train_aggregate, train_missing], axis=1
    )
    inference_new = pd.concat(
        [inference_blocks, inference_aggregate, inference_missing], axis=1
    )
    feature_names = train_new.columns.tolist()
    stats = {
        "v_blocks": len(V_BLOCKS),
        "block_features": len(block_features),
        "c_columns": len(c_columns),
        "d_columns": len(d_columns),
        "selected_v_columns": list(v_columns),
        "aggregate_features": len(train_aggregate.columns),
        "missing_profile_features": len(train_missing.columns),
        "total_features": len(feature_names),
    }
    return (
        pd.concat([train.reset_index(drop=True), train_new], axis=1),
        pd.concat([inference.reset_index(drop=True), inference_new], axis=1),
        feature_names,
        stats,
    )


Overwriting fraud_vblock_features.py


In [18]:
%%writefile honest_featureview_sources.py
"""Selected target-free feature views for the honest temporal stack."""

from __future__ import annotations

import gc
import json
from pathlib import Path
import re
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from fraud_advanced_user_features import add_advanced_user_features
from fraud_features import TARGET, build_features, read_and_merge
from fraud_multicounter_features import add_multi_counter_features
from fraud_next_features import (
    add_behavior_distribution_features,
    add_calendar_amount_features,
    add_identity_features,
)
from fraud_temporal_validation import make_four_long_gap_folds
from fraud_transaction_chain_features import add_v307_transaction_chain_features
from fraud_user_features import (
    add_target_free_user_profile_features,
    is_raw_uid_feature,
)
from fraud_vblock_features import add_vblock_user_features


ROOT = Path(__file__).resolve().parent
ADVANCED_CACHE_DIR = ROOT / "advanced_feature_ablation_models"
NEXT_CACHE_DIR = ROOT / "next_feature_ablation_models_v1"
ADVANCED_OOF_PATH = ROOT / "advanced_feature_ablation_oof.csv"
NEXT_OOF_PATH = ROOT / "next_feature_ablation_oof.csv"

LGB_PARAMS = {
    "n_estimators": 750,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 47,
    "learning_rate": 0.035,
    "min_child_samples": 90,
    "subsample": 0.78,
    "subsample_freq": 1,
    "colsample_bytree": 0.78,
    "reg_alpha": 0.75,
    "reg_lambda": 10.0,
    "max_bin": 255,
    "max_depth": -1,
    "extra_trees": True,
    "random_state": 8203,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}


def unique(*groups: list[str]) -> list[str]:
    return list(dict.fromkeys(column for group in groups for column in group))


def convert_categories(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    columns: list[str],
) -> None:
    for column in columns:
        categories = pd.Index(train[column].dropna().unique())
        dtype = pd.CategoricalDtype(categories=categories)
        train[column] = train[column].astype(dtype)
        inference[column] = inference[column].astype(dtype)


def convert_string_categories(
    train: pd.DataFrame,
    inference: pd.DataFrame,
    columns: list[str],
) -> None:
    for column in columns:
        categories = pd.Index(train[column].dropna().astype("string").unique())
        dtype = pd.CategoricalDtype(categories=categories)
        train[column] = train[column].astype("string").astype(dtype)
        inference[column] = inference[column].astype("string").astype(dtype)


def add_amount_patterns(frame: pd.DataFrame) -> list[str]:
    amount = frame["TransactionAmt"].astype("float64")
    values: dict[str, np.ndarray] = {}
    for modulus in (50.0, 100.0, 200.0):
        suffix = int(modulus)
        remainder = np.mod(amount, modulus)
        values[f"TransactionAmt_mod_{suffix}_is_zero"] = np.isclose(
            remainder, 0.0, atol=0.011
        ).astype("int8")
        values[f"TransactionAmt_mod_{suffix}_distance"] = np.minimum(
            remainder, modulus - remainder
        ).astype("float32")
    names = list(values)
    frame[names] = pd.DataFrame(values, index=frame.index)
    return names


def prepare_base() -> dict:
    print("Reading official train/test...", flush=True)
    train = read_and_merge(ROOT, "train")
    inference = read_and_merge(ROOT, "test")
    if TARGET not in train or TARGET in inference:
        raise RuntimeError("Unexpected target placement in official files")
    y = train[TARGET].astype("int8").reset_index(drop=True)

    print("Building Giba and target-free graph profiles...", flush=True)
    train, inference, base_features, base_categorical = build_features(
        train,
        inference,
        giba_features=True,
    )
    (
        train,
        inference,
        profile_features,
        train_components,
        inference_components,
        graph_stats,
    ) = add_target_free_user_profile_features(train, inference)

    print("Building improved UID, rolling and behavior features...", flush=True)
    (
        train,
        inference,
        advanced_features,
        advanced_categorical,
        train_components,
        inference_components,
        advanced_stats,
    ) = add_advanced_user_features(train, inference)

    features = unique(base_features, profile_features, advanced_features)
    categorical = unique(base_categorical, advanced_categorical)
    train_ids = train.pop("TransactionID").reset_index(drop=True)
    inference_ids = inference.pop("TransactionID").reset_index(drop=True)
    train.pop(TARGET)
    convert_categories(train, inference, categorical)

    advanced_set = set(advanced_features)
    profile_set = set(profile_features)
    dynamics_features = []
    for column in features:
        raw_v = re.fullmatch(r"V\d+", column) is not None
        raw_id = re.fullmatch(r"id_\d+", column) is not None
        if column in advanced_set or column in profile_set:
            dynamics_features.append(column)
        elif not raw_v and not raw_id and not is_raw_uid_feature(column):
            dynamics_features.append(column)

    return {
        "train": train,
        "inference": inference,
        "y": y,
        "train_ids": train_ids,
        "inference_ids": inference_ids,
        "features": features,
        "dynamics_features": dynamics_features,
        "categorical": categorical,
        "train_components": train_components,
        "inference_components": inference_components,
        "graph_stats": graph_stats,
        "advanced_stats": advanced_stats,
    }


def prepare_advanced_views() -> tuple[dict, dict[str, list[str]], dict]:
    prepared = prepare_base()
    train_amount = add_amount_patterns(prepared["train"])
    test_amount = add_amount_patterns(prepared["inference"])
    if train_amount != test_amount:
        raise RuntimeError("Train/test amount feature names differ")

    (
        prepared["train"],
        prepared["inference"],
        vblock_features,
        vblock_stats,
    ) = add_vblock_user_features(
        prepared["train"],
        prepared["inference"],
        prepared["train_components"],
        prepared["inference_components"],
    )
    dynamics = prepared["dynamics_features"]
    wide_cd = [
        column
        for column in vblock_features
        if column.startswith("wide_user_C") or column.startswith("wide_user_D")
    ]
    wide_v = [
        column for column in vblock_features if column.startswith("wide_user_V")
    ]
    missing_profiles = [
        column
        for column in vblock_features
        if column.startswith("wide_user_vblock_")
    ]
    views = {
        "vblock_dynamics": unique(dynamics, vblock_features, train_amount),
        "vblock_cd_dynamics": unique(dynamics, wide_cd, train_amount),
        "vblock_aggregates_dynamics": unique(
            dynamics,
            wide_cd,
            wide_v,
            missing_profiles,
            train_amount,
        ),
    }
    metadata = {
        "view_features": {name: len(columns) for name, columns in views.items()},
        "amount_features": train_amount,
        "vblock": vblock_stats,
    }
    return prepared, views, metadata


def prepare_next_views() -> tuple[dict, dict[str, dict], dict]:
    prepared, advanced_views, advanced_metadata = prepare_advanced_views()
    dynamics = prepared["dynamics_features"]
    vblock = advanced_views["vblock_dynamics"]

    prepared["train"].insert(
        0, "TransactionID", prepared["train_ids"].to_numpy()
    )
    prepared["inference"].insert(
        0, "TransactionID", prepared["inference_ids"].to_numpy()
    )

    print("Building target-free transaction chains...", flush=True)
    (
        prepared["train"],
        prepared["inference"],
        v307_features,
        v307_categorical,
        v307_stats,
    ) = add_v307_transaction_chain_features(
        prepared["train"],
        prepared["inference"],
        prepared["y"],
        include_target_history=False,
        max_chain_size=100,
    )
    (
        prepared["train"],
        prepared["inference"],
        multi_features,
        multi_categorical,
        multi_stats,
    ) = add_multi_counter_features(
        prepared["train"],
        prepared["inference"],
        max_chain_size=100,
    )
    chain_features = unique(v307_features, multi_features)

    (
        prepared["train"],
        prepared["inference"],
        calendar_features,
        calendar_stats,
    ) = add_calendar_amount_features(
        prepared["train"], prepared["inference"]
    )
    (
        prepared["train"],
        prepared["inference"],
        identity_features,
        identity_categorical,
        identity_stats,
    ) = add_identity_features(prepared["train"], prepared["inference"])
    (
        prepared["train"],
        prepared["inference"],
        behavior_features,
        behavior_stats,
    ) = add_behavior_distribution_features(
        prepared["train"],
        prepared["inference"],
        prepared["train_components"],
        prepared["inference_components"],
    )

    new_categorical = unique(
        v307_categorical,
        multi_categorical,
        identity_categorical,
    )
    convert_string_categories(
        prepared["train"], prepared["inference"], new_categorical
    )
    prepared["categorical"] = unique(
        prepared["categorical"], new_categorical
    )

    structured = unique(
        calendar_features,
        identity_features,
        behavior_features,
    )
    d_origin = [column for column in calendar_features if column.startswith("D")]
    c_transforms = [
        column for column in calendar_features if column.startswith("C")
    ]
    amount = [
        column for column in calendar_features if column.startswith("amount_")
    ]
    specs = {
        "vblock_d_amount": {
            "features": unique(vblock, d_origin, c_transforms, amount),
            "bayes": False,
            "candidate": True,
        },
        "vblock_structured": {
            "features": unique(vblock, structured),
            "bayes": False,
            "candidate": True,
        },
        "vblock_chains": {
            "features": unique(vblock, chain_features),
            "bayes": False,
            "candidate": True,
        },
    }
    metadata = {
        "view_features": {
            name: len(spec["features"]) for name, spec in specs.items()
        },
        "previous_vblock": advanced_metadata,
        "v307_chain": v307_stats,
        "multi_counter": multi_stats,
        "calendar_amount": calendar_stats,
        "identity": identity_stats,
        "behavior": behavior_stats,
    }
    return prepared, specs, metadata


def train_oof(
    prepared: dict,
    views: dict[str, list[str] | dict],
    cache_dir: Path,
    output_path: Path,
    seed: int,
    force: bool,
) -> tuple[pd.DataFrame, dict]:
    cache_dir.mkdir(exist_ok=True)
    folds = make_four_long_gap_folds(prepared["train"]["TransactionDT"])
    y = prepared["y"].to_numpy(dtype="int8")
    valid_rows = np.unique(
        np.concatenate([fold.validation_index for fold in folds])
    )
    fold_by_row = np.full(len(y), -1, dtype="int8")
    for fold in folds:
        fold_by_row[fold.validation_index] = fold.number
    oof = pd.DataFrame(
        {
            "row_index": valid_rows,
            "TransactionID": prepared["train_ids"].iloc[valid_rows].to_numpy(),
            TARGET: y[valid_rows],
            "fold": fold_by_row[valid_rows],
        }
    )
    metrics = {}

    for view_number, (name, specification) in enumerate(views.items(), start=1):
        features = (
            specification["features"]
            if isinstance(specification, dict)
            else specification
        )
        categorical = [
            column for column in prepared["categorical"] if column in features
        ]
        prediction_all = np.full(len(y), np.nan, dtype="float64")
        fold_rows = []
        print(
            f"View {view_number}/{len(views)}: {name} ({len(features)} features)",
            flush=True,
        )
        for fold in folds:
            prediction_path = cache_dir / f"{name}_fold_{fold.number}_prediction.npy"
            model_path = cache_dir / f"{name}_fold_{fold.number}.txt"
            started = time.time()
            if prediction_path.exists() and model_path.exists() and not force:
                prediction = np.load(prediction_path)
                reused = True
            else:
                model = lgb.LGBMClassifier(
                    **{**LGB_PARAMS, "random_state": seed + fold.number}
                )
                model.fit(
                    prepared["train"].iloc[fold.train_index][features],
                    y[fold.train_index],
                    categorical_feature=categorical,
                    callbacks=[lgb.log_evaluation(0)],
                )
                prediction = model.predict_proba(
                    prepared["train"].iloc[fold.validation_index][features]
                )[:, 1]
                model.booster_.save_model(model_path)
                np.save(prediction_path, prediction)
                del model
                gc.collect()
                reused = False
            if len(prediction) != len(fold.validation_index):
                raise RuntimeError(f"Cached {name} fold {fold.number} is misaligned")
            prediction_all[fold.validation_index] = prediction
            fold_rows.append(
                {
                    "fold": fold.number,
                    "train_rows": int(len(fold.train_index)),
                    "validation_rows": int(len(fold.validation_index)),
                    "auc": float(
                        roc_auc_score(y[fold.validation_index], prediction)
                    ),
                    "reused": reused,
                    "minutes": (time.time() - started) / 60.0,
                }
            )
            print(json.dumps(fold_rows[-1]), flush=True)
        oof[name] = prediction_all[valid_rows]
        metrics[name] = {
            "features": len(features),
            "categorical": len(categorical),
            "folds": fold_rows,
        }
        oof.to_csv(output_path, index=False)
    return oof, metrics


def train_final(
    prepared: dict,
    views: dict[str, list[str] | dict],
    names: tuple[str, ...],
    cache_dir: Path,
    seed: int,
    force: bool,
) -> dict[str, str]:
    outputs = {}
    for name in names:
        specification = views[name]
        features = (
            specification["features"]
            if isinstance(specification, dict)
            else specification
        )
        categorical = [
            column for column in prepared["categorical"] if column in features
        ]
        model_path = cache_dir / f"{name}_final.txt"
        if model_path.exists() and not force:
            print(f"Reusing final {name}", flush=True)
        else:
            print(f"Training final {name} ({len(features)} features)", flush=True)
            model = lgb.LGBMClassifier(
                **{**LGB_PARAMS, "random_state": seed}
            )
            model.fit(
                prepared["train"][features],
                prepared["y"],
                categorical_feature=categorical,
                callbacks=[lgb.log_evaluation(0)],
            )
            model.booster_.save_model(model_path)
            del model
            gc.collect()
        outputs[name] = str(model_path)
    return outputs


Overwriting honest_featureview_sources.py


In [19]:
%%writefile prepare_honest_featureview_sources.py
"""Build the train-only OOF and target-free test feature-view sources.

This is the executable subset of the earlier feature-ablation experiments.
It trains every OOF view required by the final stack, then fits only the three
full-data LightGBM sources selected by temporal validation.  No submission,
external bridge from the exploratory scripts is run.
"""

from __future__ import annotations

import argparse
import gc
import json
from pathlib import Path
import time

import honest_featureview_sources as sources


ROOT = Path(__file__).resolve().parent
REPORT_PATH = ROOT / "honest_featureview_source_report.json"

ADVANCED_FINAL_VIEWS = ("vblock_dynamics", "vblock_cd_dynamics")
NEXT_FINAL_VIEWS = ("vblock_structured",)
REQUIRED_ADVANCED_OOF = (
    "vblock_dynamics",
    "vblock_cd_dynamics",
    "vblock_aggregates_dynamics",
)
REQUIRED_NEXT_OOF = (
    "vblock_d_amount",
    "vblock_structured",
    "vblock_chains",
)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()
    started = time.time()

    print("Building official-data advanced feature views...", flush=True)
    prepared, views, advanced_metadata = sources.prepare_advanced_views()
    missing = sorted(set(REQUIRED_ADVANCED_OOF).difference(views))
    if missing:
        raise RuntimeError(f"Missing advanced views: {missing}")
    advanced_oof, advanced_metrics = sources.train_oof(
        prepared,
        views,
        sources.ADVANCED_CACHE_DIR,
        sources.ADVANCED_OOF_PATH,
        seed=8203,
        force=args.force,
    )
    advanced_models = sources.train_final(
        prepared,
        views,
        ADVANCED_FINAL_VIEWS,
        sources.ADVANCED_CACHE_DIR,
        seed=9203,
        force=args.force,
    )
    del prepared
    gc.collect()

    print("Building official-data structured feature views...", flush=True)
    prepared, specs, next_metadata = sources.prepare_next_views()
    missing = sorted(set(REQUIRED_NEXT_OOF).difference(specs))
    if missing:
        raise RuntimeError(f"Missing structured views: {missing}")

    next_oof, next_metrics = sources.train_oof(
        prepared,
        specs,
        sources.NEXT_CACHE_DIR,
        sources.NEXT_OOF_PATH,
        seed=12303,
        force=args.force,
    )
    next_models = sources.train_final(
        prepared,
        specs,
        NEXT_FINAL_VIEWS,
        sources.NEXT_CACHE_DIR,
        seed=13303,
        force=args.force,
    )
    del prepared
    gc.collect()

    required_models = (
        sources.ADVANCED_CACHE_DIR / "vblock_dynamics_final.txt",
        sources.ADVANCED_CACHE_DIR / "vblock_cd_dynamics_final.txt",
        sources.NEXT_CACHE_DIR / "vblock_structured_final.txt",
    )
    missing_models = [str(path) for path in required_models if not path.exists()]
    if missing_models:
        raise RuntimeError(f"Missing final feature-view models: {missing_models}")

    report = {
        "data_policy": (
            "official train/test covariates and official train labels only; "
            "no bridge, external labels, previous submission, or audit"
        ),
        "selection": "frozen from purged forward-time train folds",
        "advanced_oof_rows": int(len(advanced_oof)),
        "next_oof_rows": int(len(next_oof)),
        "required_advanced_oof": list(REQUIRED_ADVANCED_OOF),
        "required_next_oof": list(REQUIRED_NEXT_OOF),
        "advanced_final_views": list(ADVANCED_FINAL_VIEWS),
        "next_final_views": list(NEXT_FINAL_VIEWS),
        "advanced_metadata": advanced_metadata,
        "next_metadata": next_metadata,
        "advanced_metrics": advanced_metrics,
        "next_metrics": next_metrics,
        "advanced_models": advanced_models,
        "next_models": next_models,
        "models": [str(path.relative_to(ROOT)) for path in required_models],
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting prepare_honest_featureview_sources.py


In [20]:
%%writefile refine_honest_client_segments.py
"""Try client-segment-specific meta weights using train-only temporal folds."""

from __future__ import annotations

from itertools import product
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import build_honest_no_gap_meta as meta
import train_honest_client_meta as client


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_client_segments"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_client_segments.csv"
SEGMENTS = ("strict", "partial", "cold")
WEIGHTS = (0.0, 0.20, 0.40, 0.60, 0.80, 1.00)


def rank_match(signal: np.ndarray, reference: np.ndarray) -> np.ndarray:
    order = np.argsort(signal, kind="mergesort")
    result = np.empty(len(signal), dtype="float64")
    result[order] = np.sort(reference, kind="mergesort")
    return result


def segment_labels(features: pd.DataFrame) -> np.ndarray:
    result = np.full(len(features), "cold", dtype=object)
    partial = features["client_segment_partial"].to_numpy(dtype=bool)
    strict = features["client_segment_strict"].to_numpy(dtype=bool)
    result[partial] = "partial"
    result[strict] = "strict"
    return result


def segmented_blend(
    current: np.ndarray,
    stacked: np.ndarray,
    segments: np.ndarray,
    weights: dict[str, float],
) -> np.ndarray:
    current_rank = client.rank(current)
    stacked_rank = client.rank(stacked)
    prediction = current_rank.copy()
    for segment in SEGMENTS:
        mask = segments == segment
        weight = float(weights[segment])
        signal = (
            (1.0 - weight) * client.rank(current_rank[mask])
            + weight * client.rank(stacked_rank[mask])
        )
        prediction[mask] = rank_match(signal, current_rank[mask])
    return prediction


def search_weights(
    y: np.ndarray,
    current: np.ndarray,
    stacked: np.ndarray,
    segments: np.ndarray,
    dt: np.ndarray,
    groups: dict[str, tuple[np.ndarray, np.ndarray, int]],
    postprocess: dict,
) -> tuple[dict, list[dict]]:
    midpoint = np.median(dt)
    halves = (dt <= midpoint, dt > midpoint)
    baseline = client.apply_postprocess(
        current,
        groups,
        postprocess,
    )
    baseline_auc = float(roc_auc_score(y, baseline))
    baseline_half = [
        float(roc_auc_score(y[mask], baseline[mask])) for mask in halves
    ]
    rows = []
    for values in product(WEIGHTS, repeat=len(SEGMENTS)):
        weights = dict(zip(SEGMENTS, values))
        prediction = segmented_blend(current, stacked, segments, weights)
        prediction = client.apply_postprocess(
            prediction,
            groups,
            postprocess,
        )
        half_auc = [
            float(roc_auc_score(y[mask], prediction[mask])) for mask in halves
        ]
        half_gains = [
            score - base for score, base in zip(half_auc, baseline_half)
        ]
        score = float(roc_auc_score(y, prediction))
        rows.append(
            {
                "weights": weights,
                "auc": score,
                "gain": score - baseline_auc,
                "half_gains": half_gains,
                "min_half_gain": float(min(half_gains)),
            }
        )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(stable, key=lambda row: (row["gain"], row["min_half_gain"]))
    rows.sort(key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True)
    return selected, rows


def main() -> None:
    WORK_DIR.mkdir(exist_ok=True)
    base_report = json.loads(
        (ROOT / "honest_client_meta/report.json").read_text(encoding="utf-8")
    )
    selected_lgb = base_report["selected_lgb"]
    postprocess = base_report["selected_postprocess"]

    oof = client.build_oof_sources()
    test_sources = client.build_test_sources()
    client_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    client_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    raw_train = client.prepare_raw(
        pd.read_csv(
            ROOT / "train_transaction.csv",
            usecols=list(client.RAW_COLUMNS),
        )
    )
    raw_test = client.prepare_raw(
        pd.read_csv(
            ROOT / "test_transaction.csv",
            usecols=list(client.RAW_COLUMNS),
        )
    )
    _, _, fold_groups, test_groups = client.build_client_features(
        oof,
        raw_train,
        raw_test,
    )
    raw_uid = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(raw_uid)
    dev_current, _ = client.current_meta_prediction(
        oof, uid, client.META_DEV_FOLD
    )
    lock_current, _ = client.current_meta_prediction(
        oof, uid, client.META_LOCK_FOLD
    )

    view = selected_lgb["view"]
    features = client.make_meta_view(
        oof,
        client_oof,
        view,
        fold=oof["fold"],
    )
    dev_train = oof["fold"].lt(client.META_DEV_FOLD).to_numpy()
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    config = client.LGB_CONFIGS[int(selected_lgb["config_index"])]
    dev_model = lgb.LGBMClassifier(
        **{
            **client.LGB_BASE_PARAMS,
            **config,
            "n_estimators": int(selected_lgb["best_iteration"]),
        }
    )
    dev_model.fit(
        features.loc[dev_train],
        oof.loc[dev_train, client.TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    dev_lgb = dev_model.predict_proba(features.loc[dev_mask])[:, 1]
    lock_model = lgb.Booster(
        model_file=str(ROOT / "honest_client_meta/lock_lgb.txt")
    )
    lock_lgb = lock_model.predict(features.loc[lock_mask])

    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    selected, search = search_weights(
        oof.loc[dev_mask, client.TARGET].to_numpy(dtype="int8"),
        dev_current,
        dev_lgb,
        segment_labels(client_oof.loc[dev_mask].reset_index(drop=True)),
        raw_train.iloc[dev_rows]["TransactionDT"].to_numpy(),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
    )
    lock_prediction = segmented_blend(
        lock_current,
        lock_lgb,
        segment_labels(client_oof.loc[lock_mask].reset_index(drop=True)),
        selected["weights"],
    )
    lock_prediction = client.apply_postprocess(
        lock_prediction,
        fold_groups[client.META_LOCK_FOLD],
        postprocess,
    )
    y_lock = oof.loc[lock_mask, client.TARGET].to_numpy(dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    base_lock_auc = float(base_report["scores"]["lock_final_auc"])
    accepted = bool(selected["gain"] > 0 and lock_auc > base_lock_auc)

    test_features = client.make_meta_view(
        test_sources,
        client_test,
        view,
        fold=None,
    )
    final_model = lgb.Booster(
        model_file=str(ROOT / "honest_client_meta/final_lgb.txt")
    )
    test_lgb = final_model.predict(test_features)
    current_test = pd.read_csv(ROOT / "submission_honest_user_means.csv")
    test_prediction = segmented_blend(
        current_test[client.TARGET].to_numpy(),
        test_lgb,
        segment_labels(client_test),
        selected["weights"],
    )
    test_prediction = client.apply_postprocess(
        test_prediction,
        test_groups,
        postprocess,
    )
    output = current_test[["TransactionID"]].copy()
    output[client.TARGET] = test_prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "selection": "segment LGB weights on fold 1 halves; fold 2 lock",
        "data_policy": "official train/test only; no competition-test labels",
        "selected": selected,
        "search_top": search[:20],
        "base_client_meta_lock_auc": base_lock_auc,
        "segment_candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - base_lock_auc,
        "accepted": accepted,
        "output": OUTPUT_PATH.name,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected": report["selected"],
                "base_client_meta_lock_auc": report[
                    "base_client_meta_lock_auc"
                ],
                "segment_candidate_lock_auc": report[
                    "segment_candidate_lock_auc"
                ],
                "lock_gain": report["lock_gain"],
                "accepted": report["accepted"],
                "output": report["output"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting refine_honest_client_segments.py


In [21]:
%%writefile search_honest_cleanv2_blend.py
"""Blend the clean-v2 six-model source into the honest feature-view stack."""

from __future__ import annotations

from itertools import product
import hashlib
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import clean_v2_pipeline as clean_v2
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_cleanv2_blend"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_cleanv2_blend.csv"
SOURCES = ("featureview", "clean_v2", "fullrow_lgb")
DENOMINATOR = 10
TOP_PER_SEGMENT = 12


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def simplex(size: int, denominator: int = DENOMINATOR):
    for values in product(range(denominator + 1), repeat=size):
        if sum(values) == denominator:
            yield np.asarray(values, dtype="float64") / denominator


def load_clean_prediction(path: Path, recipe: dict) -> tuple[pd.DataFrame, np.ndarray]:
    frame = pd.read_csv(path)
    prediction = clean_v2.blend_frame(frame, recipe["horizon_weights"])
    prediction = clean_v2.apply_uid_recipe(
        frame, prediction, recipe["uid_postprocess"]
    )
    return frame, prediction


def transform_sources(values: dict[str, np.ndarray], mode: str) -> dict[str, np.ndarray]:
    if mode == "probability":
        return values
    if mode == "rank":
        return {name: client.rank(prediction) for name, prediction in values.items()}
    raise ValueError(mode)


def safe_auc(target: np.ndarray, prediction: np.ndarray) -> float:
    if np.unique(target).size < 2:
        return float("nan")
    return float(roc_auc_score(target, prediction))


def segment_candidates(
    target: np.ndarray,
    matrix: np.ndarray,
    baseline: np.ndarray,
    labels: np.ndarray,
    dt: np.ndarray,
    segment: str,
) -> list[dict]:
    segment_mask = labels == segment
    midpoint = np.median(dt)
    halves = (
        segment_mask & (dt <= midpoint),
        segment_mask & (dt > midpoint),
    )
    baseline_auc = safe_auc(target[segment_mask], baseline[segment_mask])
    baseline_halves = [safe_auc(target[mask], baseline[mask]) for mask in halves]
    rows = []
    for weights in simplex(len(SOURCES)):
        prediction = matrix @ weights
        half_auc = [safe_auc(target[mask], prediction[mask]) for mask in halves]
        half_gains = [
            score - base for score, base in zip(half_auc, baseline_halves)
        ]
        score = safe_auc(target[segment_mask], prediction[segment_mask])
        rows.append(
            {
                "weights": dict(zip(SOURCES, weights.tolist())),
                "auc": score,
                "gain": score - baseline_auc,
                "half_gains": half_gains,
                "min_half_gain": float(np.nanmin(half_gains)),
            }
        )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    if not stable:
        stable = [
            row
            for row in rows
            if row["weights"]["featureview"] == 1.0
        ]
    stable.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    return stable[:TOP_PER_SEGMENT]


def apply_segment_recipe(
    values: dict[str, np.ndarray],
    labels: np.ndarray,
    weights: dict[str, dict[str, float]],
) -> np.ndarray:
    output = np.empty(len(labels), dtype="float64")
    for segment in segments.SEGMENTS:
        mask = labels == segment
        matrix = np.column_stack([values[source][mask] for source in SOURCES])
        vector = np.asarray([weights[segment][source] for source in SOURCES])
        output[mask] = matrix @ vector
    return output


def search_recipe(
    target: np.ndarray,
    raw_values: dict[str, np.ndarray],
    labels: np.ndarray,
    dt: np.ndarray,
) -> tuple[dict, list[dict], dict]:
    rows = []
    segment_search = {}
    midpoint = np.median(dt)
    halves = (dt <= midpoint, dt > midpoint)
    for mode in ("probability", "rank"):
        values = transform_sources(raw_values, mode)
        baseline = values["featureview"]
        baseline_auc = float(roc_auc_score(target, baseline))
        baseline_halves = [
            float(roc_auc_score(target[mask], baseline[mask])) for mask in halves
        ]
        per_segment = {
            segment: segment_candidates(
                target,
                np.column_stack([values[source] for source in SOURCES]),
                baseline,
                labels,
                dt,
                segment,
            )
            for segment in segments.SEGMENTS
        }
        segment_search[mode] = per_segment
        for combination in product(
            *(per_segment[segment] for segment in segments.SEGMENTS)
        ):
            weights = {
                segment: row["weights"]
                for segment, row in zip(segments.SEGMENTS, combination)
            }
            prediction = apply_segment_recipe(values, labels, weights)
            score = float(roc_auc_score(target, prediction))
            half_auc = [
                float(roc_auc_score(target[mask], prediction[mask]))
                for mask in halves
            ]
            half_gains = [
                value - base for value, base in zip(half_auc, baseline_halves)
            ]
            rows.append(
                {
                    "mode": mode,
                    "weights": weights,
                    "auc": score,
                    "gain": score - baseline_auc,
                    "half_gains": half_gains,
                    "min_half_gain": float(min(half_gains)),
                }
            )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(
        stable, key=lambda row: (row["gain"], row["min_half_gain"])
    )
    rows.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    return selected, rows, segment_search


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    recipe = json.loads(
        (ROOT / "clean_v2/recipe.json").read_text(encoding="utf-8")
    )
    if "official train/test only" not in recipe.get("data_policy", ""):
        raise RuntimeError("clean_v2 recipe does not declare the official-data policy")

    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    feature_recipe = reference_report["recipe"]
    postprocess = feature_recipe["postprocess_locked_from_dev"]

    clean_dev_frame, clean_dev = load_clean_prediction(
        ROOT / "clean_v2/predictions/dev_h30_c.csv", recipe
    )
    clean_lock_frame, clean_lock = load_clean_prediction(
        ROOT / "clean_v2/predictions/lock_h30.csv", recipe
    )
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_rows = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    if not np.array_equal(clean_dev_frame["row_index"].to_numpy(), dev_rows):
        raise RuntimeError("clean_v2 dev rows differ from the feature-view dev fold")
    if not np.array_equal(clean_lock_frame["row_index"].to_numpy(), lock_rows):
        raise RuntimeError("clean_v2 lock rows differ from the feature-view lock fold")
    if not np.array_equal(
        clean_dev_frame[client.TARGET].to_numpy(),
        oof.loc[dev_mask, client.TARGET].to_numpy(),
    ):
        raise RuntimeError("clean_v2 dev labels are not official-train labels")

    fullrow_recipe = json.loads(
        (ROOT / "honest_fullrow_lgb/report.json").read_text(encoding="utf-8")
    )
    dev_lgb_raw = np.load(ROOT / "honest_fullrow_lgb/dev_raw.npy")
    lock_lgb_raw = np.load(ROOT / "honest_fullrow_lgb/lock_raw.npy")
    dev_lgb = fullrow.transform_variant(
        dev_lgb_raw,
        fullrow_recipe["selected_blend"]["variant"],
        fold_groups[client.META_DEV_FOLD],
        postprocess,
    )
    lock_lgb = fullrow.transform_variant(
        lock_lgb_raw,
        fullrow_recipe["selected_blend"]["variant"],
        fold_groups[client.META_LOCK_FOLD],
        postprocess,
    )

    raw_dev = {
        "featureview": reference[client.META_DEV_FOLD],
        "clean_v2": clean_dev,
        "fullrow_lgb": dev_lgb,
    }
    y_dev = oof.loc[dev_mask, client.TARGET].to_numpy(dtype="int8")
    dt_train = pd.read_csv(
        ROOT / "train_transaction.csv", usecols=["TransactionDT"]
    )["TransactionDT"].to_numpy(dtype="float64")
    labels_dev = segments.segment_labels(
        membership_oof.loc[dev_mask].reset_index(drop=True)
    )
    selected, search_rows, segment_search = search_recipe(
        y_dev,
        raw_dev,
        labels_dev,
        dt_train[dev_rows],
    )

    raw_lock = transform_sources(
        {
            "featureview": reference[client.META_LOCK_FOLD],
            "clean_v2": clean_lock,
            "fullrow_lgb": lock_lgb,
        },
        selected["mode"],
    )
    labels_lock = segments.segment_labels(
        membership_oof.loc[lock_mask].reset_index(drop=True)
    )
    lock_prediction = apply_segment_recipe(
        raw_lock, labels_lock, selected["weights"]
    )
    y_lock = oof.loc[lock_mask, client.TARGET].to_numpy(dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    reference_lock_auc = float(
        reference_report["metrics"][client.META_LOCK_FOLD]["auc"]
    )
    accepted = bool(selected["gain"] > 0.0 and lock_auc > reference_lock_auc)

    baseline_test = pd.read_csv(ROOT / "submission_honest_featureview_client.csv")
    clean_test = pd.read_csv(ROOT / "submission_clean_v2.csv")
    clean_sources = pd.read_csv(
        ROOT / "clean_v2/test_source_predictions.csv",
        usecols=["TransactionID"],
    )
    for frame, name in (
        (clean_test, "clean_v2 submission"),
        (clean_sources, "clean_v2 sources"),
    ):
        if not np.array_equal(
            frame["TransactionID"].to_numpy(),
            baseline_test["TransactionID"].to_numpy(),
        ):
            raise RuntimeError(f"{name} differs from sample order")
    test_lgb_raw = np.load(ROOT / "honest_fullrow_lgb/test_raw.npy")
    test_lgb = fullrow.transform_variant(
        test_lgb_raw,
        fullrow_recipe["selected_blend"]["variant"],
        reference_report["test_groups"],
        postprocess,
    )
    raw_test = transform_sources(
        {
            "featureview": baseline_test[client.TARGET].to_numpy(dtype="float64"),
            "clean_v2": clean_test[client.TARGET].to_numpy(dtype="float64"),
            "fullrow_lgb": test_lgb,
        },
        selected["mode"],
    )
    test_prediction = apply_segment_recipe(
        raw_test,
        segments.segment_labels(membership_test),
        selected["weights"],
    )
    output = baseline_test[["TransactionID"]].copy()
    output[client.TARGET] = test_prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test only; official train labels for dev/lock",
        "selection": "simplex and mode on days 75-90; days 90-105 one-time lock",
        "sources": list(SOURCES),
        "selected": selected,
        "search_top": search_rows[:40],
        "segment_search_top": {
            mode: {
                segment: rows[:8] for segment, rows in per_segment.items()
            }
            for mode, per_segment in segment_search.items()
        },
        "source_auc": {
            "dev": {
                source: float(roc_auc_score(y_dev, values))
                for source, values in raw_dev.items()
            },
            "lock": {
                source: float(roc_auc_score(y_lock, values))
                for source, values in {
                    "featureview": reference[client.META_LOCK_FOLD],
                    "clean_v2": clean_lock,
                    "fullrow_lgb": lock_lgb,
                }.items()
            },
        },
        "reference_lock_auc": reference_lock_auc,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - reference_lock_auc,
        "accepted": accepted,
        "output": OUTPUT_PATH.name,
        "output_sha256": file_sha256(OUTPUT_PATH),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected": selected,
                "source_auc": report["source_auc"],
                "reference_lock_auc": reference_lock_auc,
                "candidate_lock_auc": lock_auc,
                "lock_gain": lock_auc - reference_lock_auc,
                "accepted": accepted,
                "output": OUTPUT_PATH.name,
                "elapsed_minutes": report["elapsed_minutes"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting search_honest_cleanv2_blend.py


In [22]:
%%writefile search_honest_client_pooling.py
"""Target-free client pooling selected on purged temporal validation."""

from __future__ import annotations

import hashlib
from itertools import product
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import finalize_honest_xgb_magic_blend as xgb_final
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client
import train_honest_magic_heavy_stack as heavy
import train_honest_xgb_magic as magic
import train_honest_xgb_seed_subset_gkf as seed_subset


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_client_pooling"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_client_pooling.csv"
RAW_COLUMNS = (
    "TransactionID",
    "TransactionDT",
    "ProductCD",
    "card1",
    "addr1",
    "D1",
    "P_emaildomain",
)
METHODS = ("mean", "max", "q75")
WEIGHTS = tuple(np.round(np.linspace(0.0, 1.0, 11), 2))


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def auc(target: np.ndarray, prediction: np.ndarray) -> float:
    return float(roc_auc_score(target, prediction))


def token(values: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(values):
        return values.round().astype("Int64").astype("string").fillna("NA")
    return values.astype("string").fillna("NA")


def build_uid_frame(raw: pd.DataFrame) -> pd.DataFrame:
    day = raw["TransactionDT"] / 86_400.0
    origin = day - raw["D1"]
    floor_origin = token(np.floor(origin))
    round_origin = token(np.round(origin))
    card = token(raw["card1"])
    address = token(raw["addr1"])
    email = token(raw["P_emaildomain"])
    product_code = token(raw["ProductCD"])
    card_addr = card.str.cat(address, sep="|")
    return pd.DataFrame(
        {
            "floor_card_addr": card_addr.str.cat(floor_origin, sep="|"),
            "floor_card_addr_email": card_addr.str.cat(
                floor_origin, sep="|"
            ).str.cat(email, sep="|"),
            "round_card_addr_email": card_addr.str.cat(
                round_origin, sep="|"
            ).str.cat(email, sep="|"),
            "floor_card_origin_email": card.str.cat(
                floor_origin, sep="|"
            ).str.cat(email, sep="|"),
            "floor_card_addr_product": card_addr.str.cat(
                floor_origin, sep="|"
            ).str.cat(product_code, sep="|"),
            "round_card_addr_product_email": card_addr.str.cat(
                round_origin, sep="|"
            ).str.cat(product_code, sep="|").str.cat(email, sep="|"),
        }
    )


def group_pool(
    prediction: np.ndarray,
    uid: pd.Series,
    method: str,
) -> np.ndarray:
    work = pd.DataFrame(
        {"uid": uid.to_numpy(), "prediction": np.asarray(prediction)}
    )
    grouped = work.groupby("uid", sort=False, dropna=False)["prediction"]
    if method == "q75":
        pooled = grouped.transform(lambda values: values.quantile(0.75))
    else:
        pooled = grouped.transform(method)
    return pooled.to_numpy(dtype="float64")


def pooling_signals(
    prediction: np.ndarray,
    uid_frame: pd.DataFrame,
) -> dict[str, np.ndarray]:
    signals = {}
    by_method: dict[str, list[np.ndarray]] = {method: [] for method in METHODS}
    for uid_name in uid_frame.columns:
        for method in METHODS:
            values = group_pool(prediction, uid_frame[uid_name], method)
            signals[f"{uid_name}_{method}"] = values
            by_method[method].append(values)
    for method, values in by_method.items():
        ranks = np.column_stack([client.rank(value) for value in values])
        signals[f"multi_{method}_meanrank"] = ranks.mean(axis=1)
        signals[f"multi_{method}_maxrank"] = ranks.max(axis=1)
    primary = "floor_card_addr"
    primary_ranks = np.column_stack(
        [
            client.rank(signals[f"{primary}_{method}"])
            for method in METHODS
        ]
    )
    signals["floor_card_addr_method_meanrank"] = primary_ranks.mean(axis=1)
    return signals


def direct_segment_blend(
    baseline: np.ndarray,
    signal: np.ndarray,
    labels: np.ndarray,
    weights: dict[str, float],
) -> np.ndarray:
    prediction = np.asarray(baseline, dtype="float64").copy()
    for name in segments.SEGMENTS:
        mask = labels == name
        weight = float(weights[name])
        prediction[mask] = (
            (1.0 - weight) * prediction[mask] + weight * signal[mask]
        )
    return prediction


def search_pooling(
    target: np.ndarray,
    baseline: np.ndarray,
    signals: dict[str, np.ndarray],
    labels: np.ndarray,
    day: np.ndarray,
) -> tuple[dict, list[dict]]:
    midpoint = np.median(day)
    halves = (day <= midpoint, day > midpoint)
    baseline_auc = auc(target, baseline)
    baseline_halves = [auc(target[mask], baseline[mask]) for mask in halves]
    masks = {name: labels == name for name in segments.SEGMENTS}
    rows = []
    row_map = {}

    for variant, signal in signals.items():
        transformed = {
            name: {
                weight: (
                    (1.0 - weight) * baseline[mask]
                    + weight * signal[mask]
                )
                for weight in WEIGHTS
            }
            for name, mask in masks.items()
        }

        def evaluate(weights: dict[str, float]) -> dict:
            key = (
                variant,
                *(float(weights[name]) for name in segments.SEGMENTS),
            )
            if key in row_map:
                return row_map[key]
            prediction = np.asarray(baseline, dtype="float64").copy()
            for name, mask in masks.items():
                prediction[mask] = transformed[name][float(weights[name])]
            score = auc(target, prediction)
            half_auc = [
                auc(target[mask], prediction[mask]) for mask in halves
            ]
            half_gains = [
                value - base
                for value, base in zip(half_auc, baseline_halves)
            ]
            row = {
                "variant": variant,
                "weights": {
                    name: float(weights[name]) for name in segments.SEGMENTS
                },
                "auc": score,
                "gain": score - baseline_auc,
                "half_gains": half_gains,
                "min_half_gain": float(min(half_gains)),
            }
            rows.append(row)
            row_map[key] = row
            return row

        for start in (
            (0.0, 0.0, 0.0),
            (0.0, 0.0, 0.3),
            (0.2, 0.2, 0.2),
            (0.5, 0.5, 0.5),
        ):
            weights = dict(zip(segments.SEGMENTS, start))
            evaluate(weights)
            for _ in range(2):
                for name in segments.SEGMENTS:
                    local = []
                    for weight in WEIGHTS:
                        candidate = dict(weights)
                        candidate[name] = float(weight)
                        local.append(evaluate(candidate))
                    best = max(
                        local,
                        key=lambda row: (row["auc"], row["min_half_gain"]),
                    )
                    weights = dict(best["weights"])
            neighborhoods = []
            for name in segments.SEGMENTS:
                center = WEIGHTS.index(float(weights[name]))
                neighborhoods.append(WEIGHTS[max(0, center - 1) : center + 2])
            for values in product(*neighborhoods):
                evaluate(dict(zip(segments.SEGMENTS, values)))

    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(
        stable, key=lambda row: (row["gain"], row["min_half_gain"])
    )
    rows.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    return selected, rows


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    manifest, arrays = magic.load_matrix()
    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    clean_oof, clean_report = xgb_final.reconstruct_clean_oof(
        oof, membership_oof, reference, fold_groups, reference_report
    )
    xgb_report = json.loads(
        (ROOT / "honest_xgb_magic/blend_report.json").read_text(
            encoding="utf-8"
        )
    )
    heavy_report = json.loads(
        (ROOT / "honest_magic_heavy_stack/report.json").read_text(
            encoding="utf-8"
        )
    )
    xgb_recipe = xgb_report["selected"]
    heavy_recipe = heavy_report["selected_blend"]
    cat_name = heavy_report["selected_cat"]["name"]
    lgb_name = heavy_report["selected_lgb"]["name"]

    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    dev_membership = membership_oof.loc[dev_mask].reset_index(drop=True)
    lock_membership = membership_oof.loc[lock_mask].reset_index(drop=True)
    dev_xgb = seed_subset.apply_xgb_layer(
        clean_oof[client.META_DEV_FOLD],
        np.load(ROOT / "honest_xgb_magic/dev_prediction.npy"),
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
        dev_membership,
        xgb_recipe,
    )
    lock_xgb = seed_subset.apply_xgb_layer(
        clean_oof[client.META_LOCK_FOLD],
        np.load(ROOT / "honest_xgb_magic/lock_prediction.npy"),
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        xgb_recipe,
    )
    dev_current = seed_subset.apply_heavy_layer(
        dev_xgb,
        np.load(ROOT / f"honest_magic_heavy_stack/dev_cat_{cat_name}.npy"),
        np.load(ROOT / f"honest_magic_heavy_stack/dev_lgb_{lgb_name}.npy"),
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
        dev_membership,
        heavy_recipe,
    )
    lock_current = seed_subset.apply_heavy_layer(
        lock_xgb,
        np.load(ROOT / "honest_magic_heavy_stack/lock_cat.npy"),
        np.load(ROOT / "honest_magic_heavy_stack/lock_lgb.npy"),
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        heavy_recipe,
    )
    y_dev = np.asarray(arrays["target"][dev_index], dtype="int8")
    y_lock = np.asarray(arrays["target"][lock_index], dtype="int8")
    current_dev_auc = auc(y_dev, dev_current)
    current_lock_auc = auc(y_lock, lock_current)
    if abs(current_dev_auc - float(heavy_report["dev_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy-stack dev prediction")
    if abs(current_lock_auc - float(heavy_report["lock_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy-stack lock prediction")

    raw_train = pd.read_csv(ROOT / "train_transaction.csv", usecols=RAW_COLUMNS)
    raw_test = pd.read_csv(ROOT / "test_transaction.csv", usecols=RAW_COLUMNS)
    if not np.array_equal(
        raw_train["TransactionID"].to_numpy(), arrays["train_id"]
    ):
        raise RuntimeError("Raw train rows differ from the magic matrix")
    if not np.array_equal(
        raw_test["TransactionID"].to_numpy(), arrays["test_id"]
    ):
        raise RuntimeError("Raw test rows differ from the magic matrix")
    train_uid = build_uid_frame(raw_train)
    test_uid = build_uid_frame(raw_test)
    dev_uid = train_uid.iloc[dev_index].reset_index(drop=True)
    lock_uid = train_uid.iloc[lock_index].reset_index(drop=True)
    dev_signals = pooling_signals(dev_current, dev_uid)
    selected, search_rows = search_pooling(
        y_dev,
        dev_current,
        dev_signals,
        segments.segment_labels(dev_membership),
        np.asarray(arrays["day"][dev_index]),
    )
    dev_candidate = direct_segment_blend(
        dev_current,
        dev_signals[selected["variant"]],
        segments.segment_labels(dev_membership),
        selected["weights"],
    )
    lock_signals = pooling_signals(lock_current, lock_uid)
    lock_candidate = direct_segment_blend(
        lock_current,
        lock_signals[selected["variant"]],
        segments.segment_labels(lock_membership),
        selected["weights"],
    )
    dev_auc = auc(y_dev, dev_candidate)
    lock_auc = auc(y_lock, lock_candidate)
    accepted = bool(
        selected["gain"] > 0.0
        and dev_auc > current_dev_auc
        and lock_auc > current_lock_auc
    )

    if accepted:
        baseline = pd.read_csv(ROOT / "submission_honest_magic_heavy_stack.csv")
        if not np.array_equal(
            baseline["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("Pooling test rows differ from heavy stack")
        test_current = baseline[client.TARGET].to_numpy(dtype="float64")
        test_signals = pooling_signals(test_current, test_uid)
        prediction = direct_segment_blend(
            test_current,
            test_signals[selected["variant"]],
            segments.segment_labels(membership_test),
            selected["weights"],
        )
        output = baseline[["TransactionID"]].copy()
        output[client.TARGET] = prediction
        output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": (
            "official train/test covariates and train labels only; "
            "pooling uses query covariates/predictions without labels"
        ),
        "official_hashes": manifest["official_hashes"],
        "uid_definitions": list(dev_uid.columns),
        "methods": list(METHODS),
        "selection": "pooling recipe on dev; one-time unchanged lock",
        "champion_auc": {"dev": current_dev_auc, "lock": current_lock_auc},
        "selected": selected,
        "search_top": search_rows[:50],
        "dev_auc": dev_auc,
        "dev_gain": dev_auc - current_dev_auc,
        "lock_auc": lock_auc,
        "lock_gain": lock_auc - current_lock_auc,
        "accepted": accepted,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting search_honest_client_pooling.py


In [23]:
%%writefile search_honest_featureview_meta.py
"""Search clean feature-view residual sources above the honest client stack."""

from __future__ import annotations

import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import build_honest_no_gap_meta as meta
import refine_honest_client_segments as segments
import train_honest_client_meta as client
import train_honest_heavy_temporal_client as heavy


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_featureview_meta"
REPORT_PATH = WORK_DIR / "search_report.json"

FEATURE_SOURCES = (
    "fv_vblock_dynamics",
    "fv_vblock_cd_dynamics",
    "fv_vblock_aggregates_dynamics",
    "fv_vblock_d_amount",
    "fv_vblock_structured",
    "fv_vblock_chains",
)
CURRENT = tuple(client.SOURCES)
VIEWS = {
    "stable_three_clients": (
        (*CURRENT, "fv_vblock_dynamics", "fv_vblock_cd_dynamics", "fv_vblock_structured"),
        True,
    ),
    "structured_clients": (
        (*CURRENT, "fv_vblock_d_amount", "fv_vblock_structured", "fv_vblock_chains"),
        True,
    ),
    "all_featureviews_clients": ((*CURRENT, *FEATURE_SOURCES), True),
    "all_featureviews": ((*CURRENT, *FEATURE_SOURCES), False),
}


def build_oof() -> pd.DataFrame:
    oof = client.build_oof_sources()
    advanced = pd.read_csv(
        ROOT / "advanced_feature_ablation_oof.csv",
        usecols=[
            "row_index",
            "fold",
            "vblock_dynamics",
            "vblock_cd_dynamics",
            "vblock_aggregates_dynamics",
        ],
    ).rename(
        columns={
            "fold": "feature_fold",
            "vblock_dynamics": "fv_vblock_dynamics",
            "vblock_cd_dynamics": "fv_vblock_cd_dynamics",
            "vblock_aggregates_dynamics": "fv_vblock_aggregates_dynamics",
        }
    )
    next_views = pd.read_csv(
        ROOT / "next_feature_ablation_oof.csv",
        usecols=[
            "row_index",
            "vblock_d_amount",
            "vblock_structured",
            "vblock_chains",
        ],
    ).rename(
        columns={
            "vblock_d_amount": "fv_vblock_d_amount",
            "vblock_structured": "fv_vblock_structured",
            "vblock_chains": "fv_vblock_chains",
        }
    )
    merged = oof.merge(advanced, on="row_index", how="left", validate="one_to_one")
    merged = merged.merge(next_views, on="row_index", how="left", validate="one_to_one")
    if merged[list(FEATURE_SOURCES)].isna().any().any():
        raise RuntimeError("Feature-view OOF does not cover current OOF")
    if not np.array_equal(
        merged["feature_fold"].to_numpy(), merged["fold"].to_numpy() + 1
    ):
        raise RuntimeError("Feature-view and current temporal folds are misaligned")
    return merged.drop(columns="feature_fold").reset_index(drop=True)


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    oof = build_oof()
    membership = pd.read_csv(ROOT / "honest_client_meta/oof_client_features.csv")
    raw_train = client.prepare_raw(
        pd.read_csv(ROOT / "train_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    raw_test = client.prepare_raw(
        pd.read_csv(ROOT / "test_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    _, _, fold_groups, _ = client.build_client_features(oof, raw_train, raw_test)
    uid_raw = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(uid_raw)
    dev_current, dev_report = client.current_meta_prediction(oof, uid, client.META_DEV_FOLD)
    lock_current, lock_report = client.current_meta_prediction(oof, uid, client.META_LOCK_FOLD)

    original_views = heavy.VIEWS
    heavy.VIEWS = VIEWS
    try:
        selected_model, model_search, dev_feature = heavy.search_models(oof, membership)
        lock_model, lock_feature, lock_iterations = heavy.train_model(
            oof, membership, selected_model, client.META_LOCK_FOLD, 1.15
        )
    finally:
        heavy.VIEWS = original_views
    lock_model.booster_.save_model(WORK_DIR / "lock_lgb.txt")

    base_report = json.loads(
        (ROOT / "honest_client_meta/report.json").read_text(encoding="utf-8")
    )
    postprocess = base_report["selected_postprocess"]
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    selected_weights, weight_search = heavy.search_segment_weights(
        oof.loc[dev_mask, client.TARGET].to_numpy(dtype="int8"),
        dev_current,
        dev_feature,
        segments.segment_labels(membership.loc[dev_mask].reset_index(drop=True)),
        raw_train.iloc[dev_rows]["TransactionDT"].to_numpy(),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
    )
    lock_prediction = segments.segmented_blend(
        lock_current,
        lock_feature,
        segments.segment_labels(membership.loc[lock_mask].reset_index(drop=True)),
        selected_weights["weights"],
    )
    lock_prediction = client.apply_postprocess(
        lock_prediction, fold_groups[client.META_LOCK_FOLD], postprocess
    )
    y_lock = oof.loc[lock_mask, client.TARGET].to_numpy(dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    previous = float(
        json.loads((ROOT / "honest_client_segments/report.json").read_text(encoding="utf-8"))[
            "segment_candidate_lock_auc"
        ]
    )
    report = {
        "data_policy": "official train OOF only",
        "sources": list(FEATURE_SOURCES),
        "selected_model": selected_model,
        "model_search": model_search,
        "selected_segment_weights": selected_weights,
        "weight_search_top": weight_search[:30],
        "postprocess_locked_from_dev": postprocess,
        "current_meta": {"dev": dev_report, "lock": lock_report},
        "lock_iterations": lock_iterations,
        "previous_best_lock_auc": previous,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - previous,
        "accepted": bool(selected_weights["gain"] > 0 and lock_auc > previous),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected_model": selected_model,
                "selected_segment_weights": selected_weights,
                "previous_best_lock_auc": previous,
                "candidate_lock_auc": lock_auc,
                "lock_gain": lock_auc - previous,
                "accepted": report["accepted"],
                "elapsed_minutes": report["elapsed_minutes"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting search_honest_featureview_meta.py


In [24]:
%%writefile search_honest_fullrow_lgb.py
"""Train a full-row raw LightGBM with purged temporal model selection.

Only official train labels are used. Official test covariates participate in
target-free category and sequence construction, matching inference exactly.
"""

from __future__ import annotations

import gc
import hashlib
from itertools import product
import json
from pathlib import Path
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import build_honest_no_gap_meta as meta
from fraud_features import build_features, read_and_merge
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_raw_feature_meta as raw_meta
import train_honest_client_meta as client
import train_honest_heavy_temporal_client as heavy


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_fullrow_lgb"
CACHE_DIR = WORK_DIR / "matrix"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_fullrow_lgb.csv"
TARGET = client.TARGET
DAY_SECONDS = 86_400.0
DEV_TRAIN_END = 45.0
LOCK_TRAIN_END = 60.0
DEV_FOLD = client.META_DEV_FOLD
LOCK_FOLD = client.META_LOCK_FOLD
FEATURE_RECIPE_VERSION = "raw233_joint_sequence_v1"

LGB_PARAMS = {
    "n_estimators": 3_500,
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.015,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 2.0,
    "reg_lambda": 20.0,
    "random_state": 12011,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
LGB_CONFIGS = (
    {"num_leaves": 31, "max_depth": 6, "min_child_samples": 300},
    {"num_leaves": 63, "max_depth": 7, "min_child_samples": 500},
    {"num_leaves": 127, "max_depth": 8, "min_child_samples": 800},
    {
        "num_leaves": 63,
        "max_depth": 8,
        "min_child_samples": 1_000,
        "extra_trees": True,
    },
    {
        "num_leaves": 127,
        "max_depth": 9,
        "min_child_samples": 1_000,
        "extra_trees": True,
    },
)
SEGMENT_WEIGHTS = tuple(np.round(np.linspace(0.0, 1.0, 11), 2))


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def official_hashes() -> dict[str, str]:
    return {
        name: file_sha256(ROOT / name)
        for name in (
            "train_transaction.csv",
            "train_identity.csv",
            "test_transaction.csv",
            "test_identity.csv",
            "sample_submission.csv",
        )
    }


def matrix_paths() -> dict[str, Path]:
    return {
        name: CACHE_DIR / f"{name}.npy"
        for name in ("train", "test", "target", "day", "train_id", "test_id")
    }


def build_matrix_cache(hashes: dict[str, str]) -> dict:
    print("Building the full official-data feature matrix...", flush=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    train = read_and_merge(ROOT, "train")
    test = read_and_merge(ROOT, "test")
    if TARGET not in train or TARGET in test:
        raise RuntimeError("Unexpected target placement in official files")

    target = train[TARGET].to_numpy(dtype="int8")
    day = (train["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS).astype(
        "float32"
    )
    train_id = train["TransactionID"].to_numpy(dtype="int32")
    test_id = test["TransactionID"].to_numpy(dtype="int32")
    train, test, all_features, categorical = build_features(
        train,
        test,
        giba_features=True,
        frequency_mode="selected",
        v307_chain_features=True,
    )
    selected = raw_meta.select_raw_features(all_features)
    selected_categorical = [name for name in categorical if name in selected]
    raw_meta.encode_categories(train, test, selected_categorical)
    train_matrix = train[selected].astype("float32").to_numpy(copy=True)
    test_matrix = test[selected].astype("float32").to_numpy(copy=True)

    paths = matrix_paths()
    np.save(paths["train"], train_matrix)
    np.save(paths["test"], test_matrix)
    np.save(paths["target"], target)
    np.save(paths["day"], day)
    np.save(paths["train_id"], train_id)
    np.save(paths["test_id"], test_id)
    manifest = {
        "feature_recipe_version": FEATURE_RECIPE_VERSION,
        "official_hashes": hashes,
        "features": selected,
        "categorical_features": selected_categorical,
        "train_shape": list(train_matrix.shape),
        "test_shape": list(test_matrix.shape),
        "target_sum": int(target.sum()),
    }
    (CACHE_DIR / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    del train, test, train_matrix, test_matrix
    gc.collect()
    return manifest


def load_matrix() -> tuple[dict, dict[str, np.ndarray]]:
    hashes = official_hashes()
    manifest_path = CACHE_DIR / "manifest.json"
    paths = matrix_paths()
    manifest = None
    if manifest_path.exists() and all(path.exists() for path in paths.values()):
        candidate = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (
            candidate.get("feature_recipe_version") == FEATURE_RECIPE_VERSION
            and candidate.get("official_hashes") == hashes
        ):
            manifest = candidate
            print("Loading verified full-row matrix cache", flush=True)
    if manifest is None:
        manifest = build_matrix_cache(hashes)
    arrays = {
        name: np.load(path, mmap_mode="r") for name, path in paths.items()
    }
    if list(arrays["train"].shape) != manifest["train_shape"]:
        raise RuntimeError("Cached train matrix shape differs from manifest")
    if list(arrays["test"].shape) != manifest["test_shape"]:
        raise RuntimeError("Cached test matrix shape differs from manifest")
    return manifest, arrays


def make_featureview_matrix(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    view: str,
) -> pd.DataFrame:
    sources, include_clients = featureview.VIEWS[view]
    features = meta.build_meta_features(
        oof, sources, fold=oof["fold"]
    ).reset_index(drop=True)
    if include_clients:
        features = pd.concat(
            [features, membership.reset_index(drop=True)], axis=1
        )
    return features.astype("float32")


def build_reference_oof(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
) -> tuple[dict[int, np.ndarray], dict[int, dict], dict]:
    recipe = json.loads(
        (ROOT / "honest_featureview_meta/search_report.json").read_text(
            encoding="utf-8"
        )
    )
    selected = recipe["selected_model"]
    features = make_featureview_matrix(oof, membership, selected["view"])
    raw_train = client.prepare_raw(
        pd.read_csv(
            ROOT / "train_transaction.csv", usecols=list(client.RAW_COLUMNS)
        )
    )
    raw_test = client.prepare_raw(
        pd.read_csv(
            ROOT / "test_transaction.csv", usecols=list(client.RAW_COLUMNS)
        )
    )
    _, _, fold_groups, test_groups = client.build_client_features(
        oof, raw_train, raw_test
    )
    uid_frame = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(uid_frame)

    predictions = {}
    metrics = {}
    for fold, iterations in (
        (DEV_FOLD, int(selected["best_iteration"])),
        (LOCK_FOLD, int(recipe["lock_iterations"])),
    ):
        train_mask = oof["fold"].lt(fold).to_numpy()
        valid_mask = oof["fold"].eq(fold).to_numpy()
        model = lgb.LGBMClassifier(
            **{
                **heavy.LGB_PARAMS,
                **heavy.LGB_CONFIGS[int(selected["config_index"])],
                "n_estimators": iterations,
            }
        )
        model.fit(
            features.loc[train_mask],
            oof.loc[train_mask, TARGET],
            callbacks=[lgb.log_evaluation(0)],
        )
        feature_prediction = model.predict_proba(features.loc[valid_mask])[:, 1]
        current, _ = client.current_meta_prediction(oof, uid, fold)
        prediction = segments.segmented_blend(
            current,
            feature_prediction,
            segments.segment_labels(
                membership.loc[valid_mask].reset_index(drop=True)
            ),
            recipe["selected_segment_weights"]["weights"],
        )
        prediction = client.apply_postprocess(
            prediction,
            fold_groups[fold],
            recipe["postprocess_locked_from_dev"],
        )
        y_valid = oof.loc[valid_mask, TARGET].to_numpy(dtype="int8")
        predictions[fold] = prediction
        metrics[fold] = {
            "rows": int(valid_mask.sum()),
            "auc": float(roc_auc_score(y_valid, prediction)),
        }
        del model
        gc.collect()

    expected_lock = float(recipe["candidate_lock_auc"])
    if abs(metrics[LOCK_FOLD]["auc"] - expected_lock) > 1e-10:
        raise RuntimeError(
            "Feature-view reference was not reproduced: "
            f"{metrics[LOCK_FOLD]['auc']} != {expected_lock}"
        )
    return predictions, fold_groups, {
        "recipe": recipe,
        "metrics": metrics,
        "test_groups": test_groups,
    }


def search_lgb(
    train: np.ndarray,
    target: np.ndarray,
    day: np.ndarray,
    valid_index: np.ndarray,
) -> tuple[dict, list[dict], np.ndarray]:
    train_index = np.flatnonzero(day < DEV_TRAIN_END)
    rows = []
    predictions = {}
    y_valid = np.asarray(target[valid_index], dtype="int8")
    for config_index, config in enumerate(LGB_CONFIGS):
        model = lgb.LGBMClassifier(**LGB_PARAMS, **config)
        model.fit(
            train[train_index],
            target[train_index],
            eval_set=[(train[valid_index], y_valid)],
            callbacks=[
                lgb.early_stopping(220, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        prediction = model.predict_proba(train[valid_index])[:, 1]
        row = {
            "config_index": config_index,
            **config,
            "best_iteration": int(model.best_iteration_),
            "dev_auc": float(roc_auc_score(y_valid, prediction)),
        }
        print(json.dumps(row), flush=True)
        rows.append(row)
        predictions[config_index] = prediction
        del model
        gc.collect()
    rows.sort(
        key=lambda row: (row["dev_auc"], -row["num_leaves"]), reverse=True
    )
    selected = rows[0]
    return selected, rows, predictions[int(selected["config_index"])]


def train_lock(
    train: np.ndarray,
    target: np.ndarray,
    day: np.ndarray,
    valid_index: np.ndarray,
    selected: dict,
) -> tuple[lgb.LGBMClassifier, np.ndarray, int]:
    train_index = np.flatnonzero(day < LOCK_TRAIN_END)
    iterations = max(50, int(np.ceil(selected["best_iteration"] * 1.15)))
    model = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            **LGB_CONFIGS[int(selected["config_index"])],
            "n_estimators": iterations,
        }
    )
    model.fit(
        train[train_index],
        target[train_index],
        callbacks=[lgb.log_evaluation(0)],
    )
    return model, model.predict_proba(train[valid_index])[:, 1], iterations


def prediction_variants(
    prediction: np.ndarray,
    groups: dict,
    postprocess: dict,
) -> dict[str, np.ndarray]:
    ranked = client.rank(prediction)
    postprocessed = client.apply_postprocess(prediction, groups, postprocess)
    return {
        "probability": prediction,
        "rank": ranked,
        "postprocessed_probability": postprocessed,
        "postprocessed_rank": client.apply_postprocess(
            ranked, groups, postprocess
        ),
    }


def search_blend(
    target: np.ndarray,
    baseline: np.ndarray,
    raw_variants: dict[str, np.ndarray],
    segment_labels: np.ndarray,
    dt: np.ndarray,
) -> tuple[dict, list[dict]]:
    midpoint = np.median(dt)
    halves = (dt <= midpoint, dt > midpoint)
    baseline_auc = float(roc_auc_score(target, baseline))
    baseline_halves = [
        float(roc_auc_score(target[mask], baseline[mask])) for mask in halves
    ]
    rows = []
    for variant, raw_prediction in raw_variants.items():
        for weights_tuple in product(
            SEGMENT_WEIGHTS, repeat=len(segments.SEGMENTS)
        ):
            weights = dict(zip(segments.SEGMENTS, weights_tuple))
            prediction = segments.segmented_blend(
                baseline, raw_prediction, segment_labels, weights
            )
            auc = float(roc_auc_score(target, prediction))
            half_auc = [
                float(roc_auc_score(target[mask], prediction[mask]))
                for mask in halves
            ]
            half_gains = [
                score - base for score, base in zip(half_auc, baseline_halves)
            ]
            rows.append(
                {
                    "variant": variant,
                    "weights": weights,
                    "auc": auc,
                    "gain": auc - baseline_auc,
                    "half_gains": half_gains,
                    "min_half_gain": float(min(half_gains)),
                }
            )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(
        stable, key=lambda row: (row["gain"], row["min_half_gain"])
    )
    rows.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    return selected, rows


def transform_variant(
    prediction: np.ndarray,
    variant: str,
    groups: dict,
    postprocess: dict,
) -> np.ndarray:
    variants = prediction_variants(prediction, groups, postprocess)
    return variants[variant]


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    manifest, arrays = load_matrix()
    train = arrays["train"]
    test = arrays["test"]
    target = arrays["target"]
    day = arrays["day"]

    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = build_reference_oof(
        oof, membership_oof
    )
    dev_mask = oof["fold"].eq(DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    if not np.all((day[dev_index] >= 75.0) & (day[dev_index] < 90.0)):
        raise RuntimeError("Dev rows do not match the purged temporal contract")
    if not np.all((day[lock_index] >= 90.0) & (day[lock_index] < 106.0)):
        raise RuntimeError("Lock rows do not match the purged temporal contract")

    selected, model_search, dev_raw = search_lgb(
        train, target, day, dev_index
    )
    lock_model, lock_raw, lock_iterations = train_lock(
        train, target, day, lock_index, selected
    )
    lock_model_path = WORK_DIR / "lock_lgb.txt"
    lock_model.booster_.save_model(lock_model_path)
    np.save(WORK_DIR / "dev_raw.npy", dev_raw.astype("float32"))
    np.save(WORK_DIR / "lock_raw.npy", lock_raw.astype("float32"))

    postprocess = reference_report["recipe"]["postprocess_locked_from_dev"]
    dev_variants = prediction_variants(
        dev_raw, fold_groups[DEV_FOLD], postprocess
    )
    selected_blend, blend_search = search_blend(
        np.asarray(target[dev_index], dtype="int8"),
        reference[DEV_FOLD],
        dev_variants,
        segments.segment_labels(
            membership_oof.loc[dev_mask].reset_index(drop=True)
        ),
        np.asarray(day[dev_index]),
    )
    lock_variant = transform_variant(
        lock_raw,
        selected_blend["variant"],
        fold_groups[LOCK_FOLD],
        postprocess,
    )
    lock_prediction = segments.segmented_blend(
        reference[LOCK_FOLD],
        lock_variant,
        segments.segment_labels(
            membership_oof.loc[lock_mask].reset_index(drop=True)
        ),
        selected_blend["weights"],
    )
    y_lock = np.asarray(target[lock_index], dtype="int8")
    lock_raw_auc = float(roc_auc_score(y_lock, lock_raw))
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    reference_lock_auc = float(reference_report["metrics"][LOCK_FOLD]["auc"])
    accepted = bool(selected_blend["gain"] > 0 and lock_auc > reference_lock_auc)

    final_iterations = max(50, int(np.ceil(lock_iterations * 1.15)))
    final_model = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            **LGB_CONFIGS[int(selected["config_index"])],
            "n_estimators": final_iterations,
        }
    )
    final_model.fit(train, target, callbacks=[lgb.log_evaluation(0)])
    final_model_path = WORK_DIR / "final_lgb.txt"
    final_model.booster_.save_model(final_model_path)
    test_raw = final_model.predict_proba(test)[:, 1]
    np.save(WORK_DIR / "test_raw.npy", test_raw.astype("float32"))
    test_variant = transform_variant(
        test_raw,
        selected_blend["variant"],
        reference_report["test_groups"],
        postprocess,
    )
    baseline_test = pd.read_csv(ROOT / "submission_honest_featureview_client.csv")
    if not np.array_equal(
        baseline_test["TransactionID"].to_numpy(), arrays["test_id"]
    ):
        raise RuntimeError("Full-row test matrix and reference submission differ")
    test_prediction = segments.segmented_blend(
        baseline_test[TARGET].to_numpy(dtype="float64"),
        test_variant,
        segments.segment_labels(membership_test),
        selected_blend["weights"],
    )
    output = baseline_test[["TransactionID"]].copy()
    output[TARGET] = test_prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test covariates; official train labels only",
        "selection": "model and blend on days 75-90; days 90-105 one-time lock",
        "feature_recipe_version": FEATURE_RECIPE_VERSION,
        "feature_count": len(manifest["features"]),
        "selected_model": selected,
        "model_search": model_search,
        "lock_iterations": lock_iterations,
        "final_iterations": final_iterations,
        "selected_blend": selected_blend,
        "blend_search_top": blend_search[:40],
        "reference": {
            "dev_auc": reference_report["metrics"][DEV_FOLD]["auc"],
            "lock_auc": reference_lock_auc,
        },
        "candidate": {
            "dev_raw_auc": selected["dev_auc"],
            "lock_raw_auc": lock_raw_auc,
            "lock_blend_auc": lock_auc,
            "lock_gain": lock_auc - reference_lock_auc,
        },
        "accepted": accepted,
        "models": {
            "lock": {
                "path": str(lock_model_path),
                "sha256": file_sha256(lock_model_path),
            },
            "final": {
                "path": str(final_model_path),
                "sha256": file_sha256(final_model_path),
            },
        },
        "output": OUTPUT_PATH.name,
        "output_sha256": file_sha256(OUTPUT_PATH),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected_model": selected,
                "selected_blend": selected_blend,
                "reference": report["reference"],
                "candidate": report["candidate"],
                "accepted": accepted,
                "output": OUTPUT_PATH.name,
                "elapsed_minutes": report["elapsed_minutes"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting search_honest_fullrow_lgb.py


In [25]:
%%writefile search_honest_raw_feature_meta.py
"""Search a heavier raw-feature meta model on nested temporal OOF only."""

from __future__ import annotations

import gc
import json
from pathlib import Path
import re
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import build_honest_no_gap_meta as meta
from fraud_features import build_features, read_and_merge
from fraud_vblock_features import TOP_V_COLUMNS
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import train_honest_client_meta as client
import train_honest_heavy_temporal_client as heavy


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_raw_feature_meta"
REPORT_PATH = WORK_DIR / "search_report.json"
RAW_FEATURES_PATH = WORK_DIR / "raw_features.json"

META_SOURCES = (
    *client.SOURCES,
    "fv_vblock_dynamics",
    "fv_vblock_cd_dynamics",
    "fv_vblock_structured",
)
LGB_CONFIGS = (
    {"num_leaves": 15, "max_depth": 4, "min_child_samples": 300},
    {"num_leaves": 31, "max_depth": 5, "min_child_samples": 300},
    {"num_leaves": 31, "max_depth": 6, "min_child_samples": 600},
    {"num_leaves": 63, "max_depth": 7, "min_child_samples": 800},
    {"num_leaves": 63, "max_depth": 8, "min_child_samples": 1_200},
)
LGB_PARAMS = {
    **heavy.LGB_PARAMS,
    "n_estimators": 3_000,
    "learning_rate": 0.0125,
    "reg_alpha": 2.0,
    "reg_lambda": 20.0,
    "random_state": 9109,
}


def select_raw_features(columns: list[str]) -> list[str]:
    explicit = {
        "TransactionAmt",
        "TransactionAmt_log1p",
        "TransactionAmt_cents",
        "TransactionAmt_is_integer",
        "TransactionAmt_is_round_10",
        "ProductCD",
        "addr1",
        "addr2",
        "dist1",
        "dist2",
        "P_emaildomain",
        "R_emaildomain",
        "P_R_email_match",
        "P_email_suffix",
        "R_email_suffix",
        "DeviceType",
        "DeviceInfo",
        "DeviceInfo_family",
        "browser_family",
        "DT_hour",
        "DT_dayofweek",
        "D1_origin_day",
        "row_missing_count",
        "C_mean",
        "C_std",
        "C_min",
        "C_max",
        "D_missing_count",
        "D_mean",
        "D_std",
        "D_min",
        "D_max",
        "V_missing_count",
        "V_mean",
        "V_std",
        "V_min",
        "V_max",
        "id_numeric_missing_count",
        "id_numeric_mean",
        "id_numeric_std",
        "id_numeric_min",
        "id_numeric_max",
    }
    result = []
    top_v = set(TOP_V_COLUMNS)
    for column in columns:
        raw_family = bool(
            re.fullmatch(r"card[1-6]", column)
            or re.fullmatch(r"C(?:[1-9]|1[0-4])", column)
            or re.fullmatch(r"D(?:[1-9]|1[0-5])", column)
            or re.fullmatch(r"D(?:[1-9]|1[0-5])_minus_day", column)
            or re.fullmatch(r"M[1-9]", column)
            or re.fullmatch(r"id_(?:0[1-9]|[12][0-9]|3[0-8])", column)
        )
        engineered = bool(
            column.startswith("uid_d1_email_seq_")
            or column.startswith("uid_card_addr_d1_email_seq_")
            or (
                column.endswith("_freq")
                and column.startswith(
                    (
                        "card",
                        "addr",
                        "P_email",
                        "R_email",
                        "Device",
                        "uid_",
                    )
                )
            )
        )
        if column in explicit or column in top_v or raw_family or engineered:
            result.append(column)
    # This block was explicitly reported as time-inconsistent by the winning
    # solution and is removed before any local-label model selection.
    result = [
        column
        for column in result
        if not (
            re.fullmatch(r"V\d+", column)
            and 322 <= int(column[1:]) <= 339
        )
    ]
    return list(dict.fromkeys(result))


def encode_categories(
    train: pd.DataFrame,
    test: pd.DataFrame,
    categorical: list[str],
) -> None:
    for column in categorical:
        combined = pd.concat(
            [train[column], test[column]], ignore_index=True, copy=False
        ).astype("string")
        codes, _ = pd.factorize(combined, sort=False)
        train[column] = codes[: len(train)].astype("int32")
        test[column] = codes[len(train) :].astype("int32")


def prepare_raw_matrix(
    oof: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    print("Building target-free raw meta features...", flush=True)
    train = read_and_merge(ROOT, "train")
    test = read_and_merge(ROOT, "test")
    if client.TARGET in test:
        raise RuntimeError("Target reached official test")
    train, test, all_features, categorical = build_features(
        train,
        test,
        giba_features=True,
        frequency_mode="selected",
        v307_chain_features=True,
    )
    selected = select_raw_features(all_features)
    selected_categorical = [column for column in categorical if column in selected]
    encode_categories(train, test, selected_categorical)
    oof_rows = oof["row_index"].to_numpy(dtype="int64")
    oof_raw = train.iloc[oof_rows][selected].reset_index(drop=True)
    test_raw = test[selected].reset_index(drop=True)
    for frame in (oof_raw, test_raw):
        for column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").astype(
                "float32"
            )
    return oof_raw, test_raw, selected


def make_features(
    predictions: pd.DataFrame,
    membership: pd.DataFrame,
    raw: pd.DataFrame,
    fold: pd.Series | None,
) -> pd.DataFrame:
    source_features = meta.build_meta_features(
        predictions, META_SOURCES, fold=fold
    ).reset_index(drop=True)
    return pd.concat(
        [
            source_features,
            membership.reset_index(drop=True),
            raw.reset_index(drop=True),
        ],
        axis=1,
    ).astype("float32")


def search_lgb(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    raw: pd.DataFrame,
) -> tuple[dict, list[dict], np.ndarray]:
    features = make_features(oof, membership, raw, fold=oof["fold"])
    train_mask = oof["fold"].lt(client.META_DEV_FOLD).to_numpy()
    valid_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    y_train = oof.loc[train_mask, client.TARGET].to_numpy(dtype="int8")
    y_valid = oof.loc[valid_mask, client.TARGET].to_numpy(dtype="int8")
    rows = []
    predictions = {}
    for index, config in enumerate(LGB_CONFIGS):
        model = lgb.LGBMClassifier(**LGB_PARAMS, **config)
        model.fit(
            features.loc[train_mask],
            y_train,
            eval_set=[(features.loc[valid_mask], y_valid)],
            callbacks=[
                lgb.early_stopping(180, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        prediction = model.predict_proba(features.loc[valid_mask])[:, 1]
        row = {
            "config_index": index,
            **config,
            "features": int(features.shape[1]),
            "best_iteration": int(model.best_iteration_),
            "dev_auc": float(roc_auc_score(y_valid, prediction)),
        }
        print(json.dumps(row), flush=True)
        rows.append(row)
        predictions[index] = prediction
        del model
        gc.collect()
    rows.sort(key=lambda row: (row["dev_auc"], -row["num_leaves"]), reverse=True)
    selected = rows[0]
    return selected, rows, predictions[int(selected["config_index"])]


def train_lock(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    raw: pd.DataFrame,
    selected: dict,
) -> tuple[lgb.LGBMClassifier, np.ndarray, int]:
    features = make_features(oof, membership, raw, fold=oof["fold"])
    train_mask = oof["fold"].lt(client.META_LOCK_FOLD).to_numpy()
    valid_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    iterations = max(30, int(np.ceil(selected["best_iteration"] * 1.15)))
    model = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            **LGB_CONFIGS[int(selected["config_index"])],
            "n_estimators": iterations,
        }
    )
    model.fit(
        features.loc[train_mask],
        oof.loc[train_mask, client.TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    return model, model.predict_proba(features.loc[valid_mask])[:, 1], iterations


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    oof = featureview.build_oof()
    membership = pd.read_csv(ROOT / "honest_client_meta/oof_client_features.csv")
    raw_oof, raw_test_unused, selected_raw = prepare_raw_matrix(oof)
    del raw_test_unused
    gc.collect()
    RAW_FEATURES_PATH.write_text(json.dumps(selected_raw, indent=2), encoding="utf-8")

    raw_train = client.prepare_raw(
        pd.read_csv(ROOT / "train_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    raw_test = client.prepare_raw(
        pd.read_csv(ROOT / "test_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    _, _, fold_groups, _ = client.build_client_features(oof, raw_train, raw_test)
    uid_raw = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(uid_raw)
    dev_current, dev_report = client.current_meta_prediction(oof, uid, client.META_DEV_FOLD)
    lock_current, lock_report = client.current_meta_prediction(oof, uid, client.META_LOCK_FOLD)

    selected_model, model_search, dev_raw = search_lgb(oof, membership, raw_oof)
    lock_model, lock_raw, lock_iterations = train_lock(
        oof, membership, raw_oof, selected_model
    )
    lock_model.booster_.save_model(WORK_DIR / "lock_lgb.txt")
    feature_report = json.loads(
        (ROOT / "honest_featureview_meta/search_report.json").read_text(encoding="utf-8")
    )
    postprocess = feature_report["postprocess_locked_from_dev"]
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    selected_weights, weight_search = heavy.search_segment_weights(
        oof.loc[dev_mask, client.TARGET].to_numpy(dtype="int8"),
        dev_current,
        dev_raw,
        segments.segment_labels(membership.loc[dev_mask].reset_index(drop=True)),
        raw_train.iloc[dev_rows]["TransactionDT"].to_numpy(),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
    )
    lock_prediction = segments.segmented_blend(
        lock_current,
        lock_raw,
        segments.segment_labels(membership.loc[lock_mask].reset_index(drop=True)),
        selected_weights["weights"],
    )
    lock_prediction = client.apply_postprocess(
        lock_prediction, fold_groups[client.META_LOCK_FOLD], postprocess
    )
    y_lock = oof.loc[lock_mask, client.TARGET].to_numpy(dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    previous = float(feature_report["candidate_lock_auc"])
    report = {
        "data_policy": "official train/test covariates; train OOF labels only",
        "feature_policy": "family rules fixed before nested validation",
        "raw_feature_count": len(selected_raw),
        "raw_features": selected_raw,
        "meta_sources": list(META_SOURCES),
        "selected_model": selected_model,
        "model_search": model_search,
        "selected_segment_weights": selected_weights,
        "weight_search_top": weight_search[:30],
        "postprocess_locked_from_dev": postprocess,
        "current_meta": {"dev": dev_report, "lock": lock_report},
        "lock_iterations": lock_iterations,
        "previous_featureview_lock_auc": previous,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - previous,
        "accepted": bool(selected_weights["gain"] > 0 and lock_auc > previous),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "raw_feature_count": len(selected_raw),
                "selected_model": selected_model,
                "selected_segment_weights": selected_weights,
                "previous_featureview_lock_auc": previous,
                "candidate_lock_auc": lock_auc,
                "lock_gain": lock_auc - previous,
                "accepted": report["accepted"],
                "elapsed_minutes": report["elapsed_minutes"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting search_honest_raw_feature_meta.py


In [26]:
%%writefile train_boost3_neural_stack.py
from pathlib import Path
import argparse
import gc
import json
import random
import time

from catboost import CatBoostClassifier
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb

from fraud_features import TARGET, build_features, read_and_merge


DATA_DIR = Path(__file__).resolve().parent
SEED = 42
META_SEEDS = (42, 2026, 3407)
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

CAT_PARAMS = {
    "iterations": 1400,
    "depth": 8,
    "learning_rate": 0.06,
    "l2_leaf_reg": 8,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "rsm": 0.90,
    "border_count": 128,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "one_hot_max_size": 10,
    "max_ctr_complexity": 1,
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": 100,
}

LGB_PARAMS = {
    "n_estimators": 1800,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.03,
    "min_child_samples": 40,
    "subsample": 0.70,
    "subsample_freq": 1,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.5,
    "reg_lambda": 5.0,
    "max_bin": 255,
    "max_depth": -1,
    "extra_trees": True,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}

XGB_PARAMS = {
    "n_estimators": 1800,
    "learning_rate": 0.03,
    "max_depth": 7,
    "min_child_weight": 20,
    "subsample": 0.80,
    "colsample_bytree": 0.75,
    "reg_alpha": 0.5,
    "reg_lambda": 10.0,
    "gamma": 0.05,
    "max_bin": 256,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "early_stopping_rounds": 120,
    "random_state": SEED,
    "n_jobs": -1,
}


def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value)}")


def save_metrics(metrics):
    (DATA_DIR / "boost3_neural_stack_metrics.json").write_text(
        json.dumps(metrics, indent=2, default=json_default),
        encoding="utf-8",
    )


def is_giba_feature(column):
    return (
        column == "D1_origin_day"
        or column.startswith("uid_d1_email")
        or column.startswith("uid_card_addr_d1")
    )


def rank_prediction(values):
    return pd.Series(values).rank(method="average", pct=True).to_numpy()


def make_folds(transaction_dt):
    day = (transaction_dt // 86400).astype("int16")
    specs = [
        (0, 30, 60, 75),
        (1, 45, 75, 90),
        (2, 60, 90, int(day.max()) + 1),
    ]
    folds = []
    for fold, train_end, valid_start, valid_end in specs:
        train_index = day.index[day < train_end]
        valid_index = day.index[(day >= valid_start) & (day < valid_end)]
        folds.append(
            {
                "fold": fold,
                "train_end_day": train_end,
                "valid_start_day": valid_start,
                "valid_end_day": valid_end,
                "train_index": train_index,
                "valid_index": valid_index,
            }
        )
    return folds


def prediction_frame(fold, valid_index, train_ids, y, prediction, name):
    return pd.DataFrame(
        {
            "row_index": valid_index.to_numpy(),
            "TransactionID": train_ids.loc[valid_index].to_numpy(),
            "fold": fold,
            TARGET: y.loc[valid_index].to_numpy(),
            name: prediction,
        }
    )


def train_catboost_oof(
    train,
    y,
    train_ids,
    feature_columns,
    categorical_columns,
    folds,
    metrics,
    force,
):
    rows = []
    for fold_info in folds:
        fold = fold_info["fold"]
        cache_path = DATA_DIR / f"stack_catboost_fold_{fold}.csv"
        model_path = DATA_DIR / f"stack_catboost_fold_{fold}.cbm"
        if cache_path.exists() and not force:
            print(f"Loading cached CatBoost fold {fold}", flush=True)
            rows.append(pd.read_csv(cache_path))
            continue

        print(f"\nCatBoost OOF fold {fold}", flush=True)
        started = time.time()
        model = CatBoostClassifier(
            **{
                **CAT_PARAMS,
                "random_seed": SEED + fold,
            }
        )
        model.fit(
            train.loc[fold_info["train_index"], feature_columns],
            y.loc[fold_info["train_index"]],
            cat_features=categorical_columns,
            eval_set=(
                train.loc[fold_info["valid_index"], feature_columns],
                y.loc[fold_info["valid_index"]],
            ),
            early_stopping_rounds=120,
            use_best_model=True,
        )
        prediction = model.predict_proba(
            train.loc[fold_info["valid_index"], feature_columns]
        )[:, 1]
        frame = prediction_frame(
            fold,
            fold_info["valid_index"],
            train_ids,
            y,
            prediction,
            "catboost",
        )
        frame.to_csv(cache_path, index=False)
        model.save_model(model_path)
        rows.append(frame)
        metrics["catboost_folds"][str(fold)] = {
            "auc": roc_auc_score(frame[TARGET], frame["catboost"]),
            "best_iteration": model.get_best_iteration() + 1,
            "minutes": (time.time() - started) / 60,
        }
        save_metrics(metrics)
        del model, prediction, frame
        gc.collect()
    return pd.concat(rows, ignore_index=True)


def convert_categories_for_lgb(train, test, categorical_columns):
    for column in categorical_columns:
        categories = pd.Index(train[column].dropna().unique())
        dtype = pd.CategoricalDtype(categories=categories)
        train[column] = train[column].astype(dtype)
        test[column] = test[column].astype(dtype)


def train_lightgbm_oof(
    train,
    y,
    train_ids,
    feature_columns,
    categorical_columns,
    folds,
    metrics,
    force,
):
    rows = []
    for fold_info in folds:
        fold = fold_info["fold"]
        cache_path = DATA_DIR / f"stack_lightgbm_fold_{fold}.csv"
        model_path = DATA_DIR / f"stack_lightgbm_fold_{fold}.txt"
        if cache_path.exists() and not force:
            print(f"Loading cached LightGBM fold {fold}", flush=True)
            rows.append(pd.read_csv(cache_path))
            continue

        print(f"\nLightGBM OOF fold {fold}", flush=True)
        started = time.time()
        model = lgb.LGBMClassifier(
            **{
                **LGB_PARAMS,
                "random_state": SEED + fold,
            }
        )
        model.fit(
            train.loc[fold_info["train_index"], feature_columns],
            y.loc[fold_info["train_index"]],
            categorical_feature=categorical_columns,
            eval_set=[
                (
                    train.loc[fold_info["valid_index"], feature_columns],
                    y.loc[fold_info["valid_index"]],
                )
            ],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(120, verbose=False),
                lgb.log_evaluation(100),
            ],
        )
        prediction = model.predict_proba(
            train.loc[fold_info["valid_index"], feature_columns],
            num_iteration=model.best_iteration_,
        )[:, 1]
        frame = prediction_frame(
            fold,
            fold_info["valid_index"],
            train_ids,
            y,
            prediction,
            "lightgbm",
        )
        frame.to_csv(cache_path, index=False)
        model.booster_.save_model(
            model_path,
            num_iteration=model.best_iteration_,
        )
        rows.append(frame)
        metrics["lightgbm_folds"][str(fold)] = {
            "auc": roc_auc_score(frame[TARGET], frame["lightgbm"]),
            "best_iteration": model.best_iteration_,
            "minutes": (time.time() - started) / 60,
        }
        save_metrics(metrics)
        del model, prediction, frame
        gc.collect()
    return pd.concat(rows, ignore_index=True)


def select_xgb_features(feature_columns, limit=240):
    scores = pd.Series(0.0, index=feature_columns)

    cat_model = CatBoostClassifier()
    cat_model.load_model(DATA_DIR / "catboost_giba_validation.cbm")
    cat_importance = pd.Series(
        cat_model.get_feature_importance(),
        index=cat_model.feature_names_,
    )
    if cat_importance.sum() > 0:
        scores = scores.add(cat_importance / cat_importance.sum(), fill_value=0)

    lgb_model = lgb.Booster(
        model_file=str(DATA_DIR / "lightgbm_giba_validation.txt")
    )
    lgb_importance = pd.Series(
        lgb_model.feature_importance(importance_type="gain"),
        index=lgb_model.feature_name(),
    )
    if lgb_importance.sum() > 0:
        scores = scores.add(lgb_importance / lgb_importance.sum(), fill_value=0)

    top_features = scores.sort_values(ascending=False).head(limit).index.tolist()
    giba_features = [
        column for column in feature_columns if is_giba_feature(column)
    ]
    selected = list(dict.fromkeys([*top_features, *giba_features]))
    return selected, scores.sort_values(ascending=False)


def convert_categories_for_xgb(train, test, categorical_columns):
    for column in categorical_columns:
        train[column] = train[column].cat.codes.astype("int32")
        test[column] = test[column].cat.codes.astype("int32")


def train_xgboost_oof(
    train,
    y,
    train_ids,
    xgb_features,
    folds,
    metrics,
    force,
):
    rows = []
    for fold_info in folds:
        fold = fold_info["fold"]
        cache_path = DATA_DIR / f"stack_xgboost_fold_{fold}.csv"
        model_path = DATA_DIR / f"stack_xgboost_fold_{fold}.json"
        if cache_path.exists() and not force:
            print(f"Loading cached XGBoost fold {fold}", flush=True)
            rows.append(pd.read_csv(cache_path))
            continue

        print(f"\nXGBoost OOF fold {fold}", flush=True)
        started = time.time()
        model = xgb.XGBClassifier(
            **{
                **XGB_PARAMS,
                "random_state": SEED + fold,
            }
        )
        model.fit(
            train.loc[fold_info["train_index"], xgb_features],
            y.loc[fold_info["train_index"]],
            eval_set=[
                (
                    train.loc[fold_info["valid_index"], xgb_features],
                    y.loc[fold_info["valid_index"]],
                )
            ],
            verbose=100,
        )
        prediction = model.predict_proba(
            train.loc[fold_info["valid_index"], xgb_features]
        )[:, 1]
        frame = prediction_frame(
            fold,
            fold_info["valid_index"],
            train_ids,
            y,
            prediction,
            "xgboost",
        )
        frame.to_csv(cache_path, index=False)
        model.save_model(model_path)
        rows.append(frame)
        metrics["xgboost_folds"][str(fold)] = {
            "auc": roc_auc_score(frame[TARGET], frame["xgboost"]),
            "best_iteration": int(model.best_iteration + 1),
            "minutes": (time.time() - started) / 60,
        }
        save_metrics(metrics)
        del model, prediction, frame
        gc.collect()
    return pd.concat(rows, ignore_index=True)


def merge_oof(cat_oof, lgb_oof, xgb_oof):
    keys = ["row_index", "TransactionID", "fold", TARGET]
    result = cat_oof.merge(lgb_oof, on=keys, validate="one_to_one")
    result = result.merge(xgb_oof, on=keys, validate="one_to_one")
    return result.sort_values(["fold", "row_index"]).reset_index(drop=True)


def build_meta_features(predictions, fold=None):
    base_columns = ["catboost", "lightgbm", "xgboost"]
    probability = predictions[base_columns].clip(1e-6, 1 - 1e-6)
    if fold is None:
        ranks = probability.rank(method="average", pct=True)
    else:
        ranks = probability.groupby(fold).rank(method="average", pct=True)

    features = {}
    for column in base_columns:
        features[f"{column}_prob"] = probability[column]
        features[f"{column}_logit"] = np.log(
            probability[column] / (1 - probability[column])
        )
        features[f"{column}_rank"] = ranks[column]
        features[f"{column}_rank_sq"] = ranks[column] ** 2

    features["rank_mean"] = ranks.mean(axis=1)
    features["rank_std"] = ranks.std(axis=1)
    features["rank_min"] = ranks.min(axis=1)
    features["rank_max"] = ranks.max(axis=1)
    features["rank_cat_lgb_diff"] = (
        ranks["catboost"] - ranks["lightgbm"]
    ).abs()
    features["rank_cat_xgb_diff"] = (
        ranks["catboost"] - ranks["xgboost"]
    ).abs()
    features["rank_lgb_xgb_diff"] = (
        ranks["lightgbm"] - ranks["xgboost"]
    ).abs()
    return pd.DataFrame(features, index=predictions.index).astype("float32")


class MetaMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.linear = nn.Linear(input_size, 1)
        self.hidden = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.SiLU(),
            nn.Dropout(0.08),
            nn.Linear(32, 16),
            nn.SiLU(),
            nn.Dropout(0.05),
            nn.Linear(16, 1),
        )

    def forward(self, values):
        return self.linear(values) + 0.20 * self.hidden(values)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def predict_mlp(model, values):
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(values, dtype=torch.float32)
        return torch.sigmoid(model(tensor)).squeeze(1).cpu().numpy()


def train_mlp_with_validation(
    X_train,
    y_train,
    X_valid,
    y_valid,
    seed,
):
    seed_everything(seed)
    model = MetaMLP(X_train.shape[1])
    positive_weight = float(
        np.sqrt((len(y_train) - y_train.sum()) / y_train.sum())
    )
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([positive_weight], dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=2e-4,
    )
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train, dtype=torch.float32),
            torch.as_tensor(y_train, dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=8192,
        shuffle=True,
    )

    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    patience = 12
    stale_epochs = 0
    for epoch in range(1, 121):
        model.train()
        for values, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(values), labels)
            loss.backward()
            optimizer.step()

        valid_prediction = predict_mlp(model, X_valid)
        valid_auc = roc_auc_score(y_valid, valid_prediction)
        if valid_auc > best_auc + 1e-6:
            best_auc = valid_auc
            best_epoch = epoch
            best_state = {
                key: value.detach().clone()
                for key, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
        if epoch % 10 == 0:
            print(
                f"MLP seed={seed} epoch={epoch} "
                f"valid_auc={valid_auc:.9f} best={best_auc:.9f}",
                flush=True,
            )
        if stale_epochs >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_epoch, predict_mlp(model, X_valid), best_auc


def train_mlp_fixed_epochs(X, y, X_test, seed, epochs):
    seed_everything(seed)
    print(f"Final MLP seed={seed}, epochs={epochs}", flush=True)
    model = MetaMLP(X.shape[1])
    positive_weight = float(np.sqrt((len(y) - y.sum()) / y.sum()))
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([positive_weight], dtype=torch.float32)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=2e-4,
    )
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(X, dtype=torch.float32),
            torch.as_tensor(y, dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=8192,
        shuffle=True,
    )
    for _ in range(epochs):
        model.train()
        for values, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(values), labels)
            loss.backward()
            optimizer.step()
    return model, predict_mlp(model, X_test)


def best_rank_blend(y_true, first, second):
    first_rank = rank_prediction(first)
    second_rank = rank_prediction(second)
    rows = []
    for first_weight in np.linspace(0, 1, 101):
        prediction = (
            first_weight * first_rank
            + (1 - first_weight) * second_rank
        )
        rows.append(
            {
                "neural_weight": first_weight,
                "linear_weight": 1 - first_weight,
                "auc": roc_auc_score(y_true, prediction),
            }
        )
    return max(rows, key=lambda row: row["auc"])


def apply_uid_postprocess(prediction, uid, method, weight):
    frame = pd.DataFrame({"prediction": prediction, "uid": uid.to_numpy()})
    if method == "none":
        return prediction
    aggregate = frame.groupby("uid", sort=False)["prediction"].transform(method)
    return (1 - weight) * prediction + weight * aggregate.to_numpy()


def choose_uid_postprocess(y_true, prediction, uid):
    candidates = [{"method": "none", "weight": 0.0}]
    for method in ("mean", "max"):
        for weight in (0.25, 0.50, 0.75, 1.0):
            candidates.append({"method": method, "weight": weight})
    for candidate in candidates:
        processed = apply_uid_postprocess(
            prediction,
            uid,
            candidate["method"],
            candidate["weight"],
        )
        candidate["auc"] = roc_auc_score(y_true, processed)
    return max(candidates, key=lambda row: row["auc"]), candidates


parser = argparse.ArgumentParser()
parser.add_argument(
    "--force",
    action="store_true",
    help="Ignore cached OOF/test predictions and retrain every model.",
)
args = parser.parse_args()

started_at = time.time()
metrics_path = DATA_DIR / "boost3_neural_stack_metrics.json"
if metrics_path.exists() and not args.force:
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    metrics.setdefault("catboost_folds", {})
    metrics.setdefault("lightgbm_folds", {})
    metrics.setdefault("xgboost_folds", {})
else:
    metrics = {
        "scheme": "3 boosting models -> PyTorch MLP",
        "base_models": ["CatBoost", "LightGBM", "XGBoost"],
        "validation": (
            "three non-overlapping temporal OOF windows with 30-day gap"
        ),
        "catboost_folds": {},
        "lightgbm_folds": {},
        "xgboost_folds": {},
    }

print("Reading data and building Giba UID features...", flush=True)
train = read_and_merge(DATA_DIR, "train")
test = read_and_merge(DATA_DIR, "test")
train, test, feature_columns, categorical_columns = build_features(
    train,
    test,
    giba_features=True,
)
y = train.pop(TARGET)
train_ids = train.pop("TransactionID")
test_ids = test.pop("TransactionID")
transaction_dt = train["TransactionDT"].copy()
uid_train = train["uid_card_addr_d1_email"].astype("string").copy()
uid_test = test["uid_card_addr_d1_email"].astype("string").copy()
folds = make_folds(transaction_dt)

metrics["features"] = {
    "boosting_features": len(feature_columns),
    "categorical": len(categorical_columns),
}
metrics["folds"] = [
    {
        "fold": info["fold"],
        "train_rows": len(info["train_index"]),
        "valid_rows": len(info["valid_index"]),
        "train_end_day": info["train_end_day"],
        "valid_start_day": info["valid_start_day"],
        "valid_end_day": info["valid_end_day"],
    }
    for info in folds
]
save_metrics(metrics)

print("\nStage 1/4: CatBoost OOF...", flush=True)
cat_oof = train_catboost_oof(
    train,
    y,
    train_ids,
    feature_columns,
    categorical_columns,
    folds,
    metrics,
    args.force,
)

cat_test_path = DATA_DIR / "stack_test_catboost.npy"
if cat_test_path.exists() and not args.force:
    cat_test = np.load(cat_test_path)
else:
    final_cat = CatBoostClassifier()
    final_cat.load_model(DATA_DIR / "catboost_giba_final.cbm")
    cat_test = final_cat.predict_proba(
        test[final_cat.feature_names_]
    )[:, 1]
    np.save(cat_test_path, cat_test)
    del final_cat
    gc.collect()

print("\nStage 2/4: LightGBM OOF...", flush=True)
convert_categories_for_lgb(train, test, categorical_columns)
lgb_oof = train_lightgbm_oof(
    train,
    y,
    train_ids,
    feature_columns,
    categorical_columns,
    folds,
    metrics,
    args.force,
)

lgb_test_path = DATA_DIR / "stack_test_lightgbm.npy"
if lgb_test_path.exists() and not args.force:
    lgb_test = np.load(lgb_test_path)
else:
    final_lgb = lgb.Booster(
        model_file=str(DATA_DIR / "lightgbm_giba_final.txt")
    )
    lgb_test = final_lgb.predict(test[final_lgb.feature_name()])
    np.save(lgb_test_path, lgb_test)
    del final_lgb
    gc.collect()

print("\nStage 3/4: XGBoost OOF and final model...", flush=True)
xgb_features, xgb_importance = select_xgb_features(feature_columns)
xgb_importance.rename("combined_importance").to_csv(
    DATA_DIR / "stack_xgboost_feature_ranking.csv",
    header=True,
)
metrics["features"]["xgboost_selected"] = len(xgb_features)
convert_categories_for_xgb(train, test, categorical_columns)
xgb_oof = train_xgboost_oof(
    train,
    y,
    train_ids,
    xgb_features,
    folds,
    metrics,
    args.force,
)

xgb_test_path = DATA_DIR / "stack_test_xgboost.npy"
xgb_final_path = DATA_DIR / "stack_xgboost_final.json"
if xgb_test_path.exists() and xgb_final_path.exists() and not args.force:
    xgb_test = np.load(xgb_test_path)
else:
    best_iterations = [
        row["best_iteration"]
        for row in metrics["xgboost_folds"].values()
    ]
    final_iterations = min(
        XGB_PARAMS["n_estimators"],
        int(np.ceil(np.median(best_iterations) * 1.20)),
    )
    print(f"Training final XGBoost: {final_iterations} trees", flush=True)
    final_xgb = xgb.XGBClassifier(
        **{
            **XGB_PARAMS,
            "n_estimators": final_iterations,
            "early_stopping_rounds": None,
            "random_state": 2026,
        }
    )
    final_xgb.fit(train[xgb_features], y, verbose=100)
    xgb_test = final_xgb.predict_proba(test[xgb_features])[:, 1]
    final_xgb.save_model(xgb_final_path)
    np.save(xgb_test_path, xgb_test)
    metrics["xgboost_final_iterations"] = final_iterations
    save_metrics(metrics)
    del final_xgb
    gc.collect()

print("\nStage 4/4: neural meta-learner...", flush=True)
oof = merge_oof(cat_oof, lgb_oof, xgb_oof)
oof.to_csv(DATA_DIR / "boost3_oof_predictions.csv", index=False)
base_test = pd.DataFrame(
    {
        "catboost": cat_test,
        "lightgbm": lgb_test,
        "xgboost": xgb_test,
    }
)
meta_features = build_meta_features(oof, fold=oof["fold"])
test_meta_features = build_meta_features(base_test)
del (
    train,
    test,
    transaction_dt,
    cat_oof,
    lgb_oof,
    xgb_oof,
    feature_columns,
    categorical_columns,
)
gc.collect()

meta_train_mask = oof["fold"] < 2
meta_valid_mask = oof["fold"] == 2
y_meta_train = oof.loc[meta_train_mask, TARGET].to_numpy(dtype="float32")
y_meta_valid = oof.loc[meta_valid_mask, TARGET].to_numpy(dtype="float32")

scaler = StandardScaler()
X_meta_train = scaler.fit_transform(
    meta_features.loc[meta_train_mask]
).astype("float32")
X_meta_valid = scaler.transform(
    meta_features.loc[meta_valid_mask]
).astype("float32")

linear = LogisticRegression(
    C=0.20,
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)
linear.fit(X_meta_train, y_meta_train)
linear_valid = linear.predict_proba(X_meta_valid)[:, 1]

mlp_valid_parts = []
best_epochs = []
for seed in META_SEEDS:
    model, epoch, prediction, score = train_mlp_with_validation(
        X_meta_train,
        y_meta_train,
        X_meta_valid,
        y_meta_valid,
        seed,
    )
    mlp_valid_parts.append(prediction)
    best_epochs.append(epoch)
    metrics.setdefault("meta_seed_results", {})[str(seed)] = {
        "best_epoch": epoch,
        "valid_auc": score,
    }
    del model
    gc.collect()

mlp_valid = np.mean(mlp_valid_parts, axis=0)
rank_average_valid = oof.loc[
    meta_valid_mask,
    ["catboost", "lightgbm", "xgboost"],
].rank(pct=True).mean(axis=1).to_numpy()
giba_reference_valid = (
    0.775
    * rank_prediction(oof.loc[meta_valid_mask, "catboost"].to_numpy())
    + 0.225
    * rank_prediction(oof.loc[meta_valid_mask, "lightgbm"].to_numpy())
)
meta_blend = best_rank_blend(
    y_meta_valid,
    mlp_valid,
    linear_valid,
)
meta_blend_valid = (
    meta_blend["neural_weight"] * rank_prediction(mlp_valid)
    + meta_blend["linear_weight"] * rank_prediction(linear_valid)
)

valid_indices = oof.loc[meta_valid_mask, "row_index"].astype("int64")
uid_best, uid_candidates = choose_uid_postprocess(
    y_meta_valid,
    meta_blend_valid,
    uid_train.loc[valid_indices],
)
neural_uid_best, neural_uid_candidates = choose_uid_postprocess(
    y_meta_valid,
    rank_prediction(mlp_valid),
    uid_train.loc[valid_indices],
)

metrics["meta_validation"] = {
    "rows": int(meta_valid_mask.sum()),
    "catboost_auc": roc_auc_score(
        y_meta_valid,
        oof.loc[meta_valid_mask, "catboost"],
    ),
    "lightgbm_auc": roc_auc_score(
        y_meta_valid,
        oof.loc[meta_valid_mask, "lightgbm"],
    ),
    "xgboost_auc": roc_auc_score(
        y_meta_valid,
        oof.loc[meta_valid_mask, "xgboost"],
    ),
    "rank_average_auc": roc_auc_score(
        y_meta_valid,
        rank_average_valid,
    ),
    "giba_cat_lgb_reference_auc": roc_auc_score(
        y_meta_valid,
        giba_reference_valid,
    ),
    "linear_meta_auc": roc_auc_score(y_meta_valid, linear_valid),
    "neural_meta_auc": roc_auc_score(y_meta_valid, mlp_valid),
    "neural_linear_blend": meta_blend,
    "uid_postprocess": uid_best,
    "neural_only_uid_postprocess": neural_uid_best,
    "gain_over_giba_reference": (
        uid_best["auc"]
        - roc_auc_score(y_meta_valid, giba_reference_valid)
    ),
}
metrics["uid_postprocess_candidates"] = uid_candidates
metrics["neural_only_uid_postprocess_candidates"] = neural_uid_candidates
save_metrics(metrics)
print(json.dumps(metrics["meta_validation"], indent=2), flush=True)

all_scaler = StandardScaler()
X_meta_all = all_scaler.fit_transform(meta_features).astype("float32")
X_meta_test = all_scaler.transform(test_meta_features).astype("float32")
y_meta_all = oof[TARGET].to_numpy(dtype="float32")
np.savez(
    DATA_DIR / "boost3_meta_scaler.npz",
    mean=all_scaler.mean_,
    scale=all_scaler.scale_,
    features=np.array(meta_features.columns),
)

final_linear = LogisticRegression(
    C=0.20,
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)
final_linear.fit(X_meta_all, y_meta_all)
linear_test = final_linear.predict_proba(X_meta_test)[:, 1]

final_epochs = {
    str(seed): max(1, int(epoch))
    for seed, epoch in zip(META_SEEDS, best_epochs)
}
mlp_test_parts = []
for seed in META_SEEDS:
    model, prediction = train_mlp_fixed_epochs(
        X_meta_all,
        y_meta_all,
        X_meta_test,
        seed,
        final_epochs[str(seed)],
    )
    torch.save(
        model.state_dict(),
        DATA_DIR / f"boost3_meta_mlp_seed_{seed}.pt",
    )
    mlp_test_parts.append(prediction)
    del model
    gc.collect()

mlp_test = np.mean(mlp_test_parts, axis=0)
neural_only_test = rank_prediction(mlp_test)
neural_only_uid_test = apply_uid_postprocess(
    neural_only_test,
    uid_test,
    neural_uid_best["method"],
    neural_uid_best["weight"],
)
stack_test = (
    meta_blend["neural_weight"] * rank_prediction(mlp_test)
    + meta_blend["linear_weight"] * rank_prediction(linear_test)
)
stack_uid_test = apply_uid_postprocess(
    stack_test,
    uid_test,
    uid_best["method"],
    uid_best["weight"],
)

raw_submission = pd.DataFrame(
    {
        "TransactionID": test_ids,
        TARGET: stack_test,
    }
)
raw_submission.to_csv(
    DATA_DIR / "submission_boost3_neural_raw.csv",
    index=False,
)
neural_only_submission = raw_submission.copy()
neural_only_submission[TARGET] = neural_only_uid_test
neural_only_submission.to_csv(
    DATA_DIR / "submission_boost3_neural_only.csv",
    index=False,
)
submission = raw_submission.copy()
submission[TARGET] = stack_uid_test
submission.to_csv(
    DATA_DIR / "submission_boost3_neural_stack.csv",
    index=False,
)

metrics["final_meta_epochs"] = final_epochs
metrics["elapsed_minutes"] = (time.time() - started_at) / 60
metrics["submissions"] = [
    "submission_boost3_neural_stack.csv",
    "submission_boost3_neural_only.csv",
    "submission_boost3_neural_raw.csv",
]
save_metrics(metrics)

print("\nBoost3 neural stack complete", flush=True)
print(json.dumps(metrics, indent=2, default=json_default), flush=True)


Overwriting train_boost3_neural_stack.py


In [27]:
%%writefile train_clean_foundation.py
"""Train clean CatBoost and LightGBM foundation models from provided data only."""

from __future__ import annotations

import gc
import json
from pathlib import Path
import time

from catboost import CatBoostClassifier
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from fraud_features import TARGET, build_features, read_and_merge


ROOT = Path(__file__).resolve().parent
SEED = 42

CAT_PARAMS = {
    "iterations": 1400,
    "depth": 8,
    "learning_rate": 0.06,
    "l2_leaf_reg": 8,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "rsm": 0.90,
    "border_count": 128,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "one_hot_max_size": 10,
    "max_ctr_complexity": 1,
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": 100,
}

LGB_PARAMS = {
    "n_estimators": 1800,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.03,
    "min_child_samples": 40,
    "subsample": 0.70,
    "subsample_freq": 1,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.5,
    "reg_lambda": 5.0,
    "max_bin": 255,
    "max_depth": -1,
    "extra_trees": True,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}


def main() -> None:
    started = time.time()
    print("Reading provided train/test only...", flush=True)
    train = read_and_merge(ROOT, "train")
    test = read_and_merge(ROOT, "test")
    if TARGET in test:
        raise AssertionError("Competition test unexpectedly contains target")

    print("Building target-free train+test features...", flush=True)
    train, test, features, categorical = build_features(
        train,
        test,
        giba_features=True,
    )
    y = train.pop(TARGET).astype("int8")
    train.drop(columns="TransactionID", inplace=True)
    test.drop(columns="TransactionID", inplace=True)
    transaction_dt = train["TransactionDT"].copy()
    cutoff = transaction_dt.quantile(0.80)
    train_index = train.index[transaction_dt < cutoff]
    valid_index = train.index[transaction_dt >= cutoff]

    print("Selecting CatBoost iterations on the final 20% of train...", flush=True)
    cat_validation = CatBoostClassifier(**CAT_PARAMS)
    cat_validation.fit(
        train.loc[train_index, features],
        y.loc[train_index],
        cat_features=categorical,
        eval_set=(train.loc[valid_index, features], y.loc[valid_index]),
        early_stopping_rounds=120,
        use_best_model=True,
    )
    cat_valid_prediction = cat_validation.predict_proba(
        train.loc[valid_index, features]
    )[:, 1]
    cat_auc = float(roc_auc_score(y.loc[valid_index], cat_valid_prediction))
    cat_iterations = int(cat_validation.get_best_iteration() + 1)
    cat_validation.save_model(ROOT / "catboost_giba_validation.cbm")

    print(f"Training full CatBoost with {cat_iterations} iterations...", flush=True)
    cat_final = CatBoostClassifier(
        **{**CAT_PARAMS, "iterations": cat_iterations, "verbose": 100}
    )
    cat_final.fit(train[features], y, cat_features=categorical)
    cat_final.save_model(ROOT / "catboost_giba_final.cbm")
    del cat_validation, cat_final, cat_valid_prediction
    gc.collect()

    print("Converting categories and selecting LightGBM iterations...", flush=True)
    for column in categorical:
        categories = pd.Index(train[column].dropna().unique())
        dtype = pd.CategoricalDtype(categories=categories)
        train[column] = train[column].astype(dtype)
        test[column] = test[column].astype(dtype)

    lgb_validation = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_validation.fit(
        train.loc[train_index, features],
        y.loc[train_index],
        categorical_feature=categorical,
        eval_set=[(train.loc[valid_index, features], y.loc[valid_index])],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(120, verbose=False),
            lgb.log_evaluation(100),
        ],
    )
    lgb_valid_prediction = lgb_validation.predict_proba(
        train.loc[valid_index, features],
        num_iteration=lgb_validation.best_iteration_,
    )[:, 1]
    lgb_auc = float(roc_auc_score(y.loc[valid_index], lgb_valid_prediction))
    lgb_iterations = int(lgb_validation.best_iteration_)
    lgb_validation.booster_.save_model(
        ROOT / "lightgbm_giba_validation.txt",
        num_iteration=lgb_iterations,
    )

    print(f"Training full LightGBM with {lgb_iterations} iterations...", flush=True)
    lgb_final = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            "n_estimators": lgb_iterations,
            "random_state": 2026,
        }
    )
    lgb_final.fit(
        train[features],
        y,
        categorical_feature=categorical,
        callbacks=[lgb.log_evaluation(0)],
    )
    lgb_final.booster_.save_model(ROOT / "lightgbm_giba_final.txt")

    metrics = {
        "data_policy": "provided train/test only",
        "external_gap_used": False,
        "competition_test_labels_used": False,
        "train_rows": len(train),
        "test_rows": len(test),
        "features": len(features),
        "categorical": len(categorical),
        "holdout": {
            "policy": "last 20% by TransactionDT",
            "train_rows": len(train_index),
            "validation_rows": len(valid_index),
            "catboost_auc": cat_auc,
            "catboost_iterations": cat_iterations,
            "lightgbm_auc": lgb_auc,
            "lightgbm_iterations": lgb_iterations,
        },
        "models": [
            "catboost_giba_validation.cbm",
            "catboost_giba_final.cbm",
            "lightgbm_giba_validation.txt",
            "lightgbm_giba_final.txt",
        ],
        "elapsed_minutes": (time.time() - started) / 60,
    }
    (ROOT / "clean_foundation_metrics.json").write_text(
        json.dumps(metrics, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_clean_foundation.py


In [28]:
%%writefile train_honest_advanced_catboost.py
"""Train and evaluate an advanced-UID CatBoost source on clean temporal OOF."""

from __future__ import annotations

import argparse
import gc
import json
from pathlib import Path
import time

from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from build_honest_no_gap_meta import BASE_SOURCES, evaluate_source_set, make_uid
from fraud_honest_advanced_data import (
    ADVANCED_CAT_PARAMS as CAT_PARAMS,
    prepare_advanced_catboost_data,
)


ROOT = Path(__file__).resolve().parent
CACHE_DIR = ROOT / "honest_advanced_catboost"
OOF_PATH = CACHE_DIR / "oof.csv"
REPORT_PATH = CACHE_DIR / "oof_report.json"
SOURCE = "advanced_catboost"
TARGET = "isFraud"
MAX_ITERATIONS = 1_700
SEED = 1729
FOLDS = (
    (0, 30, 60, 75),
    (1, 45, 75, 90),
    (2, 60, 90, 106),
)


def prepare() -> tuple[dict, list[str], list[str]]:
    return prepare_advanced_catboost_data(ROOT)


def train_oof(prepared: dict, features: list[str], categorical: list[str], force: bool) -> tuple[pd.DataFrame, list[dict]]:
    CACHE_DIR.mkdir(exist_ok=True)
    day = prepared["train"]["TransactionDT"].to_numpy(dtype="float64") / 86_400.0
    y = prepared["y"].to_numpy(dtype="int8")
    parts = []
    metrics = []
    for fold, train_end, valid_start, valid_end in FOLDS:
        fit_index = np.flatnonzero(day < train_end)
        valid_index = np.flatnonzero((day >= valid_start) & (day < valid_end))
        prediction_path = CACHE_DIR / f"fold_{fold}_prediction.npy"
        model_path = CACHE_DIR / f"fold_{fold}.cbm"
        metadata_path = CACHE_DIR / f"fold_{fold}.json"
        started = time.time()
        if prediction_path.exists() and metadata_path.exists() and not force:
            prediction = np.load(prediction_path)
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            if len(prediction) != len(valid_index):
                raise ValueError(f"Cached fold {fold} has the wrong length")
            if prediction.dtype != np.float64:
                print(
                    f"Refreshing fold {fold} predictions as float64",
                    flush=True,
                )
                model = CatBoostClassifier()
                model.load_model(model_path)
                prediction = model.predict_proba(
                    prepared["train"].iloc[valid_index][features]
                )[:, 1]
                np.save(prediction_path, prediction)
                del model
                gc.collect()
            metadata["cached"] = True
        else:
            print(
                f"Advanced CatBoost fold {fold}: {len(fit_index):,} -> "
                f"{len(valid_index):,}",
                flush=True,
            )
            model = CatBoostClassifier(
                **{
                    **CAT_PARAMS,
                    "iterations": MAX_ITERATIONS,
                    "random_seed": SEED + fold,
                    "verbose": 200,
                }
            )
            model.fit(
                prepared["train"].iloc[fit_index][features],
                y[fit_index],
                cat_features=categorical,
                eval_set=(
                    prepared["train"].iloc[valid_index][features],
                    y[valid_index],
                ),
                early_stopping_rounds=140,
                use_best_model=True,
            )
            prediction = model.predict_proba(
                prepared["train"].iloc[valid_index][features]
            )[:, 1]
            model.save_model(model_path)
            np.save(prediction_path, prediction)
            metadata = {
                "fold": fold,
                "train_rows": int(len(fit_index)),
                "valid_rows": int(len(valid_index)),
                "best_iteration": int(model.tree_count_),
                "auc": float(roc_auc_score(y[valid_index], prediction)),
                "minutes": (time.time() - started) / 60.0,
                "cached": False,
                "model": model_path.name,
            }
            metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
            del model
            gc.collect()
        metadata["auc"] = float(roc_auc_score(y[valid_index], prediction))
        metrics.append(metadata)
        parts.append(
            pd.DataFrame(
                {
                    "row_index": valid_index,
                    "TransactionID": prepared["train_ids"].iloc[valid_index].to_numpy(),
                    "fold": fold,
                    TARGET: y[valid_index],
                    SOURCE: prediction,
                }
            )
        )
        print(f"fold {fold} AUC={metadata['auc']:.9f}", flush=True)
    oof = pd.concat(parts, ignore_index=True).sort_values(["fold", "row_index"])
    oof.to_csv(OOF_PATH, index=False)
    return oof, metrics


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()
    started = time.time()
    prepared, features, categorical = prepare()
    source_oof, fold_metrics = train_oof(
        prepared, features, categorical, args.force
    )

    base = pd.read_csv(ROOT / "boost3_oof_predictions.csv")
    oof = base.merge(
        source_oof[["row_index", SOURCE]],
        on="row_index",
        how="left",
        validate="one_to_one",
    )
    raw = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = make_uid(raw)
    baseline = evaluate_source_set(oof, uid, BASE_SOURCES)
    candidate = evaluate_source_set(oof, uid, (*BASE_SOURCES, SOURCE))
    gain = float(candidate["uid_recipe"]["auc"] - baseline["uid_recipe"]["auc"])
    report = {
        "data_policy": "official train/test only; train labels only for temporal OOF",
        "features": len(features),
        "categorical": len(categorical),
        "params": {**CAT_PARAMS, "iterations": MAX_ITERATIONS, "random_seed": SEED},
        "folds": fold_metrics,
        "baseline": baseline,
        "candidate": candidate,
        "holdout_gain": gain,
        "accepted": gain > 0,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_advanced_catboost.py


In [29]:
%%writefile train_honest_client_meta.py
"""Train a client-aware LightGBM stack on the locked honest model sources.

Model and post-processing choices are selected on temporal fold 1. Fold 2 is
evaluated once as the untouched train-only lock. Competition-test labels are
never loaded.
"""

from __future__ import annotations

import gc
import json
from pathlib import Path
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import build_honest_no_gap_meta as meta
from build_honest_user_means_final import (
    ADVANCED_WEIGHT,
    CAT_ENHANCED,
    CAT_MEANS,
    MEANS_WEIGHT,
    SOURCES,
)
from train_honest_advanced_catboost import SOURCE as ADVANCED_SOURCE
from train_honest_user_means_catboost import SOURCE as MEANS_SOURCE


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_client_meta"
TARGET = "isFraud"
DAY_SECONDS = 86_400.0
TRAIN_END_BY_FOLD = {0: 30.0, 1: 45.0, 2: 60.0}
META_DEV_FOLD = 1
META_LOCK_FOLD = 2
GENERIC_EMAILS = {"anonymous.com", "mail.com", "<MISSING>"}
OUTPUT_PATH = ROOT / "submission_honest_client_meta.csv"
REPORT_PATH = WORK_DIR / "report.json"

RAW_COLUMNS = (
    "TransactionID",
    "TransactionDT",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1",
    "D1",
    "P_emaildomain",
)
GROUP_SPECS = {
    "strict_clean_email": (
        ("card1", "addr1", "origin_day", "clean_email"),
        500,
    ),
    "card_addr_origin": (("card1", "addr1", "origin_day"), 500),
    "card_origin_clean_email": (
        ("card1", "origin_day", "clean_email"),
        250,
    ),
    "card_full_origin": (
        ("card1", "card2", "card3", "card5", "origin_day"),
        250,
    ),
    "card_addr_origin_product": (
        ("card1", "addr1", "origin_day", "ProductCD"),
        500,
    ),
    "origin_clean_email": (("origin_day", "clean_email"), 150),
}
META_VIEWS = {
    "cat_xgb": ("catboost", CAT_ENHANCED, "xgboost"),
    "all_sources": SOURCES,
    "all_sources_clients": SOURCES,
}
LGB_CONFIGS = (
    {"num_leaves": 7, "max_depth": 3, "min_child_samples": 300},
    {"num_leaves": 15, "max_depth": 4, "min_child_samples": 300},
    {"num_leaves": 15, "max_depth": 5, "min_child_samples": 600},
    {"num_leaves": 31, "max_depth": 5, "min_child_samples": 600},
    {"num_leaves": 31, "max_depth": 6, "min_child_samples": 1_000},
    {"num_leaves": 7, "max_depth": 4, "min_child_samples": 1_000},
)
LGB_BASE_PARAMS = {
    "n_estimators": 1_500,
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.02,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 1.0,
    "reg_lambda": 10.0,
    "random_state": 5309,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
PP_METHODS = ("mean", "q75", "max")
PP_WEIGHTS = (0.05, 0.10, 0.20, 0.30, 0.50, 1.00)


def rank(values: np.ndarray | pd.Series) -> np.ndarray:
    return pd.Series(np.asarray(values)).rank(
        method="average", pct=True
    ).to_numpy()


def prepare_raw(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.loc[:, list(RAW_COLUMNS)].copy().reset_index(drop=True)
    result["origin_day"] = (
        result["TransactionDT"] / DAY_SECONDS - result["D1"]
    ).round()
    email = result["P_emaildomain"].astype("string").str.lower()
    result["clean_email"] = email.mask(email.isin(GENERIC_EMAILS))
    return result


def group_hash(
    frame: pd.DataFrame,
    columns: tuple[str, ...],
) -> tuple[np.ndarray, np.ndarray]:
    values = frame.loc[:, list(columns)]
    valid = values.notna().all(axis=1).to_numpy(dtype=bool)
    hashed = pd.util.hash_pandas_object(
        values,
        index=False,
        categorize=True,
    ).to_numpy(dtype="uint64", copy=False)
    return hashed, valid


def membership_features(
    history: pd.DataFrame,
    query: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, tuple[np.ndarray, np.ndarray, int]]]:
    values: dict[str, np.ndarray] = {}
    groups: dict[str, tuple[np.ndarray, np.ndarray, int]] = {}
    seen_columns = []
    for name, (columns, max_size) in GROUP_SPECS.items():
        history_group, history_valid = group_hash(history, columns)
        query_group, query_valid = group_hash(query, columns)
        history_count = pd.Series(history_group[history_valid]).value_counts(
            sort=False
        )
        query_count = pd.Series(query_group[query_valid]).value_counts(
            sort=False
        )
        mapped_history = np.zeros(len(query), dtype="float64")
        mapped_query = np.zeros(len(query), dtype="float64")
        if query_valid.any():
            mapped_history[query_valid] = (
                pd.Series(query_group[query_valid])
                .map(history_count)
                .fillna(0)
                .to_numpy(dtype="float64")
            )
            mapped_query[query_valid] = (
                pd.Series(query_group[query_valid])
                .map(query_count)
                .fillna(0)
                .to_numpy(dtype="float64")
            )
        seen = query_valid & (mapped_history > 0)
        values[f"client_{name}_valid"] = query_valid.astype("int8")
        values[f"client_{name}_seen"] = seen.astype("int8")
        values[f"client_{name}_history_count"] = np.log1p(
            mapped_history
        ).astype("float32")
        values[f"client_{name}_query_count"] = np.log1p(
            mapped_query
        ).astype("float32")
        seen_columns.append(f"client_{name}_seen")
        groups[name] = (query_group, query_valid, max_size)

    strict = values["client_strict_clean_email_seen"].astype(bool)
    broad = np.maximum.reduce(
        [
            values["client_card_addr_origin_seen"],
            values["client_card_origin_clean_email_seen"],
            values["client_card_full_origin_seen"],
            values["client_card_addr_origin_product_seen"],
            values["client_origin_clean_email_seen"],
        ]
    ).astype(bool)
    partial = ~strict & broad
    cold = ~strict & ~broad
    values["client_segment_strict"] = strict.astype("int8")
    values["client_segment_partial"] = partial.astype("int8")
    values["client_segment_cold"] = cold.astype("int8")
    values["client_seen_key_count"] = np.column_stack(
        [values[column] for column in seen_columns]
    ).sum(axis=1).astype("int8")
    return pd.DataFrame(values), groups


def build_oof_sources() -> pd.DataFrame:
    base = pd.read_csv(ROOT / "boost3_oof_predictions.csv")
    advanced = pd.read_csv(
        ROOT / "honest_advanced_catboost/oof.csv",
        usecols=["row_index", ADVANCED_SOURCE],
    )
    means = pd.concat(
        [
            pd.read_csv(
                ROOT / f"honest_user_means_catboost/fold_{fold}_oof.csv",
                usecols=["row_index", MEANS_SOURCE],
            )
            for fold in range(3)
        ],
        ignore_index=True,
    )
    oof = base.merge(
        advanced, on="row_index", validate="one_to_one"
    ).merge(means, on="row_index", validate="one_to_one")
    old_rank = oof.groupby("fold")["catboost"].rank(pct=True)
    oof[CAT_ENHANCED] = (
        (1.0 - ADVANCED_WEIGHT) * old_rank
        + ADVANCED_WEIGHT
        * oof.groupby("fold")[ADVANCED_SOURCE].rank(pct=True)
    )
    oof[CAT_MEANS] = (
        (1.0 - MEANS_WEIGHT) * old_rank
        + MEANS_WEIGHT * oof.groupby("fold")[MEANS_SOURCE].rank(pct=True)
    )
    return oof.reset_index(drop=True)


def build_test_sources() -> pd.DataFrame:
    old_cat = np.load(ROOT / "stack_test_catboost.npy")
    old_rank = rank(old_cat)
    advanced_rank = np.mean(
        [
            rank(
                np.load(
                    ROOT
                    / f"honest_advanced_catboost/final_seed_{seed}_test.npy"
                ).astype("float32")
            )
            for seed in (1729, 2026, 3407)
        ],
        axis=0,
    )
    means_rank = rank(
        np.load(ROOT / "honest_user_means_catboost/final_seed_1729_test.npy")
    )
    return pd.DataFrame(
        {
            "catboost": old_cat,
            CAT_ENHANCED: (
                (1.0 - ADVANCED_WEIGHT) * old_rank
                + ADVANCED_WEIGHT * advanced_rank
            ),
            CAT_MEANS: (
                (1.0 - MEANS_WEIGHT) * old_rank
                + MEANS_WEIGHT * means_rank
            ),
            "lightgbm": np.load(ROOT / "stack_test_lightgbm.npy"),
            "xgboost": np.load(ROOT / "stack_test_xgboost.npy"),
        }
    )


def build_client_features(
    oof: pd.DataFrame,
    raw_train: pd.DataFrame,
    raw_test: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    dict[int, dict[str, tuple[np.ndarray, np.ndarray, int]]],
    dict[str, tuple[np.ndarray, np.ndarray, int]],
]:
    parts = []
    fold_groups = {}
    day = raw_train["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS
    for fold in sorted(TRAIN_END_BY_FOLD):
        mask = oof["fold"].eq(fold).to_numpy()
        positions = np.flatnonzero(mask)
        row_index = oof.loc[mask, "row_index"].to_numpy(dtype="int64")
        history = raw_train.loc[day < TRAIN_END_BY_FOLD[fold]].reset_index(
            drop=True
        )
        query = raw_train.iloc[row_index].reset_index(drop=True)
        features, groups = membership_features(history, query)
        features["_position"] = positions
        parts.append(features)
        fold_groups[fold] = groups
    oof_features = (
        pd.concat(parts, ignore_index=True)
        .sort_values("_position")
        .drop(columns="_position")
        .reset_index(drop=True)
    )
    test_features, test_groups = membership_features(raw_train, raw_test)
    return oof_features, test_features, fold_groups, test_groups


def make_meta_view(
    predictions: pd.DataFrame,
    client_features: pd.DataFrame,
    view: str,
    fold: pd.Series | None,
) -> pd.DataFrame:
    result = meta.build_meta_features(
        predictions,
        META_VIEWS[view],
        fold=fold,
    ).reset_index(drop=True)
    if view == "all_sources_clients":
        result = pd.concat(
            [result, client_features.reset_index(drop=True)],
            axis=1,
        )
    return result.astype("float32")


def current_meta_prediction(
    oof: pd.DataFrame,
    uid: pd.Series,
    valid_fold: int,
) -> tuple[np.ndarray, dict]:
    features = meta.build_meta_features(oof, SOURCES, fold=oof["fold"])
    train_mask = oof["fold"].lt(valid_fold)
    valid_mask = oof["fold"].eq(valid_fold)
    y_train = oof.loc[train_mask, TARGET].to_numpy(dtype="float32")
    y_valid = oof.loc[valid_mask, TARGET].to_numpy(dtype="float32")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(features.loc[train_mask]).astype("float32")
    X_valid = scaler.transform(features.loc[valid_mask]).astype("float32")
    linear = LogisticRegression(
        C=meta.LOGISTIC_C,
        max_iter=2_000,
        class_weight="balanced",
        random_state=42,
    )
    linear.fit(X_train, y_train)
    linear_prediction = linear.predict_proba(X_valid)[:, 1]
    neural_parts = []
    epochs = {}
    for seed in meta.META_SEEDS:
        epoch, prediction, _ = meta.train_mlp_with_validation(
            X_train,
            y_train,
            X_valid,
            y_valid,
            seed,
        )
        epochs[str(seed)] = int(epoch)
        neural_parts.append(prediction)
    neural_prediction = np.mean(neural_parts, axis=0)
    blend = meta.best_rank_blend(
        y_valid,
        neural_prediction,
        linear_prediction,
    )
    prediction = (
        float(blend["neural_weight"]) * rank(neural_prediction)
        + float(blend["linear_weight"]) * rank(linear_prediction)
    )
    rows = oof.loc[valid_mask, "row_index"].to_numpy(dtype="int64")
    valid_uid = uid.iloc[rows].reset_index(drop=True)
    uid_rows = []
    for weight in (0.0, 0.10, 0.25, 0.50):
        processed = meta.apply_uid_max(prediction, valid_uid, weight)
        uid_rows.append(
            {
                "weight": weight,
                "auc": float(roc_auc_score(y_valid, processed)),
            }
        )
    selected_uid = max(uid_rows, key=lambda row: row["auc"])
    prediction = meta.apply_uid_max(
        prediction,
        valid_uid,
        float(selected_uid["weight"]),
    )
    report = {
        "fold": valid_fold,
        "rows": int(valid_mask.sum()),
        "blend": blend,
        "uid": selected_uid,
        "epochs": epochs,
        "auc": float(roc_auc_score(y_valid, prediction)),
    }
    return prediction, report


def search_lgb_meta(
    oof: pd.DataFrame,
    client_features: pd.DataFrame,
) -> tuple[dict, list[dict], np.ndarray]:
    train_mask = oof["fold"].lt(META_DEV_FOLD).to_numpy()
    valid_mask = oof["fold"].eq(META_DEV_FOLD).to_numpy()
    y_train = oof.loc[train_mask, TARGET].to_numpy(dtype="int8")
    y_valid = oof.loc[valid_mask, TARGET].to_numpy(dtype="int8")
    rows = []
    predictions = {}
    for view in META_VIEWS:
        features = make_meta_view(
            oof,
            client_features,
            view,
            fold=oof["fold"],
        )
        for config_index, config in enumerate(LGB_CONFIGS):
            model = lgb.LGBMClassifier(**LGB_BASE_PARAMS, **config)
            model.fit(
                features.loc[train_mask],
                y_train,
                eval_set=[(features.loc[valid_mask], y_valid)],
                callbacks=[
                    lgb.early_stopping(100, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
            prediction = model.predict_proba(
                features.loc[valid_mask]
            )[:, 1]
            row = {
                "view": view,
                "config_index": config_index,
                **config,
                "best_iteration": int(model.best_iteration_),
                "dev_auc": float(roc_auc_score(y_valid, prediction)),
                "feature_count": int(features.shape[1]),
            }
            key = (view, config_index)
            predictions[key] = prediction
            rows.append(row)
            print(json.dumps(row), flush=True)
            del model
            gc.collect()
    rows.sort(
        key=lambda row: (
            row["dev_auc"],
            -row["num_leaves"],
            -row["feature_count"],
        ),
        reverse=True,
    )
    selected = rows[0]
    selected_prediction = predictions[
        (selected["view"], selected["config_index"])
    ]
    return selected, rows, selected_prediction


def train_lock_lgb(
    oof: pd.DataFrame,
    client_features: pd.DataFrame,
    selected: dict,
) -> tuple[lgb.LGBMClassifier, np.ndarray, int]:
    features = make_meta_view(
        oof,
        client_features,
        selected["view"],
        fold=oof["fold"],
    )
    train_mask = oof["fold"].lt(META_LOCK_FOLD).to_numpy()
    valid_mask = oof["fold"].eq(META_LOCK_FOLD).to_numpy()
    iterations = max(25, int(np.ceil(selected["best_iteration"] * 1.15)))
    config = LGB_CONFIGS[int(selected["config_index"])]
    model = lgb.LGBMClassifier(
        **{
            **LGB_BASE_PARAMS,
            **config,
            "n_estimators": iterations,
        }
    )
    model.fit(
        features.loc[train_mask],
        oof.loc[train_mask, TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    prediction = model.predict_proba(features.loc[valid_mask])[:, 1]
    return model, prediction, iterations


def select_stack_weight(
    y: np.ndarray,
    baseline: np.ndarray,
    stacked: np.ndarray,
) -> tuple[dict, list[dict]]:
    rows = []
    for weight in np.linspace(0.0, 1.0, 21):
        prediction = (1.0 - weight) * rank(baseline) + weight * rank(stacked)
        rows.append(
            {
                "lgb_weight": float(weight),
                "auc": float(roc_auc_score(y, prediction)),
            }
        )
    return max(rows, key=lambda row: row["auc"]), rows


def aggregate_prediction(
    prediction: np.ndarray,
    group: np.ndarray,
    valid: np.ndarray,
    max_size: int,
    method: str,
) -> np.ndarray:
    output = prediction.copy()
    work = pd.DataFrame(
        {"group": group[valid], "prediction": prediction[valid]}
    )
    grouped = work.groupby("group", sort=False)["prediction"]
    size = grouped.size()
    eligible = size.index[(size >= 2) & (size <= max_size)]
    eligible_mask = work["group"].isin(eligible).to_numpy()
    if method == "mean":
        statistic = grouped.mean()
    elif method == "q75":
        statistic = grouped.quantile(0.75)
    elif method == "max":
        statistic = grouped.max()
    else:
        raise ValueError(method)
    rows = np.flatnonzero(valid)[eligible_mask]
    output[rows] = work.loc[eligible_mask, "group"].map(statistic).to_numpy()
    return output


def select_postprocess(
    y: np.ndarray,
    prediction: np.ndarray,
    dt: np.ndarray,
    groups: dict[str, tuple[np.ndarray, np.ndarray, int]],
) -> tuple[dict, list[dict]]:
    baseline_auc = float(roc_auc_score(y, prediction))
    midpoint = np.median(dt)
    halves = (dt <= midpoint, dt > midpoint)
    baseline_half = [
        float(roc_auc_score(y[mask], prediction[mask])) for mask in halves
    ]
    rows = [
        {
            "group": "none",
            "method": "none",
            "weight": 0.0,
            "auc": baseline_auc,
            "gain": 0.0,
            "half_gains": [0.0, 0.0],
            "min_half_gain": 0.0,
        }
    ]
    for name, (group, valid, max_size) in groups.items():
        for method in PP_METHODS:
            grouped = aggregate_prediction(
                prediction,
                group,
                valid,
                max_size,
                method,
            )
            for weight in PP_WEIGHTS:
                candidate = (1.0 - weight) * prediction + weight * grouped
                half_auc = [
                    float(roc_auc_score(y[mask], candidate[mask]))
                    for mask in halves
                ]
                half_gains = [
                    score - base
                    for score, base in zip(half_auc, baseline_half)
                ]
                score = float(roc_auc_score(y, candidate))
                rows.append(
                    {
                        "group": name,
                        "method": method,
                        "weight": weight,
                        "auc": score,
                        "gain": score - baseline_auc,
                        "half_gains": half_gains,
                        "min_half_gain": float(min(half_gains)),
                    }
                )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(stable, key=lambda row: (row["gain"], row["min_half_gain"]))
    return selected, rows


def apply_postprocess(
    prediction: np.ndarray,
    groups: dict[str, tuple[np.ndarray, np.ndarray, int]],
    recipe: dict,
) -> np.ndarray:
    if recipe["group"] == "none":
        return prediction
    group, valid, max_size = groups[recipe["group"]]
    grouped = aggregate_prediction(
        prediction,
        group,
        valid,
        max_size,
        recipe["method"],
    )
    weight = float(recipe["weight"])
    return (1.0 - weight) * prediction + weight * grouped


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    oof = build_oof_sources()
    test_sources = build_test_sources()
    raw_train = prepare_raw(
        pd.read_csv(ROOT / "train_transaction.csv", usecols=list(RAW_COLUMNS))
    )
    raw_test = prepare_raw(
        pd.read_csv(ROOT / "test_transaction.csv", usecols=list(RAW_COLUMNS))
    )
    (
        client_oof,
        client_test,
        fold_groups,
        test_groups,
    ) = build_client_features(oof, raw_train, raw_test)
    client_oof.to_csv(WORK_DIR / "oof_client_features.csv", index=False)
    client_test.to_csv(WORK_DIR / "test_client_features.csv", index=False)

    raw_uid = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(raw_uid)
    dev_current, dev_current_report = current_meta_prediction(
        oof, uid, META_DEV_FOLD
    )
    lock_current, lock_current_report = current_meta_prediction(
        oof, uid, META_LOCK_FOLD
    )

    selected_lgb, lgb_search, dev_lgb = search_lgb_meta(oof, client_oof)
    lgb_model, lock_lgb, lock_iterations = train_lock_lgb(
        oof, client_oof, selected_lgb
    )
    lock_model_path = WORK_DIR / "lock_lgb.txt"
    lgb_model.booster_.save_model(lock_model_path)

    dev_mask = oof["fold"].eq(META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(META_LOCK_FOLD).to_numpy()
    y_dev = oof.loc[dev_mask, TARGET].to_numpy(dtype="int8")
    y_lock = oof.loc[lock_mask, TARGET].to_numpy(dtype="int8")
    selected_weight, weight_search = select_stack_weight(
        y_dev,
        dev_current,
        dev_lgb,
    )
    weight = float(selected_weight["lgb_weight"])
    dev_blend = (1.0 - weight) * rank(dev_current) + weight * rank(dev_lgb)
    lock_blend = (1.0 - weight) * rank(lock_current) + weight * rank(lock_lgb)

    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    dev_dt = raw_train.iloc[dev_rows]["TransactionDT"].to_numpy()
    selected_pp, pp_search = select_postprocess(
        y_dev,
        dev_blend,
        dev_dt,
        fold_groups[META_DEV_FOLD],
    )
    dev_final = apply_postprocess(
        dev_blend,
        fold_groups[META_DEV_FOLD],
        selected_pp,
    )
    lock_final = apply_postprocess(
        lock_blend,
        fold_groups[META_LOCK_FOLD],
        selected_pp,
    )
    lock_baseline_auc = float(roc_auc_score(y_lock, lock_current))
    lock_blend_auc = float(roc_auc_score(y_lock, lock_blend))
    lock_final_auc = float(roc_auc_score(y_lock, lock_final))
    accepted = bool(
        selected_weight["auc"] > dev_current_report["auc"]
        and float(roc_auc_score(y_dev, dev_final)) > dev_current_report["auc"]
        and lock_final_auc > lock_baseline_auc
    )

    final_iterations = max(25, int(np.ceil(lock_iterations * 1.15)))
    selected_view = selected_lgb["view"]
    all_features = make_meta_view(
        oof,
        client_oof,
        selected_view,
        fold=oof["fold"],
    )
    test_features = make_meta_view(
        test_sources,
        client_test,
        selected_view,
        fold=None,
    )
    config = LGB_CONFIGS[int(selected_lgb["config_index"])]
    final_model = lgb.LGBMClassifier(
        **{
            **LGB_BASE_PARAMS,
            **config,
            "n_estimators": final_iterations,
        }
    )
    final_model.fit(
        all_features,
        oof[TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    final_model.booster_.save_model(WORK_DIR / "final_lgb.txt")
    lgb_test = final_model.predict_proba(test_features)[:, 1]
    current_test = pd.read_csv(ROOT / "submission_honest_user_means.csv")
    prediction = (
        (1.0 - weight) * rank(current_test[TARGET])
        + weight * rank(lgb_test)
    )
    prediction = apply_postprocess(prediction, test_groups, selected_pp)
    output = current_test[["TransactionID"]].copy()
    output[TARGET] = prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test only; train labels only in temporal OOF",
        "selection": "LGB/weight/postprocess on fold 1; fold 2 one-time lock",
        "sources": list(SOURCES),
        "client_feature_count": int(client_oof.shape[1]),
        "client_segment_counts": {
            "dev": {
                name.removeprefix("client_segment_"): int(
                    client_oof.loc[dev_mask, name].sum()
                )
                for name in (
                    "client_segment_strict",
                    "client_segment_partial",
                    "client_segment_cold",
                )
            },
            "lock": {
                name.removeprefix("client_segment_"): int(
                    client_oof.loc[lock_mask, name].sum()
                )
                for name in (
                    "client_segment_strict",
                    "client_segment_partial",
                    "client_segment_cold",
                )
            },
            "test": {
                name.removeprefix("client_segment_"): int(client_test[name].sum())
                for name in (
                    "client_segment_strict",
                    "client_segment_partial",
                    "client_segment_cold",
                )
            },
        },
        "current_meta": {
            "dev": dev_current_report,
            "lock": lock_current_report,
        },
        "selected_lgb": selected_lgb,
        "lgb_search": lgb_search,
        "lock_iterations": lock_iterations,
        "final_iterations": final_iterations,
        "selected_weight": selected_weight,
        "weight_search": weight_search,
        "selected_postprocess": selected_pp,
        "postprocess_search": pp_search,
        "scores": {
            "dev_current_auc": dev_current_report["auc"],
            "dev_lgb_auc": float(roc_auc_score(y_dev, dev_lgb)),
            "dev_blend_auc": float(roc_auc_score(y_dev, dev_blend)),
            "dev_final_auc": float(roc_auc_score(y_dev, dev_final)),
            "lock_current_auc": lock_baseline_auc,
            "lock_lgb_auc": float(roc_auc_score(y_lock, lock_lgb)),
            "lock_blend_auc": lock_blend_auc,
            "lock_final_auc": lock_final_auc,
            "lock_gain": lock_final_auc - lock_baseline_auc,
        },
        "accepted": accepted,
        "output": OUTPUT_PATH.name,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected_lgb": report["selected_lgb"],
                "selected_weight": report["selected_weight"],
                "selected_postprocess": report["selected_postprocess"],
                "scores": report["scores"],
                "accepted": report["accepted"],
                "output": report["output"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting train_honest_client_meta.py


In [30]:
%%writefile train_honest_client_profile_lgb.py
"""Client-profile LightGBM trained with purged forward-time validation."""

from __future__ import annotations

import gc
import hashlib
import json
from pathlib import Path
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb

import finalize_honest_xgb_magic_blend as xgb_final
import refine_honest_client_segments as segments
import search_honest_client_pooling as pooling
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client
import train_honest_xgb_magic as magic
import train_honest_xgb_seed_subset_gkf as seed_subset


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_client_profile_lgb"
MODEL_DIR = WORK_DIR / "models"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_client_profile_lgb.csv"
DEV_TRAIN_END = 45.0
LOCK_TRAIN_END = 60.0
ITERATION_SCALE = 1.15
MIN_REQUIRED_GAIN = 0.0001
MEAN_FEATURES = 96
DETAIL_FEATURES = 32
SEED = 5903

LGB_BASE = {
    "n_estimators": 2_500,
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.02,
    "subsample": 0.84,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 1.0,
    "reg_lambda": 12.0,
    "max_bin": 255,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
LGB_CONFIGS = (
    {
        "name": "profile_leaf31",
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 160,
    },
    {
        "name": "profile_leaf63",
        "num_leaves": 63,
        "max_depth": 9,
        "min_child_samples": 260,
    },
    {
        "name": "profile_extra63",
        "num_leaves": 63,
        "max_depth": 9,
        "min_child_samples": 220,
        "extra_trees": True,
    },
)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def auc(target: np.ndarray, prediction: np.ndarray) -> float:
    return float(roc_auc_score(target, prediction))


def select_feature_indices(feature_names: list[str]) -> tuple[list[int], list[int]]:
    model = xgb.XGBClassifier(**magic.XGB_PARAMS)
    model.load_model(ROOT / "honest_xgb_magic/models/dev.json")
    importance = model.get_booster().get_score(importance_type="gain")
    ranked = [
        index
        for index, _ in sorted(
            (
                (int(name[1:]), float(value))
                for name, value in importance.items()
            ),
            key=lambda item: item[1],
            reverse=True,
        )
    ]
    manual_names = (
        "TransactionAmt",
        "card1",
        "card2",
        "card3",
        "card5",
        "addr1",
        "addr2",
        "P_emaildomain",
        "R_emaildomain",
        "uid_FE",
        "TransactionAmt_uid_mean",
        "TransactionAmt_uid_std",
        "D4_uid_mean",
        "D10_uid_mean",
        "D15_uid_mean",
        "C13_uid_mean",
        "C14_uid_std",
    )
    manual = [
        feature_names.index(name) for name in manual_names if name in feature_names
    ]
    mean_indices = list(dict.fromkeys([*manual, *ranked]))[:MEAN_FEATURES]
    detail_indices = mean_indices[:DETAIL_FEATURES]
    del model
    gc.collect()
    return mean_indices, detail_indices


def build_profile_matrix(
    matrix: np.ndarray,
    row_index: np.ndarray | None,
    uid: pd.Series,
    day: np.ndarray,
    feature_names: list[str],
    mean_indices: list[int],
    detail_indices: list[int],
) -> tuple[np.ndarray, list[str], dict]:
    if row_index is None:
        values = np.asarray(matrix[:, mean_indices], dtype="float32").copy()
        uid_rows = uid.reset_index(drop=True)
        day_rows = np.asarray(day, dtype="float32")
    else:
        values = np.asarray(
            matrix[np.ix_(row_index, mean_indices)], dtype="float32"
        ).copy()
        uid_rows = uid.iloc[row_index].reset_index(drop=True)
        day_rows = np.asarray(day[row_index], dtype="float32")
    values[values == -1.0] = np.nan
    codes, unique_uid = pd.factorize(uid_rows, sort=False)
    if np.any(codes < 0):
        raise RuntimeError("Unexpected missing UID in client profile")
    mean_names = [feature_names[index] for index in mean_indices]
    detail_names = [feature_names[index] for index in detail_indices]
    frame = pd.DataFrame(values, columns=mean_names, copy=False)
    grouped = frame.groupby(codes, sort=False, observed=True)
    group_mean = grouped.mean().to_numpy(dtype="float32")

    detail_positions = [mean_indices.index(index) for index in detail_indices]
    detail = frame.iloc[:, detail_positions]
    detail_grouped = detail.groupby(codes, sort=False, observed=True)
    group_std = detail_grouped.std().to_numpy(dtype="float32")
    group_min = detail_grouped.min().to_numpy(dtype="float32")
    group_max = detail_grouped.max().to_numpy(dtype="float32")
    group_first = detail_grouped.first().to_numpy(dtype="float32")
    group_last = detail_grouped.last().to_numpy(dtype="float32")

    counts = np.bincount(codes, minlength=len(unique_uid)).astype("float32")
    day_frame = pd.DataFrame({"group": codes, "day": day_rows})
    day_stats = day_frame.groupby("group", sort=False, observed=True)["day"].agg(
        ["min", "max", "std"]
    )
    span = (day_stats["max"] - day_stats["min"]).to_numpy(dtype="float32")
    day_std = day_stats["std"].to_numpy(dtype="float32")
    group_meta = np.column_stack(
        [
            counts,
            np.log1p(counts),
            span,
            day_std,
            counts / (span + 1.0),
        ]
    ).astype("float32")
    group_matrix = np.column_stack(
        [
            group_mean,
            group_std,
            group_min,
            group_max,
            group_first,
            group_last,
            group_meta,
        ]
    ).astype("float32")
    row_matrix = group_matrix[codes]
    names = [
        *(f"mean_{name}" for name in mean_names),
        *(f"std_{name}" for name in detail_names),
        *(f"min_{name}" for name in detail_names),
        *(f"max_{name}" for name in detail_names),
        *(f"first_{name}" for name in detail_names),
        *(f"last_{name}" for name in detail_names),
        "client_count",
        "client_log_count",
        "client_day_span",
        "client_day_std",
        "client_transactions_per_day",
    ]
    report = {
        "rows": len(row_matrix),
        "clients": len(unique_uid),
        "features": len(names),
        "repeated_row_rate": float(np.mean(counts[codes] > 1)),
        "max_client_rows": int(counts.max()),
    }
    del frame, detail, grouped, detail_grouped, group_matrix, values
    gc.collect()
    return row_matrix, names, report


def fit_lgb(
    train: np.ndarray,
    target: np.ndarray,
    predict: np.ndarray,
    predict_target: np.ndarray | None,
    config: dict,
    model_path: Path,
    fixed_iterations: int | None,
) -> tuple[np.ndarray, int, float]:
    params = {
        **LGB_BASE,
        **{key: value for key, value in config.items() if key != "name"},
    }
    if fixed_iterations is not None:
        params["n_estimators"] = fixed_iterations
    model = lgb.LGBMClassifier(**params)
    fit_kwargs = {"X": train, "y": target}
    if fixed_iterations is None:
        fit_kwargs.update(
            {
                "eval_set": [(predict, predict_target)],
                "eval_metric": "auc",
                "callbacks": [
                    lgb.early_stopping(180, verbose=False),
                    lgb.log_evaluation(200),
                ],
            }
        )
    else:
        fit_kwargs["callbacks"] = [lgb.log_evaluation(0)]
    started = time.time()
    model.fit(**fit_kwargs)
    best = int(model.best_iteration_ or params["n_estimators"])
    prediction = model.predict_proba(predict, num_iteration=best)[:, 1]
    model.booster_.save_model(model_path, num_iteration=best)
    minutes = (time.time() - started) / 60.0
    del model
    gc.collect()
    return prediction, best, minutes


def model_signals(predictions: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    signals = dict(predictions)
    names = list(predictions)
    ranks = {name: client.rank(predictions[name]) for name in names}
    signals["all_probability_mean"] = np.mean(
        [predictions[name] for name in names], axis=0
    )
    signals["all_rank_mean"] = np.mean([ranks[name] for name in names], axis=0)
    signals["all_rank_max"] = np.max([ranks[name] for name in names], axis=0)
    for left_index in range(len(names)):
        for right_index in range(left_index + 1, len(names)):
            left = names[left_index]
            right = names[right_index]
            signals[f"rank_mean_{left}_{right}"] = 0.5 * (
                ranks[left] + ranks[right]
            )
    return signals


def reconstruct_champion(
    arrays: dict[str, np.ndarray],
    oof: pd.DataFrame,
    membership_oof: pd.DataFrame,
    clean_oof: dict[int, np.ndarray],
    fold_groups: dict,
    postprocess: dict,
) -> tuple[np.ndarray, np.ndarray, dict]:
    xgb_report = json.loads(
        (ROOT / "honest_xgb_magic/blend_report.json").read_text(
            encoding="utf-8"
        )
    )
    heavy_report = json.loads(
        (ROOT / "honest_magic_heavy_stack/report.json").read_text(
            encoding="utf-8"
        )
    )
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_membership = membership_oof.loc[dev_mask].reset_index(drop=True)
    lock_membership = membership_oof.loc[lock_mask].reset_index(drop=True)
    dev_xgb = seed_subset.apply_xgb_layer(
        clean_oof[client.META_DEV_FOLD],
        np.load(ROOT / "honest_xgb_magic/dev_prediction.npy"),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
        dev_membership,
        xgb_report["selected"],
    )
    lock_xgb = seed_subset.apply_xgb_layer(
        clean_oof[client.META_LOCK_FOLD],
        np.load(ROOT / "honest_xgb_magic/lock_prediction.npy"),
        fold_groups[client.META_LOCK_FOLD],
        postprocess,
        lock_membership,
        xgb_report["selected"],
    )
    cat_name = heavy_report["selected_cat"]["name"]
    lgb_name = heavy_report["selected_lgb"]["name"]
    dev = seed_subset.apply_heavy_layer(
        dev_xgb,
        np.load(ROOT / f"honest_magic_heavy_stack/dev_cat_{cat_name}.npy"),
        np.load(ROOT / f"honest_magic_heavy_stack/dev_lgb_{lgb_name}.npy"),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
        dev_membership,
        heavy_report["selected_blend"],
    )
    lock = seed_subset.apply_heavy_layer(
        lock_xgb,
        np.load(ROOT / "honest_magic_heavy_stack/lock_cat.npy"),
        np.load(ROOT / "honest_magic_heavy_stack/lock_lgb.npy"),
        fold_groups[client.META_LOCK_FOLD],
        postprocess,
        lock_membership,
        heavy_report["selected_blend"],
    )
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    metrics = {
        "dev": auc(arrays["target"][dev_index], dev),
        "lock": auc(arrays["target"][lock_index], lock),
    }
    if abs(metrics["dev"] - float(heavy_report["dev_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy champion on dev")
    if abs(metrics["lock"] - float(heavy_report["lock_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy champion on lock")
    return dev, lock, metrics


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    MODEL_DIR.mkdir(exist_ok=True)
    manifest, arrays = magic.load_matrix()
    feature_names = list(manifest["features"])
    mean_indices, detail_indices = select_feature_indices(feature_names)
    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    clean_oof, clean_report = xgb_final.reconstruct_clean_oof(
        oof, membership_oof, reference, fold_groups, reference_report
    )
    champion_dev, champion_lock, champion_metrics = reconstruct_champion(
        arrays,
        oof,
        membership_oof,
        clean_oof,
        fold_groups,
        clean_report["postprocess"],
    )

    raw_train = pd.read_csv(
        ROOT / "train_transaction.csv", usecols=pooling.RAW_COLUMNS
    )
    raw_test = pd.read_csv(
        ROOT / "test_transaction.csv", usecols=pooling.RAW_COLUMNS
    )
    train_uid = pooling.build_uid_frame(raw_train)["floor_card_addr_email"]
    test_uid = pooling.build_uid_frame(raw_test)["floor_card_addr_email"]
    day = arrays["day"]
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    dev_train_index = np.flatnonzero(day < DEV_TRAIN_END)
    lock_train_index = np.flatnonzero(day < LOCK_TRAIN_END)
    y_dev = np.asarray(arrays["target"][dev_index], dtype="int8")
    y_lock = np.asarray(arrays["target"][lock_index], dtype="int8")
    dev_labels = segments.segment_labels(
        membership_oof.loc[dev_mask].reset_index(drop=True)
    )
    lock_labels = segments.segment_labels(
        membership_oof.loc[lock_mask].reset_index(drop=True)
    )

    dev_history, profile_names, dev_history_report = build_profile_matrix(
        arrays["train"],
        dev_train_index,
        train_uid,
        day,
        feature_names,
        mean_indices,
        detail_indices,
    )
    dev_query, query_names, dev_query_report = build_profile_matrix(
        arrays["train"],
        dev_index,
        train_uid,
        day,
        feature_names,
        mean_indices,
        detail_indices,
    )
    if query_names != profile_names:
        raise RuntimeError("Dev profile feature names differ")
    dev_predictions = {}
    dev_models = []
    for config in LGB_CONFIGS:
        name = config["name"]
        prediction, iterations, minutes = fit_lgb(
            dev_history,
            np.asarray(arrays["target"][dev_train_index], dtype="int8"),
            dev_query,
            y_dev,
            config,
            MODEL_DIR / f"dev_{name}.txt",
            None,
        )
        dev_predictions[name] = prediction
        np.save(WORK_DIR / f"dev_{name}.npy", prediction.astype("float32"))
        row = {
            "name": name,
            "config": config,
            "source_auc": auc(y_dev, prediction),
            "best_iteration": iterations,
            "minutes": minutes,
        }
        dev_models.append(row)
        print(json.dumps(row), flush=True)
    dev_signals = model_signals(dev_predictions)
    selected, search_rows = pooling.search_pooling(
        y_dev,
        champion_dev,
        dev_signals,
        dev_labels,
        np.asarray(day[dev_index]),
    )
    dev_candidate = pooling.direct_segment_blend(
        champion_dev,
        dev_signals[selected["variant"]],
        dev_labels,
        selected["weights"],
    )
    del dev_history, dev_query, dev_predictions, dev_signals
    gc.collect()

    lock_history, lock_names, lock_history_report = build_profile_matrix(
        arrays["train"],
        lock_train_index,
        train_uid,
        day,
        feature_names,
        mean_indices,
        detail_indices,
    )
    lock_query, lock_query_names, lock_query_report = build_profile_matrix(
        arrays["train"],
        lock_index,
        train_uid,
        day,
        feature_names,
        mean_indices,
        detail_indices,
    )
    if lock_names != profile_names or lock_query_names != profile_names:
        raise RuntimeError("Lock profile feature names differ")
    lock_predictions = {}
    lock_models = []
    for config, dev_row in zip(LGB_CONFIGS, dev_models):
        iterations = max(
            50, int(np.ceil(dev_row["best_iteration"] * ITERATION_SCALE))
        )
        prediction, _, minutes = fit_lgb(
            lock_history,
            np.asarray(arrays["target"][lock_train_index], dtype="int8"),
            lock_query,
            None,
            config,
            MODEL_DIR / f"lock_{config['name']}.txt",
            iterations,
        )
        lock_predictions[config["name"]] = prediction
        np.save(
            WORK_DIR / f"lock_{config['name']}.npy",
            prediction.astype("float32"),
        )
        lock_models.append(
            {"name": config["name"], "iterations": iterations, "minutes": minutes}
        )
    lock_signals = model_signals(lock_predictions)
    lock_candidate = pooling.direct_segment_blend(
        champion_lock,
        lock_signals[selected["variant"]],
        lock_labels,
        selected["weights"],
    )
    dev_auc = auc(y_dev, dev_candidate)
    lock_auc = auc(y_lock, lock_candidate)
    dev_gain = dev_auc - champion_metrics["dev"]
    lock_gain = lock_auc - champion_metrics["lock"]
    accepted = bool(
        dev_gain >= MIN_REQUIRED_GAIN and lock_gain >= MIN_REQUIRED_GAIN
    )
    del lock_history, lock_query, lock_predictions, lock_signals
    gc.collect()

    final_models = []
    if accepted:
        final_history, final_names, final_history_report = build_profile_matrix(
            arrays["train"],
            None,
            train_uid,
            day,
            feature_names,
            mean_indices,
            detail_indices,
        )
        test_query, test_names, test_query_report = build_profile_matrix(
            arrays["test"],
            None,
            test_uid,
            np.asarray(
                raw_test["TransactionDT"].to_numpy(dtype="float64") / 86_400.0,
                dtype="float32",
            ),
            feature_names,
            mean_indices,
            detail_indices,
        )
        if final_names != profile_names or test_names != profile_names:
            raise RuntimeError("Final profile feature names differ")
        test_predictions = {}
        for config, lock_row in zip(LGB_CONFIGS, lock_models):
            iterations = max(
                50, int(np.ceil(lock_row["iterations"] * ITERATION_SCALE))
            )
            prediction, _, minutes = fit_lgb(
                final_history,
                np.asarray(arrays["target"], dtype="int8"),
                test_query,
                None,
                config,
                MODEL_DIR / f"final_{config['name']}.txt",
                iterations,
            )
            test_predictions[config["name"]] = prediction
            final_models.append(
                {"name": config["name"], "iterations": iterations, "minutes": minutes}
            )
        test_signals = model_signals(test_predictions)
        baseline = pd.read_csv(ROOT / "submission_honest_magic_heavy_stack.csv")
        if not np.array_equal(
            baseline["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("Client-profile test rows differ from champion")
        prediction = pooling.direct_segment_blend(
            baseline[client.TARGET].to_numpy(dtype="float64"),
            test_signals[selected["variant"]],
            segments.segment_labels(membership_test),
            selected["weights"],
        )
        output = baseline[["TransactionID"]].copy()
        output[client.TARGET] = prediction
        output.to_csv(OUTPUT_PATH, index=False)
    else:
        final_history_report = None
        test_query_report = None

    report = {
        "data_policy": (
            "official train/test covariates; official train labels only; "
            "history/query profiles built separately"
        ),
        "official_hashes": manifest["official_hashes"],
        "uid": "card1|addr1|floor(day-D1)|P_emaildomain",
        "profile": {
            "mean_features": [feature_names[index] for index in mean_indices],
            "detail_features": [feature_names[index] for index in detail_indices],
            "feature_count": len(profile_names),
        },
        "profile_reports": {
            "dev_history": dev_history_report,
            "dev_query": dev_query_report,
            "lock_history": lock_history_report,
            "lock_query": lock_query_report,
            "final_history": final_history_report,
            "test_query": test_query_report,
        },
        "dev_models": dev_models,
        "selected": selected,
        "search_top": search_rows[:40],
        "champion_auc": champion_metrics,
        "dev_auc": dev_auc,
        "dev_gain": dev_gain,
        "lock_models": lock_models,
        "lock_auc": lock_auc,
        "lock_gain": lock_gain,
        "minimum_required_gain": MIN_REQUIRED_GAIN,
        "accepted": accepted,
        "final_models": final_models,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_client_profile_lgb.py


In [31]:
%%writefile train_honest_heavy_temporal_client.py
"""Train a heavy client-aware temporal stack using official data only.

The additional seven CatBoost/LightGBM sources are out-of-fold predictions
from purged forward-time splits. Their final test counterparts come from the
clean temporal run. Model choices are made on fold 1; fold 2 stays locked.
"""

from __future__ import annotations

import gc
from itertools import product
import json
from pathlib import Path
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import build_honest_no_gap_meta as meta
import refine_honest_client_segments as segments
import train_honest_client_meta as client


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_heavy_temporal_client"
OUTPUT_PATH = ROOT / "submission_honest_heavy_temporal_client.csv"
REPORT_PATH = WORK_DIR / "report.json"

TEMPORAL_BASE_SOURCES = (
    "cat_giba",
    "cat_multi_full",
    "cat_recent45",
    "cat_recent30",
    "cat_plain",
    "cat_time_plain",
    "lgb_giba",
)
TEMPORAL_SOURCES = tuple(f"t7_{source}" for source in TEMPORAL_BASE_SOURCES)
TEMPORAL_STACK = "t7_stack"
ALL_HEAVY_SOURCES = (*client.SOURCES, *TEMPORAL_SOURCES, TEMPORAL_STACK)

VIEWS = {
    "diverse_clients": (
        (*client.SOURCES, "t7_cat_giba", "t7_cat_recent30", "t7_lgb_giba", TEMPORAL_STACK),
        True,
    ),
    "temporal_core_clients": (
        (*client.SOURCES, "t7_cat_giba", "t7_cat_multi_full", "t7_cat_recent45",
         "t7_cat_recent30", "t7_lgb_giba", TEMPORAL_STACK),
        True,
    ),
    "all_heavy_clients": (ALL_HEAVY_SOURCES, True),
    "all_heavy": (ALL_HEAVY_SOURCES, False),
}

LGB_CONFIGS = (
    {"num_leaves": 15, "max_depth": 4, "min_child_samples": 300},
    {"num_leaves": 31, "max_depth": 5, "min_child_samples": 300},
    {"num_leaves": 31, "max_depth": 6, "min_child_samples": 600},
    {"num_leaves": 63, "max_depth": 6, "min_child_samples": 600},
    {"num_leaves": 63, "max_depth": 7, "min_child_samples": 1_000},
    {"num_leaves": 127, "max_depth": 8, "min_child_samples": 1_000},
)
LGB_PARAMS = {
    "n_estimators": 2_500,
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.015,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 1.5,
    "reg_lambda": 15.0,
    "random_state": 7907,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
SEGMENT_WEIGHTS = tuple(np.round(np.linspace(0.0, 1.0, 11), 2))


def verify_no_gap_artifacts() -> dict:
    metrics_path = ROOT / "temporal7_no_gap_metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    training = metrics["training"]
    required = {
        "unlabeled_bridge_rows": 0,
        "uses_external_labels": False,
        "uses_competition_test_labels": False,
    }
    observed = {key: training.get(key) for key in required}
    if observed != required:
        raise RuntimeError(f"Temporal source is not the clean no-gap run: {observed}")
    test_header = pd.read_csv(ROOT / "test_transaction.csv", nrows=1)
    if client.TARGET in test_header.columns:
        raise RuntimeError("Target leaked into official test_transaction.csv")
    return training


def build_heavy_oof() -> pd.DataFrame:
    oof = client.build_oof_sources()
    temporal_columns = [
        "row_index",
        "fold",
        *TEMPORAL_BASE_SOURCES,
        "stack_prediction",
    ]
    temporal = pd.read_csv(
        ROOT / "temporal7_oof_predictions.csv",
        usecols=temporal_columns,
    ).rename(
        columns={
            "fold": "temporal_fold",
            "stack_prediction": TEMPORAL_STACK,
            **{source: f"t7_{source}" for source in TEMPORAL_BASE_SOURCES},
        }
    )
    merged = oof.merge(temporal, on="row_index", how="left", validate="one_to_one")
    if merged[list(TEMPORAL_SOURCES) + [TEMPORAL_STACK]].isna().any().any():
        raise RuntimeError("Temporal7 OOF does not cover every current OOF row")
    if not np.array_equal(
        merged["temporal_fold"].to_numpy(),
        merged["fold"].to_numpy() + 1,
    ):
        raise RuntimeError("Temporal7 and current purged folds are misaligned")
    return merged.drop(columns="temporal_fold").reset_index(drop=True)


def build_heavy_test() -> pd.DataFrame:
    result = client.build_test_sources()
    temporal = pd.read_csv(ROOT / "temporal7_no_gap_source_predictions.csv")
    stack = pd.read_csv(ROOT / "submission_temporal7_no_gap.csv")
    sample = pd.read_csv(ROOT / "sample_submission.csv", usecols=["TransactionID"])
    for frame, name in ((temporal, "temporal sources"), (stack, "temporal stack")):
        if not np.array_equal(frame["TransactionID"].to_numpy(), sample["TransactionID"].to_numpy()):
            raise RuntimeError(f"TransactionID order differs for {name}")
    for source in TEMPORAL_BASE_SOURCES:
        result[f"t7_{source}"] = temporal[source].to_numpy(dtype="float64")
    result[TEMPORAL_STACK] = stack[client.TARGET].to_numpy(dtype="float64")
    return result


def make_view(
    predictions: pd.DataFrame,
    membership: pd.DataFrame,
    view: str,
    fold: pd.Series | None,
) -> pd.DataFrame:
    sources, include_clients = VIEWS[view]
    features = meta.build_meta_features(predictions, sources, fold=fold).reset_index(drop=True)
    if include_clients:
        features = pd.concat([features, membership.reset_index(drop=True)], axis=1)
    return features.astype("float32")


def search_models(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
) -> tuple[dict, list[dict], np.ndarray]:
    train_mask = oof["fold"].lt(client.META_DEV_FOLD).to_numpy()
    valid_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    y_train = oof.loc[train_mask, client.TARGET].to_numpy(dtype="int8")
    y_valid = oof.loc[valid_mask, client.TARGET].to_numpy(dtype="int8")
    rows: list[dict] = []
    predictions: dict[tuple[str, int], np.ndarray] = {}
    for view in VIEWS:
        features = make_view(oof, membership, view, fold=oof["fold"])
        for config_index, config in enumerate(LGB_CONFIGS):
            model = lgb.LGBMClassifier(**LGB_PARAMS, **config)
            model.fit(
                features.loc[train_mask],
                y_train,
                eval_set=[(features.loc[valid_mask], y_valid)],
                callbacks=[
                    lgb.early_stopping(150, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
            prediction = model.predict_proba(features.loc[valid_mask])[:, 1]
            row = {
                "view": view,
                "config_index": config_index,
                **config,
                "features": int(features.shape[1]),
                "best_iteration": int(model.best_iteration_),
                "dev_auc": float(roc_auc_score(y_valid, prediction)),
            }
            rows.append(row)
            predictions[(view, config_index)] = prediction
            print(json.dumps(row), flush=True)
            del model
            gc.collect()
        del features
        gc.collect()
    rows.sort(key=lambda row: (row["dev_auc"], -row["num_leaves"]), reverse=True)
    selected = rows[0]
    return selected, rows, predictions[(selected["view"], selected["config_index"])]


def train_model(
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    selected: dict,
    valid_fold: int,
    iteration_scale: float,
) -> tuple[lgb.LGBMClassifier, np.ndarray, int]:
    features = make_view(oof, membership, selected["view"], fold=oof["fold"])
    train_mask = oof["fold"].lt(valid_fold).to_numpy()
    valid_mask = oof["fold"].eq(valid_fold).to_numpy()
    iterations = max(30, int(np.ceil(selected["best_iteration"] * iteration_scale)))
    model = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            **LGB_CONFIGS[int(selected["config_index"])],
            "n_estimators": iterations,
        }
    )
    model.fit(
        features.loc[train_mask],
        oof.loc[train_mask, client.TARGET],
        callbacks=[lgb.log_evaluation(0)],
    )
    prediction = model.predict_proba(features.loc[valid_mask])[:, 1]
    return model, prediction, iterations


def search_segment_weights(
    y: np.ndarray,
    baseline: np.ndarray,
    stacked: np.ndarray,
    labels: np.ndarray,
    dt: np.ndarray,
    groups: dict[str, tuple[np.ndarray, np.ndarray, int]],
    postprocess: dict,
) -> tuple[dict, list[dict]]:
    midpoint = np.median(dt)
    halves = (dt <= midpoint, dt > midpoint)
    baseline_final = client.apply_postprocess(baseline, groups, postprocess)
    baseline_auc = float(roc_auc_score(y, baseline_final))
    baseline_halves = [float(roc_auc_score(y[mask], baseline_final[mask])) for mask in halves]
    rows = []
    for values in product(SEGMENT_WEIGHTS, repeat=len(segments.SEGMENTS)):
        weights = dict(zip(segments.SEGMENTS, values))
        prediction = segments.segmented_blend(baseline, stacked, labels, weights)
        prediction = client.apply_postprocess(prediction, groups, postprocess)
        half_auc = [float(roc_auc_score(y[mask], prediction[mask])) for mask in halves]
        half_gains = [score - base for score, base in zip(half_auc, baseline_halves)]
        score = float(roc_auc_score(y, prediction))
        rows.append(
            {
                "weights": weights,
                "auc": score,
                "gain": score - baseline_auc,
                "half_gains": half_gains,
                "min_half_gain": float(min(half_gains)),
            }
        )
    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    selected = max(stable, key=lambda row: (row["gain"], row["min_half_gain"]))
    rows.sort(key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True)
    return selected, rows


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    clean_training = verify_no_gap_artifacts()
    oof = build_heavy_oof()
    test_sources = build_heavy_test()
    membership_oof = pd.read_csv(ROOT / "honest_client_meta/oof_client_features.csv")
    membership_test = pd.read_csv(ROOT / "honest_client_meta/test_client_features.csv")

    raw_train = client.prepare_raw(
        pd.read_csv(ROOT / "train_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    raw_test = client.prepare_raw(
        pd.read_csv(ROOT / "test_transaction.csv", usecols=list(client.RAW_COLUMNS))
    )
    _, _, fold_groups, test_groups = client.build_client_features(oof, raw_train, raw_test)
    raw_uid = pd.read_csv(
        ROOT / "train_transaction.csv",
        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain"],
    )
    uid = meta.make_uid(raw_uid)
    dev_current, dev_current_report = client.current_meta_prediction(oof, uid, client.META_DEV_FOLD)
    lock_current, lock_current_report = client.current_meta_prediction(oof, uid, client.META_LOCK_FOLD)

    selected_model, model_search, dev_heavy = search_models(oof, membership_oof)
    lock_model, lock_heavy, lock_iterations = train_model(
        oof, membership_oof, selected_model, client.META_LOCK_FOLD, 1.15
    )
    lock_model.booster_.save_model(WORK_DIR / "lock_lgb.txt")

    base_report = json.loads(
        (ROOT / "honest_client_meta/report.json").read_text(encoding="utf-8")
    )
    postprocess = base_report["selected_postprocess"]
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_rows = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    selected_weights, weight_search = search_segment_weights(
        oof.loc[dev_mask, client.TARGET].to_numpy(dtype="int8"),
        dev_current,
        dev_heavy,
        segments.segment_labels(membership_oof.loc[dev_mask].reset_index(drop=True)),
        raw_train.iloc[dev_rows]["TransactionDT"].to_numpy(),
        fold_groups[client.META_DEV_FOLD],
        postprocess,
    )
    lock_prediction = segments.segmented_blend(
        lock_current,
        lock_heavy,
        segments.segment_labels(membership_oof.loc[lock_mask].reset_index(drop=True)),
        selected_weights["weights"],
    )
    lock_prediction = client.apply_postprocess(
        lock_prediction, fold_groups[client.META_LOCK_FOLD], postprocess
    )
    y_lock = oof.loc[lock_mask, client.TARGET].to_numpy(dtype="int8")
    lock_auc = float(roc_auc_score(y_lock, lock_prediction))
    previous_lock_auc = float(
        json.loads((ROOT / "honest_client_segments/report.json").read_text(encoding="utf-8"))[
            "segment_candidate_lock_auc"
        ]
    )
    accepted = bool(selected_weights["gain"] > 0.0 and lock_auc > previous_lock_auc)

    final_iterations = max(30, int(np.ceil(lock_iterations * 1.15)))
    all_features = make_view(oof, membership_oof, selected_model["view"], fold=oof["fold"])
    test_features = make_view(test_sources, membership_test, selected_model["view"], fold=None)
    final_model = lgb.LGBMClassifier(
        **{
            **LGB_PARAMS,
            **LGB_CONFIGS[int(selected_model["config_index"])],
            "n_estimators": final_iterations,
        }
    )
    final_model.fit(all_features, oof[client.TARGET], callbacks=[lgb.log_evaluation(0)])
    final_model.booster_.save_model(WORK_DIR / "final_lgb.txt")
    heavy_test = final_model.predict_proba(test_features)[:, 1]
    current_test = pd.read_csv(ROOT / "submission_honest_user_means.csv")
    test_prediction = segments.segmented_blend(
        current_test[client.TARGET].to_numpy(dtype="float64"),
        heavy_test,
        segments.segment_labels(membership_test),
        selected_weights["weights"],
    )
    test_prediction = client.apply_postprocess(test_prediction, test_groups, postprocess)
    output = current_test[["TransactionID"]].copy()
    output[client.TARGET] = test_prediction
    output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test only",
        "no_gap_artifact_training": clean_training,
        "fold_contract": "temporal7_fold == current_fold + 1",
        "sources": list(ALL_HEAVY_SOURCES),
        "selected_model": selected_model,
        "model_search": model_search,
        "selected_segment_weights": selected_weights,
        "weight_search_top": weight_search[:30],
        "postprocess_locked_from_dev": postprocess,
        "current_meta": {"dev": dev_current_report, "lock": lock_current_report},
        "lock_iterations": lock_iterations,
        "final_iterations": final_iterations,
        "previous_best_lock_auc": previous_lock_auc,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - previous_lock_auc,
        "accepted": accepted,
        "output": OUTPUT_PATH.name,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(
        json.dumps(
            {
                "selected_model": selected_model,
                "selected_segment_weights": selected_weights,
                "previous_best_lock_auc": previous_lock_auc,
                "candidate_lock_auc": lock_auc,
                "lock_gain": lock_auc - previous_lock_auc,
                "accepted": accepted,
                "output": OUTPUT_PATH.name,
                "elapsed_minutes": report["elapsed_minutes"],
            },
            indent=2,
        ),
        flush=True,
    )


if __name__ == "__main__":
    main()


Overwriting train_honest_heavy_temporal_client.py


In [32]:
%%writefile train_honest_magic_heavy_stack.py
"""Heavy CatBoost/LightGBM stack on the honest XGB-magic feature view.

Model and blend selection use two purged forward-time windows from official
train. Official test covariates are used only by target-free feature builders.
"""

from __future__ import annotations

import gc
import hashlib
from itertools import product
import json
from pathlib import Path
import time

from catboost import CatBoostClassifier
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import finalize_honest_xgb_magic_blend as xgb_final
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client
import train_honest_xgb_magic as magic


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_magic_heavy_stack"
MODEL_DIR = WORK_DIR / "models"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_magic_heavy_stack.csv"

DEV_TRAIN_END = 45.0
LOCK_TRAIN_END = 60.0
ITERATION_SCALE = 1.15
SEED = 4513

CAT_BASE = {
    "iterations": 2_400,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "bootstrap_type": "Bernoulli",
    "subsample": 0.82,
    "border_count": 128,
    "one_hot_max_size": 16,
    "thread_count": -1,
    "allow_writing_files": False,
    "random_seed": SEED,
    "verbose": 200,
}
CAT_CONFIGS = (
    {
        "name": "ctr_d8",
        "depth": 8,
        "learning_rate": 0.045,
        "l2_leaf_reg": 10.0,
        "random_strength": 0.35,
        "rsm": 0.85,
        "max_ctr_complexity": 2,
    },
    {
        "name": "ctr_d9",
        "depth": 9,
        "learning_rate": 0.035,
        "l2_leaf_reg": 12.0,
        "random_strength": 0.50,
        "rsm": 0.75,
        "max_ctr_complexity": 1,
    },
)

LGB_BASE = {
    "n_estimators": 3_500,
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.015,
    "subsample": 0.84,
    "subsample_freq": 1,
    "colsample_bytree": 0.78,
    "reg_alpha": 1.0,
    "reg_lambda": 12.0,
    "max_bin": 255,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}
LGB_CONFIGS = (
    {
        "name": "leaf63",
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 120,
    },
    {
        "name": "leaf127",
        "num_leaves": 127,
        "max_depth": 10,
        "min_child_samples": 240,
    },
    {
        "name": "extra127",
        "num_leaves": 127,
        "max_depth": 10,
        "min_child_samples": 300,
        "extra_trees": True,
    },
)

CATEGORICAL_NAMES = {
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "card1_addr1",
    "card1_addr1_P_emaildomain",
    *(f"M{i}" for i in range(1, 10)),
    "id_12",
    "id_15",
    "id_16",
    "id_28",
    "id_29",
    "id_31",
    "id_35",
    "id_36",
    "id_37",
    "id_38",
}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def auc(target: np.ndarray, prediction: np.ndarray) -> float:
    return float(roc_auc_score(target, prediction))


def make_cat_frame(
    matrix: np.ndarray,
    row_index: np.ndarray | None,
    feature_names: list[str],
    categorical: list[str],
) -> pd.DataFrame:
    values = matrix if row_index is None else matrix[row_index]
    frame = pd.DataFrame(values, columns=feature_names, copy=False)
    for column in categorical:
        frame[column] = np.rint(frame[column]).astype("int32")
    return frame


def fit_cat(
    matrix: np.ndarray,
    target: np.ndarray,
    train_index: np.ndarray | None,
    predict_matrix: np.ndarray,
    predict_index: np.ndarray | None,
    feature_names: list[str],
    categorical: list[str],
    config: dict,
    model_path: Path,
    fixed_iterations: int | None,
) -> tuple[np.ndarray, int, float]:
    params = {
        **CAT_BASE,
        **{key: value for key, value in config.items() if key != "name"},
    }
    if fixed_iterations is not None:
        params["iterations"] = fixed_iterations
    fit_x = make_cat_frame(matrix, train_index, feature_names, categorical)
    fit_y = target if train_index is None else target[train_index]
    predict_x = make_cat_frame(
        predict_matrix, predict_index, feature_names, categorical
    )
    model = CatBoostClassifier(**params)
    fit_kwargs = {
        "X": fit_x,
        "y": fit_y,
        "cat_features": categorical,
    }
    if fixed_iterations is None:
        fit_kwargs.update(
            {
                "eval_set": (predict_x, target[predict_index]),
                "early_stopping_rounds": 180,
                "use_best_model": True,
            }
        )
    started = time.time()
    model.fit(**fit_kwargs)
    prediction = model.predict_proba(predict_x)[:, 1]
    iterations = int(model.tree_count_)
    model.save_model(model_path)
    minutes = (time.time() - started) / 60.0
    del model, fit_x, predict_x
    gc.collect()
    return prediction, iterations, minutes


def fit_lgb(
    matrix: np.ndarray,
    target: np.ndarray,
    train_index: np.ndarray | None,
    predict_matrix: np.ndarray,
    predict_index: np.ndarray | None,
    config: dict,
    model_path: Path,
    fixed_iterations: int | None,
) -> tuple[np.ndarray, int, float]:
    params = {
        **LGB_BASE,
        **{key: value for key, value in config.items() if key != "name"},
    }
    if fixed_iterations is not None:
        params["n_estimators"] = fixed_iterations
    fit_x = matrix if train_index is None else matrix[train_index]
    fit_y = target if train_index is None else target[train_index]
    predict_x = (
        predict_matrix
        if predict_index is None
        else predict_matrix[predict_index]
    )
    model = lgb.LGBMClassifier(**params)
    fit_kwargs = {"X": fit_x, "y": fit_y}
    if fixed_iterations is None:
        fit_kwargs.update(
            {
                "eval_set": [(predict_x, target[predict_index])],
                "eval_metric": "auc",
                "callbacks": [
                    lgb.early_stopping(180, verbose=False),
                    lgb.log_evaluation(200),
                ],
            }
        )
    else:
        fit_kwargs["callbacks"] = [lgb.log_evaluation(0)]
    started = time.time()
    model.fit(**fit_kwargs)
    best = int(model.best_iteration_ or params["n_estimators"])
    prediction = model.predict_proba(predict_x, num_iteration=best)[:, 1]
    model.booster_.save_model(model_path, num_iteration=best)
    minutes = (time.time() - started) / 60.0
    del model, fit_x, predict_x
    gc.collect()
    return prediction, best, minutes


def reconstruct_champion(
    arrays: dict[str, np.ndarray],
    oof: pd.DataFrame,
    membership: pd.DataFrame,
    clean_oof: dict[int, np.ndarray],
    fold_groups: dict,
    postprocess: dict,
) -> tuple[dict[int, np.ndarray], dict]:
    report = json.loads(
        (ROOT / "honest_xgb_magic/blend_report.json").read_text(
            encoding="utf-8"
        )
    )
    recipe = report["selected"]
    predictions = {}
    for fold, filename in (
        (client.META_DEV_FOLD, "dev_prediction.npy"),
        (client.META_LOCK_FOLD, "lock_prediction.npy"),
    ):
        mask = oof["fold"].eq(fold).to_numpy()
        source = np.load(ROOT / "honest_xgb_magic" / filename)
        transformed = fullrow.transform_variant(
            source, recipe["variant"], fold_groups[fold], postprocess
        )
        predictions[fold] = segments.segmented_blend(
            clean_oof[fold],
            transformed,
            segments.segment_labels(
                membership.loc[mask].reset_index(drop=True)
            ),
            recipe["weights"],
        )
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    metrics = {
        "dev": auc(
            np.asarray(arrays["target"])[
                oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
            ],
            predictions[client.META_DEV_FOLD],
        ),
        "lock": auc(
            np.asarray(arrays["target"])[
                oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
            ],
            predictions[client.META_LOCK_FOLD],
        ),
    }
    if abs(metrics["dev"] - float(recipe["auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce champion dev prediction")
    if abs(metrics["lock"] - float(report["candidate_lock_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce champion lock prediction")
    return predictions, {"recipe": recipe, "metrics": metrics}


def screen_source(
    target: np.ndarray,
    baseline: np.ndarray,
    prediction: np.ndarray,
    groups: dict,
    postprocess: dict,
    segment_labels: np.ndarray,
    day: np.ndarray,
) -> tuple[dict, list[dict]]:
    variants = fullrow.prediction_variants(prediction, groups, postprocess)
    return fast_search_blend(
        target,
        baseline,
        variants,
        segment_labels,
        day,
        starts=((0.0, 0.0, 0.0), (0.2, 0.2, 0.4)),
    )


def fast_search_blend(
    target: np.ndarray,
    baseline: np.ndarray,
    raw_variants: dict[str, np.ndarray],
    segment_labels: np.ndarray,
    day: np.ndarray,
    starts: tuple[tuple[float, float, float], ...],
) -> tuple[dict, list[dict]]:
    """Coordinate search with cached rank transforms for segment weights."""
    weight_grid = tuple(np.round(np.linspace(0.0, 1.0, 11), 2))
    midpoint = np.median(day)
    halves = (day <= midpoint, day > midpoint)
    baseline_rank = client.rank(baseline)
    baseline_auc = auc(target, baseline_rank)
    baseline_halves = [
        auc(target[mask], baseline_rank[mask]) for mask in halves
    ]
    masks = {
        name: segment_labels == name for name in segments.SEGMENTS
    }
    baseline_segment_ranks = {
        name: client.rank(baseline_rank[mask]) for name, mask in masks.items()
    }
    rows = []
    seen = set()

    for variant, raw_prediction in raw_variants.items():
        stacked_rank = client.rank(raw_prediction)
        transformed = {}
        for name, mask in masks.items():
            current = baseline_segment_ranks[name]
            stacked = client.rank(stacked_rank[mask])
            transformed[name] = {}
            for weight in weight_grid:
                signal = (1.0 - weight) * current + weight * stacked
                transformed[name][weight] = segments.rank_match(
                    signal, baseline_rank[mask]
                )

        def evaluate(weights: dict[str, float]) -> dict:
            key = (
                variant,
                *(float(weights[name]) for name in segments.SEGMENTS),
            )
            if key in seen:
                return next(row for row in rows if row["_key"] == key)
            prediction = baseline_rank.copy()
            for name, mask in masks.items():
                prediction[mask] = transformed[name][float(weights[name])]
            score = auc(target, prediction)
            half_auc = [
                auc(target[mask], prediction[mask]) for mask in halves
            ]
            half_gains = [
                value - base
                for value, base in zip(half_auc, baseline_halves)
            ]
            row = {
                "variant": variant,
                "weights": {
                    name: float(weights[name]) for name in segments.SEGMENTS
                },
                "auc": score,
                "gain": score - baseline_auc,
                "half_gains": half_gains,
                "min_half_gain": float(min(half_gains)),
                "_key": key,
            }
            rows.append(row)
            seen.add(key)
            return row

        for start in starts:
            weights = dict(zip(segments.SEGMENTS, start))
            evaluate(weights)
            for _ in range(2):
                for name in segments.SEGMENTS:
                    local = []
                    for weight in weight_grid:
                        candidate = dict(weights)
                        candidate[name] = float(weight)
                        local.append(evaluate(candidate))
                    best = max(
                        local,
                        key=lambda row: (row["auc"], row["min_half_gain"]),
                    )
                    weights = dict(best["weights"])
            neighborhoods = []
            for name in segments.SEGMENTS:
                center = weight_grid.index(float(weights[name]))
                neighborhoods.append(
                    weight_grid[max(0, center - 1) : center + 2]
                )
            for values in product(*neighborhoods):
                evaluate(dict(zip(segments.SEGMENTS, values)))

    stable = [row for row in rows if row["min_half_gain"] >= 0.0]
    if not stable:
        raise RuntimeError("No temporally stable blend candidate")
    selected = max(
        stable, key=lambda row: (row["gain"], row["min_half_gain"])
    )
    rows.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    for row in rows:
        row.pop("_key", None)
    return selected, rows


def source_signals(
    cat_prediction: np.ndarray,
    lgb_prediction: np.ndarray,
    groups: dict,
    postprocess: dict,
) -> dict[str, np.ndarray]:
    cat_rank = client.rank(cat_prediction)
    lgb_rank = client.rank(lgb_prediction)
    cat_post = client.apply_postprocess(cat_rank, groups, postprocess)
    lgb_post = client.apply_postprocess(lgb_rank, groups, postprocess)
    signals = {}
    for mode, left, right in (
        ("rank", cat_rank, lgb_rank),
        ("postprocessed_rank", cat_post, lgb_post),
    ):
        for alpha in (0.0, 0.25, 0.50, 0.75, 1.0):
            signals[f"{mode}_cat{alpha:.2f}"] = (
                alpha * left + (1.0 - alpha) * right
            )
        signals[f"{mode}_max"] = np.maximum(left, right)
        signals[f"{mode}_min"] = np.minimum(left, right)
        signals[f"{mode}_geo"] = np.sqrt(
            np.clip(left, 0.0, None) * np.clip(right, 0.0, None)
        )
    return signals


def search_signals(
    target: np.ndarray,
    baseline: np.ndarray,
    signals: dict[str, np.ndarray],
    segment_labels: np.ndarray,
    day: np.ndarray,
) -> tuple[dict, list[dict]]:
    return fast_search_blend(
        target,
        baseline,
        signals,
        segment_labels,
        day,
        starts=(
            (0.0, 0.0, 0.0),
            (0.0, 0.0, 0.4),
            (0.2, 0.2, 0.4),
            (0.5, 0.5, 0.5),
        ),
    )


def segment_metrics(
    target: np.ndarray,
    baseline: np.ndarray,
    candidate: np.ndarray,
    labels: np.ndarray,
) -> dict[str, dict[str, float | int]]:
    result = {}
    for name in segments.SEGMENTS:
        mask = labels == name
        result[name] = {
            "rows": int(mask.sum()),
            "frauds": int(target[mask].sum()),
            "baseline_auc": auc(target[mask], baseline[mask]),
            "candidate_auc": auc(target[mask], candidate[mask]),
        }
        result[name]["gain"] = (
            result[name]["candidate_auc"] - result[name]["baseline_auc"]
        )
    return result


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    MODEL_DIR.mkdir(exist_ok=True)
    manifest, arrays = magic.load_matrix()
    train = arrays["train"]
    target = arrays["target"]
    day = arrays["day"]
    feature_names = list(manifest["features"])
    categorical = sorted(CATEGORICAL_NAMES.intersection(feature_names))

    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    clean_oof, clean_report = xgb_final.reconstruct_clean_oof(
        oof, membership_oof, reference, fold_groups, reference_report
    )
    champion, champion_report = reconstruct_champion(
        arrays,
        oof,
        membership_oof,
        clean_oof,
        fold_groups,
        clean_report["postprocess"],
    )

    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    dev_train_index = np.flatnonzero(day < DEV_TRAIN_END)
    lock_train_index = np.flatnonzero(day < LOCK_TRAIN_END)
    y_dev = np.asarray(target[dev_index], dtype="int8")
    y_lock = np.asarray(target[lock_index], dtype="int8")
    dev_segments = segments.segment_labels(
        membership_oof.loc[dev_mask].reset_index(drop=True)
    )
    lock_segments = segments.segment_labels(
        membership_oof.loc[lock_mask].reset_index(drop=True)
    )

    dev_rows = []
    dev_predictions: dict[str, np.ndarray] = {}
    for config in CAT_CONFIGS:
        name = config["name"]
        prediction_path = WORK_DIR / f"dev_cat_{name}.npy"
        model_path = MODEL_DIR / f"dev_cat_{name}.cbm"
        cached = prediction_path.exists() and model_path.exists()
        if cached:
            prediction = np.load(prediction_path).astype("float64")
            cached_model = CatBoostClassifier()
            cached_model.load_model(model_path)
            iterations = int(cached_model.tree_count_)
            minutes = 0.0
            del cached_model
        else:
            prediction, iterations, minutes = fit_cat(
                train,
                target,
                dev_train_index,
                train,
                dev_index,
                feature_names,
                categorical,
                config,
                model_path,
                None,
            )
            np.save(prediction_path, prediction.astype("float32"))
        selected, search = screen_source(
            y_dev,
            champion[client.META_DEV_FOLD],
            prediction,
            fold_groups[client.META_DEV_FOLD],
            clean_report["postprocess"],
            dev_segments,
            np.asarray(day[dev_index]),
        )
        row = {
            "family": "catboost",
            "name": name,
            "config": config,
            "source_auc": auc(y_dev, prediction),
            "best_iteration": iterations,
            "minutes": minutes,
            "cached": cached,
            "best_blend": selected,
            "search_top": search[:5],
        }
        dev_rows.append(row)
        dev_predictions[f"catboost:{name}"] = prediction
        print(json.dumps(row), flush=True)

    for config in LGB_CONFIGS:
        name = config["name"]
        prediction_path = WORK_DIR / f"dev_lgb_{name}.npy"
        model_path = MODEL_DIR / f"dev_lgb_{name}.txt"
        cached = prediction_path.exists() and model_path.exists()
        if cached:
            prediction = np.load(prediction_path).astype("float64")
            cached_model = lgb.Booster(model_file=str(model_path))
            iterations = int(cached_model.num_trees())
            minutes = 0.0
            del cached_model
        else:
            prediction, iterations, minutes = fit_lgb(
                train,
                target,
                dev_train_index,
                train,
                dev_index,
                config,
                model_path,
                None,
            )
            np.save(prediction_path, prediction.astype("float32"))
        selected, search = screen_source(
            y_dev,
            champion[client.META_DEV_FOLD],
            prediction,
            fold_groups[client.META_DEV_FOLD],
            clean_report["postprocess"],
            dev_segments,
            np.asarray(day[dev_index]),
        )
        row = {
            "family": "lightgbm",
            "name": name,
            "config": config,
            "source_auc": auc(y_dev, prediction),
            "best_iteration": iterations,
            "minutes": minutes,
            "cached": cached,
            "best_blend": selected,
            "search_top": search[:5],
        }
        dev_rows.append(row)
        dev_predictions[f"lightgbm:{name}"] = prediction
        print(json.dumps(row), flush=True)

    best_cat = max(
        (row for row in dev_rows if row["family"] == "catboost"),
        key=lambda row: (
            row["best_blend"]["gain"],
            row["best_blend"]["min_half_gain"],
        ),
    )
    best_lgb = max(
        (row for row in dev_rows if row["family"] == "lightgbm"),
        key=lambda row: (
            row["best_blend"]["gain"],
            row["best_blend"]["min_half_gain"],
        ),
    )
    dev_cat = dev_predictions[f"catboost:{best_cat['name']}"]
    dev_lgb = dev_predictions[f"lightgbm:{best_lgb['name']}"]
    dev_signals = source_signals(
        dev_cat,
        dev_lgb,
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
    )
    selected, search_rows = search_signals(
        y_dev,
        champion[client.META_DEV_FOLD],
        dev_signals,
        dev_segments,
        np.asarray(day[dev_index]),
    )
    dev_candidate = segments.segmented_blend(
        champion[client.META_DEV_FOLD],
        dev_signals[selected["variant"]],
        dev_segments,
        selected["weights"],
    )

    cat_config = next(
        config for config in CAT_CONFIGS if config["name"] == best_cat["name"]
    )
    lgb_config = next(
        config for config in LGB_CONFIGS if config["name"] == best_lgb["name"]
    )
    cat_lock_iterations = max(
        50, int(np.ceil(best_cat["best_iteration"] * ITERATION_SCALE))
    )
    lgb_lock_iterations = max(
        50, int(np.ceil(best_lgb["best_iteration"] * ITERATION_SCALE))
    )
    lock_cat, _, cat_lock_minutes = fit_cat(
        train,
        target,
        lock_train_index,
        train,
        lock_index,
        feature_names,
        categorical,
        cat_config,
        MODEL_DIR / "lock_cat.cbm",
        cat_lock_iterations,
    )
    lock_lgb, _, lgb_lock_minutes = fit_lgb(
        train,
        target,
        lock_train_index,
        train,
        lock_index,
        lgb_config,
        MODEL_DIR / "lock_lgb.txt",
        lgb_lock_iterations,
    )
    np.save(WORK_DIR / "lock_cat.npy", lock_cat.astype("float32"))
    np.save(WORK_DIR / "lock_lgb.npy", lock_lgb.astype("float32"))
    lock_signals = source_signals(
        lock_cat,
        lock_lgb,
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
    )
    lock_candidate = segments.segmented_blend(
        champion[client.META_LOCK_FOLD],
        lock_signals[selected["variant"]],
        lock_segments,
        selected["weights"],
    )
    dev_auc = auc(y_dev, dev_candidate)
    lock_auc = auc(y_lock, lock_candidate)
    accepted = bool(
        dev_auc > champion_report["metrics"]["dev"]
        and lock_auc > champion_report["metrics"]["lock"]
        and selected["min_half_gain"] >= 0.0
    )

    final_rows = []
    if accepted:
        cat_final_iterations = max(
            50, int(np.ceil(cat_lock_iterations * ITERATION_SCALE))
        )
        lgb_final_iterations = max(
            50, int(np.ceil(lgb_lock_iterations * ITERATION_SCALE))
        )
        test_cat, _, cat_minutes = fit_cat(
            train,
            target,
            None,
            arrays["test"],
            None,
            feature_names,
            categorical,
            cat_config,
            MODEL_DIR / "final_cat.cbm",
            cat_final_iterations,
        )
        final_rows.append(
            {
                "family": "catboost",
                "iterations": cat_final_iterations,
                "minutes": cat_minutes,
            }
        )
        test_lgb, _, lgb_minutes = fit_lgb(
            train,
            target,
            None,
            arrays["test"],
            None,
            lgb_config,
            MODEL_DIR / "final_lgb.txt",
            lgb_final_iterations,
        )
        final_rows.append(
            {
                "family": "lightgbm",
                "iterations": lgb_final_iterations,
                "minutes": lgb_minutes,
            }
        )
        np.save(WORK_DIR / "test_cat.npy", test_cat.astype("float32"))
        np.save(WORK_DIR / "test_lgb.npy", test_lgb.astype("float32"))
        test_signals = source_signals(
            test_cat,
            test_lgb,
            reference_report["test_groups"],
            clean_report["postprocess"],
        )
        baseline = pd.read_csv(ROOT / "submission_honest_xgb_magic_blend.csv")
        if not np.array_equal(
            baseline["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("Heavy-stack test rows differ from champion")
        prediction = segments.segmented_blend(
            baseline[client.TARGET].to_numpy(dtype="float64"),
            test_signals[selected["variant"]],
            segments.segment_labels(membership_test),
            selected["weights"],
        )
        output = baseline[["TransactionID"]].copy()
        output[client.TARGET] = prediction
        output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": (
            "official train/test covariates; official train labels only; "
            "official train labels only"
        ),
        "official_hashes": manifest["official_hashes"],
        "feature_recipe": manifest["source"],
        "feature_count": len(feature_names),
        "categorical_count": len(categorical),
        "categorical": categorical,
        "fold_contract": {
            "dev_train": f"day < {DEV_TRAIN_END:g}",
            "dev_valid": "day 75-90",
            "lock_train": f"day < {LOCK_TRAIN_END:g}",
            "lock_valid": "day 90-105",
        },
        "champion": champion_report,
        "development_screen": dev_rows,
        "selected_cat": best_cat,
        "selected_lgb": best_lgb,
        "selected_blend": selected,
        "search_top": search_rows[:40],
        "dev_auc": dev_auc,
        "dev_gain": dev_auc - champion_report["metrics"]["dev"],
        "lock_auc": lock_auc,
        "lock_gain": lock_auc - champion_report["metrics"]["lock"],
        "lock_source_auc": {
            "catboost": auc(y_lock, lock_cat),
            "lightgbm": auc(y_lock, lock_lgb),
        },
        "segment_metrics": {
            "dev": segment_metrics(
                y_dev,
                champion[client.META_DEV_FOLD],
                dev_candidate,
                dev_segments,
            ),
            "lock": segment_metrics(
                y_lock,
                champion[client.META_LOCK_FOLD],
                lock_candidate,
                lock_segments,
            ),
        },
        "lock_iterations": {
            "catboost": cat_lock_iterations,
            "lightgbm": lgb_lock_iterations,
        },
        "lock_minutes": {
            "catboost": cat_lock_minutes,
            "lightgbm": lgb_lock_minutes,
        },
        "acceptance_rule": "strictly improve both dev and untouched lock",
        "accepted": accepted,
        "final_models": final_rows,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_magic_heavy_stack.py


In [33]:
%%writefile train_honest_user_means_catboost.py
"""Ablate broad per-user C/D/V means in the advanced CatBoost."""

from __future__ import annotations

import argparse
import gc
import json
from pathlib import Path
import re
import time

from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from fraud_honest_advanced_data import ADVANCED_CAT_PARAMS as CAT_PARAMS
from fraud_vblock_features import add_vblock_user_features
from train_honest_advanced_catboost import FOLDS, prepare


ROOT = Path(__file__).resolve().parent
CACHE_DIR = ROOT / "honest_user_means_catboost"
REPORT_PATH = CACHE_DIR / "report.json"
SOURCE = "user_means_catboost"
TARGET = "isFraud"
MAX_ITERATIONS = 1_900
SEED = 1729


def parse_folds(value: str) -> tuple[int, ...]:
    result = tuple(int(item) for item in value.split(",") if item.strip())
    if not result or any(fold not in {0, 1, 2} for fold in result):
        raise argparse.ArgumentTypeError("Use a comma-separated subset of 0,1,2")
    return result


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--folds", type=parse_folds, default=(2,))
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()
    CACHE_DIR.mkdir(exist_ok=True)
    started = time.time()

    prepared, features, categorical = prepare()
    (
        prepared["train"],
        prepared["inference"],
        vblock_features,
        vblock_stats,
    ) = add_vblock_user_features(
        prepared["train"],
        prepared["inference"],
        prepared["train_components"],
        prepared["inference_components"],
    )
    mean_features = [
        column
        for column in vblock_features
        if re.fullmatch(r"wide_user_[CDV]\d+_mean", column)
    ]
    features = list(dict.fromkeys([*features, *mean_features]))
    day = prepared["train"]["TransactionDT"].to_numpy(dtype="float64") / 86_400.0
    y = prepared["y"].to_numpy(dtype="int8")
    rows = []
    for fold, train_end, valid_start, valid_end in FOLDS:
        if fold not in args.folds:
            continue
        fit_index = np.flatnonzero(day < train_end)
        valid_index = np.flatnonzero((day >= valid_start) & (day < valid_end))
        model_path = CACHE_DIR / f"fold_{fold}.cbm"
        prediction_path = CACHE_DIR / f"fold_{fold}_prediction.npy"
        fold_started = time.time()
        if model_path.exists() and prediction_path.exists() and not args.force:
            prediction = np.load(prediction_path)
            model = CatBoostClassifier()
            model.load_model(model_path)
            iteration = int(model.tree_count_)
            if prediction.dtype != np.float64:
                print(
                    f"Refreshing fold {fold} predictions as float64",
                    flush=True,
                )
                prediction = model.predict_proba(
                    prepared["train"].iloc[valid_index][features]
                )[:, 1]
                np.save(prediction_path, prediction)
            del model
            cached = True
        else:
            print(
                f"User-means CatBoost fold {fold}: {len(fit_index):,} -> "
                f"{len(valid_index):,}; {len(features)} features",
                flush=True,
            )
            model = CatBoostClassifier(
                **{
                    **CAT_PARAMS,
                    "iterations": MAX_ITERATIONS,
                    "random_seed": SEED + fold,
                    "verbose": 200,
                }
            )
            model.fit(
                prepared["train"].iloc[fit_index][features],
                y[fit_index],
                cat_features=categorical,
                eval_set=(
                    prepared["train"].iloc[valid_index][features],
                    y[valid_index],
                ),
                early_stopping_rounds=140,
                use_best_model=True,
            )
            prediction = model.predict_proba(
                prepared["train"].iloc[valid_index][features]
            )[:, 1]
            iteration = int(model.tree_count_)
            model.save_model(model_path)
            np.save(prediction_path, prediction)
            del model
            gc.collect()
            cached = False
        score = float(roc_auc_score(y[valid_index], prediction))
        pd.DataFrame(
            {
                "row_index": valid_index,
                "TransactionID": prepared["train_ids"].iloc[valid_index],
                "fold": fold,
                TARGET: y[valid_index],
                SOURCE: prediction,
            }
        ).to_csv(CACHE_DIR / f"fold_{fold}_oof.csv", index=False)
        row = {
            "fold": fold,
            "train_rows": int(len(fit_index)),
            "valid_rows": int(len(valid_index)),
            "auc": score,
            "best_iteration": iteration,
            "cached": cached,
            "minutes": (time.time() - fold_started) / 60.0,
        }
        rows.append(row)
        print(json.dumps(row, indent=2), flush=True)

    report = {
        "data_policy": "official train/test only",
        "folds_requested": list(args.folds),
        "base_features": len(features) - len(mean_features),
        "user_mean_features": mean_features,
        "total_features": len(features),
        "categorical": len(categorical),
        "vblock_stats": vblock_stats,
        "models": rows,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_user_means_catboost.py


In [34]:
%%writefile train_honest_xgb_magic.py
"""Rebuild Chris Deotte's XGB magic recipe on the official local data only."""

from __future__ import annotations

import gc
import hashlib
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

from fraud_features import read_and_merge
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_xgb_magic"
CACHE_DIR = WORK_DIR / "matrix"
MODEL_DIR = WORK_DIR / "models"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_xgb_magic.csv"
TARGET = client.TARGET
DAY_SECONDS = 86_400.0
RECIPE_VERSION = "deotte_xgb_magic_v1"

V_NUMBERS = (
    1, 3, 4, 6, 8, 11,
    13, 14, 17, 20, 23, 26, 27, 30,
    36, 37, 40, 41, 44, 47, 48,
    54, 56, 59, 62, 65, 67, 68, 70,
    76, 78, 80, 82, 86, 88, 89, 91,
    107, 108, 111, 115, 117, 120, 121, 123,
    124, 127, 129, 130, 136,
    138, 139, 142, 147, 156, 162,
    165, 160, 166,
    178, 176, 173, 182,
    187, 203, 205, 207, 215,
    169, 171, 175, 180, 185, 188, 198, 210, 209,
    218, 223, 224, 226, 228, 229, 235,
    240, 258, 257, 253, 252, 260, 261,
    264, 266, 267, 274, 277,
    220, 221, 234, 238, 250, 271,
    294, 284, 285, 286, 291, 297,
    303, 305, 307, 309, 310, 320,
    281, 283, 289, 296, 301, 314,
)
TRANSACTION_COLUMNS = (
    "TransactionDT", "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    *(f"C{i}" for i in range(1, 15)),
    *(f"D{i}" for i in range(1, 16)),
    *(f"M{i}" for i in range(1, 10)),
    *(f"V{i}" for i in V_NUMBERS),
)
FAILED_TIME_FEATURES = {
    "C3", "M5", "id_08", "id_33", "card4", "id_07", "id_14",
    "id_21", "id_30", "id_32", "id_34",
    *(f"id_{i:02d}" for i in range(22, 28)),
}
REMOVED_D = {"D6", "D7", "D8", "D9", "D12", "D13", "D14"}
XGB_PARAMS = {
    "n_estimators": 3_500,
    "max_depth": 12,
    "learning_rate": 0.02,
    "subsample": 0.8,
    "colsample_bytree": 0.4,
    "missing": -1,
    "eval_metric": "auc",
    "tree_method": "hist",
    "max_bin": 256,
    "objective": "binary:logistic",
    "random_state": 2027,
    "n_jobs": -1,
}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def official_hashes() -> dict[str, str]:
    return {
        name: file_sha256(ROOT / name)
        for name in (
            "train_transaction.csv",
            "train_identity.csv",
            "test_transaction.csv",
            "test_identity.csv",
            "sample_submission.csv",
        )
    }


def cache_paths() -> dict[str, Path]:
    return {
        name: CACHE_DIR / f"{name}.npy"
        for name in (
            "train", "test", "target", "day", "month", "train_id", "test_id"
        )
    }


def joint_factorize(train: pd.DataFrame, test: pd.DataFrame, column: str) -> None:
    combined = pd.concat(
        [train[column], test[column]], ignore_index=True, copy=False
    )
    codes, _ = pd.factorize(combined, sort=True)
    train[column] = codes[: len(train)].astype("int32")
    test[column] = codes[len(train) :].astype("int32")


def frequency_encode(
    train: pd.DataFrame,
    test: pd.DataFrame,
    columns: list[str],
) -> list[str]:
    names = []
    for column in columns:
        combined = pd.concat(
            [train[column], test[column]], ignore_index=True, copy=False
        )
        frequency = combined.value_counts(dropna=True, normalize=True)
        name = f"{column}_FE"
        train[name] = train[column].map(frequency).fillna(-1).astype("float32")
        test[name] = test[column].map(frequency).fillna(-1).astype("float32")
        names.append(name)
    return names


def combine_columns(
    train: pd.DataFrame,
    test: pd.DataFrame,
    left: str,
    right: str,
) -> str:
    name = f"{left}_{right}"
    train[name] = train[left].astype(str).str.cat(train[right].astype(str), sep="_")
    test[name] = test[left].astype(str).str.cat(test[right].astype(str), sep="_")
    joint_factorize(train, test, name)
    return name


def aggregate_values(
    train: pd.DataFrame,
    test: pd.DataFrame,
    main_columns: list[str],
    group_columns: list[str],
    aggregations: tuple[str, ...],
    use_na: bool,
) -> list[str]:
    names = []
    for main_column in main_columns:
        for group_column in group_columns:
            combined = pd.concat(
                [
                    train[[group_column, main_column]],
                    test[[group_column, main_column]],
                ],
                ignore_index=True,
                copy=False,
            ).copy()
            if use_na:
                combined.loc[combined[main_column].eq(-1), main_column] = np.nan
            grouped = combined.groupby(group_column, dropna=False)[main_column]
            for aggregation in aggregations:
                name = f"{main_column}_{group_column}_{aggregation}"
                mapping = grouped.agg(aggregation)
                train[name] = (
                    train[group_column].map(mapping).fillna(-1).astype("float32")
                )
                test[name] = (
                    test[group_column].map(mapping).fillna(-1).astype("float32")
                )
                names.append(name)
    return names


def aggregate_nunique(
    train: pd.DataFrame,
    test: pd.DataFrame,
    main_columns: list[str],
    group_columns: list[str],
) -> list[str]:
    names = []
    for main_column in main_columns:
        for group_column in group_columns:
            combined = pd.concat(
                [
                    train[[group_column, main_column]],
                    test[[group_column, main_column]],
                ],
                ignore_index=True,
                copy=False,
            )
            mapping = combined.groupby(group_column, dropna=False)[
                main_column
            ].nunique(dropna=True)
            name = f"{group_column}_{main_column}_ct"
            train[name] = train[group_column].map(mapping).fillna(0).astype("float32")
            test[name] = test[group_column].map(mapping).fillna(0).astype("float32")
            names.append(name)
    return names


def transaction_month(transaction_dt: pd.Series) -> np.ndarray:
    timestamp = pd.Timestamp("2017-11-30") + pd.to_timedelta(
        transaction_dt, unit="s"
    )
    return ((timestamp.dt.year - 2017) * 12 + timestamp.dt.month).to_numpy(
        dtype="int16"
    )


def build_cache(hashes: dict[str, str]) -> dict:
    print("Building the exact XGB-magic feature matrix...", flush=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    train_all = read_and_merge(ROOT, "train").reset_index(drop=True)
    test_all = read_and_merge(ROOT, "test").reset_index(drop=True)
    if TARGET not in train_all or TARGET in test_all:
        raise RuntimeError("Unexpected target placement in official files")
    identity_columns = [
        column for column in train_all.columns if column.startswith("id_")
    ]
    selected_input = [
        column
        for column in (*TRANSACTION_COLUMNS, *identity_columns)
        if column in train_all and column in test_all
    ]
    train = train_all[selected_input].copy()
    test = test_all[selected_input].copy()
    target = train_all[TARGET].to_numpy(dtype="int8")
    day = (train["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS).astype(
        "float32"
    )
    month = transaction_month(train["TransactionDT"])
    test_month = transaction_month(test["TransactionDT"])
    train_id = train_all["TransactionID"].to_numpy(dtype="int32")
    test_id = test_all["TransactionID"].to_numpy(dtype="int32")
    del train_all, test_all
    gc.collect()

    for number in range(1, 16):
        if number in {1, 2, 3, 5, 9}:
            continue
        column = f"D{number}"
        train[column] = train[column] - train["TransactionDT"] / DAY_SECONDS
        test[column] = test[column] - test["TransactionDT"] / DAY_SECONDS

    for column in list(train.columns):
        if not pd.api.types.is_numeric_dtype(train[column]):
            joint_factorize(train, test, column)
        elif column not in {"TransactionAmt", "TransactionDT"}:
            minimum = min(train[column].min(skipna=True), test[column].min(skipna=True))
            if pd.isna(minimum):
                minimum = 0.0
            train[column] = (train[column] - minimum).fillna(-1)
            test[column] = (test[column] - minimum).fillna(-1)

    train["cents"] = (
        train["TransactionAmt"] - np.floor(train["TransactionAmt"])
    ).astype("float32")
    test["cents"] = (
        test["TransactionAmt"] - np.floor(test["TransactionAmt"])
    ).astype("float32")
    frequency_encode(
        train, test, ["addr1", "card1", "card2", "card3", "P_emaildomain"]
    )
    card_addr = combine_columns(train, test, "card1", "addr1")
    card_addr_email = combine_columns(train, test, card_addr, "P_emaildomain")
    frequency_encode(train, test, [card_addr, card_addr_email])
    aggregate_values(
        train,
        test,
        ["TransactionAmt", "D9", "D11"],
        ["card1", card_addr, card_addr_email],
        ("mean", "std"),
        use_na=True,
    )

    train["DT_M"] = month
    test["DT_M"] = test_month
    train["day"] = day
    test["day"] = (
        test["TransactionDT"].to_numpy(dtype="float64") / DAY_SECONDS
    ).astype("float32")
    train["uid"] = train[card_addr].astype(str).str.cat(
        np.floor(train["day"] - train["D1"]).astype(str), sep="_"
    )
    test["uid"] = test[card_addr].astype(str).str.cat(
        np.floor(test["day"] - test["D1"]).astype(str), sep="_"
    )
    frequency_encode(train, test, ["uid"])
    aggregate_values(
        train,
        test,
        ["TransactionAmt", "D4", "D9", "D10", "D15"],
        ["uid"],
        ("mean", "std"),
        use_na=True,
    )
    aggregate_values(
        train,
        test,
        [f"C{i}" for i in range(1, 15) if i != 3],
        ["uid"],
        ("mean",),
        use_na=True,
    )
    aggregate_values(
        train,
        test,
        [f"M{i}" for i in range(1, 10)],
        ["uid"],
        ("mean",),
        use_na=True,
    )
    aggregate_nunique(
        train,
        test,
        ["P_emaildomain", "dist1", "DT_M", "id_02", "cents"],
        ["uid"],
    )
    aggregate_values(
        train, test, ["C14"], ["uid"], ("std",), use_na=True
    )
    aggregate_nunique(train, test, ["C13", "V314"], ["uid"])
    aggregate_nunique(
        train, test, ["V127", "V136", "V309", "V307", "V320"], ["uid"]
    )
    train["outsider15"] = (train["D1"] - train["D15"]).abs().gt(3).astype("int8")
    test["outsider15"] = (test["D1"] - test["D15"]).abs().gt(3).astype("int8")

    excluded = {
        "TransactionDT", "DT_M", "day", "uid", *REMOVED_D, *FAILED_TIME_FEATURES
    }
    features = [column for column in train.columns if column not in excluded]
    missing_numeric = [
        column
        for column in features
        if not pd.api.types.is_numeric_dtype(train[column])
    ]
    if missing_numeric:
        raise RuntimeError(f"Non-numeric XGB features remain: {missing_numeric[:10]}")
    train_matrix = train[features].astype("float32").to_numpy(copy=True)
    test_matrix = test[features].astype("float32").to_numpy(copy=True)
    paths = cache_paths()
    for name, values in (
        ("train", train_matrix),
        ("test", test_matrix),
        ("target", target),
        ("day", day),
        ("month", month),
        ("train_id", train_id),
        ("test_id", test_id),
    ):
        np.save(paths[name], values)
    manifest = {
        "recipe_version": RECIPE_VERSION,
        "official_hashes": hashes,
        "features": features,
        "train_shape": list(train_matrix.shape),
        "test_shape": list(test_matrix.shape),
        "target_sum": int(target.sum()),
        "source": "https://www.kaggle.com/code/cdeotte/xgb-fraud-with-magic-0-9600",
    }
    (CACHE_DIR / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    del train, test, train_matrix, test_matrix
    gc.collect()
    return manifest


def load_matrix() -> tuple[dict, dict[str, np.ndarray]]:
    hashes = official_hashes()
    paths = cache_paths()
    manifest_path = CACHE_DIR / "manifest.json"
    manifest = None
    if manifest_path.exists() and all(path.exists() for path in paths.values()):
        candidate = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (
            candidate.get("recipe_version") == RECIPE_VERSION
            and candidate.get("official_hashes") == hashes
        ):
            manifest = candidate
            print("Loading verified XGB-magic matrix cache", flush=True)
    if manifest is None:
        manifest = build_cache(hashes)
    arrays = {
        name: np.load(path, mmap_mode="r") for name, path in paths.items()
    }
    return manifest, arrays


def fit_temporal_source(
    train: np.ndarray,
    target: np.ndarray,
    day: np.ndarray,
    train_end: float,
    valid_index: np.ndarray,
    model_path: Path,
    iterations: int | None,
) -> tuple[np.ndarray, int, float]:
    train_index = np.flatnonzero(day < train_end)
    params = {**XGB_PARAMS}
    if iterations is None:
        params["early_stopping_rounds"] = 150
    else:
        params["n_estimators"] = iterations
    model = xgb.XGBClassifier(**params)
    fit_kwargs = {
        "X": train[train_index],
        "y": target[train_index],
        "verbose": False,
    }
    if iterations is None:
        fit_kwargs["eval_set"] = [(train[valid_index], target[valid_index])]
    started = time.time()
    model.fit(**fit_kwargs)
    prediction = model.predict_proba(train[valid_index])[:, 1]
    best = int(getattr(model, "best_iteration", params["n_estimators"] - 1) + 1)
    model.save_model(model_path)
    minutes = (time.time() - started) / 60.0
    del model
    gc.collect()
    return prediction, best, minutes


def train_group_models(
    train: np.ndarray,
    test: np.ndarray,
    target: np.ndarray,
    month: np.ndarray,
    fixed_iterations: int | None = None,
) -> tuple[np.ndarray, list[dict]]:
    unique_months = np.unique(month)
    splitter = GroupKFold(n_splits=len(unique_months))
    predictions = []
    rows = []
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(np.zeros(len(target)), target, groups=month)
    ):
        held_month = int(np.unique(month[valid_index])[0])
        params = {
            **XGB_PARAMS,
            "random_state": XGB_PARAMS["random_state"] + fold,
        }
        if fixed_iterations is None:
            params["early_stopping_rounds"] = 180
        else:
            params["n_estimators"] = fixed_iterations
        model = xgb.XGBClassifier(**params)
        started = time.time()
        fit_kwargs = {
            "X": train[train_index],
            "y": target[train_index],
            "verbose": False,
        }
        if fixed_iterations is None:
            fit_kwargs["eval_set"] = [
                (train[valid_index], target[valid_index])
            ]
        model.fit(**fit_kwargs)
        prediction = model.predict_proba(test)[:, 1]
        predictions.append(prediction)
        model_path = MODEL_DIR / f"month_fold_{fold}.json"
        model.save_model(model_path)
        rows.append(
            {
                "fold": fold,
                "held_month": held_month,
                "train_rows": int(len(train_index)),
                "valid_rows": int(len(valid_index)),
                "best_iteration": int(
                    model.best_iteration + 1
                    if fixed_iterations is None
                    else fixed_iterations
                ),
                "valid_auc": float(
                    roc_auc_score(
                        target[valid_index],
                        model.predict_proba(train[valid_index])[:, 1],
                    )
                ),
                "minutes": (time.time() - started) / 60.0,
                "model": str(model_path),
            }
        )
        print(json.dumps(rows[-1]), flush=True)
        del model
        gc.collect()
    return np.mean(predictions, axis=0), rows


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    MODEL_DIR.mkdir(exist_ok=True)
    manifest, arrays = load_matrix()
    train = arrays["train"]
    target = arrays["target"]
    day = arrays["day"]
    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")

    dev_prediction, dev_iterations, dev_minutes = fit_temporal_source(
        train,
        target,
        day,
        45.0,
        dev_index,
        MODEL_DIR / "dev.json",
        iterations=None,
    )
    lock_iterations = max(50, int(np.ceil(dev_iterations * 1.15)))
    lock_prediction, _, lock_minutes = fit_temporal_source(
        train,
        target,
        day,
        60.0,
        lock_index,
        MODEL_DIR / "lock.json",
        iterations=lock_iterations,
    )
    np.save(WORK_DIR / "dev_prediction.npy", dev_prediction.astype("float32"))
    np.save(WORK_DIR / "lock_prediction.npy", lock_prediction.astype("float32"))

    postprocess = reference_report["recipe"]["postprocess_locked_from_dev"]
    dev_variants = fullrow.prediction_variants(
        dev_prediction, fold_groups[client.META_DEV_FOLD], postprocess
    )
    selected_blend, blend_search = fullrow.search_blend(
        np.asarray(target[dev_index], dtype="int8"),
        reference[client.META_DEV_FOLD],
        dev_variants,
        segments.segment_labels(
            membership_oof.loc[dev_mask].reset_index(drop=True)
        ),
        np.asarray(day[dev_index]),
    )
    lock_variant = fullrow.transform_variant(
        lock_prediction,
        selected_blend["variant"],
        fold_groups[client.META_LOCK_FOLD],
        postprocess,
    )
    lock_blend = segments.segmented_blend(
        reference[client.META_LOCK_FOLD],
        lock_variant,
        segments.segment_labels(
            membership_oof.loc[lock_mask].reset_index(drop=True)
        ),
        selected_blend["weights"],
    )
    y_dev = np.asarray(target[dev_index], dtype="int8")
    y_lock = np.asarray(target[lock_index], dtype="int8")
    source_metrics = {
        "dev_auc": float(roc_auc_score(y_dev, dev_prediction)),
        "lock_auc": float(roc_auc_score(y_lock, lock_prediction)),
        "lock_blend_auc": float(roc_auc_score(y_lock, lock_blend)),
    }
    previous_lock = float(
        json.loads(
            (ROOT / "honest_cleanv2_blend/report.json").read_text(encoding="utf-8")
        )["candidate_lock_auc"]
    )
    accepted = bool(
        selected_blend["gain"] > 0
        and source_metrics["lock_blend_auc"] > previous_lock
    )

    group_models = []
    if accepted:
        test_prediction, group_models = train_group_models(
            train, arrays["test"], target, arrays["month"]
        )
        np.save(WORK_DIR / "test_prediction.npy", test_prediction.astype("float32"))
        test_variant = fullrow.transform_variant(
            test_prediction,
            selected_blend["variant"],
            reference_report["test_groups"],
            postprocess,
        )
        baseline_test = pd.read_csv(
            ROOT / "submission_honest_featureview_client.csv"
        )
        if not np.array_equal(
            baseline_test["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("XGB test rows differ from the reference submission")
        output_prediction = segments.segmented_blend(
            baseline_test[TARGET].to_numpy(dtype="float64"),
            test_variant,
            segments.segment_labels(membership_test),
            selected_blend["weights"],
        )
        output = baseline_test[["TransactionID"]].copy()
        output[TARGET] = output_prediction
        output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": "official train/test covariates; official train labels only",
        "source_recipe": manifest["source"],
        "selection": "published XGB recipe; blend on days 75-90; days 90-105 lock",
        "features": len(manifest["features"]),
        "params": XGB_PARAMS,
        "dev_iterations": dev_iterations,
        "lock_iterations": lock_iterations,
        "dev_minutes": dev_minutes,
        "lock_minutes": lock_minutes,
        "source_metrics": source_metrics,
        "selected_blend": selected_blend,
        "blend_search_top": blend_search[:30],
        "previous_cleanv2_lock_auc": previous_lock,
        "lock_gain": source_metrics["lock_blend_auc"] - previous_lock,
        "accepted": accepted,
        "group_models": group_models,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_xgb_magic.py


In [35]:
%%writefile train_honest_xgb_seed_subset_gkf.py
"""Select an XGB seed subset on temporal dev and preserve month-GroupKFold."""

from __future__ import annotations

import gc
import hashlib
from itertools import combinations
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

import finalize_honest_xgb_magic_blend as xgb_final
import refine_honest_client_segments as segments
import search_honest_featureview_meta as featureview
import search_honest_fullrow_lgb as fullrow
import train_honest_client_meta as client
import train_honest_magic_heavy_stack as heavy
import train_honest_xgb_magic as magic


ROOT = Path(__file__).resolve().parent
WORK_DIR = ROOT / "honest_xgb_seed_subset_gkf"
MODEL_DIR = WORK_DIR / "models"
REPORT_PATH = WORK_DIR / "report.json"
OUTPUT_PATH = ROOT / "submission_honest_xgb_seed_subset_gkf.csv"
SEEDS = (2027, 3407, 7907, 12011)
PREDICTION_CACHE_VERSION = 2


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def auc(target: np.ndarray, prediction: np.ndarray) -> float:
    return float(roc_auc_score(target, prediction))


def load_xgb_prediction(
    model_path: Path,
    matrix: np.ndarray,
    row_index: np.ndarray,
) -> np.ndarray:
    # ``missing`` is a sklearn-wrapper inference parameter and is not restored
    # by load_model, so construct the wrapper with the original recipe first.
    model = xgb.XGBClassifier(**magic.XGB_PARAMS)
    model.load_model(model_path)
    prediction = model.predict_proba(matrix[row_index])[:, 1]
    del model
    gc.collect()
    return prediction


def load_seed_predictions(
    matrix: np.ndarray,
    dev_index: np.ndarray,
    lock_index: np.ndarray,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    dev = {
        2027: np.load(ROOT / "honest_xgb_magic/dev_prediction.npy").astype(
            "float64"
        )
    }
    lock = {
        2027: np.load(ROOT / "honest_xgb_magic/lock_prediction.npy").astype(
            "float64"
        )
    }
    for seed in SEEDS[1:]:
        dev_path = (
            WORK_DIR / f"dev_seed_{seed}_v{PREDICTION_CACHE_VERSION}.npy"
        )
        lock_path = (
            WORK_DIR / f"lock_seed_{seed}_v{PREDICTION_CACHE_VERSION}.npy"
        )
        if dev_path.exists() and lock_path.exists():
            dev[seed] = np.load(dev_path).astype("float64")
            lock[seed] = np.load(lock_path).astype("float64")
            continue
        dev[seed] = load_xgb_prediction(
            ROOT / f"honest_xgb_magic_multiseed/models/dev_seed_{seed}.json",
            matrix,
            dev_index,
        )
        lock[seed] = load_xgb_prediction(
            ROOT / f"honest_xgb_magic_multiseed/models/lock_seed_{seed}.json",
            matrix,
            lock_index,
        )
        np.save(dev_path, dev[seed].astype("float32"))
        np.save(lock_path, lock[seed].astype("float32"))
    return dev, lock


def apply_xgb_layer(
    clean_prediction: np.ndarray,
    source_prediction: np.ndarray,
    groups: dict,
    postprocess: dict,
    membership: pd.DataFrame,
    recipe: dict,
) -> np.ndarray:
    transformed = fullrow.transform_variant(
        source_prediction, recipe["variant"], groups, postprocess
    )
    return segments.segmented_blend(
        clean_prediction,
        transformed,
        segments.segment_labels(membership),
        recipe["weights"],
    )


def apply_heavy_layer(
    xgb_prediction: np.ndarray,
    cat_prediction: np.ndarray,
    lgb_prediction: np.ndarray,
    groups: dict,
    postprocess: dict,
    membership: pd.DataFrame,
    recipe: dict,
) -> np.ndarray:
    signals = heavy.source_signals(
        cat_prediction, lgb_prediction, groups, postprocess
    )
    return segments.segmented_blend(
        xgb_prediction,
        signals[recipe["variant"]],
        segments.segment_labels(membership),
        recipe["weights"],
    )


def train_seed_group_models(
    train: np.ndarray,
    test: np.ndarray,
    target: np.ndarray,
    month: np.ndarray,
    seed: int,
    iterations: int,
) -> tuple[np.ndarray, list[dict]]:
    cache_path = WORK_DIR / f"test_seed_{seed}.npy"
    report_path = WORK_DIR / f"test_seed_{seed}.json"
    if cache_path.exists() and report_path.exists():
        return (
            np.load(cache_path).astype("float64"),
            json.loads(report_path.read_text(encoding="utf-8")),
        )
    unique_months = np.unique(month)
    splitter = GroupKFold(n_splits=len(unique_months))
    predictions = []
    rows = []
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(np.zeros(len(target)), target, groups=month)
    ):
        params = {
            **magic.XGB_PARAMS,
            "n_estimators": iterations,
            "random_state": seed + fold,
        }
        model = xgb.XGBClassifier(**params)
        started = time.time()
        model.fit(train[train_index], target[train_index], verbose=False)
        predictions.append(model.predict_proba(test)[:, 1])
        model_path = MODEL_DIR / f"seed_{seed}_fold_{fold}.json"
        model.save_model(model_path)
        valid_prediction = model.predict_proba(train[valid_index])[:, 1]
        row = {
            "seed": seed,
            "fold": fold,
            "held_month": int(np.unique(month[valid_index])[0]),
            "train_rows": int(len(train_index)),
            "valid_rows": int(len(valid_index)),
            "iterations": iterations,
            "valid_auc": auc(target[valid_index], valid_prediction),
            "minutes": (time.time() - started) / 60.0,
            "model": str(model_path.relative_to(ROOT)),
        }
        rows.append(row)
        print(json.dumps(row), flush=True)
        del model
        gc.collect()
    prediction = np.mean(predictions, axis=0)
    np.save(cache_path, prediction.astype("float32"))
    report_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    return prediction, rows


def main() -> None:
    started = time.time()
    WORK_DIR.mkdir(exist_ok=True)
    MODEL_DIR.mkdir(exist_ok=True)
    manifest, arrays = magic.load_matrix()
    oof = featureview.build_oof()
    membership_oof = pd.read_csv(
        ROOT / "honest_client_meta/oof_client_features.csv"
    )
    membership_test = pd.read_csv(
        ROOT / "honest_client_meta/test_client_features.csv"
    )
    reference, fold_groups, reference_report = fullrow.build_reference_oof(
        oof, membership_oof
    )
    clean_oof, clean_report = xgb_final.reconstruct_clean_oof(
        oof, membership_oof, reference, fold_groups, reference_report
    )

    dev_mask = oof["fold"].eq(client.META_DEV_FOLD).to_numpy()
    lock_mask = oof["fold"].eq(client.META_LOCK_FOLD).to_numpy()
    dev_index = oof.loc[dev_mask, "row_index"].to_numpy(dtype="int64")
    lock_index = oof.loc[lock_mask, "row_index"].to_numpy(dtype="int64")
    y_dev = np.asarray(arrays["target"][dev_index], dtype="int8")
    y_lock = np.asarray(arrays["target"][lock_index], dtype="int8")
    day_dev = np.asarray(arrays["day"][dev_index])
    dev_membership = membership_oof.loc[dev_mask].reset_index(drop=True)
    lock_membership = membership_oof.loc[lock_mask].reset_index(drop=True)

    xgb_report = json.loads(
        (ROOT / "honest_xgb_magic/blend_report.json").read_text(
            encoding="utf-8"
        )
    )
    heavy_report = json.loads(
        (ROOT / "honest_magic_heavy_stack/report.json").read_text(
            encoding="utf-8"
        )
    )
    xgb_recipe = xgb_report["selected"]
    heavy_recipe = heavy_report["selected_blend"]
    cat_name = heavy_report["selected_cat"]["name"]
    lgb_name = heavy_report["selected_lgb"]["name"]
    dev_cat = np.load(WORK_DIR.parent / f"honest_magic_heavy_stack/dev_cat_{cat_name}.npy")
    dev_lgb = np.load(WORK_DIR.parent / f"honest_magic_heavy_stack/dev_lgb_{lgb_name}.npy")
    lock_cat = np.load(ROOT / "honest_magic_heavy_stack/lock_cat.npy")
    lock_lgb = np.load(ROOT / "honest_magic_heavy_stack/lock_lgb.npy")

    dev_seed, lock_seed = load_seed_predictions(
        arrays["train"], dev_index, lock_index
    )
    current_dev_xgb = apply_xgb_layer(
        clean_oof[client.META_DEV_FOLD],
        dev_seed[2027],
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
        dev_membership,
        xgb_recipe,
    )
    current_lock_xgb = apply_xgb_layer(
        clean_oof[client.META_LOCK_FOLD],
        lock_seed[2027],
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        xgb_recipe,
    )
    current_dev = apply_heavy_layer(
        current_dev_xgb,
        dev_cat,
        dev_lgb,
        fold_groups[client.META_DEV_FOLD],
        clean_report["postprocess"],
        dev_membership,
        heavy_recipe,
    )
    current_lock = apply_heavy_layer(
        current_lock_xgb,
        lock_cat,
        lock_lgb,
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        heavy_recipe,
    )
    current_dev_auc = auc(y_dev, current_dev)
    current_lock_auc = auc(y_lock, current_lock)
    if abs(current_dev_auc - float(heavy_report["dev_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy-stack dev prediction")
    if abs(current_lock_auc - float(heavy_report["lock_auc"])) > 1e-12:
        raise RuntimeError("Could not reproduce heavy-stack lock prediction")

    midpoint = np.median(day_dev)
    halves = (day_dev <= midpoint, day_dev > midpoint)
    current_half_auc = [auc(y_dev[mask], current_dev[mask]) for mask in halves]
    subset_rows = []
    subset_predictions = {}
    for size in range(1, len(SEEDS) + 1):
        for subset in combinations(SEEDS, size):
            source = np.mean([dev_seed[seed] for seed in subset], axis=0)
            candidate_xgb = apply_xgb_layer(
                clean_oof[client.META_DEV_FOLD],
                source,
                fold_groups[client.META_DEV_FOLD],
                clean_report["postprocess"],
                dev_membership,
                xgb_recipe,
            )
            candidate = apply_heavy_layer(
                candidate_xgb,
                dev_cat,
                dev_lgb,
                fold_groups[client.META_DEV_FOLD],
                clean_report["postprocess"],
                dev_membership,
                heavy_recipe,
            )
            score = auc(y_dev, candidate)
            half_auc = [auc(y_dev[mask], candidate[mask]) for mask in halves]
            half_gains = [
                value - base for value, base in zip(half_auc, current_half_auc)
            ]
            row = {
                "seeds": list(subset),
                "source_auc": auc(y_dev, source),
                "auc": score,
                "gain": score - current_dev_auc,
                "half_gains": half_gains,
                "min_half_gain": float(min(half_gains)),
            }
            subset_rows.append(row)
            subset_predictions[subset] = candidate
    stable = [row for row in subset_rows if row["min_half_gain"] >= 0.0]
    selected = max(
        stable, key=lambda row: (row["gain"], row["min_half_gain"])
    )
    subset_rows.sort(
        key=lambda row: (row["gain"], row["min_half_gain"]), reverse=True
    )
    selected_seeds = tuple(selected["seeds"])

    lock_source = np.mean([lock_seed[seed] for seed in selected_seeds], axis=0)
    lock_xgb = apply_xgb_layer(
        clean_oof[client.META_LOCK_FOLD],
        lock_source,
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        xgb_recipe,
    )
    lock_candidate = apply_heavy_layer(
        lock_xgb,
        lock_cat,
        lock_lgb,
        fold_groups[client.META_LOCK_FOLD],
        clean_report["postprocess"],
        lock_membership,
        heavy_recipe,
    )
    lock_auc = auc(y_lock, lock_candidate)
    accepted = bool(selected["gain"] > 0.0 and lock_auc > current_lock_auc)

    final_rows = []
    if accepted:
        iterations = int(xgb_report["final_group_iterations"])
        test_members = []
        for seed in selected_seeds:
            if seed == 2027:
                test_prediction = np.load(
                    ROOT / "honest_xgb_magic/test_prediction.npy"
                ).astype("float64")
                rows = [{"seed": seed, "cached_champion": True}]
            else:
                test_prediction, rows = train_seed_group_models(
                    arrays["train"],
                    arrays["test"],
                    arrays["target"],
                    arrays["month"],
                    seed,
                    iterations,
                )
            test_members.append(test_prediction)
            final_rows.extend(rows)
        test_source = np.mean(test_members, axis=0)
        np.save(WORK_DIR / "test_prediction.npy", test_source.astype("float32"))

        clean_submission = pd.read_csv(
            ROOT / "submission_honest_cleanv2_blend.csv"
        )
        if not np.array_equal(
            clean_submission["TransactionID"].to_numpy(), arrays["test_id"]
        ):
            raise RuntimeError("Seed-subset test rows differ from clean blend")
        test_xgb = apply_xgb_layer(
            clean_submission[client.TARGET].to_numpy(dtype="float64"),
            test_source,
            reference_report["test_groups"],
            clean_report["postprocess"],
            membership_test,
            xgb_recipe,
        )
        test_cat = np.load(ROOT / "honest_magic_heavy_stack/test_cat.npy")
        test_lgb = np.load(ROOT / "honest_magic_heavy_stack/test_lgb.npy")
        prediction = apply_heavy_layer(
            test_xgb,
            test_cat,
            test_lgb,
            reference_report["test_groups"],
            clean_report["postprocess"],
            membership_test,
            heavy_recipe,
        )
        output = clean_submission[["TransactionID"]].copy()
        output[client.TARGET] = prediction
        output.to_csv(OUTPUT_PATH, index=False)

    report = {
        "data_policy": (
            "official train/test covariates; official train labels only; "
            "official train labels only"
        ),
        "official_hashes": manifest["official_hashes"],
        "selection": (
            "15 fixed seed subsets on dev; unchanged XGB/heavy recipes; "
            "one-time lock"
        ),
        "seeds": list(SEEDS),
        "prediction_cache_version": PREDICTION_CACHE_VERSION,
        "champion_auc": {
            "dev": current_dev_auc,
            "lock": current_lock_auc,
        },
        "subset_search": subset_rows,
        "selected": selected,
        "candidate_lock_auc": lock_auc,
        "lock_gain": lock_auc - current_lock_auc,
        "accepted": accepted,
        "final_models": final_rows,
        "output": OUTPUT_PATH.name if accepted else None,
        "output_sha256": file_sha256(OUTPUT_PATH) if accepted else None,
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting train_honest_xgb_seed_subset_gkf.py


In [36]:
SOURCE_FILES = ('build_honest_advanced_cat_final.py', 'build_honest_no_gap_meta.py', 'build_honest_user_means_final.py', 'clean_v2_pipeline.py', 'finalize_honest_featureview_meta.py', 'finalize_honest_xgb_magic_blend.py', 'fraud_advanced_user_features.py', 'fraud_features.py', 'fraud_honest_advanced_data.py', 'fraud_multicounter_features.py', 'fraud_next_features.py', 'fraud_overlap_recipe.py', 'fraud_temporal_validation.py', 'fraud_transaction_chain_features.py', 'fraud_user_features.py', 'fraud_vblock_features.py', 'honest_featureview_sources.py', 'prepare_honest_featureview_sources.py', 'refine_honest_client_segments.py', 'search_honest_cleanv2_blend.py', 'search_honest_client_pooling.py', 'search_honest_featureview_meta.py', 'search_honest_fullrow_lgb.py', 'search_honest_raw_feature_meta.py', 'train_boost3_neural_stack.py', 'train_clean_foundation.py', 'train_honest_advanced_catboost.py', 'train_honest_client_meta.py', 'train_honest_client_profile_lgb.py', 'train_honest_heavy_temporal_client.py', 'train_honest_magic_heavy_stack.py', 'train_honest_user_means_catboost.py', 'train_honest_xgb_magic.py', 'train_honest_xgb_seed_subset_gkf.py')
FORBIDDEN_TOKENS = ('external_ieee', 'external_test_labels_AUDIT_ONLY', 'gap_transaction.csv', 'gap_identity.csv')
for filename in SOURCE_FILES:
    source = Path(filename).read_text(encoding="utf-8")
    assert not any(token in source for token in FORBIDDEN_TOKENS), filename
print("Input-policy source check passed for", len(SOURCE_FILES), "files")

Input-policy source check passed for 34 files


## 3. Foundation and client-aware temporal OOF

Three tree families create independent predictions. Advanced UID, graph-user,
and broad C/D/V profile views are added with fixed seeds. The client meta-model
and consistency postprocess are selected on labeled train time windows only.

In [37]:
run_stage("train_clean_foundation.py")
run_stage("train_boost3_neural_stack.py", supports_force=True)
run_stage("train_honest_advanced_catboost.py", supports_force=True)
run_stage("build_honest_advanced_cat_final.py", supports_force=True)
run_stage(
    "train_honest_user_means_catboost.py",
    "--folds", "0,1,2",
    supports_force=True,
)
run_stage(
    "build_honest_user_means_final.py",
    "--seeds", "1729",
    supports_force=True,
)
run_stage("train_honest_client_meta.py")
run_stage("refine_honest_client_segments.py")

Running: <path> x МТС Kaggle/.venv/bin/python train_clean_foundation.py


Reading provided train/test only...


Building target-free train+test features...


Selecting CatBoost iterations on the final 20% of train...



bestTest = 0.9400001724
bestIteration = 1393

Shrink model to first 1394 iterations.
Training full CatBoost with 1394 iterations...
    ... строк с итерациями обучения скрыто: 15


...

<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"view": "all_sources_clients", "config_index": 5, "num_leaves": 7, "max_depth": 4, "min_child_samples": 1000, "best_iteration": 153, "dev_auc": 0.9238955223074397, "feature_count": 62}


{
  "selected_lgb": {
    "view": "all_sources_clients",
    "config_index": 1,
    "num_leaves": 15,
    "max_depth": 4,
    "min_child_samples": 300,
    "best_iteration": 109,
    "dev_auc": 0.924256717401685,
    "feature_count": 62
  },
  "selected_weight": {
    "lgb_weight": 0.6000000000000001,
    "auc": 0.9251555417375738
  },
  "selected_postprocess": {
    "group": "card_addr_origin_product",
    "method": "max",
    "weight": 0.5,
    "auc": 0.9264365921236896,
    "gain": 0.0012810503861158118,
    "half_gains": [
      0.0008510205564521423,
      0.0017215739643298145
    ],
    "min_half_gain": 0.0008510205564521423
  },
  "scores": {
    "dev_current_auc": 0.9235641970632845,
    "dev_lgb_auc": 0.924256717401685,
    "dev_blend_auc": 0.9251555417375738,
    "dev_final_auc": 0.9264365921236896,
    "lock_current_auc": 0.9209672106542669,
    "lock_lgb_auc": 0.9189006653637299,
    "lock_blend_auc": 0.920902891243474,
    "lock_final_auc": 0.9218517495145931,
    "lock_g

Running: <path> x МТС Kaggle/.venv/bin/python refine_honest_client_segments.py


{
  "selected": {
    "weights": {
      "strict": 0.8,
      "partial": 1.0,
      "cold": 0.4
    },
    "auc": 0.9265898487265363,
    "gain": 0.0017428685732909788,
    "half_gains": [
      0.0004795992591898335,
      0.00300577857018014
    ],
    "min_half_gain": 0.0004795992591898335
  },
  "base_client_meta_lock_auc": 0.9218517495145931,
  "segment_candidate_lock_auc": 0.9219770608964069,
  "lock_gain": 0.0001253113818137086,
  "accepted": true,
  "output": "submission_honest_client_segments.csv"
}


## 4. Selected V-block and structured feature views

Only the six views used by the final stack are trained. Four purged folds use
`past -> 30-day embargo -> future`; three full-data LightGBMs provide test
signals. A compact client-aware LightGBM stacks these sources.

In [38]:
run_stage("prepare_honest_featureview_sources.py", supports_force=True)
run_stage("search_honest_featureview_meta.py")
run_stage("finalize_honest_featureview_meta.py")
run_stage("search_honest_fullrow_lgb.py")

Running: <path> x МТС Kaggle/.venv/bin/python prepare_honest_featureview_sources.py


Building official-data advanced feature views...
Reading official train/test...


Building Giba and target-free graph profiles...


Building improved UID, rolling and behavior features...


<path> x МТС Kaggle/honest_best_4files_run/honest_featureview_sources.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[names] = pd.DataFrame(values, index=frame.index)
<path> x МТС Kaggle/honest_best_4files_run/honest_featureview_sources.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[names] = pd.DataFrame(values, index=frame.index)
<path> x МТС Kaggle/honest_best_4files_run/honest_featureview_sources.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.

...

<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"config_index": 3, "num_leaves": 63, "max_depth": 8, "min_child_samples": 1000, "extra_trees": true, "best_iteration": 3454, "dev_auc": 0.8998963955340576}


<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"config_index": 4, "num_leaves": 127, "max_depth": 9, "min_child_samples": 1000, "extra_trees": true, "best_iteration": 3481, "dev_auc": 0.900381875300461}


{
  "selected_model": {
    "config_index": 2,
    "num_leaves": 127,
    "max_depth": 8,
    "min_child_samples": 800,
    "best_iteration": 1251,
    "dev_auc": 0.9042659800015111
  },
  "selected_blend": {
    "variant": "postprocessed_rank",
    "weights": {
      "strict": 0.0,
      "partial": 0.0,
      "cold": 0.3
    },
    "auc": 0.9275705591028438,
    "gain": 0.00041405308401298857,
    "half_gains": [
      1.796882835058966e-05,
      0.0008861524979858482
    ],
    "min_half_gain": 1.796882835058966e-05
  },
  "reference": {
    "dev_auc": 0.9271565060188308,
    "lock_auc": 0.9229394341187334
  },
  "candidate": {
    "dev_raw_auc": 0.9042659800015111,
    "lock_raw_auc": 0.8922104169095699,
    "lock_blend_auc": 0.9235975240653755,
    "lock_gain": 0.0006580899466421819
  },
  "accepted": true,
  "output": "submission_honest_fullrow_lgb.csv",
  "elapsed_minutes": 11.131419082482656
}


## 5. Heavy clean-v2 and XGB-magic ensemble

Clean-v2 trains horizon-aware CatBoost/LightGBM/XGBoost sources with embargoed
history. The magic view adds D-origin UID aggregates. A second CatBoost/LGB
pair supplies a small cold-client residual.

In [39]:
clean_arguments = ["--mode", "all"]
if FORCE_RETRAIN:
    clean_arguments.append("--force")
run_stage("clean_v2_pipeline.py", *clean_arguments)
run_stage("search_honest_cleanv2_blend.py")
run_stage("train_honest_xgb_magic.py")
run_stage("finalize_honest_xgb_magic_blend.py")
run_stage("train_honest_magic_heavy_stack.py")

Running: <path> x МТС Kaggle/.venv/bin/python clean_v2_pipeline.py --mode all



dev_h30_a: history=56,059, future=42,290, physically skipped=122,457


Building row-level V-block encodings...


Building broad user C/D aggregates...
Building selected user V aggregates...


Building user-level V-block missingness profiles...


...

<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"family": "lightgbm", "name": "leaf127", "config": {"name": "leaf127", "num_leaves": 127, "max_depth": 10, "min_child_samples": 240}, "source_auc": 0.9122317316719393, "best_iteration": 508, "minutes": 0.4986189007759094, "cached": false, "best_blend": {"variant": "probability", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.0}, "auc": 0.9330364961445237, "gain": 0.0, "half_gains": [0.0, 0.0], "min_half_gain": 0.0}, "search_top": [{"variant": "postprocessed_rank", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.1}, "auc": 0.9330442898043286, "gain": 7.793659804966602e-06, "half_gains": [3.843644689316417e-05, -8.483215628352081e-05], "min_half_gain": -8.483215628352081e-05}, {"variant": "postprocessed_probability", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.1}, "auc": 0.9330370152541775, "gain": 5.191096538181839e-07, "half_gains": [-2.2461035431575738e-07, -7.005313269303759e-05], "min_half_gain": -7.005313269303759e-05}, {"variant": "probability", "weights": {"s

<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"family": "lightgbm", "name": "extra127", "config": {"name": "extra127", "num_leaves": 127, "max_depth": 10, "min_child_samples": 300, "extra_trees": true}, "source_auc": 0.9055072413341482, "best_iteration": 1586, "minutes": 0.7747231642405192, "cached": false, "best_blend": {"variant": "probability", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.0}, "auc": 0.9330364961445237, "gain": 0.0, "half_gains": [0.0, 0.0], "min_half_gain": 0.0}, "search_top": [{"variant": "postprocessed_rank", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.1}, "auc": 0.9330414627612131, "gain": 4.966616689472403e-06, "half_gains": [7.580599460266235e-06, -3.3315901376895773e-05], "min_half_gain": -3.3315901376895773e-05}, {"variant": "probability", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.0}, "auc": 0.9330364961445237, "gain": 0.0, "half_gains": [0.0, 0.0], "min_half_gain": 0.0}, {"variant": "rank", "weights": {"strict": 0.0, "partial": 0.0, "cold": 0.0}, "auc": 0.9330364961445237, "

0:	total: 118ms	remaining: 1m 23s
200:	total: 23.8s	remaining: 1m
400:	total: 47.4s	remaining: 36.7s
600:	total: 1m 10s	remaining: 13s
710:	total: 1m 23s	remaining: 0us
0:	total: 213ms	remaining: 2m 54s
200:	total: 37.4s	remaining: 1m 54s
400:	total: 1m 14s	remaining: 1m 17s
600:	total: 1m 52s	remaining: 40.6s
800:	total: 2m 29s	remaining: 3.17s
817:	total: 2m 32s	remaining: 0us
{
  "data_policy": "official train/test covariates; official train labels only; no gap or audit labels",
  "official_hashes": {
    "train_transaction.csv": "d75f0cedaf354a750bc148372b63835efef53445abe017aad3d93dc2ebfc7da7",
    "train_identity.csv": "8cece20966dc33be033ace5821ec183ea019a2d18008a9590e30a202d2da19d0",
    "test_transaction.csv": "7116dd600f0f3b6fcc62292988a4997c02727265836baa816383dcf4202bdadd",
    "test_identity.csv": "5ceabcf3453372247372399e3ff083f3727dd69675040235bc1ef00c27f05a81",
    "sample_submission.csv": "c2d4a3a53db995f95f7a10fe1785683303146d683ed71b8c396c8d31d4859e83"
  },
  "featur

## 6. Final client-profile residual and submission

For each target-free client UID, 96 selected transaction features are reduced
to means plus 32-feature dispersion/range/first/last profiles. Three LightGBMs
are rank-combined, and the locked `0.10` residual is applied only to cold
clients.

In [40]:
run_stage("train_honest_client_profile_lgb.py")

source_path = WORK_DIR / "submission_honest_client_profile_lgb.csv"
submission_path = (
    Path("/kaggle/working/submission_honest_best.csv")
    if Path("/kaggle/working").exists()
    else WORK_DIR / "submission_honest_best.csv"
)
if submission_path != source_path:
    shutil.copy2(source_path, submission_path)
else:
    submission_path = source_path

submission = pd.read_csv(submission_path)
sample = pd.read_csv(WORK_DIR / "sample_submission.csv")
assert len(submission) == len(sample) == EXPECTED_ROWS["test_transaction.csv"]
assert submission["TransactionID"].equals(sample["TransactionID"])
assert submission["TransactionID"].is_unique
assert submission["isFraud"].between(0.0, 1.0).all()
assert submission["isFraud"].notna().all()

profile_report = json.loads(
    (WORK_DIR / "honest_client_profile_lgb/report.json").read_text()
)
assert profile_report["accepted"]
summary = {
    "temporal_dev_auc": profile_report["dev_auc"],
    "temporal_lock_auc": profile_report["lock_auc"],
    "selected_residual": profile_report["selected"],
    "rows": len(submission),
    "sha256": hashlib.sha256(submission_path.read_bytes()).hexdigest(),
    "submission": str(submission_path),
}
print(json.dumps(summary, indent=2))
submission.head()

Running: <path> x МТС Kaggle/.venv/bin/python train_honest_client_profile_lgb.py


Loading verified XGB-magic matrix cache


<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"name": "profile_leaf31", "config": {"name": "profile_leaf31", "num_leaves": 31, "max_depth": -1, "min_child_samples": 160}, "source_auc": 0.9037874240354925, "best_iteration": 608, "minutes": 0.4019191821416219}
    ... строк с итерациями обучения скрыто: 3


<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

<path> x МТС Kaggle/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{"name": "profile_extra63", "config": {"name": "profile_extra63", "num_leaves": 63, "max_depth": 9, "min_child_samples": 220, "extra_trees": true}, "source_auc": 0.8980401926963095, "best_iteration": 836, "minutes": 0.5692484021186829}
    ... строк с итерациями обучения скрыто: 5


{
  "data_policy": "official train/test covariates; official train labels only; history/query profiles built separately",
  "official_hashes": {
    "train_transaction.csv": "d75f0cedaf354a750bc148372b63835efef53445abe017aad3d93dc2ebfc7da7",
    "train_identity.csv": "8cece20966dc33be033ace5821ec183ea019a2d18008a9590e30a202d2da19d0",
    "test_transaction.csv": "7116dd600f0f3b6fcc62292988a4997c02727265836baa816383dcf4202bdadd",
    "test_identity.csv": "5ceabcf3453372247372399e3ff083f3727dd69675040235bc1ef00c27f05a81",
    "sample_submission.csv": "c2d4a3a53db995f95f7a10fe1785683303146d683ed71b8c396c8d31d4859e83"
  },
  "uid": "card1|addr1|floor(day-D1)|P_emaildomain",
  "profile": {
    "mean_features": [
      "TransactionAmt",
      "card1",
      "card2",
      "card3",
      "card5",
      "addr1",
      "addr2",
      "P_emaildomain",
      "R_emaildomain",
      "uid_FE",
      "TransactionAmt_uid_mean",
      "TransactionAmt_uid_std",
      "D4_uid_mean",
      "D10_uid_mean",


{
  "temporal_dev_auc": 0.9334955925135129,
  "temporal_lock_auc": 0.9261184312567802,
  "selected_residual": {
    "variant": "all_rank_max",
    "weights": {
      "strict": 0.0,
      "partial": 0.0,
      "cold": 0.1
    },
    "auc": 0.9334955925135129,
    "gain": 0.000346582858985256,
    "half_gains": [
      0.0003757169702893748,
      0.000286634535330621
    ],
    "min_half_gain": 0.000286634535330621
  },
  "rows": 129736,
  "sha256": "94e23083a2d58e84b800493c6ec03e5b188c9be0136d74db753fdf2957ad36f0",
  "submission": "<path> x \u041c\u0422\u0421 Kaggle/honest_best_4files_run/submission_honest_best.csv"
}


,TransactionID,isFraud
0,3447804,0.589496
1,3447805,0.577489
2,3447806,0.021235
3,3447807,0.400190
4,3447808,0.144806


## Output

Submit `/kaggle/working/submission_honest_best.csv`.

The notebook intentionally cannot calculate a test ROC AUC. Its only labels
come from `train_transaction.csv`, and every reported executable metric is a
forward-time train validation metric.